# Export Trained Models to ONNX (Debugging Log)

**Why?:** For running real-time inference on your local machine  
**Date:** 21 to 30 May 2026  
**Demo:** for AI (Fundamental) Class Students  
**Author:** Ye Kyaw Thu, Lab. Leader, Language Understanding Lab., Myanmar  

## export_to_onnx.py Code

In [2]:
%pwd

'/mnt/disk1/ye/exp/msl_recognition'

## ONNX Python Library Installation

Running on my GPU machine ...  

In [3]:
!pip install onnx onnxruntime

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.6/17.6 MB 48.6 MB/s  0:00:00 52.5 MB/s eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 66.9 MB/s  0:00:00 78.5 MB/s eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [onnx]━━━━━━ 1/2 [onnx]


## Installation of ONNX Runtime GPU (Optional)  

If you want to verify the exported model runs correctly on the server ...  

In [4]:
!pip install onnxruntime-gpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 277.0/277.0 MB 62.5 MB/s  0:00:05 eta 0:00:010:01:02


## export_to_onnx.py Code

In [19]:
from IPython.display import Markdown

file_path = './src/export_to_onnx.py'

with open(file_path, 'r') as f:
    code = f.read()

# 'python' keyword triggers the syntax highlighter
display(Markdown(f"```python\n{code}\n```"))

```python
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
export_to_onnx.py  –  Export trained MSL models to ONNX format
                       (runs on Linux/HPC server side)

Bugs fixed vs original
───────────────────────
  1. ST-GCN: edge_importance in-place copy_ during torch.onnx.export broke
     the export (same root cause as the training autograd bug).  Fixed by
     pre-computing the scaled adjacency before export so no in-place op
     appears in the traced graph.
  2. BiLSTM: pack_padded_sequence is NOT supported in ONNX opset 14; the
     original wrapper silently fell back to an empty-graph export.  Fixed
     by bypassing packing entirely (no-op path) and exporting with a simple
     mask-sum length calculation that ONNX can trace.
  3. input_dim for ST-GCN was referencing an undefined variable; fixed with
     explicit constant (225 for sequence models, None for graph models).
  4. num_joints / num_coords were leaking between branches; fixed with clear
     if/elif blocks and explicit constants.
  5. The metadata JSON now includes 'model_type' (was missing), which
     infer_onnx.py reads to choose the correct input layout.
  6. Transformer: PyTorch 2.x fuses TransformerEncoderLayer and
     MultiheadAttention into single CUDA kernels 
     (`aten::_transformer_encoder_layer_fwd` and `aten::_native_multi_head_attention`)
     which have NO ONNX mapping in opset 14. Also, `aten::gelu` requires opset >= 20.
     Fixed by unrolling the encoder loop, replacing MHA with a manual math 
     implementation, and replacing GELU with an `erf`-based equivalent.

Usage (on Linux / HPC)
───────────────────────
  python src/export_to_onnx.py --config config/config.yaml --all
  python src/export_to_onnx.py --config config/config.yaml --model transformer
  python src/export_to_onnx.py --config config/config.yaml --model bilstm --no-validate
"""

import argparse
import json
import shutil
import sys
from pathlib import Path

import numpy as np
import torch
import yaml

sys.path.insert(0, str(Path(__file__).parent))
from models import build_model
from utils  import get_logger, get_device, load_label_map

# ── Experiment → output name mapping ─────────────────────────────────────────
MODEL_CONFIGS = {
    "bilstm":       {"exp_dir": "results/exp_bilstm",       "output_name": "bilstm_msl.onnx"},
    "transformer":  {"exp_dir": "results/exp_transformer",  "output_name": "transformer_msl.onnx"},
    "stgcn":        {"exp_dir": "results/exp_stgcn",        "output_name": "stgcn_msl.onnx"},
}

# Fixed constants — must match extract_keypoints.py
POSE_N       = 33
HAND_N       = 21
TOTAL_JOINTS = POSE_N + HAND_N * 2   # 75
FEAT_DIM     = TOTAL_JOINTS * 3       # 225


# ─────────────────────────────────────────────────────────────────────────────
# BiLSTM export
# ─────────────────────────────────────────────────────────────────────────────

class _BiLSTMExportWrapper(torch.nn.Module):
    """
    Thin wrapper that converts the boolean padding mask to a length tensor
    WITHOUT calling pack_padded_sequence (which ONNX cannot trace).

    The mask convention (True = padded position) is the same as training.
    We count the valid (non-padded) positions to get sequence lengths,
    then call the model in 'no-pack' mode by setting lengths=None and
    passing the mask directly so the attention layer masks out padding.
    """
    def __init__(self, model):
        super().__init__()
        self.model = model

    def forward(self, keypoints: torch.Tensor, mask: torch.Tensor) -> torch.Tensor:
        # mask: (B, T) True where PADDED  →  lengths = number of valid frames
        # We do NOT pass lengths so the LSTM runs on the full padded sequence
        # (slightly less efficient, but fully ONNX-traceable).
        # The additive attention mask still zeros out padded positions.
        return self.model(keypoints, lengths=None, mask=mask)


def export_bilstm(model, out_path: Path, max_seq_len: int, device: str):
    model.eval()
    wrapper = _BiLSTMExportWrapper(model).to(device)
    wrapper.eval()

    dummy_kp   = torch.randn(1, max_seq_len, FEAT_DIM, device=device)
    # mask: True = padded.  Use all-False (nothing padded) for the trace.
    dummy_mask = torch.zeros(1, max_seq_len, dtype=torch.bool, device=device)

    print(f"  BiLSTM dummy inputs: kp={tuple(dummy_kp.shape)}, mask={tuple(dummy_mask.shape)}")

    with torch.no_grad():
        torch.onnx.export(
            wrapper,
            (dummy_kp, dummy_mask),
            str(out_path),
            opset_version=14,
            input_names=["keypoints", "mask"],
            output_names=["logits"],
            dynamic_axes={
                "keypoints": {0: "batch"},
                "mask":      {0: "batch"},
                "logits":    {0: "batch"},
            },
            do_constant_folding=True,
            verbose=False,
        )
    print(f"  Exported → {out_path}")


# ─────────────────────────────────────────────────────────────────────────────
# Transformer export
# ─────────────────────────────────────────────────────────────────────────────

def _onnx_safe_gelu(x: torch.Tensor) -> torch.Tensor:
    """GELU implementation using erf, fully supported in ONNX opset 14+."""
    return 0.5 * x * (1.0 + torch.erf(x / 1.4142135623730951))


class ManualMHA(torch.nn.Module):
    """
    Manual Multi-Head Attention implementation to bypass PyTorch 2.x's
    fused `aten::_native_multi_head_attention` which lacks ONNX support.
    """
    def __init__(self, mha: torch.nn.MultiheadAttention):
        super().__init__()
        self.embed_dim = mha.embed_dim
        self.num_heads = mha.num_heads
        self.head_dim = self.embed_dim // self.num_heads
        
        self.in_proj_weight = torch.nn.Parameter(mha.in_proj_weight.data.clone())
        self.in_proj_bias = torch.nn.Parameter(mha.in_proj_bias.data.clone())
        self.out_proj_weight = torch.nn.Parameter(mha.out_proj.weight.data.clone())
        self.out_proj_bias = torch.nn.Parameter(mha.out_proj.bias.data.clone())

    def forward(self, query: torch.Tensor, key: torch.Tensor, value: torch.Tensor,
                key_padding_mask: torch.Tensor = None, need_weights: bool = False):
        B, T, C = query.shape
        
        # Split projection weights for Q, K, V
        q_w = self.in_proj_weight[:C]
        k_w = self.in_proj_weight[C:2*C]
        v_w = self.in_proj_weight[2*C:]
        
        q_b = self.in_proj_bias[:C]
        k_b = self.in_proj_bias[C:2*C]
        v_b = self.in_proj_bias[2*C:]
        
        Q = torch.nn.functional.linear(query, q_w, q_b)
        K = torch.nn.functional.linear(key, k_w, k_b)
        V = torch.nn.functional.linear(value, v_w, v_b)
        
        # Use -1 for dynamic sequence length compatibility in ONNX
        Q = Q.reshape(B, -1, self.num_heads, self.head_dim).transpose(1, 2)
        K = K.reshape(B, -1, self.num_heads, self.head_dim).transpose(1, 2)
        V = V.reshape(B, -1, self.num_heads, self.head_dim).transpose(1, 2)
        
        # Scaled dot-product attention
        scale = self.head_dim ** 0.5
        scores = torch.matmul(Q, K.transpose(-2, -1)) / scale
        
        if key_padding_mask is not None:
            # key_padding_mask: (B, T) -> (B, 1, 1, T) for broadcasting
            mask = key_padding_mask.unsqueeze(1).unsqueeze(2)
            scores = scores.masked_fill(mask, float('-inf'))
            
        attn_weights = torch.nn.functional.softmax(scores, dim=-1)
        attn_out = torch.matmul(attn_weights, V)
        
        # Reshape back to (B, T, C)
        attn_out = attn_out.transpose(1, 2).reshape(B, -1, C)
        
        out = torch.nn.functional.linear(attn_out, self.out_proj_weight, self.out_proj_bias)
        
        if need_weights:
            return out, attn_weights
        return out, None


class _TransformerExportWrapper(torch.nn.Module):
    """
    Wrapper that completely bypasses PyTorch 2.x fused operators by unrolling
    the encoder loop and calling basic sub-modules directly.
    """
    def __init__(self, model):
        super().__init__()
        # Copy all the model's input/head components
        self.input_norm = model.input_norm
        self.input_proj = model.input_proj
        self.proj_norm = model.proj_norm
        self.cls_token = model.cls_token
        self.pos_enc = model.pos_enc
        self.encoder_norm = model.encoder.norm
        self.head_norm = model.head_norm
        self.head_dropout = model.head_dropout
        self.fc = model.fc
        
        # Replace fused MultiheadAttention with ONNX-safe manual implementation
        self.layer_self_attn = torch.nn.ModuleList([ManualMHA(l.self_attn) for l in model.encoder.layers])
        self.layer_linear1 = torch.nn.ModuleList([l.linear1 for l in model.encoder.layers])
        self.layer_linear2 = torch.nn.ModuleList([l.linear2 for l in model.encoder.layers])
        self.layer_norm1 = torch.nn.ModuleList([l.norm1 for l in model.encoder.layers])
        self.layer_norm2 = torch.nn.ModuleList([l.norm2 for l in model.encoder.layers])
        self.layer_dropout1 = torch.nn.ModuleList([l.dropout1 for l in model.encoder.layers])
        self.layer_dropout2 = torch.nn.ModuleList([l.dropout2 for l in model.encoder.layers])
        
        self.num_layers = len(model.encoder.layers)
        
        # Check architecture type from the first layer
        self.norm_first = model.encoder.layers[0].norm_first
        
        # Determine activation type
        act = model.encoder.layers[0].activation
        if act is torch.nn.functional.relu or isinstance(act, torch.nn.ReLU):
            self._use_gelu = False
        else:
            self._use_gelu = True

    def forward(self, keypoints: torch.Tensor, mask: torch.Tensor) -> torch.Tensor:
        # ── Input processing ──
        x = self.input_norm(keypoints)       # (B, T, 225)
        x = self.input_proj(x)               # (B, T, 256)
        x = self.proj_norm(x)                # (B, T, 256)
        
        # ── Add CLS token ──
        B = x.shape[0]
        cls_tokens = self.cls_token.expand(B, -1, -1)  # (B, 1, 256)
        x = torch.cat([cls_tokens, x], dim=1)          # (B, T+1, 256)
        
        # ── Update mask for CLS token (CLS is never padded) ──
        cls_mask = torch.zeros(B, 1, dtype=torch.bool, device=x.device)
        padding_mask = torch.cat([cls_mask, mask], dim=1)  # (B, T+1)
        
        # ── Add positional encoding ──
        x = self.pos_enc(x)
        
        # ── Pass through encoder layers MANUALLY ──
        for i in range(self.num_layers):
            if self.norm_first:
                # Pre-norm architecture
                nx = self.layer_norm1[i](x)
                attn_out, _ = self.layer_self_attn[i](
                    nx, nx, nx,
                    key_padding_mask=padding_mask,
                    need_weights=False
                )
                x = x + self.layer_dropout1[i](attn_out)
                
                nx = self.layer_norm2[i](x)
                ff = self.layer_linear1[i](nx)
                if self._use_gelu:
                    ff = _onnx_safe_gelu(ff)
                else:
                    ff = torch.nn.functional.relu(ff)
                ff = self.layer_linear2[i](self.layer_dropout2[i](ff))
                x = x + self.layer_dropout2[i](ff)
            else:
                # Post-norm architecture (PyTorch default)
                attn_out, _ = self.layer_self_attn[i](
                    x, x, x,
                    key_padding_mask=padding_mask,
                    need_weights=False
                )
                x = self.layer_norm1[i](x + self.layer_dropout1[i](attn_out))
                
                ff = self.layer_linear1[i](x)
                if self._use_gelu:
                    ff = _onnx_safe_gelu(ff)
                else:
                    ff = torch.nn.functional.relu(ff)
                ff = self.layer_linear2[i](self.layer_dropout2[i](ff))
                x = self.layer_norm2[i](x + self.layer_dropout2[i](ff))
        
        # ── Final encoder norm ──
        x = self.encoder_norm(x)
        
        # ── Extract CLS token output ──
        cls_out = x[:, 0, :]  # (B, 256)
        
        # ── Classification head ──
        cls_out = self.head_norm(cls_out)
        cls_out = self.head_dropout(cls_out)
        logits = self.fc(cls_out)
        
        return logits


def export_transformer(model, out_path: Path, max_seq_len: int, device: str):
    model.eval()

    # ── CRITICAL FIX ──────────────────────────────────────────────────────────
    # PyTorch 2.x aggressively fuses elements into unsupported C++ kernels:
    # 1. `aten::_transformer_encoder_layer_fwd` (bypassed by unrolling loop)
    # 2. `aten::_native_multi_head_attention` (bypassed by ManualMHA class)
    # 3. `aten::gelu` (requires opset >= 20, bypassed by _onnx_safe_gelu)
    # ──────────────────────────────────────────────────────────────────────────
    
    wrapper = _TransformerExportWrapper(model).to(device)
    wrapper.eval()

    dummy_kp   = torch.randn(1, max_seq_len, FEAT_DIM, device=device)
    dummy_mask = torch.zeros(1, max_seq_len, dtype=torch.bool, device=device)

    print(f"  Transformer dummy inputs: kp={tuple(dummy_kp.shape)}, mask={tuple(dummy_mask.shape)}")

    with torch.no_grad():
        torch.onnx.export(
            wrapper,  
            (dummy_kp, dummy_mask),
            str(out_path),
            opset_version=14,
            input_names=["keypoints", "mask"],
            output_names=["logits"],
            dynamic_axes={
                "keypoints": {0: "batch", 1: "seq_len"},
                "mask":      {0: "batch", 1: "seq_len"},
                "logits":    {0: "batch"},
            },
            do_constant_folding=True,
            verbose=False,
        )
    print(f"  Exported → {out_path}")


# ─────────────────────────────────────────────────────────────────────────────
# ST-GCN export
# ─────────────────────────────────────────────────────────────────────────────

class _STGCNExportWrapper(torch.nn.Module):
    """
    Pre-compute the edge-importance-scaled adjacency matrices so no
    in-place copy_() appears in the traced computation graph.
    This was the root cause of the ST-GCN export failure.
    """
    def __init__(self, model):
        super().__init__()
        self.model = model

        # Pre-bake all scaled adjacencies into registered buffers
        for i, block in enumerate(model.st_gcn_blocks):
            A_base = block.gcn.A.detach().clone()
            if model.edge_importance_weighting:
                ei    = model.edge_importance[i].detach().clone()
                A_scaled = (A_base * ei).clamp(min=0)
            else:
                A_scaled = A_base
            # Replace the block's GCN adjacency with the pre-scaled version
            block.gcn.A.data.copy_(A_scaled)

        # Disable edge_importance so the forward pass uses A directly
        self.model.edge_importance_weighting = False

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.model(x)


def export_stgcn(model, out_path: Path, max_seq_len: int, device: str):
    model.eval()

    # Bake edge importance into adjacency buffers so no in-place op in graph
    wrapper = _STGCNExportWrapper(model).to(device)
    wrapper.eval()

    # ST-GCN input: (N, C, T, V) = (batch, 3, max_seq_len, 75)
    dummy_kp = torch.randn(1, 3, max_seq_len, TOTAL_JOINTS, device=device)

    print(f"  ST-GCN dummy input: kp={tuple(dummy_kp.shape)}")

    with torch.no_grad():
        torch.onnx.export(
            wrapper,
            dummy_kp,
            str(out_path),
            opset_version=14,
            input_names=["keypoints"],
            output_names=["logits"],
            dynamic_axes={
                "keypoints": {0: "batch"},
                "logits":    {0: "batch"},
            },
            do_constant_folding=True,
            verbose=False,
        )
    print(f"  Exported → {out_path}")


# ─────────────────────────────────────────────────────────────────────────────
# Validation
# ─────────────────────────────────────────────────────────────────────────────

def validate_onnx(out_path: Path, model_type: str, max_seq_len: int, num_classes: int):
    try:
        import onnx
        import onnxruntime as ort
    except ImportError:
        print("  [SKIP] onnx / onnxruntime not installed — skipping validation")
        return True

    print("  Validating ONNX model...")
    mdl = onnx.load(str(out_path))
    onnx.checker.check_model(mdl)
    print("  ✓ ONNX graph is valid")

    sess = ort.InferenceSession(str(out_path), providers=["CPUExecutionProvider"])

    if model_type == "stgcn":
        dummy = {"keypoints": np.random.randn(1, 3, max_seq_len, TOTAL_JOINTS).astype(np.float32)}
    else:
        dummy = {
            "keypoints": np.random.randn(1, max_seq_len, FEAT_DIM).astype(np.float32),
            "mask":      np.zeros((1, max_seq_len), dtype=bool),
        }

    out = sess.run(None, dummy)
    logits = out[0]
    assert logits.shape == (1, num_classes), \
        f"Shape mismatch: expected (1,{num_classes}), got {logits.shape}"
    print(f"  ✓ Inference OK — output shape: {logits.shape}")
    print(f"  ✓ Top-5 class indices: {np.argsort(logits[0])[::-1][:5].tolist()}")
    return True


# ─────────────────────────────────────────────────────────────────────────────
# Orchestrator
# ─────────────────────────────────────────────────────────────────────────────

def export_model(model_type: str, cfg: dict, output_dir: Path, device: str,
                 validate: bool = True) -> bool:
    print(f"\n{'='*60}")
    print(f"  Exporting  {model_type.upper()}")
    print(f"{'='*60}")

    mcfg     = MODEL_CONFIGS[model_type]
    ckpt_path = Path(mcfg["exp_dir"]) / "checkpoints" / "best.pth"
    out_path  = output_dir / mcfg["output_name"]

    if not ckpt_path.exists():
        print(f"  [ERROR] Checkpoint not found: {ckpt_path}")
        return False

    ckpt        = torch.load(str(ckpt_path), map_location=device)
    num_classes = ckpt.get("num_classes", 556)
    max_seq_len = cfg["data"]["max_seq_len"]

    print(f"  Classes: {num_classes}   MaxSeqLen: {max_seq_len}")

    model = build_model(model_type, cfg, num_classes)
    model.load_state_dict(ckpt["state_dict"])
    model = model.to(device)
    model.eval()

    output_dir.mkdir(parents=True, exist_ok=True)

    if model_type == "bilstm":
        export_bilstm(model, out_path, max_seq_len, device)
    elif model_type == "transformer":
        export_transformer(model, out_path, max_seq_len, device)
    elif model_type == "stgcn":
        export_stgcn(model, out_path, max_seq_len, device)

    # Validate
    if validate:
        try:
            validate_onnx(out_path, model_type, max_seq_len, num_classes)
        except Exception as e:
            print(f"  [WARNING] Validation error: {e}")

    # Save metadata (infer_onnx.py reads this)
    metadata = {
        "model_type":  model_type,     # ← was missing in original!
        "num_classes": num_classes,
        "max_seq_len": max_seq_len,
        "feat_dim":    FEAT_DIM,       # 225 for sequence models
        "num_joints":  TOTAL_JOINTS,   # 75  for ST-GCN
        "num_coords":  3,
    }
    meta_path = out_path.with_suffix(".json")
    with open(meta_path, "w") as f:
        json.dump(metadata, f, indent=2)
    print(f"  Metadata  → {meta_path}")

    # Copy label map alongside ONNX file for convenience
    lm_src = Path(cfg["data"]["label_map_file"])
    lm_dst = output_dir / "label_map.json"
    if lm_src.exists() and not lm_dst.exists():
        shutil.copy(str(lm_src), str(lm_dst))
        print(f"  Label map → {lm_dst}")

    size_mb = out_path.stat().st_size / 1024**2
    print(f"  ✓ Done  ({size_mb:.1f} MB)")
    return True


# ─────────────────────────────────────────────────────────────────────────────
# CLI
# ─────────────────────────────────────────────────────────────────────────────

def main():
    parser = argparse.ArgumentParser(description="Export MSL models to ONNX")
    parser.add_argument("--config",     default="config/config.yaml")
    parser.add_argument("--model",      choices=list(MODEL_CONFIGS.keys()),
                        help="Export one specific model")
    parser.add_argument("--all",        action="store_true",
                        help="Export all three models")
    parser.add_argument("--output_dir", default="onnx_models")
    parser.add_argument("--no-validate",action="store_true")
    parser.add_argument("--device",     default=None,
                        help="cuda | cpu (auto-detected if omitted)")
    args = parser.parse_args()

    if not args.model and not args.all:
        parser.error("Specify --model <name> or --all")

    with open(args.config) as f:
        cfg = yaml.safe_load(f)

    device = args.device or ("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Device: {device}")

    models = list(MODEL_CONFIGS.keys()) if args.all else [args.model]
    output_dir = Path(args.output_dir)

    ok = 0
    for m in models:
        if export_model(m, cfg, output_dir, device, validate=not args.no_validate):
            ok += 1

    print(f"\n{'='*60}")
    print(f"  SUMMARY: {ok}/{len(models)} models exported to '{output_dir}/'")
    if output_dir.exists():
        for f in sorted(output_dir.glob("*")):
            mb = f.stat().st_size / 1024**2
            print(f"    {f.name:<35} {mb:6.1f} MB")
    print(f"{'='*60}")


if __name__ == "__main__":
    main()


```

## Exporting Bi-LSTM Model  

In [20]:
!time python src/export_to_onnx.py --config config/config.yaml --model bilstm

Device: cuda

  Exporting  BILSTM
  Classes: 556   MaxSeqLen: 200
  BiLSTM dummy inputs: kp=(1, 200, 225), mask=(1, 200)
/mnt/disk1/ye/exp/msl_recognition/.venv/lib/python3.12/site-packages/torch/onnx/symbolic_opset9.py:4244: UserWarning: Exporting a model to ONNX with a batch_size other than 1, with a variable length with LSTM can cause an error when running the ONNX model with a different batch size. Make sure to save the model with a batch size of 1, or define the initial states (h0/c0) as inputs of the model. 
  warnings.warn(
  Exported → onnx_models/bilstm_msl.onnx
  Validating ONNX model...
  ✓ ONNX graph is valid
  ✓ Inference OK — output shape: (1, 556)
  ✓ Top-5 class indices: [213, 208, 226, 109, 96]
  Metadata  → onnx_models/bilstm_msl.json
  ✓ Done  (18.4 MB)

  SUMMARY: 1/1 models exported to 'onnx_models/'
    bilstm_msl.json                        0.0 MB
    bilstm_msl.onnx                       18.4 MB
    label_map.json                         0.1 MB

real	0m2.210s
us

In [21]:
!ls ./onnx_models/* -lh

-rw-rw-r-- 1 ye ye  130 May 28 15:31 ./onnx_models/bilstm_msl.json
-rw-rw-r-- 1 ye ye  19M May 28 15:31 ./onnx_models/bilstm_msl.onnx
-rw-rw-r-- 1 ye ye 114K May 28 13:43 ./onnx_models/label_map.json


In [22]:
!cat ./onnx_models/bilstm_msl.json

{
  "model_type": "bilstm",
  "num_classes": 556,
  "max_seq_len": 200,
  "feat_dim": 225,
  "num_joints": 75,
  "num_coords": 3
}

In [23]:
!cat ./onnx_models/label_map.json | head -n 30

{
  "label2idx": {
    "မီး ။": 0,
    "မီး ငြှိမ်း ။": 1,
    "မီးချိတ် ။": 2,
    "မီးကပ် ။": 3,
    "ရေပုံး ။": 4,
    "သဲ အိတ် ။": 5,
    "မီးသတ်ဆေးဘူး ။": 6,
    "မီးသတ်ရေကန် ။": 7,
    "မီးလောင် လွယ် သော ပစ္စည်း များ ။": 8,
    "လောင်စာဆီ ။": 9,
    "အမှိုက် ။": 10,
    "လျှပ်စစ် မီး ။": 11,
    "ဂက်စ် မီး ။": 12,
    "မီးကြွင်းမီးကျန် ။": 13,
    "မီးခလုပ် ။": 14,
    "မီးခြစ် ။": 15,
    "ဖယောင်းတိုင် ။": 16,
    "ခြင်ဆေးခွေ ။": 17,
    "အမွှေးတိုင် ။": 18,
    "မီးခိုး ။": 19,
    "အောက်စီဂျင် ။": 20,
    "ထွက်ပေါက် ။": 21,
    "မီးသတ်ကား ။": 22,
    "မီးသတ်သမား ။": 23,
    "မီးသတ်ဌာန ။": 24,
    "လျှပ်စစ် ဝါယာ ရှော့ ။": 25,
    "ရေဘေး ။": 26,
    "ဘေးအန္တရာယ် ။": 27,
cat: write error: Broken pipe


## Exporting Transformer Model

In [24]:
!time python ./src/export_to_onnx.py --config ./config/config.yaml --model transformer

Device: cuda

  Exporting  TRANSFORMER
  Classes: 556   MaxSeqLen: 200
  Transformer dummy inputs: kp=(1, 200, 225), mask=(1, 200)
  Exported → onnx_models/transformer_msl.onnx
  Validating ONNX model...
  ✓ ONNX graph is valid
  ✓ Inference OK — output shape: (1, 556)
  ✓ Top-5 class indices: [103, 201, 94, 119, 233]
  Metadata  → onnx_models/transformer_msl.json
  ✓ Done  (13.1 MB)

  SUMMARY: 1/1 models exported to 'onnx_models/'
    bilstm_msl.json                        0.0 MB
    bilstm_msl.onnx                       18.4 MB
    label_map.json                         0.1 MB
    transformer_msl.json                   0.0 MB
    transformer_msl.onnx                  13.1 MB

real	0m2.062s
user	0m5.627s
sys	0m2.292s


In [25]:
!ls ./onnx_models/* -lh

-rw-rw-r-- 1 ye ye  130 May 28 15:31 ./onnx_models/bilstm_msl.json
-rw-rw-r-- 1 ye ye  19M May 28 15:31 ./onnx_models/bilstm_msl.onnx
-rw-rw-r-- 1 ye ye 114K May 28 13:43 ./onnx_models/label_map.json
-rw-rw-r-- 1 ye ye  135 May 28 15:31 ./onnx_models/transformer_msl.json
-rw-rw-r-- 1 ye ye  14M May 28 15:31 ./onnx_models/transformer_msl.onnx


## Exporting ST-GCN Model

In [26]:
!time python ./src/export_to_onnx.py --config ./config/config.yaml --model stgcn

Device: cuda

  Exporting  STGCN
  Classes: 556   MaxSeqLen: 200
  ST-GCN dummy input: kp=(1, 3, 200, 75)
  Exported → onnx_models/stgcn_msl.onnx
  Validating ONNX model...
  ✓ ONNX graph is valid
  ✓ Inference OK — output shape: (1, 556)
  ✓ Top-5 class indices: [216, 208, 233, 470, 486]
  Metadata  → onnx_models/stgcn_msl.json
  ✓ Done  (10.6 MB)

  SUMMARY: 1/1 models exported to 'onnx_models/'
    bilstm_msl.json                        0.0 MB
    bilstm_msl.onnx                       18.4 MB
    label_map.json                         0.1 MB
    stgcn_msl.json                         0.0 MB
    stgcn_msl.onnx                        10.6 MB
    transformer_msl.json                   0.0 MB
    transformer_msl.onnx                  13.1 MB

real	0m3.353s
user	0m6.302s
sys	0m3.224s


In [27]:
!ls ./onnx_models/* -lh 

-rw-rw-r-- 1 ye ye  130 May 28 15:31 ./onnx_models/bilstm_msl.json
-rw-rw-r-- 1 ye ye  19M May 28 15:31 ./onnx_models/bilstm_msl.onnx
-rw-rw-r-- 1 ye ye 114K May 28 13:43 ./onnx_models/label_map.json
-rw-rw-r-- 1 ye ye  129 May 28 15:31 ./onnx_models/stgcn_msl.json
-rw-rw-r-- 1 ye ye  11M May 28 15:31 ./onnx_models/stgcn_msl.onnx
-rw-rw-r-- 1 ye ye  135 May 28 15:31 ./onnx_models/transformer_msl.json
-rw-rw-r-- 1 ye ye  14M May 28 15:31 ./onnx_models/transformer_msl.onnx


In [28]:
!cat ./onnx_models/stgcn_msl.json

{
  "model_type": "stgcn",
  "num_classes": 556,
  "max_seq_len": 200,
  "feat_dim": 225,
  "num_joints": 75,
  "num_coords": 3
}

## Notes for Running on Windows OS

ONNX မော်ဒယ်သုံးခုနဲ့ python code နှစ်ဖိုင်ကို ကိုယ့် local စက်ပေါ်ကို ကော်ပီကူးယူပါ။  

```
(base) C:\Users\801680>scp ye@xx.xxx.xx.xxx:/mnt/disk1/ye/exp/msl_recognition/onnx_models.zip Downloads
ye@xx.xxx.xx.xxx's password:
onnx_models.zip                                                        100%   39MB   2.0MB/s   00:19

(base) C:\Users\801680>scp ye@xx.xxx.xx.xxx:/mnt/disk1/ye/exp/msl_recognition/src/export_to_onnx.py Downloads
ye@xx.xxx.xx.xxx's password:
export_to_onnx.py                                                      100%   23KB 126.6KB/s   00:00

(base) C:\Users\801680>scp ye@xx.xxx.xx.xxx:/mnt/disk1/ye/exp/msl_recognition/src/infer_onnx.py Downloads
ye@xx.xxx.xx.xxx's password:
infer_onnx.py                                                          100%   24KB 131.7KB/s   00:00
```

ONNX က ကိုယ့် Windows OS စက်ထဲမှာ run လို့ရအောင် လိုအပ်တဲ့ python library တွေကို install လုပ်ထားရပါမယ်။  

```
pip install onnxruntime opencv-python mediapipe numpy Pillow
```

ဆရာကတော့ Anaconda environment အသစ်ဆောက်လိုက်ပါတယ်။ အရင်ဆုံး infer_onnx.py ကို run လို့ ရနိုင်မလား confirmation လုပ်ပါ။  

```
(base) E:\LU_Lab\MSL_Tran_Demo\onnx_models>conda activate msl_onnx

(msl_onnx) E:\LU_Lab\MSL_Tran_Demo\onnx_models>python infer_onnx.py --help
usage: infer_onnx.py [-h] --onnx_model ONNX_MODEL [--metadata METADATA] [--label_map LABEL_MAP]
                     (--video VIDEO | --webcam CAM_IDX) [--top_k TOP_K]
                     [--classify_every CLASSIFY_EVERY] [--log_file LOG_FILE] [--font_size FONT_SIZE]

MSL Sign Language Inference — ONNX (Windows/Linux)

options:
  -h, --help            show this help message and exit
  --onnx_model ONNX_MODEL
                        Path to .onnx model file
  --metadata METADATA   Path to .json metadata (auto-detected if omitted)
  --label_map LABEL_MAP
                        Path to label_map.json (auto-detected if omitted)
  --video VIDEO         Path to a video file
  --webcam CAM_IDX      Webcam device index (e.g. 0)
  --top_k TOP_K
  --classify_every CLASSIFY_EVERY
                        Run model every N webcam frames (default: 15)
  --log_file LOG_FILE   Append predictions to this file
  --font_size FONT_SIZE

(msl_onnx) E:\LU_Lab\MSL_Tran_Demo\onnx_models>
```

## Infer with Video on Windows Machine (with Transformer Model)

```
(msl_onnx) E:\LU_Lab\MSL_Tran_Demo\onnx_models>python infer_onnx.py --onnx_model transformer_msl.onnx --video ..\sample4inference\idx20-30.mp4
[INFO] Label map: 556 classes
C:\Users\801680\Anaconda3\envs\msl_onnx\lib\site-packages\onnxruntime\capi\onnxruntime_inference_collection.py:123: UserWarning: Specified provider 'CUDAExecutionProvider' is not in available provider names.Available providers: 'AzureExecutionProvider, CPUExecutionProvider'
  warnings.warn(
[INFO] ONNX loaded: transformer_msl.onnx  (provider: CPUExecutionProvider)
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
[INFO] Myanmar font loaded: C:\Windows\Fonts\seguisym.ttf
[INFO] Processing 127 frames from idx20-30.mp4 ...
C:\Users\801680\Anaconda3\envs\msl_onnx\lib\site-packages\google\protobuf\symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '

Video: idx20-30.mp4
──────────────────────────────────────────────────
  Top-1: ခင်ဗျား ခံစား နေ ရ တာ ဘယ်လောက် ကြာ ပြီ လဲ ။  (4.47%)
  Top-2: လမ်းလျှောက် ဖို့ လိုအပ် တယ် ။  (4.30%)
  Top-3: မနက် ၉ နာရီ က နေ ည ၁၀ နာရီ ထိ ပါ ။  (2.32%)
  Top-4: ဒဏ်ရာ ကို ရေခဲ ကပ် ပေး ပါ ။  (1.98%)
  Top-5: သူ့ မိသားစု ကို ငါ တို့ ဖုန်းဆက် ရ မယ် ။  (1.69%)
──────────────────────────────────────────────────
[INFO] Done.

(msl_onnx) E:\LU_Lab\MSL_Tran_Demo\onnx_models>
```

```
(msl_onnx) E:\LU_Lab\MSL_Tran_Demo\onnx_models>python infer_onnx.py --onnx_model transformer_msl.onnx --video ..\sample4inference\idx20-86.mp4
[INFO] Label map: 556 classes
C:\Users\801680\Anaconda3\envs\msl_onnx\lib\site-packages\onnxruntime\capi\onnxruntime_inference_collection.py:123: UserWarning: Specified provider 'CUDAExecutionProvider' is not in available provider names.Available providers: 'AzureExecutionProvider, CPUExecutionProvider'
  warnings.warn(
[INFO] ONNX loaded: transformer_msl.onnx  (provider: CPUExecutionProvider)
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
[INFO] Myanmar font loaded: C:\Windows\Fonts\seguisym.ttf
[INFO] Processing 169 frames from idx20-86.mp4 ...
C:\Users\801680\Anaconda3\envs\msl_onnx\lib\site-packages\google\protobuf\symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '

Video: idx20-86.mp4
──────────────────────────────────────────────────
  Top-1: လူနာ က ကျွန်တော့် အမေ ပါ ။  (5.48%)
  Top-2: ကောလဟာလ သတင်း တွေ ကို မ ယုံ နဲ့ ။  (4.22%)
  Top-3: သူ့ မိသားစု ကို ငါ တို့ ဖုန်းဆက် ရ မယ် ။  (3.68%)
  Top-4: ကျွန်မ ကလေး မှန်ကြည်ဆေး တွေ သောက် ပစ် လိုက် လို့ ပါ ။  (3.32%)
  Top-5: လတ်ဆတ် တဲ့ ဟင်းသီးဟင်းရွက် တွေ ကို စား ပါ ။  (2.74%)
──────────────────────────────────────────────────
[INFO] Done.

(msl_onnx) E:\LU_Lab\MSL_Tran_Demo\onnx_models>
```

```
(msl_onnx) E:\LU_Lab\MSL_Tran_Demo\onnx_models>python infer_onnx.py --onnx_model transformer_msl.onnx --video ..\sample4inference\idx20-244.mp4
[INFO] Label map: 556 classes
C:\Users\801680\Anaconda3\envs\msl_onnx\lib\site-packages\onnxruntime\capi\onnxruntime_inference_collection.py:123: UserWarning: Specified provider 'CUDAExecutionProvider' is not in available provider names.Available providers: 'AzureExecutionProvider, CPUExecutionProvider'
  warnings.warn(
[INFO] ONNX loaded: transformer_msl.onnx  (provider: CPUExecutionProvider)
[INFO] Myanmar font loaded: C:\Windows\Fonts\seguisym.ttf
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
[INFO] Processing 55 frames from idx20-244.mp4 ...
C:\Users\801680\Anaconda3\envs\msl_onnx\lib\site-packages\google\protobuf\symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '

Video: idx20-244.mp4
──────────────────────────────────────────────────
  Top-1: လူနာ နဲ့ ဘယ်လို တော် တာ လဲ ။  (5.14%)
  Top-2: လတ်ဆတ် တဲ့ ဟင်းသီးဟင်းရွက် တွေ ကို စား ပါ ။  (3.30%)
  Top-3: ငါ ဒီ ဆေး တွေ ကို ဆက် သောက် သင့် သ လား ။  (2.75%)
  Top-4: သူ ကျွန်တော့် ကို ထိုး တယ် ။  (2.68%)
  Top-5: အစာ စား ပြီး မှ သောက် ပါ ။  (2.42%)
──────────────────────────────────────────────────
[INFO] Done.

(msl_onnx) E:\LU_Lab\MSL_Tran_Demo\onnx_models>
```

```
(msl_onnx) E:\LU_Lab\MSL_Tran_Demo\onnx_models>python infer_onnx.py --onnx_model transformer_msl.onnx --video ..\sample4inference\idx20-291.mp4
[INFO] Label map: 556 classes
C:\Users\801680\Anaconda3\envs\msl_onnx\lib\site-packages\onnxruntime\capi\onnxruntime_inference_collection.py:123: UserWarning: Specified provider 'CUDAExecutionProvider' is not in available provider names.Available providers: 'AzureExecutionProvider, CPUExecutionProvider'
  warnings.warn(
[INFO] ONNX loaded: transformer_msl.onnx  (provider: CPUExecutionProvider)
[INFO] Myanmar font loaded: C:\Windows\Fonts\seguisym.ttf
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
[INFO] Processing 211 frames from idx20-291.mp4 ...
C:\Users\801680\Anaconda3\envs\msl_onnx\lib\site-packages\google\protobuf\symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '

Video: idx20-291.mp4
──────────────────────────────────────────────────
  Top-1: သူ့ မိသားစု ကို ငါ တို့ ဖုန်းဆက် ရ မယ် ။  (8.28%)
  Top-2: လတ်ဆတ် တဲ့ ဟင်းသီးဟင်းရွက် တွေ ကို စား ပါ ။  (6.06%)
  Top-3: လူနာစောင့် တစ်ယောက် ပဲ ည အိပ် စောင့် ခွင့် ရှိ ပါ မယ် ။  (3.09%)
  Top-4: အမျိုးသမီး လူနာစောင့် ပဲ ည အိပ် စောင့် ရ ပါ မယ် ။  (1.93%)
  Top-5: ပျောက် သွား တဲ့ လူ ရဲ့ ပုံပန်းသဏ္ဌာန် ကို ပြောပြ နိုင် မ လား ။  (1.08%)
──────────────────────────────────────────────────
[INFO] Done.

(msl_onnx) E:\LU_Lab\MSL_Tran_Demo\onnx_models>
```

```
(msl_onnx) E:\LU_Lab\MSL_Tran_Demo\onnx_models>python infer_onnx.py --onnx_model transformer_msl.onnx --video ..\sample4inference\idx20-440.mp4
[INFO] Label map: 556 classes
C:\Users\801680\Anaconda3\envs\msl_onnx\lib\site-packages\onnxruntime\capi\onnxruntime_inference_collection.py:123: UserWarning: Specified provider 'CUDAExecutionProvider' is not in available provider names.Available providers: 'AzureExecutionProvider, CPUExecutionProvider'
  warnings.warn(
[INFO] ONNX loaded: transformer_msl.onnx  (provider: CPUExecutionProvider)
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
[INFO] Myanmar font loaded: C:\Windows\Fonts\seguisym.ttf
[INFO] Processing 248 frames from idx20-440.mp4 ...
C:\Users\801680\Anaconda3\envs\msl_onnx\lib\site-packages\google\protobuf\symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '

Video: idx20-440.mp4
──────────────────────────────────────────────────
  Top-1: တိုက် သွား တဲ့ ကား ရဲ့ နံပါတ် နဲ့ အရောင် ကို ကောင်းကောင်း မှတ်မိ လား ။  (13.84%)
  Top-2: လတ်ဆတ် တဲ့ ဟင်းသီးဟင်းရွက် တွေ ကို စား ပါ ။  (6.91%)
  Top-3: သူ့ မိသားစု ကို ငါ တို့ ဖုန်းဆက် ရ မယ် ။  (2.18%)
  Top-4: မိသားစုဝင် တယောက်ယောက် ကို အကြောင်းကြား ။  (1.94%)
  Top-5: အရေပြား အထူးကု ဒေါက်တာ ဒေါ်ခင်မြလွင် နဲ့ ပြ ချင် လို့ ပါ ။  (1.24%)
──────────────────────────────────────────────────
[INFO] Done.

(msl_onnx) E:\LU_Lab\MSL_Tran_Demo\onnx_models>
```

## Infer with BiLSTM Model

```
(msl_onnx) E:\LU_Lab\MSL_Tran_Demo\onnx_models>python infer_onnx.py --onnx_model bilstm_msl.onnx --video ..\sample4inference\idx20-30.mp4
[INFO] Label map: 556 classes
C:\Users\801680\Anaconda3\envs\msl_onnx\lib\site-packages\onnxruntime\capi\onnxruntime_inference_collection.py:123: UserWarning: Specified provider 'CUDAExecutionProvider' is not in available provider names.Available providers: 'AzureExecutionProvider, CPUExecutionProvider'
  warnings.warn(
[INFO] ONNX loaded: bilstm_msl.onnx  (provider: CPUExecutionProvider)
[INFO] Myanmar font loaded: C:\Windows\Fonts\seguisym.ttf
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
[INFO] Processing 127 frames from idx20-30.mp4 ...
C:\Users\801680\Anaconda3\envs\msl_onnx\lib\site-packages\google\protobuf\symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '

Video: idx20-30.mp4
──────────────────────────────────────────────────
  Top-1: မနက် ၉ နာရီ က နေ ည ၁၀ နာရီ ထိ ပါ ။  (5.93%)
  Top-2: အမျိုးသမီး လူနာစောင့် ပဲ ည အိပ် စောင့် ရ ပါ မယ် ။  (4.63%)
  Top-3: ည ၁၀ နာရီ နောက် ပိုင်း အမျိုးသား နေ ခွင့် မ ရှိ ပါ ဘူး ။  (3.46%)
  Top-4: မိုးလေဝသ သတင်း များ ကို စဉ်ဆက်မပြတ် နားထောင် ပါ ။  (2.17%)
  Top-5: လူနာ တွေ့ ချိန် က ဘယ် အချိန် လဲ ။  (0.96%)
──────────────────────────────────────────────────
[INFO] Done.

(msl_onnx) E:\LU_Lab\MSL_Tran_Demo\onnx_models>
```

```
(msl_onnx) E:\LU_Lab\MSL_Tran_Demo\onnx_models>python infer_onnx.py --onnx_model bilstm_msl.onnx --video ..\sample4inference\idx20-86.mp4
[INFO] Label map: 556 classes
C:\Users\801680\Anaconda3\envs\msl_onnx\lib\site-packages\onnxruntime\capi\onnxruntime_inference_collection.py:123: UserWarning: Specified provider 'CUDAExecutionProvider' is not in available provider names.Available providers: 'AzureExecutionProvider, CPUExecutionProvider'
  warnings.warn(
[INFO] ONNX loaded: bilstm_msl.onnx  (provider: CPUExecutionProvider)
[INFO] Myanmar font loaded: C:\Windows\Fonts\seguisym.ttf
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
[INFO] Processing 169 frames from idx20-86.mp4 ...
C:\Users\801680\Anaconda3\envs\msl_onnx\lib\site-packages\google\protobuf\symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '

Video: idx20-86.mp4
──────────────────────────────────────────────────
  Top-1: အစားသောက် မ တည့် တာ တွေ ရှိ လား ။  (2.56%)
  Top-2: မိသားစုဝင် တယောက်ယောက် ကို အကြောင်းကြား ။  (2.47%)
  Top-3: ဆရာဝန် ဖုန်း နံပါတ် ခင်ဗျား မှာ ရှိ လား ။  (2.11%)
  Top-4: ည ၁၀ နာရီ နောက် ပိုင်း အမျိုးသား နေ ခွင့် မ ရှိ ပါ ဘူး ။  (1.76%)
  Top-5: သွားဆရာဝန် နဲ့ ဒီ နေ့ မနက် တွေ့ နိုင် မ လား ခင်ဗျာ ။  (1.27%)
──────────────────────────────────────────────────
[INFO] Done.

(msl_onnx) E:\LU_Lab\MSL_Tran_Demo\onnx_models>
```

```
(msl_onnx) E:\LU_Lab\MSL_Tran_Demo\onnx_models>python infer_onnx.py --onnx_model bilstm_msl.onnx --video ..\sample4inference\idx20-244.mp4
[INFO] Label map: 556 classes
C:\Users\801680\Anaconda3\envs\msl_onnx\lib\site-packages\onnxruntime\capi\onnxruntime_inference_collection.py:123: UserWarning: Specified provider 'CUDAExecutionProvider' is not in available provider names.Available providers: 'AzureExecutionProvider, CPUExecutionProvider'
  warnings.warn(
[INFO] ONNX loaded: bilstm_msl.onnx  (provider: CPUExecutionProvider)
[INFO] Myanmar font loaded: C:\Windows\Fonts\seguisym.ttf
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
[INFO] Processing 55 frames from idx20-244.mp4 ...
C:\Users\801680\Anaconda3\envs\msl_onnx\lib\site-packages\google\protobuf\symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '

Video: idx20-244.mp4
──────────────────────────────────────────────────
  Top-1: လူနာ နဲ့ ဘယ်လို တော် တာ လဲ ။  (6.74%)
  Top-2: ငါ ဒီ ဆေး တွေ ကို ဆက် သောက် သင့် သ လား ။  (1.35%)
  Top-3: ဘယ် နေရာ က နာ တာ လဲ ။  (1.01%)
  Top-4: ငါ့ ညီလေး အသက် မ ရှူ တော့ ဘူး ။  (0.92%)
  Top-5: ဘာ အကူညီ ပေး ရ မ လဲ ။  (0.81%)
──────────────────────────────────────────────────
[INFO] Done.

(msl_onnx) E:\LU_Lab\MSL_Tran_Demo\onnx_models>
```

```
(msl_onnx) E:\LU_Lab\MSL_Tran_Demo\onnx_models>python infer_onnx.py --onnx_model bilstm_msl.onnx --video ..\sample4inference\idx20-291.mp4
[INFO] Label map: 556 classes
C:\Users\801680\Anaconda3\envs\msl_onnx\lib\site-packages\onnxruntime\capi\onnxruntime_inference_collection.py:123: UserWarning: Specified provider 'CUDAExecutionProvider' is not in available provider names.Available providers: 'AzureExecutionProvider, CPUExecutionProvider'
  warnings.warn(
[INFO] ONNX loaded: bilstm_msl.onnx  (provider: CPUExecutionProvider)
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
[INFO] Myanmar font loaded: C:\Windows\Fonts\seguisym.ttf
[INFO] Processing 211 frames from idx20-291.mp4 ...
C:\Users\801680\Anaconda3\envs\msl_onnx\lib\site-packages\google\protobuf\symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '

Video: idx20-291.mp4
──────────────────────────────────────────────────
  Top-1: မနက် ၉ နာရီ က နေ ည ၁၀ နာရီ ထိ ပါ ။  (8.98%)
  Top-2: လူနာ တွေ့ ချိန် က ဘယ် အချိန် လဲ ။  (6.14%)
  Top-3: ကြမ်းပြင် တစ်ခု လုံး မှာ ရေ တွေ ကြည့် ပဲ ။  (1.78%)
  Top-4: လူနာစောင့် တစ်ယောက် ပဲ ည အိပ် စောင့် ခွင့် ရှိ ပါ မယ် ။  (1.03%)
  Top-5: ဆေး မ တည့် တာ တွေ ရှိ လား ။  (0.97%)
──────────────────────────────────────────────────
[INFO] Done.

(msl_onnx) E:\LU_Lab\MSL_Tran_Demo\onnx_models>
```

```
(msl_onnx) E:\LU_Lab\MSL_Tran_Demo\onnx_models>python infer_onnx.py --onnx_model bilstm_msl.onnx --video ..\sample4inference\idx20-440.mp4
[INFO] Label map: 556 classes
C:\Users\801680\Anaconda3\envs\msl_onnx\lib\site-packages\onnxruntime\capi\onnxruntime_inference_collection.py:123: UserWarning: Specified provider 'CUDAExecutionProvider' is not in available provider names.Available providers: 'AzureExecutionProvider, CPUExecutionProvider'
  warnings.warn(
[INFO] ONNX loaded: bilstm_msl.onnx  (provider: CPUExecutionProvider)
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
[INFO] Myanmar font loaded: C:\Windows\Fonts\seguisym.ttf
[INFO] Processing 248 frames from idx20-440.mp4 ...
C:\Users\801680\Anaconda3\envs\msl_onnx\lib\site-packages\google\protobuf\symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '

Video: idx20-440.mp4
──────────────────────────────────────────────────
  Top-1: ကား က ဘာ အရောင် လဲ ။  (1.50%)
  Top-2: ဆရာဝန် ဖုန်း နံပါတ် ခင်ဗျား မှာ ရှိ လား ။  (1.25%)
  Top-3: တိုက် သွား တဲ့ ကား ရဲ့ နံပါတ် နဲ့ အရောင် ကို ကောင်းကောင်း မှတ်မိ လား ။  (1.21%)
  Top-4: အနီးဆုံး ဆေးရုံ ကို ပို့ ပေး ပါ ။  (1.13%)
  Top-5: အရေပြား တွေ ခြောက် နေ တာ အတွက် ခရင်မ် လိမ်းဆေး လို ချင် ပါ တယ် ။  (1.08%)
──────────────────────────────────────────────────
[INFO] Done.

(msl_onnx) E:\LU_Lab\MSL_Tran_Demo\onnx_models>
```


## Infer with STGCN Model

```
(msl_onnx) E:\LU_Lab\MSL_Tran_Demo\onnx_models>python infer_onnx.py --onnx_model stgcn_msl.onnx --video ..\sample4inference\idx20-30.mp4
[INFO] Label map: 556 classes
C:\Users\801680\Anaconda3\envs\msl_onnx\lib\site-packages\onnxruntime\capi\onnxruntime_inference_collection.py:123: UserWarning: Specified provider 'CUDAExecutionProvider' is not in available provider names.Available providers: 'AzureExecutionProvider, CPUExecutionProvider'
  warnings.warn(
[INFO] ONNX loaded: stgcn_msl.onnx  (provider: CPUExecutionProvider)
[INFO] Myanmar font loaded: C:\Windows\Fonts\seguisym.ttf
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
[INFO] Processing 127 frames from idx20-30.mp4 ...
C:\Users\801680\Anaconda3\envs\msl_onnx\lib\site-packages\google\protobuf\symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '

Video: idx20-30.mp4
──────────────────────────────────────────────────
  Top-1: မြေနိမ့်ဒေသ ။  (8.49%)
  Top-2: တောမီး က အရမ်း အန္တရာယ် ကြီး တယ် ။  (2.76%)
  Top-3: ရေမြုပ် နေ တဲ့ နေရာ မှာ ကားမောင်း ဖို့ မ ကြိုးစား နဲ့ ။  (1.93%)
  Top-4: မင်း မီးလောင် တဲ့ အနံ့ မ ရ ဘူး လား ။  (1.91%)
  Top-5: တောင်စောင်း ။  (1.78%)
──────────────────────────────────────────────────
[INFO] Done.

(msl_onnx) E:\LU_Lab\MSL_Tran_Demo\onnx_models>
```

```
(msl_onnx) E:\LU_Lab\MSL_Tran_Demo\onnx_models>python infer_onnx.py --onnx_model stgcn_msl.onnx --video ..\sample4inference\idx20-86.mp4
[INFO] Label map: 556 classes
C:\Users\801680\Anaconda3\envs\msl_onnx\lib\site-packages\onnxruntime\capi\onnxruntime_inference_collection.py:123: UserWarning: Specified provider 'CUDAExecutionProvider' is not in available provider names.Available providers: 'AzureExecutionProvider, CPUExecutionProvider'
  warnings.warn(
[INFO] ONNX loaded: stgcn_msl.onnx  (provider: CPUExecutionProvider)
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
[INFO] Myanmar font loaded: C:\Windows\Fonts\seguisym.ttf
[INFO] Processing 169 frames from idx20-86.mp4 ...
C:\Users\801680\Anaconda3\envs\msl_onnx\lib\site-packages\google\protobuf\symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '

Video: idx20-86.mp4
──────────────────────────────────────────────────
  Top-1: မနက် ၉ နာရီ က နေ ည ၁၀ နာရီ ထိ ပါ ။  (24.19%)
  Top-2: အစာ မ စား ခင် နာရီ ဝက် အလို မှာ သောက် ပါ ။  (8.06%)
  Top-3: ခေါင်း တွေ ခဏခဏ ကိုက် နေ တယ် ။  (7.14%)
  Top-4: သွားဆရာဝန် နဲ့ ဒီ နေ့ မနက် တွေ့ နိုင် မ လား ခင်ဗျာ ။  (3.21%)
  Top-5: 199 ကို ဖုန်းဆက် ပေး ပါ ။  (3.03%)
──────────────────────────────────────────────────
[INFO] Done.

(msl_onnx) E:\LU_Lab\MSL_Tran_Demo\onnx_models>
```

```
(msl_onnx) E:\LU_Lab\MSL_Tran_Demo\onnx_models>python infer_onnx.py --onnx_model stgcn_msl.onnx --video ..\sample4inference\idx20-244.mp4
[INFO] Label map: 556 classes
C:\Users\801680\Anaconda3\envs\msl_onnx\lib\site-packages\onnxruntime\capi\onnxruntime_inference_collection.py:123: UserWarning: Specified provider 'CUDAExecutionProvider' is not in available provider names.Available providers: 'AzureExecutionProvider, CPUExecutionProvider'
  warnings.warn(
[INFO] ONNX loaded: stgcn_msl.onnx  (provider: CPUExecutionProvider)
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
[INFO] Myanmar font loaded: C:\Windows\Fonts\seguisym.ttf
[INFO] Processing 55 frames from idx20-244.mp4 ...
C:\Users\801680\Anaconda3\envs\msl_onnx\lib\site-packages\google\protobuf\symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '

Video: idx20-244.mp4
──────────────────────────────────────────────────
  Top-1: မ တပ် ထား ပါ ဘူး ။  (25.61%)
  Top-2: မြန်မြန် လုပ် ပါ ။  (7.39%)
  Top-3: ကား ကို ရပ် ထား ။  (3.51%)
  Top-4: ဂက်စ် မီး ။  (3.38%)
  Top-5: သူ ကျွန်တော့် ကို ထိုး တယ် ။  (2.96%)
──────────────────────────────────────────────────
[INFO] Done.

(msl_onnx) E:\LU_Lab\MSL_Tran_Demo\onnx_models>
```

```
(msl_onnx) E:\LU_Lab\MSL_Tran_Demo\onnx_models>python infer_onnx.py --onnx_model stgcn_msl.onnx --video ..\sample4inference\idx20-291.mp4
[INFO] Label map: 556 classes
C:\Users\801680\Anaconda3\envs\msl_onnx\lib\site-packages\onnxruntime\capi\onnxruntime_inference_collection.py:123: UserWarning: Specified provider 'CUDAExecutionProvider' is not in available provider names.Available providers: 'AzureExecutionProvider, CPUExecutionProvider'
  warnings.warn(
[INFO] ONNX loaded: stgcn_msl.onnx  (provider: CPUExecutionProvider)
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
[INFO] Myanmar font loaded: C:\Windows\Fonts\seguisym.ttf
[INFO] Processing 211 frames from idx20-291.mp4 ...
C:\Users\801680\Anaconda3\envs\msl_onnx\lib\site-packages\google\protobuf\symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '

Video: idx20-291.mp4
──────────────────────────────────────────────────
  Top-1: ညစ်ညမ်း နေ တဲ့ နေရာ များ ကို သန့်ရှင်းရေး လုပ် ပြီး ရောဂါပိုး သတ် ရ ပါ မယ် ။  (4.19%)
  Top-2: ၄ ရက် ကြာ မယ် ။  (3.24%)
  Top-3: မနက် ၆ နာရီ နောက် ပိုင်း အစား မ စား ပါ နဲ့ ။  (3.21%)
  Top-4: ဖန်ကွဲစ ရှ မိ ပြီး သွေး တွေ တအား ထွက် နေ တယ် ။  (3.17%)
  Top-5: ဆေးစစ်ချက် တွေ ဘယ် အချိန် ရ မ လဲ ။  (2.53%)
──────────────────────────────────────────────────
[INFO] Done.

(msl_onnx) E:\LU_Lab\MSL_Tran_Demo\onnx_models>
```

```
(msl_onnx) E:\LU_Lab\MSL_Tran_Demo\onnx_models>python infer_onnx.py --onnx_model stgcn_msl.onnx --video ..\sample4inference\idx20-440.mp4
[INFO] Label map: 556 classes
C:\Users\801680\Anaconda3\envs\msl_onnx\lib\site-packages\onnxruntime\capi\onnxruntime_inference_collection.py:123: UserWarning: Specified provider 'CUDAExecutionProvider' is not in available provider names.Available providers: 'AzureExecutionProvider, CPUExecutionProvider'
  warnings.warn(
[INFO] ONNX loaded: stgcn_msl.onnx  (provider: CPUExecutionProvider)
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
[INFO] Myanmar font loaded: C:\Windows\Fonts\seguisym.ttf
[INFO] Processing 248 frames from idx20-440.mp4 ...
C:\Users\801680\Anaconda3\envs\msl_onnx\lib\site-packages\google\protobuf\symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '

Video: idx20-440.mp4
──────────────────────────────────────────────────
  Top-1: ကား ထဲ မှာ လူ တစ်ယောက် ညှပ်မိ နေ တယ် ။  (10.89%)
  Top-2: ဘေး အိမ် ကို ဓားပြ ဝင် တိုက် နေ ပါ တယ် ။  (6.00%)
  Top-3: မိုးလေဝသ သတင်း များ ကို စဉ်ဆက်မပြတ် နားထောင် ပါ ။  (4.51%)
  Top-4: ပြိုကျ နိုင် တဲ့ အရာ များ နဲ့ ဝေးဝေး နေ ပါ ။  (4.19%)
  Top-5: သူ အခြေအနေ ကောင်း ပါ့ မ လား ။  (3.54%)
──────────────────────────────────────────────────
[INFO] Done.

(msl_onnx) E:\LU_Lab\MSL_Tran_Demo\onnx_models>
```

## Comments

- အထက်မှာ မြင်ရတဲ့အတိုင်းပဲ ONNX ပြောင်းလိုက်ပြီး Windows OS ဘက်မှာ ဗီဒီယိုဖိုင် input လုပ်ပြီး စမ်းသပ်ကြည့်တော့ ရလဒ်တွေက မကောင်းဘူး။ အထူးသဖြင့် BiLSTM နဲ့ Transformer မော်ဒယ်တွေရဲ့ ခန့်မှန်းချက်က တစ်ခုမှ မမှန်တာကို တွေ့ရတယ်။ STGCN မော်ဒယ်ကတော့ ဗီဒီယို ၅ ဖိုင်မှာ ၂ ခုမှန်ပါတယ်။  

- WebCam နဲ့ စမ်းတာကကျတော့ Jupyter Notebook ထဲကနေ အဆင်မပြေဘူး။ အဲဒါကြောင့် commandline ကနေပဲ testing လုပ်ကြည့်ခဲ့တယ်။


## Debugging/Updating

Code တွေကို အကြိမ်ကြိမ်အခါခါ ဝင်ပြင်လိုက် run ကြည့်လိုက် လုပ်ခဲ့တယ်။  
လက်ရှိ ဗားရှင်းနဲ့ Jupyter Notebook မှာ ထပ် စမ်းကြည့်ပြီး log လုပ်ထားမယ်။ ဘယ်လောက် ရလဒ် ကောင်းလာမလဲ ဆိုတာ ကြည့်မယ်။  

In [29]:
from IPython.display import Markdown

file_path = './src/export_to_onnx.py'

with open(file_path, 'r') as f:
    code = f.read()

# 'python' keyword triggers the syntax highlighter
display(Markdown(f"```python\n{code}\n```"))

```python
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
export_to_onnx.py  –  Export trained MSL models to ONNX format
                       (runs on Linux/HPC server side)

Bugs fixed vs original
───────────────────────
  1. ST-GCN: edge_importance in-place copy_ during torch.onnx.export broke
     the export (same root cause as the training autograd bug).  Fixed by
     pre-computing the scaled adjacency before export so no in-place op
     appears in the traced graph.
  2. BiLSTM: pack_padded_sequence is NOT supported in ONNX opset 14; the
     original wrapper silently fell back to an empty-graph export.  Fixed
     by bypassing packing entirely (no-op path) and exporting with a simple
     mask-sum length calculation that ONNX can trace.
  3. input_dim for ST-GCN was referencing an undefined variable; fixed with
     explicit constant (225 for sequence models, None for graph models).
  4. num_joints / num_coords were leaking between branches; fixed with clear
     if/elif blocks and explicit constants.
  5. The metadata JSON now includes 'model_type' (was missing), which
     infer_onnx.py reads to choose the correct input layout.
  6. Transformer: PyTorch 2.x fuses TransformerEncoderLayer and
     MultiheadAttention into single CUDA kernels 
     (`aten::_transformer_encoder_layer_fwd` and `aten::_native_multi_head_attention`)
     which have NO ONNX mapping in opset 14. Also, `aten::gelu` requires opset >= 20.
     Fixed by unrolling the encoder loop, replacing MHA with a manual math 
     implementation, and replacing GELU with an `erf`-based equivalent.

Usage (on Linux / HPC)
───────────────────────
  python src/export_to_onnx.py --config config/config.yaml --all
  python src/export_to_onnx.py --config config/config.yaml --model transformer
  python src/export_to_onnx.py --config config/config.yaml --model bilstm --no-validate
"""

import argparse
import json
import shutil
import sys
from pathlib import Path

import numpy as np
import torch
import yaml

sys.path.insert(0, str(Path(__file__).parent))
from models import build_model
from utils  import get_logger, get_device, load_label_map

# ── Experiment → output name mapping ─────────────────────────────────────────
MODEL_CONFIGS = {
    "bilstm":       {"exp_dir": "results/exp_bilstm",       "output_name": "bilstm_msl.onnx"},
    "transformer":  {"exp_dir": "results/exp_transformer",  "output_name": "transformer_msl.onnx"},
    "stgcn":        {"exp_dir": "results/exp_stgcn",        "output_name": "stgcn_msl.onnx"},
}

# Fixed constants — must match extract_keypoints.py
POSE_N       = 33
HAND_N       = 21
TOTAL_JOINTS = POSE_N + HAND_N * 2   # 75
FEAT_DIM     = TOTAL_JOINTS * 3       # 225


# ─────────────────────────────────────────────────────────────────────────────
# BiLSTM export
# ─────────────────────────────────────────────────────────────────────────────

#class _BiLSTMExportWrapper(torch.nn.Module):
#    """
#    Thin wrapper that converts the boolean padding mask to a length tensor
#    WITHOUT calling pack_padded_sequence (which ONNX cannot trace).
#
#    The mask convention (True = padded position) is the same as training.
#    We count the valid (non-padded) positions to get sequence lengths,
#    then call the model in 'no-pack' mode by setting lengths=None and
#    passing the mask directly so the attention layer masks out padding.
#    """
#    def __init__(self, model):
#        super().__init__()
#        self.model = model

#    def forward(self, keypoints: torch.Tensor, mask: torch.Tensor) -> torch.Tensor:
        # mask: (B, T) True where PADDED  →  lengths = number of valid frames
        # We do NOT pass lengths so the LSTM runs on the full padded sequence
        # (slightly less efficient, but fully ONNX-traceable).
        # The additive attention mask still zeros out padded positions.
#        return self.model(keypoints, lengths=None, mask=mask)


#def export_bilstm(model, out_path: Path, max_seq_len: int, device: str):
#    model.eval()
#    wrapper = _BiLSTMExportWrapper(model).to(device)
#    wrapper.eval()

#    dummy_kp   = torch.randn(1, max_seq_len, FEAT_DIM, device=device)
    # mask: True = padded.  Use all-False (nothing padded) for the trace.
#    dummy_mask = torch.zeros(1, max_seq_len, dtype=torch.bool, device=device)

#    print(f"  BiLSTM dummy inputs: kp={tuple(dummy_kp.shape)}, mask={tuple(dummy_mask.shape)}")

#    with torch.no_grad():
#        torch.onnx.export(
#            wrapper,
#            (dummy_kp, dummy_mask),
#            str(out_path),
#            opset_version=14,
#            input_names=["keypoints", "mask"],
#            output_names=["logits"],
#            dynamic_axes={
#                "keypoints": {0: "batch"},
#                "mask":      {0: "batch"},
#                "logits":    {0: "batch"},
#            },
#            do_constant_folding=True,
#            verbose=False,
#        )
#    print(f"  Exported → {out_path}")

class _BiLSTMExportWrapper(torch.nn.Module):
    """
    Traces pack_padded_sequence safely by treating lengths as an explicit tensor input.
    """
    def __init__(self, model):
        super().__init__()
        self.model = model

    def forward(self, keypoints: torch.Tensor, lengths: torch.Tensor) -> torch.Tensor:
        # Pass lengths as an explicit Tensor which ONNX opset 14 maps perfectly
        return self.model(keypoints, lengths=lengths, mask=None)


def export_bilstm(model, out_path: Path, max_seq_len: int, device: str):
    model.eval()
    wrapper = _BiLSTMExportWrapper(model).to(device)
    wrapper.eval()

    dummy_kp = torch.randn(1, max_seq_len, FEAT_DIM, device=device)
    # Default trace expects full sequence length
    dummy_len = torch.tensor([max_seq_len], dtype=torch.int64, device=device)

    print(f"  BiLSTM dummy inputs: kp={tuple(dummy_kp.shape)}, lengths={tuple(dummy_len.shape)}")

    with torch.no_grad():
        torch.onnx.export(
            wrapper,
            (dummy_kp, dummy_len),
            str(out_path),
            opset_version=14,
            input_names=["keypoints", "lengths"],
            output_names=["logits"],
            dynamic_axes={
                "keypoints": {0: "batch"},
                "lengths":   {0: "batch"},
                "logits":    {0: "batch"},
            },
            do_constant_folding=True,
            verbose=False,
        )
    print(f"  Exported safely → {out_path}")

# ─────────────────────────────────────────────────────────────────────────────
# Transformer export
# ─────────────────────────────────────────────────────────────────────────────

def _onnx_safe_gelu(x: torch.Tensor) -> torch.Tensor:
    """GELU implementation using erf, fully supported in ONNX opset 14+."""
    return 0.5 * x * (1.0 + torch.erf(x / 1.4142135623730951))


class ManualMHA(torch.nn.Module):
    """
    Manual Multi-Head Attention implementation to bypass PyTorch 2.x's
    fused `aten::_native_multi_head_attention` which lacks ONNX support.
    """
    def __init__(self, mha: torch.nn.MultiheadAttention):
        super().__init__()
        self.embed_dim = mha.embed_dim
        self.num_heads = mha.num_heads
        self.head_dim = self.embed_dim // self.num_heads
        
        self.in_proj_weight = torch.nn.Parameter(mha.in_proj_weight.data.clone())
        self.in_proj_bias = torch.nn.Parameter(mha.in_proj_bias.data.clone())
        self.out_proj_weight = torch.nn.Parameter(mha.out_proj.weight.data.clone())
        self.out_proj_bias = torch.nn.Parameter(mha.out_proj.bias.data.clone())

    def forward(self, query: torch.Tensor, key: torch.Tensor, value: torch.Tensor,
                key_padding_mask: torch.Tensor = None, need_weights: bool = False):
        B, T, C = query.shape
        
        # Split projection weights for Q, K, V
        q_w = self.in_proj_weight[:C]
        k_w = self.in_proj_weight[C:2*C]
        v_w = self.in_proj_weight[2*C:]
        
        q_b = self.in_proj_bias[:C]
        k_b = self.in_proj_bias[C:2*C]
        v_b = self.in_proj_bias[2*C:]
        
        Q = torch.nn.functional.linear(query, q_w, q_b)
        K = torch.nn.functional.linear(key, k_w, k_b)
        V = torch.nn.functional.linear(value, v_w, v_b)
        
        # Use -1 for dynamic sequence length compatibility in ONNX
        Q = Q.reshape(B, -1, self.num_heads, self.head_dim).transpose(1, 2)
        K = K.reshape(B, -1, self.num_heads, self.head_dim).transpose(1, 2)
        V = V.reshape(B, -1, self.num_heads, self.head_dim).transpose(1, 2)
        
        # Scaled dot-product attention
        scale = self.head_dim ** 0.5
        scores = torch.matmul(Q, K.transpose(-2, -1)) / scale
        
        if key_padding_mask is not None:
            # key_padding_mask: (B, T) -> (B, 1, 1, T) for broadcasting
            mask = key_padding_mask.unsqueeze(1).unsqueeze(2)
            scores = scores.masked_fill(mask, float('-inf'))
            
        attn_weights = torch.nn.functional.softmax(scores, dim=-1)
        attn_out = torch.matmul(attn_weights, V)
        
        # Reshape back to (B, T, C)
        attn_out = attn_out.transpose(1, 2).reshape(B, -1, C)
        
        out = torch.nn.functional.linear(attn_out, self.out_proj_weight, self.out_proj_bias)
        
        if need_weights:
            return out, attn_weights
        return out, None


class _TransformerExportWrapper(torch.nn.Module):
    """
    Wrapper that completely bypasses PyTorch 2.x fused operators by unrolling
    the encoder loop and calling basic sub-modules directly.
    """
    def __init__(self, model):
        super().__init__()
        # Copy all the model's input/head components
        self.input_norm = model.input_norm
        self.input_proj = model.input_proj
        self.proj_norm = model.proj_norm
        self.cls_token = model.cls_token
        self.pos_enc = model.pos_enc
        self.encoder_norm = model.encoder.norm
        self.head_norm = model.head_norm
        self.head_dropout = model.head_dropout
        self.fc = model.fc
        
        # Replace fused MultiheadAttention with ONNX-safe manual implementation
        self.layer_self_attn = torch.nn.ModuleList([ManualMHA(l.self_attn) for l in model.encoder.layers])
        self.layer_linear1 = torch.nn.ModuleList([l.linear1 for l in model.encoder.layers])
        self.layer_linear2 = torch.nn.ModuleList([l.linear2 for l in model.encoder.layers])
        self.layer_norm1 = torch.nn.ModuleList([l.norm1 for l in model.encoder.layers])
        self.layer_norm2 = torch.nn.ModuleList([l.norm2 for l in model.encoder.layers])
        self.layer_dropout1 = torch.nn.ModuleList([l.dropout1 for l in model.encoder.layers])
        self.layer_dropout2 = torch.nn.ModuleList([l.dropout2 for l in model.encoder.layers])
        
        self.num_layers = len(model.encoder.layers)
        
        # Check architecture type from the first layer
        self.norm_first = model.encoder.layers[0].norm_first
        
        # Determine activation type
        act = model.encoder.layers[0].activation
        if act is torch.nn.functional.relu or isinstance(act, torch.nn.ReLU):
            self._use_gelu = False
        else:
            self._use_gelu = True

    def forward(self, keypoints: torch.Tensor, mask: torch.Tensor) -> torch.Tensor:
        # ── Input processing ──
        x = self.input_norm(keypoints)       # (B, T, 225)
        x = self.input_proj(x)               # (B, T, 256)
        x = self.proj_norm(x)                # (B, T, 256)
        
        # ── Add CLS token ──
        B = x.shape[0]
        cls_tokens = self.cls_token.expand(B, -1, -1)  # (B, 1, 256)
        x = torch.cat([cls_tokens, x], dim=1)          # (B, T+1, 256)
        
        # ── Update mask for CLS token (CLS is never padded) ──
        cls_mask = torch.zeros(B, 1, dtype=torch.bool, device=x.device)
        padding_mask = torch.cat([cls_mask, mask], dim=1)  # (B, T+1)
        
        # ── Add positional encoding ──
        x = self.pos_enc(x)
        
        # ── Pass through encoder layers MANUALLY ──
        for i in range(self.num_layers):
            if self.norm_first:
                # Pre-norm architecture
                nx = self.layer_norm1[i](x)
                attn_out, _ = self.layer_self_attn[i](
                    nx, nx, nx,
                    key_padding_mask=padding_mask,
                    need_weights=False
                )
                x = x + self.layer_dropout1[i](attn_out)
                
                nx = self.layer_norm2[i](x)
                ff = self.layer_linear1[i](nx)
                if self._use_gelu:
                    ff = _onnx_safe_gelu(ff)
                else:
                    ff = torch.nn.functional.relu(ff)
                ff = self.layer_linear2[i](self.layer_dropout2[i](ff))
                x = x + self.layer_dropout2[i](ff)
            else:
                # Post-norm architecture (PyTorch default)
                attn_out, _ = self.layer_self_attn[i](
                    x, x, x,
                    key_padding_mask=padding_mask,
                    need_weights=False
                )
                x = self.layer_norm1[i](x + self.layer_dropout1[i](attn_out))
                
                ff = self.layer_linear1[i](x)
                if self._use_gelu:
                    ff = _onnx_safe_gelu(ff)
                else:
                    ff = torch.nn.functional.relu(ff)
                ff = self.layer_linear2[i](self.layer_dropout2[i](ff))
                x = self.layer_norm2[i](x + self.layer_dropout2[i](ff))
        
        # ── Final encoder norm ──
        x = self.encoder_norm(x)
        
        # ── Extract CLS token output ──
        cls_out = x[:, 0, :]  # (B, 256)
        
        # ── Classification head ──
        cls_out = self.head_norm(cls_out)
        cls_out = self.head_dropout(cls_out)
        logits = self.fc(cls_out)
        
        return logits


def export_transformer(model, out_path: Path, max_seq_len: int, device: str):
    model.eval()

    # ── CRITICAL FIX ──────────────────────────────────────────────────────────
    # PyTorch 2.x aggressively fuses elements into unsupported C++ kernels:
    # 1. `aten::_transformer_encoder_layer_fwd` (bypassed by unrolling loop)
    # 2. `aten::_native_multi_head_attention` (bypassed by ManualMHA class)
    # 3. `aten::gelu` (requires opset >= 20, bypassed by _onnx_safe_gelu)
    # ──────────────────────────────────────────────────────────────────────────
    
    wrapper = _TransformerExportWrapper(model).to(device)
    wrapper.eval()

    dummy_kp   = torch.randn(1, max_seq_len, FEAT_DIM, device=device)
    dummy_mask = torch.zeros(1, max_seq_len, dtype=torch.bool, device=device)

    print(f"  Transformer dummy inputs: kp={tuple(dummy_kp.shape)}, mask={tuple(dummy_mask.shape)}")

    with torch.no_grad():
        torch.onnx.export(
            wrapper,  
            (dummy_kp, dummy_mask),
            str(out_path),
            opset_version=14,
            input_names=["keypoints", "mask"],
            output_names=["logits"],
            dynamic_axes={
                "keypoints": {0: "batch", 1: "seq_len"},
                "mask":      {0: "batch", 1: "seq_len"},
                "logits":    {0: "batch"},
            },
            do_constant_folding=True,
            verbose=False,
        )
    print(f"  Exported → {out_path}")


# ─────────────────────────────────────────────────────────────────────────────
# ST-GCN export
# ─────────────────────────────────────────────────────────────────────────────

class _STGCNExportWrapper(torch.nn.Module):
    """
    Pre-compute the edge-importance-scaled adjacency matrices so no
    in-place copy_() appears in the traced computation graph.
    This was the root cause of the ST-GCN export failure.
    """
    def __init__(self, model):
        super().__init__()
        self.model = model

        # Pre-bake all scaled adjacencies into registered buffers
        for i, block in enumerate(model.st_gcn_blocks):
            A_base = block.gcn.A.detach().clone()
            if model.edge_importance_weighting:
                ei    = model.edge_importance[i].detach().clone()
                A_scaled = (A_base * ei).clamp(min=0)
            else:
                A_scaled = A_base
            # Replace the block's GCN adjacency with the pre-scaled version
            block.gcn.A.data.copy_(A_scaled)

        # Disable edge_importance so the forward pass uses A directly
        self.model.edge_importance_weighting = False

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.model(x)


def export_stgcn(model, out_path: Path, max_seq_len: int, device: str):
    model.eval()

    # Bake edge importance into adjacency buffers so no in-place op in graph
    wrapper = _STGCNExportWrapper(model).to(device)
    wrapper.eval()

    # ST-GCN input: (N, C, T, V) = (batch, 3, max_seq_len, 75)
    dummy_kp = torch.randn(1, 3, max_seq_len, TOTAL_JOINTS, device=device)

    print(f"  ST-GCN dummy input: kp={tuple(dummy_kp.shape)}")

    with torch.no_grad():
        torch.onnx.export(
            wrapper,
            dummy_kp,
            str(out_path),
            opset_version=14,
            input_names=["keypoints"],
            output_names=["logits"],
            dynamic_axes={
                "keypoints": {0: "batch"},
                "logits":    {0: "batch"},
            },
            do_constant_folding=True,
            verbose=False,
        )
    print(f"  Exported → {out_path}")


# ─────────────────────────────────────────────────────────────────────────────
# Validation
# ─────────────────────────────────────────────────────────────────────────────

def validate_onnx(out_path: Path, model_type: str, max_seq_len: int, num_classes: int):
    try:
        import onnx
        import onnxruntime as ort
    except ImportError:
        print("  [SKIP] onnx / onnxruntime not installed — skipping validation")
        return True

    print("  Validating ONNX model...")
    mdl = onnx.load(str(out_path))
    onnx.checker.check_model(mdl)
    print("  ✓ ONNX graph is valid")

    sess = ort.InferenceSession(str(out_path), providers=["CPUExecutionProvider"])

    if model_type == "stgcn":
        dummy = {"keypoints": np.random.randn(1, 3, max_seq_len, TOTAL_JOINTS).astype(np.float32)}
    else:
        dummy = {
            "keypoints": np.random.randn(1, max_seq_len, FEAT_DIM).astype(np.float32),
            "mask":      np.zeros((1, max_seq_len), dtype=bool),
        }

    out = sess.run(None, dummy)
    logits = out[0]
    assert logits.shape == (1, num_classes), \
        f"Shape mismatch: expected (1,{num_classes}), got {logits.shape}"
    print(f"  ✓ Inference OK — output shape: {logits.shape}")
    print(f"  ✓ Top-5 class indices: {np.argsort(logits[0])[::-1][:5].tolist()}")
    return True


# ─────────────────────────────────────────────────────────────────────────────
# Orchestrator
# ─────────────────────────────────────────────────────────────────────────────

def export_model(model_type: str, cfg: dict, output_dir: Path, device: str,
                 validate: bool = True) -> bool:
    print(f"\n{'='*60}")
    print(f"  Exporting  {model_type.upper()}")
    print(f"{'='*60}")

    mcfg     = MODEL_CONFIGS[model_type]
    ckpt_path = Path(mcfg["exp_dir"]) / "checkpoints" / "best.pth"
    out_path  = output_dir / mcfg["output_name"]

    if not ckpt_path.exists():
        print(f"  [ERROR] Checkpoint not found: {ckpt_path}")
        return False

    ckpt        = torch.load(str(ckpt_path), map_location=device)
    num_classes = ckpt.get("num_classes", 556)
    max_seq_len = cfg["data"]["max_seq_len"]

    print(f"  Classes: {num_classes}   MaxSeqLen: {max_seq_len}")

    model = build_model(model_type, cfg, num_classes)
    model.load_state_dict(ckpt["state_dict"])
    model = model.to(device)
    model.eval()

    output_dir.mkdir(parents=True, exist_ok=True)

    if model_type == "bilstm":
        export_bilstm(model, out_path, max_seq_len, device)
    elif model_type == "transformer":
        export_transformer(model, out_path, max_seq_len, device)
    elif model_type == "stgcn":
        export_stgcn(model, out_path, max_seq_len, device)

    # Validate
    if validate:
        try:
            validate_onnx(out_path, model_type, max_seq_len, num_classes)
        except Exception as e:
            print(f"  [WARNING] Validation error: {e}")

    # Save metadata (infer_onnx.py reads this)
    metadata = {
        "model_type":  model_type,     # ← was missing in original!
        "num_classes": num_classes,
        "max_seq_len": max_seq_len,
        "feat_dim":    FEAT_DIM,       # 225 for sequence models
        "num_joints":  TOTAL_JOINTS,   # 75  for ST-GCN
        "num_coords":  3,
    }
    meta_path = out_path.with_suffix(".json")
    with open(meta_path, "w") as f:
        json.dump(metadata, f, indent=2)
    print(f"  Metadata  → {meta_path}")

    # Copy label map alongside ONNX file for convenience
    lm_src = Path(cfg["data"]["label_map_file"])
    lm_dst = output_dir / "label_map.json"
    if lm_src.exists() and not lm_dst.exists():
        shutil.copy(str(lm_src), str(lm_dst))
        print(f"  Label map → {lm_dst}")

    size_mb = out_path.stat().st_size / 1024**2
    print(f"  ✓ Done  ({size_mb:.1f} MB)")
    return True


# ─────────────────────────────────────────────────────────────────────────────
# CLI
# ─────────────────────────────────────────────────────────────────────────────

def main():
    parser = argparse.ArgumentParser(description="Export MSL models to ONNX")
    parser.add_argument("--config",     default="config/config.yaml")
    parser.add_argument("--model",      choices=list(MODEL_CONFIGS.keys()),
                        help="Export one specific model")
    parser.add_argument("--all",        action="store_true",
                        help="Export all three models")
    parser.add_argument("--output_dir", default="onnx_models")
    parser.add_argument("--no-validate",action="store_true")
    parser.add_argument("--device",     default=None,
                        help="cuda | cpu (auto-detected if omitted)")
    args = parser.parse_args()

    if not args.model and not args.all:
        parser.error("Specify --model <name> or --all")

    with open(args.config) as f:
        cfg = yaml.safe_load(f)

    device = args.device or ("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Device: {device}")

    models = list(MODEL_CONFIGS.keys()) if args.all else [args.model]
    output_dir = Path(args.output_dir)

    ok = 0
    for m in models:
        if export_model(m, cfg, output_dir, device, validate=not args.no_validate):
            ok += 1

    print(f"\n{'='*60}")
    print(f"  SUMMARY: {ok}/{len(models)} models exported to '{output_dir}/'")
    if output_dir.exists():
        for f in sorted(output_dir.glob("*")):
            mb = f.stat().st_size / 1024**2
            print(f"    {f.name:<35} {mb:6.1f} MB")
    print(f"{'='*60}")


if __name__ == "__main__":
    main()


```

In [30]:
from IPython.display import Markdown

file_path = './src/infer_onnx.py'

with open(file_path, 'r') as f:
    code = f.read()

# 'python' keyword triggers the syntax highlighter
display(Markdown(f"```python\n{code}\n```"))

```python
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
infer_onnx.py  –  Production-ready Real-time MSL Inference with Output Recording
"""

import argparse
import json
import sys
import importlib
from datetime import datetime
from pathlib import Path

import cv2
import numpy as np
import onnxruntime as ort

try:
    from PIL import Image, ImageDraw, ImageFont
    HAS_PIL = True
except ImportError:
    HAS_PIL = False

# Topology Constants
POSE_N, HAND_N = 33, 21
TOTAL_JOINTS = POSE_N + HAND_N * 2
COORDS = 3
FEAT_DIM = TOTAL_JOINTS * COORDS

POSE_LEFT_SHOULDER, POSE_RIGHT_SHOULDER = 11, 12
LEFT_HAND_START = POSE_N
RIGHT_HAND_START = POSE_N + HAND_N


def _import_mp():
    for prefix in ["mediapipe.solutions", "mediapipe.python.solutions"]:
        try:
            return (importlib.import_module(f"{prefix}.pose"),
                    importlib.import_module(f"{prefix}.hands"),
                    importlib.import_module(f"{prefix}.drawing_utils"))
        except (ImportError, ModuleNotFoundError): continue
    raise ImportError("MediaPipe installation missing or corrupt.")

mp_pose, mp_hands, mp_drawing = _import_mp()


class TextRenderer:
    """Renders complex Myanmar script ensuring proper layout shaping rules."""
    _FONT_CANDIDATES = [
        "C:\\Windows\\Fonts\\Padauk.ttf",
        "C:\\Windows\\Fonts\\NotoSansMyanmar-Regular.ttf",
        "fonts\\Padauk.ttf",
    ]

    def __init__(self, font_size: int = 28):
        self.font = self._load_font(font_size)
        if not self.font:
            print("[WARNING] Complex font asset missing. Visual ordering bugs may persist.")

    def _load_font(self, size: int):
        if not HAS_PIL: return None
        for path in self._FONT_CANDIDATES:
            if Path(path).exists():
                try: return ImageFont.truetype(path, size)
                except: continue
        return None

    def put_text(self, frame: np.ndarray, text: str, position: tuple = (20, 30), color: tuple = (0, 255, 0)) -> np.ndarray:
        if self.font and HAS_PIL:
            pil_img = Image.fromarray(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
            draw = ImageDraw.Draw(pil_img)
            # CRITICAL: layout_engine=ImageFont.Layout.RAQM executes OpenType ligature reordering
            try:
                draw.text(position, text, font=self.font, fill=(color[2], color[1], color[0]), layout_engine=ImageFont.Layout.RAQM)
            except AttributeError:
                draw.text(position, text, font=self.font, fill=(color[2], color[1], color[0])) # Fallback
            return cv2.cvtColor(np.array(pil_img), cv2.COLOR_RGB2BGR)
        else:
            cv2.putText(frame, text, position, cv2.FONT_HERSHEY_SIMPLEX, 0.7, color, 2, cv2.LINE_AA)
            return frame


class KeypointExtractor:
    def __init__(self):
        self.pose = mp_pose.Pose(static_image_mode=False, model_complexity=1, smooth_landmarks=True)
        self.hands = mp_hands.Hands(static_image_mode=False, max_num_hands=2, model_complexity=1)

    def process_frame(self, frame: np.ndarray):
        rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        p_res = self.pose.process(rgb)
        h_res = self.hands.process(rgb)
        
        kp = np.zeros((TOTAL_JOINTS, COORDS), dtype=np.float32)
        detected = False

        if p_res.pose_landmarks:
            detected = True
            for i, lm in enumerate(p_res.pose_landmarks.landmark):
                kp[i] = [lm.x, lm.y, lm.z]

        if h_res.multi_hand_landmarks and h_res.multi_handedness:
            for lm_list, handedness in zip(h_res.multi_hand_landmarks, h_res.multi_handedness):
                label = handedness.classification[0].label
                # Flip camera perspective mapping cleanly
                offset = LEFT_HAND_START if label == 'Right' else RIGHT_HAND_START
                for i, lm in enumerate(lm_list.landmark):
                    kp[offset + i] = [lm.x, lm.y, lm.z]
                    
        return kp if detected else None


def normalize_sequence(seq: np.ndarray) -> np.ndarray:
    seq = seq.copy()
    mid_2d = (seq[:, POSE_LEFT_SHOULDER, :2] + seq[:, POSE_RIGHT_SHOULDER, :2]) / 2.0
    scale = np.linalg.norm(seq[:, POSE_RIGHT_SHOULDER, :2] - seq[:, POSE_LEFT_SHOULDER, :2], axis=1, keepdims=True)
    scale = np.where(scale < 1e-6, 1.0, scale)

    seq[:, :, 0] = (seq[:, :, 0] - mid_2d[:, None, 0]) / scale
    seq[:, :, 1] = (seq[:, :, 1] - mid_2d[:, None, 1]) / scale
    
    mid_z = (seq[:, POSE_LEFT_SHOULDER, 2] + seq[:, POSE_RIGHT_SHOULDER, 2]) / 2.0
    seq[:, :, 2] = seq[:, :, 2] - mid_z[:, None]
    return seq


class Preprocessor:
    def __init__(self, max_seq_len: int = 200):
        self.max_seq_len = max_seq_len

    def __call__(self, kp_list: list, model_type: str) -> dict:
        T = len(kp_list)
        seq = np.stack(kp_list, axis=0)

        # FIX: Uniform temporal downsampling/upsampling via interpolation index spaces
        if T != self.max_seq_len:
            indices = np.linspace(0, T - 1, self.max_seq_len).astype(int)
            seq = seq[indices]

        seq = normalize_sequence(seq)

        if model_type == "stgcn":
            return {"keypoints": seq.transpose(2, 0, 1)[np.newaxis].astype(np.float32)}
        elif model_type == "bilstm":
            # Pass explicit length array to satisfy our fixed BiLSTM wrapper graph
            lengths = np.array([self.max_seq_len], dtype=np.int64)
            return {"keypoints": seq.reshape(self.max_seq_len, -1)[np.newaxis].astype(np.float32), "lengths": lengths}
        else: # Transformer
            mask = np.zeros((1, self.max_seq_len), dtype=bool) # No positions padded under resampled logic
            return {"keypoints": seq.reshape(self.max_seq_len, -1)[np.newaxis].astype(np.float32), "mask": mask}


class ONNXModel:
    def __init__(self, onnx_path: str, meta_path: str):
        self.session = ort.InferenceSession(onnx_path, providers=["CUDAExecutionProvider", "CPUExecutionProvider"])
        with open(meta_path, encoding="utf-8") as f:
            meta = json.load(f)
        self.model_type = meta.get("model_type", "transformer")
        self.max_seq_len = meta.get("max_seq_len", 200)

    def predict(self, inputs: dict) -> np.ndarray:
        return self.session.run(None, inputs)[0][0]


# ─── Orchestrated Recording Inference Runners ───────────────────────────────

def run_video(model, prep, extractor, renderer, video_path, idx2label, output_dir):
    cap = cv2.VideoCapture(video_path)
    w  = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps = int(cap.get(cv2.CAP_PROP_FPS)) or 30
    
    out_path = Path(output_dir) / f"out_{Path(video_path).name}"
    writer = cv2.VideoWriter(str(out_path), cv2.VideoWriter_fourcc(*'mp4v'), fps, (w, h))

    frames_buffer, kp_buffer = [], []
    print(f"[INFO] Extracting features sequentially from: {Path(video_path).name}")

    while True:
        ret, frame = cap.read()
        if not ret: break
        frames_buffer.append(frame.copy())
        kp = extractor.process_frame(frame)
        # Fallback to zero frame if tracking drops temporarily
        kp_buffer.append(kp if kp is not None else np.zeros((TOTAL_JOINTS, COORDS), dtype=np.float32))

    cap.release()
    if not kp_buffer: return

    # Inference step
    inputs = prep(kp_buffer, model.model_type)
    logits = model.predict(inputs)
    probs = np.exp(logits - logits.max()) / np.exp(logits - logits.max()).sum()
    best_idx = np.argsort(probs)[::-1][0]
    label_text = idx2label.get(best_idx, f"Class {best_idx}") + f" ({probs[best_idx]*100:.1f}%)"

    print(f"[RESULT] Predicted: {label_text} -> Saved visualization output to file.")

    # Write out annotated clip + render live frame stream
    for frame in frames_buffer:
        annotated = renderer.put_text(frame, label_text, (30, 50), (0, 255, 0))
        writer.write(annotated)
        cv2.imshow("MSL Offline Video Evaluation [Q=close]", annotated)
        if cv2.waitKey(1) & 0xFF == ord('q'): break

    writer.release()
    cv2.destroyAllWindows()


def run_webcam(model, prep, extractor, renderer, idx2label, cam_idx, output_dir):
    cap = cv2.VideoCapture(cam_idx)
    w  = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    
    out_path = Path(output_dir) / f"webcam_session_{datetime.now().strftime('%Y%m%d_%H%M%S')}.mp4"
    writer = cv2.VideoWriter(str(out_path), cv2.VideoWriter_fourcc(*'mp4v'), 25, (w, h))

    kp_window = []
    display_text = "Buffering sequence..."
    color = (200, 200, 200)
    frame_idx = 0

    print("[INFO] Live tracking running. Press 'Q' inside screen window to quit session cleanly.")

    while True:
        ret, frame = cap.read()
        if not ret: break
        frame_idx += 1
        
        kp = extractor.process_frame(frame)
        if kp is not None:
            kp_window.append(kp)
            if len(kp_window) > model.max_seq_len: kp_window.pop(0)

        # Continuous prediction update step every 12 frames
        if len(kp_window) >= 30 and frame_idx % 12 == 0:
            inputs = prep(list(kp_window), model.model_type)
            logits = model.predict(inputs)
            probs = np.exp(logits - logits.max()) / np.exp(logits - logits.max()).sum()
            best_idx = np.argsort(probs)[::-1][0]
            
            display_text = idx2label.get(best_idx, f"Class {best_idx}") + f" ({probs[best_idx]*100:.1f}%)"
            color = (0, 255, 0) if probs[best_idx] > 0.75 else (0, 255, 255)

        annotated = renderer.put_text(frame.copy(), display_text, (20, 40), color)
        
        # Draw status tracking metrics overlay directly to stream
        cv2.putText(annotated, f"Buffer Window: {len(kp_window)}/{model.max_seq_len}", (20, h - 20),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 1, cv2.LINE_AA)
        
        writer.write(annotated)
        cv2.imshow("MSL Real-Time Recognition Window", annotated)

        key = cv2.waitKey(1) & 0xFF
        if key == ord('q'): break
        elif key == ord('c'): kp_window.clear()

    cap.release()
    writer.release()
    cv2.destroyAllWindows()


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--onnx_model", required=True)
    parser.add_argument("--output_dir", default="predictions_output")
    grp = parser.add_mutually_exclusive_group(required=True)
    grp.add_argument("--video")
    grp.add_argument("--webcam", type=int)
    args = parser.parse_args()

    model_p = Path(args.onnx_model)
    meta_p = model_p.with_suffix(".json")
    lmap_p = model_p.parent / "label_map.json"

    with open(lmap_p, encoding="utf-8") as f:
        idx2label = {int(k): v for k, v in json.load(f).get("idx2label", {}).items()}

    Path(args.output_dir).mkdir(parents=True, exist_ok=True)
    
    model = ONNXModel(str(model_p), str(meta_p))
    prep = Preprocessor(max_seq_len=model.max_seq_len)
    extractor = KeypointExtractor()
    renderer = TextRenderer()

    if args.video:
        run_video(model, prep, extractor, renderer, args.video, idx2label, args.output_dir)
    else:
        run_webcam(model, prep, extractor, renderer, idx2label, args.webcam, args.output_dir)


if __name__ == "__main__":
    main()


```

## Export Models Again

In [32]:
!rm -rf onnx_models

In [33]:
!python src/export_to_onnx.py --config config/config.yaml --all

Device: cuda

  Exporting  BILSTM
  Classes: 556   MaxSeqLen: 200
  BiLSTM dummy inputs: kp=(1, 200, 225), lengths=(1,)
[W528 20:14:43.279503986 shape_type_inference.cpp:1995] Warning: The shape inference of prim::PackPadded type is missing, so it may result in wrong shape inference for the exported graph. Please consider adding it in symbolic function. (function UpdateReliable)
[W528 20:14:43.280297669 shape_type_inference.cpp:1995] Warning: The shape inference of prim::PackPadded type is missing, so it may result in wrong shape inference for the exported graph. Please consider adding it in symbolic function. (function UpdateReliable)
/mnt/disk1/ye/exp/msl_recognition/.venv/lib/python3.12/site-packages/torch/onnx/symbolic_opset9.py:4244: UserWarning: Exporting a model to ONNX with a batch_size other than 1, with a variable length with LSTM can cause an error when running the ONNX model with a different batch size. Make sure to save the model with a batch size of 1, or define the initi

In [34]:
!ls ./onnx_models/*

./onnx_models/bilstm_msl.json  ./onnx_models/stgcn_msl.onnx
./onnx_models/bilstm_msl.onnx  ./onnx_models/transformer_msl.json
./onnx_models/label_map.json   ./onnx_models/transformer_msl.onnx
./onnx_models/stgcn_msl.json


## Testing Again on Windows Machine  

```
(msl_onnx) E:\LU_Lab\MSL_Tran_Demo>python infer_onnx.py --onnx_model .\onnx_models\transformer_msl.onnx --video .\sample4inference\idx20-30.mp4
C:\Users\801680\Anaconda3\envs\msl_onnx\lib\site-packages\onnxruntime\capi\onnxruntime_inference_collection.py:123: UserWarning: Specified provider 'CUDAExecutionProvider' is not in available provider names.Available providers: 'AzureExecutionProvider, CPUExecutionProvider'
  warnings.warn(
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
[INFO] Extracting features sequentially from: idx20-30.mp4
C:\Users\801680\Anaconda3\envs\msl_onnx\lib\site-packages\google\protobuf\symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '
[RESULT] Predicted: ဘယ် အချိန် တွေ ဆရာဝန် နဲ့ ပြ လို့ ရ လဲ သိ ချင် ပါ တယ် ။ (5.3%) -> Saved visualization output to file.

(msl_onnx) E:\LU_Lab\MSL_Tran_Demo>
```

```
(msl_onnx) E:\LU_Lab\MSL_Tran_Demo>python infer_onnx.py --onnx_model .\onnx_models\bilstm_msl.onnx --video .\sample4inference\idx20-30.mp4
C:\Users\801680\Anaconda3\envs\msl_onnx\lib\site-packages\onnxruntime\capi\onnxruntime_inference_collection.py:123: UserWarning: Specified provider 'CUDAExecutionProvider' is not in available provider names.Available providers: 'AzureExecutionProvider, CPUExecutionProvider'
  warnings.warn(
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
[INFO] Extracting features sequentially from: idx20-30.mp4
C:\Users\801680\Anaconda3\envs\msl_onnx\lib\site-packages\google\protobuf\symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '
[RESULT] Predicted: မီးချိတ် ။ (3.7%) -> Saved visualization output to file.

(msl_onnx) E:\LU_Lab\MSL_Tran_Demo>
```

```
(msl_onnx) E:\LU_Lab\MSL_Tran_Demo>python infer_onnx.py --onnx_model .\onnx_models\stgcn_msl.onnx --video .\sample4inference\idx20-30.mp4
C:\Users\801680\Anaconda3\envs\msl_onnx\lib\site-packages\onnxruntime\capi\onnxruntime_inference_collection.py:123: UserWarning: Specified provider 'CUDAExecutionProvider' is not in available provider names.Available providers: 'AzureExecutionProvider, CPUExecutionProvider'
  warnings.warn(
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
[INFO] Extracting features sequentially from: idx20-30.mp4
C:\Users\801680\Anaconda3\envs\msl_onnx\lib\site-packages\google\protobuf\symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '
[RESULT] Predicted: သွားဆရာဝန် ရဲ့ လိပ်စာ ပြော ပေး ပါ ဦး ။ (13.3%) -> Saved visualization output to file.

(msl_onnx) E:\LU_Lab\MSL_Tran_Demo>
```

ရလဒ် အားလုံးက မှားနေသေးတယ်။ :(  

## Debugging/Updating

code နှစ်ဖိုင်လုံးကို ထပ် debugging လုပ်ခဲ့တယ်။  


In [61]:
from IPython.display import Markdown

file_path = './src/export_to_onnx.py'

with open(file_path, 'r') as f:
    code = f.read()

# 'python' keyword triggers the syntax highlighter
display(Markdown(f"```python\n{code}\n```"))

```python
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
export_to_onnx.py  –  Export trained MSL models to ONNX format
                       (runs on Linux/HPC server side)

Bugs fixed vs original
───────────────────────
  1. ST-GCN: edge_importance in-place copy_ during torch.onnx.export broke
     the export (same root cause as the training autograd bug).  Fixed by
     pre-computing the scaled adjacency before export so no in-place op
     appears in the traced graph.
  2. BiLSTM: pack_padded_sequence is NOT supported in ONNX opset 14; the
     original wrapper silently fell back to an empty-graph export.  Fixed
     by bypassing packing entirely (no-op path) and exporting with a simple
     mask-sum length calculation that ONNX can trace.
  3. input_dim for ST-GCN was referencing an undefined variable; fixed with
     explicit constant (225 for sequence models, None for graph models).
  4. num_joints / num_coords were leaking between branches; fixed with clear
     if/elif blocks and explicit constants.
  5. The metadata JSON now includes 'model_type' (was missing), which
     infer_onnx.py reads to choose the correct input layout.
  6. Transformer: PyTorch 2.x fuses TransformerEncoderLayer and
     MultiheadAttention into single CUDA kernels 
     (`aten::_transformer_encoder_layer_fwd` and `aten::_native_multi_head_attention`)
     which have NO ONNX mapping in opset 14. Also, `aten::gelu` requires opset >= 20.
     Fixed by unrolling the encoder loop, replacing MHA with a manual math 
     implementation, and replacing GELU with an `erf`-based equivalent.
  7. Vector Dimension Tracking Fix: Set tracer dummy inputs to batch size 2 across
     models to prevent scalar-collapse graph parsing crashes (index_select exceptions).
"""

import argparse
import json
import shutil
import sys
from pathlib import Path

import numpy as np
import torch
import yaml

sys.path.insert(0, str(Path(__file__).parent))
from models import build_model
from utils  import get_logger, get_device, load_label_map

# ── Experiment → output name mapping ─────────────────────────────────────────
MODEL_CONFIGS = {
    "bilstm":       {"exp_dir": "results/exp_bilstm",       "output_name": "bilstm_msl.onnx"},
    "transformer":  {"exp_dir": "results/exp_transformer",  "output_name": "transformer_msl.onnx"},
    "stgcn":        {"exp_dir": "results/exp_stgcn",        "output_name": "stgcn_msl.onnx"},
}

# Fixed constants — must match extract_keypoints.py
POSE_N       = 33
HAND_N       = 21
TOTAL_JOINTS = POSE_N + HAND_N * 2   # 75
FEAT_DIM     = TOTAL_JOINTS * 3       # 225


# ─────────────────────────────────────────────────────────────────────────────
# BiLSTM export
# ─────────────────────────────────────────────────────────────────────────────

class _BiLSTMExportWrapper(torch.nn.Module):
    """
    Thin wrapper that converts the boolean padding mask to a length tensor
    WITHOUT calling pack_padded_sequence (which ONNX cannot trace).

    The mask convention (True = padded position) is the same as training.
    We count the valid (non-padded) positions to get sequence lengths,
    then call the model in 'no-pack' mode by setting lengths=None and
    passing the mask directly so the attention layer masks out padding.
    """
    def __init__(self, model):
        super().__init__()
        self.model = model

    def forward(self, keypoints: torch.Tensor, mask: torch.Tensor) -> torch.Tensor:
        # mask: (B, T) True where PADDED  →  lengths = number of valid frames
        # We do NOT pass lengths so the LSTM runs on the full padded sequence
        # (slightly less efficient, but fully ONNX-traceable).
        # The additive attention mask still zeros out padded positions.
        return self.model(keypoints, lengths=None, mask=mask)


def export_bilstm(model, out_path: Path, max_seq_len: int, device: str):
    model.eval()
    wrapper = _BiLSTMExportWrapper(model).to(device)
    wrapper.eval()

    # FIX: Batch size 2 prevents tracking indices from collapsing into 0D scalar tensors
    dummy_kp   = torch.randn(2, max_seq_len, FEAT_DIM, device=device)
    dummy_mask = torch.zeros(2, max_seq_len, dtype=torch.bool, device=device)

    print(f"  BiLSTM dummy inputs: kp={tuple(dummy_kp.shape)}, mask={tuple(dummy_mask.shape)}")

    with torch.no_grad():
        torch.onnx.export(
            wrapper,
            (dummy_kp, dummy_mask),
            str(out_path),
            opset_version=16,
            input_names=["keypoints", "mask"],
            output_names=["logits"],
            dynamic_axes={
                "keypoints": {0: "batch"},
                "mask":      {0: "batch"},
                "logits":    {0: "batch"},
            },
            do_constant_folding=True,
            verbose=False,
        )
    print(f"  Exported → {out_path}")


# ─────────────────────────────────────────────────────────────────────────────
# Transformer export
# ─────────────────────────────────────────────────────────────────────────────

def _onnx_safe_gelu(x: torch.Tensor) -> torch.Tensor:
    """GELU implementation using erf, fully supported in ONNX opset 14+."""
    return 0.5 * x * (1.0 + torch.erf(x / 1.4142135623730951))


class ManualMHA(torch.nn.Module):
    """
    Manual Multi-Head Attention implementation to bypass PyTorch 2.x's
    fused `aten::_native_multi_head_attention` which lacks ONNX support.
    """
    def __init__(self, mha: torch.nn.MultiheadAttention):
        super().__init__()
        self.embed_dim = mha.embed_dim
        self.num_heads = mha.num_heads
        self.head_dim = self.embed_dim // self.num_heads
        
        self.in_proj_weight = torch.nn.Parameter(mha.in_proj_weight.data.clone())
        self.in_proj_bias = torch.nn.Parameter(mha.in_proj_bias.data.clone())
        self.out_proj_weight = torch.nn.Parameter(mha.out_proj.weight.data.clone())
        self.out_proj_bias = torch.nn.Parameter(mha.out_proj.bias.data.clone())

    def forward(self, query: torch.Tensor, key: torch.Tensor, value: torch.Tensor,
                key_padding_mask: torch.Tensor = None, need_weights: bool = False):
        B, T, C = query.shape
        
        # Split projection weights for Q, K, V
        q_w = self.in_proj_weight[:C]
        k_w = self.in_proj_weight[C:2*C]
        v_w = self.in_proj_weight[2*C:]
        
        q_b = self.in_proj_bias[:C]
        k_b = self.in_proj_bias[C:2*C]
        v_b = self.in_proj_bias[2*C:]
        
        Q = torch.nn.functional.linear(query, q_w, q_b)
        K = torch.nn.functional.linear(key, k_w, k_b)
        V = torch.nn.functional.linear(value, v_w, v_b)
        
        # Use -1 for dynamic sequence length compatibility in ONNX
        Q = Q.reshape(B, -1, self.num_heads, self.head_dim).transpose(1, 2)
        K = K.reshape(B, -1, self.num_heads, self.head_dim).transpose(1, 2)
        V = V.reshape(B, -1, self.num_heads, self.head_dim).transpose(1, 2)
        
        # Scaled dot-product attention
        scale = self.head_dim ** 0.5
        scores = torch.matmul(Q, K.transpose(-2, -1)) / scale
        
        if key_padding_mask is not None:
            # key_padding_mask: (B, T) -> (B, 1, 1, T) for broadcasting
            mask = key_padding_mask.unsqueeze(1).unsqueeze(2)
            scores = scores.masked_fill(mask, float('-inf'))
            
        attn_weights = torch.nn.functional.softmax(scores, dim=-1)
        attn_out = torch.matmul(attn_weights, V)
        
        # Reshape back to (B, T, C)
        attn_out = attn_out.transpose(1, 2).reshape(B, -1, C)
        
        out = torch.nn.functional.linear(attn_out, self.out_proj_weight, self.out_proj_bias)
        
        if need_weights:
            return out, attn_weights
        return out, None


class _TransformerExportWrapper(torch.nn.Module):
    """
    Wrapper that completely bypasses PyTorch 2.x fused operators by unrolling
    the encoder loop and calling basic sub-modules directly.
    """
    def __init__(self, model):
        super().__init__()
        # Copy all the model's input/head components
        self.input_norm = model.input_norm
        self.input_proj = model.input_proj
        self.proj_norm = model.proj_norm
        self.cls_token = model.cls_token
        self.pos_enc = model.pos_enc
        self.encoder_norm = model.encoder.norm
        self.head_norm = model.head_norm
        self.head_dropout = model.head_dropout
        self.fc = model.fc
        
        # Replace fused MultiheadAttention with ONNX-safe manual implementation
        self.layer_self_attn = torch.nn.ModuleList([ManualMHA(l.self_attn) for l in model.encoder.layers])
        self.layer_linear1 = torch.nn.ModuleList([l.linear1 for l in model.encoder.layers])
        self.layer_linear2 = torch.nn.ModuleList([l.linear2 for l in model.encoder.layers])
        self.layer_norm1 = torch.nn.ModuleList([l.norm1 for l in model.encoder.layers])
        self.layer_norm2 = torch.nn.ModuleList([l.norm2 for l in model.encoder.layers])
        self.layer_dropout1 = torch.nn.ModuleList([l.dropout1 for l in model.encoder.layers])
        self.layer_dropout2 = torch.nn.ModuleList([l.dropout2 for l in model.encoder.layers])
        
        self.num_layers = len(model.encoder.layers)
        
        # Check architecture type from the first layer
        self.norm_first = model.encoder.layers[0].norm_first
        
        # Determine activation type
        act = model.encoder.layers[0].activation
        if act is torch.nn.functional.relu or isinstance(act, torch.nn.ReLU):
            self._use_gelu = False
        else:
            self._use_gelu = True

    def forward(self, keypoints: torch.Tensor, mask: torch.Tensor) -> torch.Tensor:
        # ── Input processing ──
        x = self.input_norm(keypoints)       # (B, T, 225)
        x = self.input_proj(x)               # (B, T, 256)
        x = self.proj_norm(x)                # (B, T, 256)
        
        # ── Add CLS token ──
        B = x.shape[0]
        cls_tokens = self.cls_token.expand(B, -1, -1)  # (B, 1, 256)
        x = torch.cat([cls_tokens, x], dim=1)          # (B, T+1, 256)
        
        # ── Update mask for CLS token (CLS is never padded) ──
        cls_mask = torch.zeros(B, 1, dtype=torch.bool, device=x.device)
        padding_mask = torch.cat([cls_mask, mask], dim=1)  # (B, T+1)
        
        # ── Add positional encoding ──
        x = self.pos_enc(x)
        
        # ── Pass through encoder layers MANUALLY ──
        for i in range(self.num_layers):
            if self.norm_first:
                # Pre-norm architecture
                nx = self.layer_norm1[i](x)
                attn_out, _ = self.layer_self_attn[i](
                    nx, nx, nx,
                    key_padding_mask=padding_mask,
                    need_weights=False
                )
                x = x + self.layer_dropout1[i](attn_out)
                
                nx = self.layer_norm2[i](x)
                ff = self.layer_linear1[i](nx)
                if self._use_gelu:
                    ff = _onnx_safe_gelu(ff)
                else:
                    ff = torch.nn.functional.relu(ff)
                ff = self.layer_linear2[i](self.layer_dropout2[i](ff))
                x = x + self.layer_dropout2[i](ff)
            else:
                # Post-norm architecture (PyTorch default)
                attn_out, _ = self.layer_self_attn[i](
                    x, x, x,
                    key_padding_mask=padding_mask,
                    need_weights=False
                )
                x = self.layer_norm1[i](x + self.layer_dropout1[i](attn_out))
                
                ff = self.layer_linear1[i](x)
                if self._use_gelu:
                    ff = _onnx_safe_gelu(ff)
                else:
                    ff = torch.nn.functional.relu(ff)
                ff = self.layer_linear2[i](self.layer_dropout2[i](ff))
                x = self.layer_norm2[i](x + self.layer_dropout2[i](ff))
        
        # ── Final encoder norm ──
        x = self.encoder_norm(x)
        
        # ── Extract CLS token output ──
        cls_out = x[:, 0, :]  # (B, 256)
        
        # ── Classification head ──
        cls_out = self.head_norm(cls_out)
        cls_out = self.head_dropout(cls_out)
        logits = self.fc(cls_out)
        
        return logits


def export_transformer(model, out_path: Path, max_seq_len: int, device: str):
    model.eval()
    wrapper = _TransformerExportWrapper(model).to(device)
    wrapper.eval()

    # FIX: Batch size 2 ensures dimensions match downstream dynamic axis parsing expectations
    dummy_kp   = torch.randn(2, max_seq_len, FEAT_DIM, device=device)
    dummy_mask = torch.zeros(2, max_seq_len, dtype=torch.bool, device=device)

    print(f"  Transformer dummy inputs: kp={tuple(dummy_kp.shape)}, mask={tuple(dummy_mask.shape)}")

    with torch.no_grad():
        # FIX: Enforce core math execution backend to suppress PyTorch 2.x native kernel fusion
        try:
            from torch.nn.attention import sdp_kernel
            attention_context = sdp_kernel(enable_flash=False, enable_mem_efficient=False, enable_math=True)
        except ImportError:
            import contextlib
            attention_context = contextlib.nullcontext()

        with attention_context:
            torch.onnx.export(
                wrapper,  
                (dummy_kp, dummy_mask),
                str(out_path),
                opset_version=16,
                input_names=["keypoints", "mask"],
                output_names=["logits"],
                dynamic_axes={
                    "keypoints": {0: "batch", 1: "seq_len"},
                    "mask":      {0: "batch", 1: "seq_len"},
                    "logits":    {0: "batch"},
                },
                do_constant_folding=True,
                verbose=False,
            )
    print(f"  Exported → {out_path}")


# ─────────────────────────────────────────────────────────────────────────────
# ST-GCN export
# ─────────────────────────────────────────────────────────────────────────────

class _STGCNExportWrapper(torch.nn.Module):
    """
    Pre-compute the edge-importance-scaled adjacency matrices so no
    in-place copy_() appears in the traced computation graph.
    This was the root cause of the ST-GCN export failure.
    """
    def __init__(self, model):
        super().__init__()
        self.model = model

        # Pre-bake all scaled adjacencies into registered buffers
        for i, block in enumerate(model.st_gcn_blocks):
            A_base = block.gcn.A.detach().clone()
            if model.edge_importance_weighting:
                ei    = model.edge_importance[i].detach().clone()
                A_scaled = (A_base * ei).clamp(min=0)
            else:
                A_scaled = A_base
            # Replace the block's GCN adjacency with the pre-scaled version
            block.gcn.A.data.copy_(A_scaled)

        # Disable edge_importance so the forward pass uses A directly
        self.model.edge_importance_weighting = False

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.model(x)


def export_stgcn(model, out_path: Path, max_seq_len: int, device: str):
    model.eval()

    # Bake edge importance into adjacency buffers so no in-place op in graph
    wrapper = _STGCNExportWrapper(model).to(device)
    wrapper.eval()

    # ST-GCN input: (N, C, T, V) = (batch, 3, max_seq_len, 75)
    dummy_kp = torch.randn(2, 3, max_seq_len, TOTAL_JOINTS, device=device)

    print(f"  ST-GCN dummy input: kp={tuple(dummy_kp.shape)}")

    with torch.no_grad():
        torch.onnx.export(
            wrapper,
            dummy_kp,
            str(out_path),
            opset_version=16,
            input_names=["keypoints"],
            output_names=["logits"],
            dynamic_axes={
                "keypoints": {0: "batch"},
                "logits":    {0: "batch"},
            },
            do_constant_folding=True,
            verbose=False,
        )
    print(f"  Exported → {out_path}")


# ─────────────────────────────────────────────────────────────────────────────
# Validation
# ─────────────────────────────────────────────────────────────────────────────

def validate_onnx(out_path: Path, model_type: str, max_seq_len: int, num_classes: int):
    try:
        import onnx
        import onnxruntime as ort
    except ImportError:
        print("  [SKIP] onnx / onnxruntime not installed — skipping validation")
        return True

    print("  Validating ONNX model...")
    mdl = onnx.load(str(out_path))
    onnx.checker.check_model(mdl)
    print("  ✓ ONNX graph is valid")

    sess = ort.InferenceSession(str(out_path), providers=["CPUExecutionProvider"])

    if model_type == "stgcn":
        dummy = {"keypoints": np.random.randn(1, 3, max_seq_len, TOTAL_JOINTS).astype(np.float32)}
    else:
        dummy = {
            "keypoints": np.random.randn(1, max_seq_len, FEAT_DIM).astype(np.float32),
            "mask":      np.zeros((1, max_seq_len), dtype=bool),
        }

    out = sess.run(None, dummy)
    logits = out[0]
    assert logits.shape == (1, num_classes), \
        f"Shape mismatch: expected (1,{num_classes}), got {logits.shape}"
    print(f"  ✓ Inference OK — output shape: {logits.shape}")
    print(f"  ✓ Top-5 class indices: {np.argsort(logits[0])[::-1][:5].tolist()}")
    return True


# ─────────────────────────────────────────────────────────────────────────────
# Orchestrator
# ─────────────────────────────────────────────────────────────────────────────

def export_model(model_type: str, cfg: dict, output_dir: Path, device: str,
                 validate: bool = True) -> bool:
    print(f"\n{'='*60}")
    print(f"  Exporting  {model_type.upper()}")
    print(f"{'='*60}")

    mcfg     = MODEL_CONFIGS[model_type]
    ckpt_path = Path(mcfg["exp_dir"]) / "checkpoints" / "best.pth"
    out_path  = output_dir / mcfg["output_name"]

    if not ckpt_path.exists():
        print(f"  [ERROR] Checkpoint not found: {ckpt_path}")
        return False

    ckpt        = torch.load(str(ckpt_path), map_location=device)
    num_classes = ckpt.get("num_classes", 556)
    max_seq_len = cfg["data"]["max_seq_len"]

    print(f"  Classes: {num_classes}   MaxSeqLen: {max_seq_len}")

    model = build_model(model_type, cfg, num_classes)
    model.load_state_dict(ckpt["state_dict"])
    model = model.to(device)
    model.eval()

    output_dir.mkdir(parents=True, exist_ok=True)

    if model_type == "bilstm":
        export_bilstm(model, out_path, max_seq_len, device)
    elif model_type == "transformer":
        export_transformer(model, out_path, max_seq_len, device)
    elif model_type == "stgcn":
        export_stgcn(model, out_path, max_seq_len, device)

    # Validate
    if validate:
        try:
            validate_onnx(out_path, model_type, max_seq_len, num_classes)
        except Exception as e:
            print(f"  [WARNING] Validation error: {e}")

    # Save metadata (infer_onnx.py reads this)
    metadata = {
        "model_type":  model_type,
        "num_classes": num_classes,
        "max_seq_len": max_seq_len,
        "feat_dim":    FEAT_DIM,       # 225 for sequence models
        "num_joints":  TOTAL_JOINTS,   # 75  for ST-GCN
        "num_coords":  3,
    }
    meta_path = out_path.with_suffix(".json")
    with open(meta_path, "w") as f:
        json.dump(metadata, f, indent=2)
    print(f"  Metadata  → {meta_path}")

    # Copy label map alongside ONNX file for convenience
    lm_src = Path(cfg["data"]["label_map_file"])
    lm_dst = output_dir / "label_map.json"
    if lm_src.exists() and not lm_dst.exists():
        shutil.copy(str(lm_src), str(lm_dst))
        print(f"  Label map → {lm_dst}")

    size_mb = out_path.stat().st_size / 1024**2
    print(f"  ✓ Done  ({size_mb:.1f} MB)")
    return True


# ─────────────────────────────────────────────────────────────────────────────
# CLI
# ─────────────────────────────────────────────────────────────────────────────

def main():
    parser = argparse.ArgumentParser(description="Export MSL models to ONNX")
    parser.add_argument("--config",     default="config/config.yaml")
    parser.add_argument("--model",      choices=list(MODEL_CONFIGS.keys()),
                        help="Export one specific model")
    parser.add_argument("--all",        action="store_true",
                        help="Export all three models")
    parser.add_argument("--output_dir", default="onnx_models")
    parser.add_argument("--no-validate",action="store_true")
    parser.add_argument("--device",     default=None,
                        help="cuda | cpu (auto-detected if omitted)")
    args = parser.parse_args()

    if not args.model and not args.all:
        parser.error("Specify --model <name> or --all")

    with open(args.config) as f:
        cfg = yaml.safe_load(f)

    device = args.device or ("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Device: {device}")

    models = list(MODEL_CONFIGS.keys()) if args.all else [args.model]
    output_dir = Path(args.output_dir)

    ok = 0
    for m in models:
        if export_model(m, cfg, output_dir, device, validate=not args.no_validate):
            ok += 1

    print(f"\n{'='*60}")
    print(f"  SUMMARY: {ok}/{len(models)} models exported to '{output_dir}/'")
    if output_dir.exists():
        for f in sorted(output_dir.glob("*")):
            mb = f.stat().st_size / 1024**2
            print(f"    {f.name:<35} {mb:6.1f} MB")
    print(f"{'='*60}")


if __name__ == "__main__":
    main()


```

In [67]:
from IPython.display import Markdown

file_path = './src/infer_onnx.py'

with open(file_path, 'r') as f:
    code = f.read()

# 'python' keyword triggers the syntax highlighter
display(Markdown(f"```python\n{code}\n```"))

```python
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
infer_onnx.py  –  Real-time MSL recognition on Windows using ONNX Runtime
                   (Fully Debugged & Optimized Production Version)
"""

import argparse
import glob
import importlib
import json
import sys
from datetime import datetime
from pathlib import Path

import cv2
import numpy as np
import onnxruntime as ort

try:
    from PIL import Image, ImageDraw, ImageFont
    HAS_PIL = True
except ImportError:
    HAS_PIL = False

# ── Constants (must match training exactly) ───────────────────────────────────
POSE_N          = 33
HAND_N          = 21
TOTAL_JOINTS    = POSE_N + HAND_N * 2   # 75
COORDS          = 3
FEAT_DIM        = TOTAL_JOINTS * COORDS  # 225

# Indices for normalization (MediaPipe Pose topology)
POSE_LEFT_SHOULDER  = 11
POSE_RIGHT_SHOULDER = 12
LEFT_HAND_START     = POSE_N            # 33
RIGHT_HAND_START    = POSE_N + HAND_N   # 54


# ─── MediaPipe import (compatible with mediapipe >= 0.10.13) ─────────────────

def _import_mp():
    """Try all known import paths; return (pose_mod, hands_mod, drawing_mod)."""
    for prefix in ["mediapipe.solutions", "mediapipe.python.solutions"]:
        try:
            pose  = importlib.import_module(f"{prefix}.pose")
            hands = importlib.import_module(f"{prefix}.hands")
            draw  = importlib.import_module(f"{prefix}.drawing_utils")
            return pose, hands, draw
        except (ImportError, ModuleNotFoundError):
            continue
    try:
        import mediapipe as _mp
        return _mp.solutions.pose, _mp.solutions.hands, _mp.solutions.drawing_utils
    except AttributeError:
        pass
    raise ImportError(
        "Cannot import MediaPipe solutions.\n"
        "Try: pip uninstall mediapipe -y && pip install mediapipe==0.10.18"
    )

mp_pose, mp_hands, mp_drawing = _import_mp()


# ─── Myanmar text rendering ──────────────────────────────────────────────────

class TextRenderer:
    """Renders text onto OpenCV BGR frames via Pillow with OpenType shaping support."""

    _FONT_CANDIDATES = [
        "C:\\Windows\\Fonts\\Padauk.ttf",
        "C:\\Windows\\Fonts\\NotoSansMyanmar-Regular.ttf",
        "C:\\Windows\\Fonts\\NotoSansMyanmar.ttf",
        "C:\\Windows\\Fonts\\myanmar3.ttf",
        "/usr/share/fonts/truetype/padauk/Padauk-Regular.ttf",
        "/usr/share/fonts/truetype/noto/NotoSansMyanmar-Regular.ttf",
        "fonts\\Padauk.ttf",
        "fonts/NotoSansMyanmar-Regular.ttf",
    ]

    def __init__(self, font_size: int = 32):
        self.font = self._load_font(font_size)
        self.can_render_myanmar = (self.font is not None)

        if not self.can_render_myanmar:
            print()
            print("  ╔═══════════════════════════════════════════════════════════╗")
            print("  ║  INFO: No Myanmar shaping font found.                    ║")
            print("  ║  Displaying readable Class IDs instead of Myanmar text.  ║")
            print("  ╚═══════════════════════════════════════════════════════════╝")
            print()

    def _load_font(self, size: int):
        if not HAS_PIL:
            return None
        for path in self._FONT_CANDIDATES:
            if Path(path).exists():
                try:
                    return ImageFont.truetype(path, size)
                except Exception:
                    continue
        return None

    def put_text(self, frame: np.ndarray, text: str,
                 position: tuple = (20, 20),
                 color: tuple = (0, 255, 0)) -> np.ndarray:
        if self.font is not None and HAS_PIL:
            pil_img = Image.fromarray(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
            draw    = ImageDraw.Draw(pil_img)
            r, g, b = color[2], color[1], color[0]
            for dx in [-1, 0, 1]:
                for dy in [-1, 0, 1]:
                    if dx or dy:
                        draw.text((position[0]+dx, position[1]+dy),
                                  text, font=self.font, fill=(0, 0, 0))
            draw.text(position, text, font=self.font, fill=(r, g, b))
            return cv2.cvtColor(np.array(pil_img), cv2.COLOR_RGB2BGR)
        else:
            cv2.putText(frame, text, position,
                        cv2.FONT_HERSHEY_SIMPLEX, 0.75, color, 2, cv2.LINE_AA)
            return frame


# ─── Keypoint extraction ──────────────────────────────────────────────────────

class KeypointExtractor:
    def __init__(self, model_complexity: int = 1):
        self.pose  = mp_pose.Pose(
            static_image_mode=False, model_complexity=model_complexity,
            smooth_landmarks=True, enable_segmentation=False,
            min_detection_confidence=0.5, min_tracking_confidence=0.5,
        )
        self.hands = mp_hands.Hands(
            static_image_mode=False, max_num_hands=2,
            model_complexity=min(model_complexity, 1),
            min_detection_confidence=0.5, min_tracking_confidence=0.5,
        )
        self._last_pose  = None
        self._last_hands = None

    def process_frame(self, frame: np.ndarray):
        rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        rgb.flags.writeable = False
        p_res = self.pose.process(rgb)
        h_res = self.hands.process(rgb)
        self._last_pose  = p_res
        self._last_hands = h_res

        kp       = np.zeros((TOTAL_JOINTS, COORDS), dtype=np.float32)
        detected = False

        if p_res.pose_landmarks:
            detected = True
            for i, lm in enumerate(p_res.pose_landmarks.landmark):
                kp[i, 0] = lm.x
                kp[i, 1] = lm.y
                kp[i, 2] = lm.z

        if h_res.multi_hand_landmarks and h_res.multi_handedness:
            for lm_list, handedness in zip(h_res.multi_hand_landmarks, h_res.multi_handedness):
                label = handedness.classification[0].label
                offset = LEFT_HAND_START if label == 'Right' else RIGHT_HAND_START
                for i, lm in enumerate(lm_list.landmark):
                    kp[offset + i, 0] = lm.x
                    kp[offset + i, 1] = lm.y
                    kp[offset + i, 2] = lm.z
        return kp if detected else None

    def draw_landmarks(self, frame: np.ndarray):
        if self._last_pose and self._last_pose.pose_landmarks:
            mp_drawing.draw_landmarks(frame, self._last_pose.pose_landmarks, mp_pose.POSE_CONNECTIONS)
        if self._last_hands and self._last_hands.multi_hand_landmarks:
            for lm in self._last_hands.multi_hand_landmarks:
                mp_drawing.draw_landmarks(frame, lm, mp_hands.HAND_CONNECTIONS)

    def close(self):
        self.pose.close()
        self.hands.close()


# ─── Normalization ────────────────────────────────────────────────────────────

def normalize_sequence(seq: np.ndarray) -> np.ndarray:
    """Translate and scale spatial sequence dimensions according to training distribution."""
    seq = seq.copy()
    l_sh = seq[:, POSE_LEFT_SHOULDER, :2]
    r_sh = seq[:, POSE_RIGHT_SHOULDER, :2]
    mid_2d = (l_sh + r_sh) / 2.0
    scale = np.linalg.norm(r_sh - l_sh, axis=1, keepdims=True)
    scale = np.where(scale < 1e-6, 1.0, scale)

    seq[:, :, 0] = (seq[:, :, 0] - mid_2d[:, None, 0]) / scale
    seq[:, :, 1] = (seq[:, :, 1] - mid_2d[:, None, 1]) / scale

    mid_z = (seq[:, POSE_LEFT_SHOULDER, 2] + seq[:, POSE_RIGHT_SHOULDER, 2]) / 2.0
    seq[:, :, 2] = seq[:, :, 2] - mid_z[:, None]
    
    return seq.astype(np.float32)


# ─── Preprocessor ─────────────────────────────────────────────────────────────

class Preprocessor:
    def __init__(self, max_seq_len: int = 200):
        self.max_seq_len = max_seq_len

    def __call__(self, kp_list: list, model_type: str) -> dict:
        if not kp_list:
            return {}
        seq = np.stack(kp_list, axis=0)
        seq = normalize_sequence(seq)
        T = seq.shape[0]

        if T > self.max_seq_len:
            start = (T - self.max_seq_len) // 2
            seq   = seq[start: start + self.max_seq_len]
            T     = self.max_seq_len

        real_len = T
        if T < self.max_seq_len:
            pad = np.zeros((self.max_seq_len - T, 75, 3), dtype=np.float32)
            seq = np.concatenate([seq, pad], axis=0)

        mask = np.zeros(self.max_seq_len, dtype=bool)
        mask[real_len:] = True

        if model_type == "stgcn":
            kp_out = seq.transpose(2, 0, 1)[np.newaxis].astype(np.float32)
            return {"keypoints": kp_out}
        else:
            kp_out = seq.reshape(self.max_seq_len, -1)[np.newaxis].astype(np.float32)
            return {"keypoints": kp_out, "mask": mask[np.newaxis]}


# ─── ONNX inference wrapper ───────────────────────────────────────────────────

class ONNXModel:
    def __init__(self, onnx_path: str, meta_path: str = None):
        # Dynamically query available providers to avoid system execution mismatch warnings
        available_providers = ort.get_available_providers()
        preferred_providers = ["CUDAExecutionProvider", "DmlExecutionProvider", "CPUExecutionProvider"]
        providers = [p for p in preferred_providers if p in available_providers]
        
        self.session = ort.InferenceSession(onnx_path, providers=providers)
        print(f"[INFO] ONNX loaded: {Path(onnx_path).name}  (provider: {self.session.get_providers()[0]})")

        # FIX: Unconditional attribute declaration guarantees self.meta always exists
        self.meta = {}
        if meta_path and Path(meta_path).exists():
            try:
                with open(meta_path, encoding="utf-8") as f:
                    self.meta = json.load(f)
            except Exception as e:
                print(f"[WARNING] Could not parse metadata json: {e}")
        else:
            print(f"[INFO] Metadata JSON file not found at {meta_path}. Using fallback defaults.")

        # Safely assign parameters via dictionary lookups
        self.model_type  = self.meta.get("model_type", "transformer")
        self.max_seq_len = self.meta.get("max_seq_len", 200)
        self.num_classes = self.meta.get("num_classes", 556)

    def predict(self, inputs: dict) -> np.ndarray:
        return self.session.run(None, inputs)[0][0]

    @staticmethod
    def softmax(logits: np.ndarray) -> np.ndarray:
        e = np.exp(logits - logits.max())
        return e / e.sum()


def load_idx2label(path: str) -> dict:
    with open(path, encoding="utf-8") as f:
        data = json.load(f)
    return {int(k): v for k, v in data.get("idx2label", {}).items()}


def _build_text(class_id: int, label: str, confidence: float, can_my: bool,
                top_k: int = 1, all_ids: list = None, probs: np.ndarray = None, lbl_map: dict = None):
    if top_k == 1:
        return f"{label}  ({confidence*100:.1f}%)" if can_my else f"Class {class_id}  ({confidence*100:.1f}%)"
    parts = []
    for i, cid in enumerate(all_ids):
        lbl = lbl_map.get(cid, "?") if (can_my and lbl_map) else f"Class {cid}"
        parts.append(f"#{i+1} {lbl} ({probs[cid]*100:.0f}%)")
    return "  |  ".join(parts)


def _build_log(class_id: int, label: str, confidence: float,
               top_k: int = 1, all_ids: list = None, probs: np.ndarray = None, lbl_map: dict = None):
    safe_lbl = label.encode('utf-8', errors='replace').decode('utf-8')
    if top_k == 1:
        return f"  Class {class_id:>4d} | {safe_lbl:<40s} | {confidence*100:6.2f}%"
    lines = []
    for i, cid in enumerate(all_ids):
        lbl = lbl_map.get(cid, "?").encode('utf-8', errors='replace').decode('utf-8') if lbl_map else "?"
        lines.append(f"  Top-{i+1}: Class {cid:>4d} | {lbl:<40s} | {probs[cid]*100:6.2f}%")
    return "\n".join(lines)


# ─── Video processing ─────────────────────────────────────────────────────────

def run_video(model, prep, extractor, renderer, video_path, top_k=5, log_file=None, idx2label=None, debug=False):
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened(): 
        print(f"[ERROR] Could not open video file: {video_path}")
        return
        
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    print(f"[INFO] Processing {total} frames from {Path(video_path).name} ...")

    # OPTIMIZED: Process video frames in a single efficient execution loop
    kp_list = []
    dropped = 0
    while True:
        ret, frame = cap.read()
        if not ret: 
            break
        kp = extractor.process_frame(frame)
        if kp is not None: 
            kp_list.append(kp)
        else: 
            dropped += 1
    cap.release()

    if not kp_list: 
        print("[WARNING] No person detected in video sequence.")
        return

    valid = len(kp_list)
    pad = max(0, model.max_seq_len - valid)
    if valid > model.max_seq_len:
        pad = 0
        valid = model.max_seq_len
        
    print(f"[INFO] Frames: {valid} valid / {pad} padded (dropped {dropped}/{total})")
    if valid / model.max_seq_len < 0.25:
        print("  [WARNING] Very low valid frame ratio. Predictions may be inaccurate.")

    inputs = prep(kp_list, model.model_type)
    logits = model.predict(inputs)
    probs = ONNXModel.softmax(logits)
    top_idx = np.argsort(probs)[::-1][:top_k]

    sep = "─" * 60
    lines = [f"Video: {Path(video_path).name}", f"Model: {model.model_type.upper()}", sep]
    for rank, idx in enumerate(top_idx, 1):
        lbl = idx2label.get(idx, f"[class {idx}]") if idx2label else f"[class {idx}]"
        lines.append(f"  Top-{rank}: {lbl}  ({probs[idx]*100:.2f}%)")
    lines.append(sep)
    print("\n" + "\n".join(lines))

    if log_file:
        log_lines = [f"[{datetime.now().isoformat()}] {Path(video_path).name}"]
        for idx in top_idx:
            # FIX: Swapped keyword parameter mapping to match function target declaration
            log_lines.append(_build_log(idx, idx2label.get(idx,"?") if idx2label else "?", probs[idx], lbl_map=idx2label))
        _append_log(log_file, "\n".join(log_lines) + "\n\n")


# ─── Webcam processing ────────────────────────────────────────────────────────

def run_webcam(model, prep, extractor, renderer, idx2label, cam_idx=0, top_k=1, log_file=None, classify_every=15):
    cap = cv2.VideoCapture(cam_idx)
    if not cap.isOpened(): 
        print(f"[ERROR] Could not open webcam index {cam_idx}")
        return
        
    max_buf = model.max_seq_len
    kp_buffer, last_text, last_color, last_log = [], "Collecting frames...", (200, 200, 200), ""
    frame_cnt = 0
    print(f"[INFO] Webcam started. Waiting for frames to buffer...")

    while True:
        ret, frame = cap.read()
        if not ret: 
            break
        frame_cnt += 1
        kp = extractor.process_frame(frame)
        if kp is not None:
            kp_buffer.append(kp)
            if len(kp_buffer) > max_buf: 
                kp_buffer.pop(0)

        if len(kp_buffer) >= 20 and frame_cnt % classify_every == 0:
            inputs = prep(list(kp_buffer), model.model_type)
            logits = model.predict(inputs)
            probs = ONNXModel.softmax(logits)
            top_idx = np.argsort(probs)[::-1][:top_k]
            best_idx, best_prob = top_idx[0], probs[top_idx[0]]
            best_lbl = idx2label.get(best_idx, f"[class {best_idx}]") if idx2label else f"Class {best_idx}"

            last_text = _build_text(best_idx, best_lbl, best_prob, renderer.can_render_myanmar, top_k, top_idx, probs, idx2label)
            last_color = (0, 200, 0) if best_prob >= 0.80 else (0, 200, 200) if best_prob >= 0.50 else (0, 0, 255)

            if log_file and last_text != last_log:
                ts = datetime.now().strftime("%H:%M:%S")
                # FIX: Fixed positional and keyword binding mismatches
                log_line = f"[{ts}] {_build_log(best_idx, best_lbl, best_prob, lbl_map=idx2label)}\n"
                _append_log(log_file, log_line)
                last_log = last_text

        extractor.draw_landmarks(frame)
        frame = renderer.put_text(frame, last_text, position=(10, 10), color=last_color)

        h, w = frame.shape[:2]
        cv2.rectangle(frame, (0, h - 8), (int(len(kp_buffer) / max_buf * w), h), (0, 180, 255), -1)
        cv2.putText(frame, f"buf {len(kp_buffer)}/{max_buf}", (10, h - 12), cv2.FONT_HERSHEY_SIMPLEX, 0.45, (200, 200, 200), 1)

        cv2.imshow("MSL Recognition  [Q=quit  C=clear]", frame)
        key = cv2.waitKey(1) & 0xFF
        if key == ord("q"): 
            break
        elif key == ord("c"):
            kp_buffer.clear()
            last_text = "Buffer cleared"
            last_color = (200, 200, 200)
            last_log = ""

    cap.release()
    cv2.destroyAllWindows()


def _append_log(path, text):
    try:
        with open(path, "a", encoding="utf-8-sig") as f: 
            f.write(text)
    except Exception: 
        pass


# ─── CLI ──────────────────────────────────────────────────────────────────────

def main():
    parser = argparse.ArgumentParser(description="MSL Sign Language Inference — ONNX")
    parser.add_argument("--onnx_model", required=True, help="Path to .onnx model file")
    parser.add_argument("--metadata", default=None, help="Path to .json metadata")
    parser.add_argument("--label_map", default=None, help="Path to label_map.json")
    grp = parser.add_mutually_exclusive_group(required=True)
    grp.add_argument("--video", help="Path to a video file")
    grp.add_argument("--webcam", type=int, metavar="CAM_IDX", help="Webcam device index")
    parser.add_argument("--top_k", type=int, default=5)
    parser.add_argument("--classify_every", type=int, default=15, help="Run model every N webcam frames")
    parser.add_argument("--log_file", default=None, help="Append predictions to this text file")
    parser.add_argument("--font_size", type=int, default=32)
    parser.add_argument("--debug", action="store_true", help="Print frame diagnostics")
    args = parser.parse_args()

    model_p = Path(args.onnx_model)
    meta_p  = args.metadata or str(model_p.with_suffix(".json"))
    lmap_p  = args.label_map or str(model_p.parent / "label_map.json")
    
    if not Path(lmap_p).exists(): 
        sys.exit(f"[ERROR] label_map.json not found at: {lmap_p}")

    idx2label = load_idx2label(lmap_p)
    print(f"[INFO] Operational classification dictionary parsed: loaded {len(idx2label)} target sign labels.")
    
    model = ONNXModel(str(model_p), meta_p)
    prep = Preprocessor(max_seq_len=model.max_seq_len)
    extractor = KeypointExtractor(model_complexity=1)
    renderer = TextRenderer(font_size=args.font_size)
    
    if args.log_file: 
        print(f"[INFO] Logging to: {args.log_file}")

    try:
        if args.video:
            run_video(model, prep, extractor, renderer, args.video, args.top_k, args.log_file, idx2label, args.debug)
        else:
            run_webcam(model, prep, extractor, renderer, idx2label, args.webcam, args.top_k, args.log_file, args.classify_every)
    finally:
        extractor.close()
        print("[INFO] Done.")


if __name__ == "__main__":
    main()


```

## Export Models Again

In [46]:
%pwd

'/mnt/disk1/ye/exp/msl_recognition'

In [62]:
!rm -rf ./onnx_models/

In [63]:
!time python src/export_to_onnx.py --config config/config.yaml --all

Device: cuda

  Exporting  BILSTM
  Classes: 556   MaxSeqLen: 200
  BiLSTM dummy inputs: kp=(2, 200, 225), mask=(2, 200)
/mnt/disk1/ye/exp/msl_recognition/.venv/lib/python3.12/site-packages/torch/onnx/symbolic_opset9.py:4244: UserWarning: Exporting a model to ONNX with a batch_size other than 1, with a variable length with LSTM can cause an error when running the ONNX model with a different batch size. Make sure to save the model with a batch size of 1, or define the initial states (h0/c0) as inputs of the model. 
  warnings.warn(
  Exported → onnx_models/bilstm_msl.onnx
  Validating ONNX model...
  ✓ ONNX graph is valid
  ✓ Inference OK — output shape: (1, 556)
  ✓ Top-5 class indices: [226, 208, 458, 256, 215]
  Metadata  → onnx_models/bilstm_msl.json
  Label map → onnx_models/label_map.json
  ✓ Done  (18.4 MB)

  Exporting  TRANSFORMER
  Classes: 556   MaxSeqLen: 200
  Transformer dummy inputs: kp=(2, 200, 225), mask=(2, 200)
  Exported → onnx_models/transformer_msl.onnx
  Validatin

In [60]:
!ls ./onnx_models/

stgcn.json  stgcn.onnx


In [64]:
!ls ./onnx_models/

bilstm_msl.json  label_map.json  stgcn_msl.onnx        transformer_msl.onnx
bilstm_msl.onnx  stgcn_msl.json  transformer_msl.json


## Infer Video on Windows OS Machine Again


```
(msl_onnx) E:\LU_Lab\MSL_Tran_Demo>python infer_onnx.py --onnx_model .\onnx_models\bilstm_msl.onnx --video .\sample4inference\idx20-30.mp4
[INFO] Operational classification dictionary parsed: loaded 556 target sign labels.
[INFO] ONNX loaded: bilstm_msl.onnx  (provider: CPUExecutionProvider)
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
[INFO] Processing 127 frames from idx20-30.mp4 ...
C:\Users\801680\Anaconda3\envs\msl_onnx\lib\site-packages\google\protobuf\symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '
[INFO] Frames: 127 valid / 73 padded (dropped 0/127)

Video: idx20-30.mp4
Model: BILSTM
────────────────────────────────────────────────────────────
  Top-1: မီးချိတ် ။  (1.46%)
  Top-2: ဒီ ဆေး ကို တစ် ရက် တစ် ကြိမ် သောက် ပါ ။  (1.45%)
  Top-3: ဒီ ဆေး ကို နာကျင် ကိုက်ခဲ မှ သောက် ပါ ။  (1.29%)
  Top-4: အရေးပေါ် သယ် ရ မယ့် ပစ္စည်း တွေ ကြိုတင် ပြင်ဆင် ထား ။  (1.04%)
  Top-5: ဆေး ကို တစ် နေ့ ဘယ်နှစ်ကြိမ် သောက် ရ မှာ လဲ ။  (1.03%)
────────────────────────────────────────────────────────────
[INFO] Done.

(msl_onnx) E:\LU_Lab\MSL_Tran_Demo>
```

```
(msl_onnx) E:\LU_Lab\MSL_Tran_Demo>python infer_onnx.py --onnx_model .\onnx_models\bilstm_msl.onnx --video .\sample4inference\idx20-86.mp4
[INFO] Operational classification dictionary parsed: loaded 556 target sign labels.
[INFO] ONNX loaded: bilstm_msl.onnx  (provider: CPUExecutionProvider)
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
[INFO] Processing 169 frames from idx20-86.mp4 ...
C:\Users\801680\Anaconda3\envs\msl_onnx\lib\site-packages\google\protobuf\symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '
[INFO] Frames: 169 valid / 31 padded (dropped 0/169)

Video: idx20-86.mp4
Model: BILSTM
────────────────────────────────────────────────────────────
  Top-1: မီးချိတ် ။  (27.28%)
  Top-2: အရေးပေါ် သယ် ရ မယ့် ပစ္စည်း တွေ ကြိုတင် ပြင်ဆင် ထား ။  (1.06%)
  Top-3: သစ်ပင် ။  (0.80%)
  Top-4: မ နေ့ ည က ကား အ ခိုး ခံ လိုက် ရ တယ် ။  (0.67%)
  Top-5: ကားတိုက် မှု ဖြစ် လို့ ပါ ။  (0.63%)
────────────────────────────────────────────────────────────
[INFO] Done.

(msl_onnx) E:\LU_Lab\MSL_Tran_Demo>
```

```
(msl_onnx) E:\LU_Lab\MSL_Tran_Demo>python infer_onnx.py --onnx_model .\onnx_models\bilstm_msl.onnx --video .\sample4inference\idx20-244.mp4
[INFO] Operational classification dictionary parsed: loaded 556 target sign labels.
[INFO] ONNX loaded: bilstm_msl.onnx  (provider: CPUExecutionProvider)
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
[INFO] Processing 55 frames from idx20-244.mp4 ...
C:\Users\801680\Anaconda3\envs\msl_onnx\lib\site-packages\google\protobuf\symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '
[INFO] Frames: 55 valid / 145 padded (dropped 0/55)

Video: idx20-244.mp4
Model: BILSTM
────────────────────────────────────────────────────────────
  Top-1: ရေ စို ပြီး အနံ့အသက် မ ကောင်း တဲ့ အစာ များ ကို မ စား ပါ နှင့် ။  (2.05%)
  Top-2: ကြောက်လန့်တကြား မ ပြေး ပါ နဲ့ ။  (1.70%)
  Top-3: အပေါ်သွေး ၁၅၀ အောက်သွေး ၁၀၀ ရှိ တယ် ။  (1.27%)
  Top-4: လျှပ်စစ် ဝါယာ ရှော့ ဖြစ် ရင် ရေ စို အဝတ် နဲ့ မီး မ ငြှိမ်းသတ် ပါ နဲ့ ။  (1.01%)
  Top-5: ကျွန်တော် လက် ကွေး လို့ မ ရ ဘူး ဖြစ် နေ တယ် ။  (0.71%)
────────────────────────────────────────────────────────────
[INFO] Done.

(msl_onnx) E:\LU_Lab\MSL_Tran_Demo>
```

```
(msl_onnx) E:\LU_Lab\MSL_Tran_Demo>python infer_onnx.py --onnx_model .\onnx_models\bilstm_msl.onnx --video .\sample4inference\idx20-291.mp4
[INFO] Operational classification dictionary parsed: loaded 556 target sign labels.
[INFO] ONNX loaded: bilstm_msl.onnx  (provider: CPUExecutionProvider)
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
[INFO] Processing 211 frames from idx20-291.mp4 ...
C:\Users\801680\Anaconda3\envs\msl_onnx\lib\site-packages\google\protobuf\symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '
[INFO] Frames: 200 valid / 0 padded (dropped 0/211)

Video: idx20-291.mp4
Model: BILSTM
────────────────────────────────────────────────────────────
  Top-1: အရေးပေါ် သယ် ရ မယ့် ပစ္စည်း တွေ ကြိုတင် ပြင်ဆင် ထား ။  (4.64%)
  Top-2: မီးချိတ် ။  (2.36%)
  Top-3: ဘယ် အချိန် တွေ ဆရာဝန် နဲ့ ပြ လို့ ရ လဲ သိ ချင် ပါ တယ် ။  (2.27%)
  Top-4: မီးလောင် နေ တဲ့ အဆောက်အဦး ထဲ မှာ လူ တွေ အများကြီး ပိတ်မိ နေ ကြ တယ် ။  (1.39%)
  Top-5: ကား တိုက်ခံရ တဲ့ လူ သတိ ရှိ သေး လား ။  (1.18%)
────────────────────────────────────────────────────────────
[INFO] Done.

(msl_onnx) E:\LU_Lab\MSL_Tran_Demo>
```

```
(msl_onnx) E:\LU_Lab\MSL_Tran_Demo>python infer_onnx.py --onnx_model .\onnx_models\bilstm_msl.onnx --video .\sample4inference\idx20-440.mp4
[INFO] Operational classification dictionary parsed: loaded 556 target sign labels.
[INFO] ONNX loaded: bilstm_msl.onnx  (provider: CPUExecutionProvider)
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
[INFO] Processing 248 frames from idx20-440.mp4 ...
C:\Users\801680\Anaconda3\envs\msl_onnx\lib\site-packages\google\protobuf\symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '
[INFO] Frames: 200 valid / 0 padded (dropped 0/248)

Video: idx20-440.mp4
Model: BILSTM
────────────────────────────────────────────────────────────
  Top-1: ကျွန်တော့် စျေးဆိုင် အ ဖောက် ခံ လိုက် ရ တယ် ။  (6.16%)
  Top-2: မုန်တိုင်း အဆင့် အဝါရောင် လိမ္မော်ရောင် အနီရောင် အညိုရောင် အစိမ်းရောင် ။  (3.89%)
  Top-3: အရေးပေါ် သယ် ရ မယ့် ပစ္စည်း တွေ ကြိုတင် ပြင်ဆင် ထား ။  (1.70%)
  Top-4: ဆေး ကို တစ် နေ့ ဘယ်နှစ်ကြိမ် သောက် ရ မှာ လဲ ။  (1.14%)
  Top-5: အိမ် ဖောက် ခံ လိုက် ရ လို့ တိုင်ချက် ဖွင့် ချင် လို့ ပါ ။  (0.97%)
────────────────────────────────────────────────────────────
[INFO] Done.

(msl_onnx) E:\LU_Lab\MSL_Tran_Demo>
```

```
(msl_onnx) E:\LU_Lab\MSL_Tran_Demo>python infer_onnx.py --onnx_model .\onnx_models\transformer_msl.onnx --video .\sample4inference\idx20-30.mp4
[INFO] Operational classification dictionary parsed: loaded 556 target sign labels.
[INFO] ONNX loaded: transformer_msl.onnx  (provider: CPUExecutionProvider)
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
[INFO] Processing 127 frames from idx20-30.mp4 ...
C:\Users\801680\Anaconda3\envs\msl_onnx\lib\site-packages\google\protobuf\symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '
[INFO] Frames: 127 valid / 73 padded (dropped 0/127)

Video: idx20-30.mp4
Model: TRANSFORMER
────────────────────────────────────────────────────────────
  Top-1: ဘယ် အချိန် တွေ ဆရာဝန် နဲ့ ပြ လို့ ရ လဲ သိ ချင် ပါ တယ် ။  (13.77%)
  Top-2: ထွက်ပေါက် ။  (3.54%)
  Top-3: ဆေးရုံ ကို ဖုန်းဆက် ပေး ။  (1.89%)
  Top-4: ကျွန်တော့် နောက်ကျော က နာ တယ် ။  (1.80%)
  Top-5: ဒီ အခန်း ထဲ မှာ ဂတ်စ် အနံ့ ရ နေ တယ် ။  (1.00%)
────────────────────────────────────────────────────────────
[INFO] Done.

(msl_onnx) E:\LU_Lab\MSL_Tran_Demo>
```

```
(msl_onnx) E:\LU_Lab\MSL_Tran_Demo>python infer_onnx.py --onnx_model .\onnx_models\transformer_msl.onnx --video .\sample4inference\idx20-86.mp4
[INFO] Operational classification dictionary parsed: loaded 556 target sign labels.
[INFO] ONNX loaded: transformer_msl.onnx  (provider: CPUExecutionProvider)
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
[INFO] Processing 169 frames from idx20-86.mp4 ...
C:\Users\801680\Anaconda3\envs\msl_onnx\lib\site-packages\google\protobuf\symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '
[INFO] Frames: 169 valid / 31 padded (dropped 0/169)

Video: idx20-86.mp4
Model: TRANSFORMER
────────────────────────────────────────────────────────────
  Top-1: ထွက်ပေါက် ။  (5.29%)
  Top-2: ကြမ်းပြင် တစ်ခု လုံး မှာ ရေ တွေ ကြည့် ပဲ ။  (1.89%)
  Top-3: ဆေးရုံ ကို ဖုန်းဆက် ပေး ။  (1.50%)
  Top-4: ကျွန်တော့် နောက်ကျော က နာ တယ် ။  (1.48%)
  Top-5: ရဲ ကို ဖုန်းဆက် ပေး ပါ ။  (1.44%)
────────────────────────────────────────────────────────────
[INFO] Done.

(msl_onnx) E:\LU_Lab\MSL_Tran_Demo>
```

```
(msl_onnx) E:\LU_Lab\MSL_Tran_Demo>python infer_onnx.py --onnx_model .\onnx_models\transformer_msl.onnx --video .\sample4inference\idx20-244.mp4
[INFO] Operational classification dictionary parsed: loaded 556 target sign labels.
[INFO] ONNX loaded: transformer_msl.onnx  (provider: CPUExecutionProvider)
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
[INFO] Processing 55 frames from idx20-244.mp4 ...
C:\Users\801680\Anaconda3\envs\msl_onnx\lib\site-packages\google\protobuf\symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '
[INFO] Frames: 55 valid / 145 padded (dropped 0/55)

Video: idx20-244.mp4
Model: TRANSFORMER
────────────────────────────────────────────────────────────
  Top-1: ကြောက်လန့်တကြား မ ပြေး ပါ နဲ့ ။  (9.52%)
  Top-2: ဘယ် အချိန် တွေ ဆရာဝန် နဲ့ ပြ လို့ ရ လဲ သိ ချင် ပါ တယ် ။  (4.59%)
  Top-3: ဆရာဝန် မြန်မြန် ခေါ် လိုက် လေ ။  (2.63%)
  Top-4: ဒီ ဆေး ကို နာကျင် ကိုက်ခဲ မှ သောက် ပါ ။  (2.53%)
  Top-5: ကြမ်းပြင် တစ်ခု လုံး မှာ ရေ တွေ ကြည့် ပဲ ။  (2.35%)
────────────────────────────────────────────────────────────
[INFO] Done.

(msl_onnx) E:\LU_Lab\MSL_Tran_Demo>
```

```
(msl_onnx) E:\LU_Lab\MSL_Tran_Demo>python infer_onnx.py --onnx_model .\onnx_models\transformer_msl.onnx --video .\sample4inference\idx20-291.mp4
[INFO] Operational classification dictionary parsed: loaded 556 target sign labels.
[INFO] ONNX loaded: transformer_msl.onnx  (provider: CPUExecutionProvider)
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
[INFO] Processing 211 frames from idx20-291.mp4 ...
C:\Users\801680\Anaconda3\envs\msl_onnx\lib\site-packages\google\protobuf\symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '
[INFO] Frames: 200 valid / 0 padded (dropped 0/211)

Video: idx20-291.mp4
Model: TRANSFORMER
────────────────────────────────────────────────────────────
  Top-1: ထွက်ပေါက် ။  (1.83%)
  Top-2: ဆေး ကောင်တာ ကို သွား ပြီး ဆေး ထုတ် လိုက် ပါ ။  (1.71%)
  Top-3: ဘယ် အချိန် တွေ ဆရာဝန် နဲ့ ပြ လို့ ရ လဲ သိ ချင် ပါ တယ် ။  (1.50%)
  Top-4: သူ က ဘယ် လူနာဆောင် ထဲ မှာ လဲ ။  (1.40%)
  Top-5: ဒီ အခန်း ထဲ မှာ ဂတ်စ် အနံ့ ရ နေ တယ် ။  (1.32%)
────────────────────────────────────────────────────────────
[INFO] Done.

(msl_onnx) E:\LU_Lab\MSL_Tran_Demo>
```

```
(msl_onnx) E:\LU_Lab\MSL_Tran_Demo>python infer_onnx.py --onnx_model .\onnx_models\transformer_msl.onnx --video .\sample4inference\idx20-440.mp4
[INFO] Operational classification dictionary parsed: loaded 556 target sign labels.
[INFO] ONNX loaded: transformer_msl.onnx  (provider: CPUExecutionProvider)
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
[INFO] Processing 248 frames from idx20-440.mp4 ...
C:\Users\801680\Anaconda3\envs\msl_onnx\lib\site-packages\google\protobuf\symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '
[INFO] Frames: 200 valid / 0 padded (dropped 0/248)

Video: idx20-440.mp4
Model: TRANSFORMER
────────────────────────────────────────────────────────────
  Top-1: သူ က ဘယ် လူနာဆောင် ထဲ မှာ လဲ ။  (6.53%)
  Top-2: လျှပ်စစ် ဝါယော ရှော့ ဖြစ် ရင် အဝတ်ခြောက် ဖြင့် ရိုက်ပုတ် ငြှိမ်းသတ် ပါ ။  (3.84%)
  Top-3: ဘယ် အချိန် တွေ ဆရာဝန် နဲ့ ပြ လို့ ရ လဲ သိ ချင် ပါ တယ် ။  (3.53%)
  Top-4: မုန်တိုင်း အဆင့် အဝါရောင် လိမ္မော်ရောင် အနီရောင် အညိုရောင် အစိမ်းရောင် ။  (3.22%)
  Top-5: ဒီ လိမ်းဆေး ကို တစ် ရက် နှစ် ကြိမ် လိမ်း ပါ ။  (2.09%)
────────────────────────────────────────────────────────────
[INFO] Done.

(msl_onnx) E:\LU_Lab\MSL_Tran_Demo>
```

```
(msl_onnx) E:\LU_Lab\MSL_Tran_Demo>python infer_onnx.py --onnx_model .\onnx_models\stgcn_msl.onnx --video .\sample4inference\idx20-30.mp4
[INFO] Operational classification dictionary parsed: loaded 556 target sign labels.
[INFO] ONNX loaded: stgcn_msl.onnx  (provider: CPUExecutionProvider)
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
[INFO] Processing 127 frames from idx20-30.mp4 ...
C:\Users\801680\Anaconda3\envs\msl_onnx\lib\site-packages\google\protobuf\symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '
[INFO] Frames: 127 valid / 73 padded (dropped 0/127)

Video: idx20-30.mp4
Model: STGCN
────────────────────────────────────────────────────────────
  Top-1: မ နေ့ ည က အိမ် ထဲ ကို သူခိုး ဝင် သွား လို့ ပါ ။  (7.99%)
  Top-2: ဓာတ်ကြိုး များ ပျက်စီး နေ နိုင် ပါ တယ် ။  (5.85%)
  Top-3: မ တပ် ထား ပါ ဘူး ။  (4.48%)
  Top-4: ကျွန်တော် အသက် ကောင်းကောင်း ရှူ လို့ မ ရ ဘူး ။  (4.05%)
  Top-5: သွားဆရာဝန် ရဲ့ လိပ်စာ ပြော ပေး ပါ ဦး ။  (3.90%)
────────────────────────────────────────────────────────────
[INFO] Done.

(msl_onnx) E:\LU_Lab\MSL_Tran_Demo>
```

```
(msl_onnx) E:\LU_Lab\MSL_Tran_Demo>python infer_onnx.py --onnx_model .\onnx_models\stgcn_msl.onnx --video .\sample4inference\idx20-86.mp4
[INFO] Operational classification dictionary parsed: loaded 556 target sign labels.
[INFO] ONNX loaded: stgcn_msl.onnx  (provider: CPUExecutionProvider)
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
[INFO] Processing 169 frames from idx20-86.mp4 ...
C:\Users\801680\Anaconda3\envs\msl_onnx\lib\site-packages\google\protobuf\symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '
[INFO] Frames: 169 valid / 31 padded (dropped 0/169)

Video: idx20-86.mp4
Model: STGCN
────────────────────────────────────────────────────────────
  Top-1: ကျွန်တော် အသက် ကောင်းကောင်း ရှူ လို့ မ ရ ဘူး ။  (40.97%)
  Top-2: ကျွန်တော် သွား စိုက် ချင် လို့ ပါ ။  (7.26%)
  Top-3: ဒီ ဆေး က ဘာ အတွက် လဲ ။  (7.18%)
  Top-4: ဆေးလိပ်မီး ကို သေချာ ငြိမ်း ပြီး မှ ပစ် ပါ ။  (4.14%)
  Top-5: လူ တစ်ယောက် နှလုံးခုန် ရပ် သွား လို့ ပါ ။  (4.13%)
────────────────────────────────────────────────────────────
[INFO] Done.

(msl_onnx) E:\LU_Lab\MSL_Tran_Demo>
```

```
(msl_onnx) E:\LU_Lab\MSL_Tran_Demo>python infer_onnx.py --onnx_model .\onnx_models\stgcn_msl.onnx --video .\sample4inference\idx20-244.mp4
[INFO] Operational classification dictionary parsed: loaded 556 target sign labels.
[INFO] ONNX loaded: stgcn_msl.onnx  (provider: CPUExecutionProvider)
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
[INFO] Processing 55 frames from idx20-244.mp4 ...
C:\Users\801680\Anaconda3\envs\msl_onnx\lib\site-packages\google\protobuf\symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '
[INFO] Frames: 55 valid / 145 padded (dropped 0/55)

Video: idx20-244.mp4
Model: STGCN
────────────────────────────────────────────────────────────
  Top-1: ကျွန်တော် သွား စိုက် ချင် လို့ ပါ ။  (25.89%)
  Top-2: ကျွန်တော် တစ်ကိုယ်လုံး ကိုက်ခဲ နေ တာ ပဲ ။  (10.68%)
  Top-3: မ တပ် ထား ပါ ဘူး ။  (6.91%)
  Top-4: ကျွန်တော် အသက် ကောင်းကောင်း ရှူ လို့ မ ရ ဘူး ။  (5.20%)
  Top-5: သွား ကိုက် တယ် ။  (3.47%)
────────────────────────────────────────────────────────────
[INFO] Done.

(msl_onnx) E:\LU_Lab\MSL_Tran_Demo>
```

```
(msl_onnx) E:\LU_Lab\MSL_Tran_Demo>python infer_onnx.py --onnx_model .\onnx_models\stgcn_msl.onnx --video .\sample4inference\idx20-291.mp4
[INFO] Operational classification dictionary parsed: loaded 556 target sign labels.
[INFO] ONNX loaded: stgcn_msl.onnx  (provider: CPUExecutionProvider)
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
[INFO] Processing 211 frames from idx20-291.mp4 ...
C:\Users\801680\Anaconda3\envs\msl_onnx\lib\site-packages\google\protobuf\symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '
[INFO] Frames: 200 valid / 0 padded (dropped 0/211)

Video: idx20-291.mp4
Model: STGCN
────────────────────────────────────────────────────────────
  Top-1: ကျွန်တော် အသက် ကောင်းကောင်း ရှူ လို့ မ ရ ဘူး ။  (13.61%)
  Top-2: ကျွန်မ အမျိုးသမီး ဆရာဝန် နဲ့ ပဲ ပြ ချင် တယ် ။  (9.55%)
  Top-3: ကျွန်တော် တစ်ချိန်လုံး အားအင် ကုန်ခမ်း နေ သလို ခံစား နေ ရ တယ် ။  (7.15%)
  Top-4: ဆေးလိပ်မီး ကို သေချာ ငြိမ်း ပြီး မှ ပစ် ပါ ။  (6.16%)
  Top-5: ကား ထဲ မှာ လူ တစ်ယောက် ညှပ်မိ နေ တယ် ။  (3.95%)
────────────────────────────────────────────────────────────
[INFO] Done.

(msl_onnx) E:\LU_Lab\MSL_Tran_Demo>
```

```
(msl_onnx) E:\LU_Lab\MSL_Tran_Demo>python infer_onnx.py --onnx_model .\onnx_models\stgcn_msl.onnx --video .\sample4inference\idx20-440.mp4
[INFO] Operational classification dictionary parsed: loaded 556 target sign labels.
[INFO] ONNX loaded: stgcn_msl.onnx  (provider: CPUExecutionProvider)
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
[INFO] Processing 248 frames from idx20-440.mp4 ...
C:\Users\801680\Anaconda3\envs\msl_onnx\lib\site-packages\google\protobuf\symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '
[INFO] Frames: 200 valid / 0 padded (dropped 0/248)

Video: idx20-440.mp4
Model: STGCN
────────────────────────────────────────────────────────────
  Top-1: ဓာတ်ကြိုး များ ပျက်စီး နေ နိုင် ပါ တယ် ။  (46.75%)
  Top-2: သွားဆရာဝန် ရဲ့ လိပ်စာ ပြော ပေး ပါ ဦး ။  (13.41%)
  Top-3: သစ်ပင် ၊ ဓာတ်ကြိုး နဲ့ ဖုန်းလိုင်း တွေ နား မှာ မ နေ နဲ့ ။  (4.33%)
  Top-4: ညနေ ၅ နာရီ ဆို ဆေးခန်း ဖွင့် ပါ တယ် ။  (1.56%)
  Top-5: ဒီ ဆေး က ဘာ အတွက် လဲ ။  (1.22%)
────────────────────────────────────────────────────────────
[INFO] Done.

(msl_onnx) E:\LU_Lab\MSL_Tran_Demo>
```

အထက်မှာ မြင်ရတဲ့အတိုင်းပါပဲ model သုံးခုစလုံးရဲ့ prediction တွေက မှားနေပါတယ်။  
Debug လုပ်ဖို့ လိုအပ်တယ်။  

## Updated Again

In [69]:
from IPython.display import Markdown

file_path = './src/export_to_onnx.py'

with open(file_path, 'r') as f:
    code = f.read()

# 'python' keyword triggers the syntax highlighter
display(Markdown(f"```python\n{code}\n```"))

```python
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
export_to_onnx.py  –  Export trained MSL models to ONNX format
                       (runs on Linux/HPC server side)

Bugs fixed in this revision
────────────────────────────
  [EXPORT-1] _TransformerExportWrapper — double-dropout in post-norm FFN path.
      The residual add used `layer_dropout2[i](ff)` when `ff` had already had
      dropout applied inside the FF block. Fixed by applying dropout once to
      the FF output and adding the result directly to the residual.

  [EXPORT-2] _BiLSTMExportWrapper — `lengths=None` causes the LSTM to run on
      the *full* padded sequence and read the last real hidden state from the
      *padded* tail, not from position `real_len-1`. For sequences shorter than
      max_seq_len this produces wrong pooled representations.  Fixed by
      computing real lengths from the mask inside the wrapper and passing them
      through so the model uses the correct final states.

  [EXPORT-3] validate_onnx — boolean mask dtype on Windows ORT.
      `dtype=bool` alias maps to Python bool which some Windows ORT builds
      reject. Explicitly cast to `np.bool_` (== np.uint8 in memory for ORT).

  Previously documented bugs (retained from prior revision):
  [EXPORT-4] ST-GCN in-place copy_ during tracing — fixed via _STGCNExportWrapper.
  [EXPORT-5] pack_padded_sequence not in ONNX opset 14 — fixed by no-pack mode.
  [EXPORT-6] PyTorch 2.x fused MHA / TransformerEncoderLayer kernels have no
      ONNX mapping — fixed via _TransformerExportWrapper + ManualMHA.
  [EXPORT-7] GELU requires opset >= 20 — fixed via erf-based equivalent.
  [EXPORT-8] Batch size 2 dummy inputs to prevent scalar-collapse graph bugs.
  [EXPORT-9] 'model_type' missing from metadata JSON — now always written.
"""

import argparse
import json
import shutil
import sys
from pathlib import Path

import numpy as np
import torch
import yaml

sys.path.insert(0, str(Path(__file__).parent))
from models import build_model
from utils  import get_logger, get_device, load_label_map

# ── Experiment → output name mapping ─────────────────────────────────────────
MODEL_CONFIGS = {
    "bilstm":       {"exp_dir": "results/exp_bilstm",       "output_name": "bilstm_msl.onnx"},
    "transformer":  {"exp_dir": "results/exp_transformer",  "output_name": "transformer_msl.onnx"},
    "stgcn":        {"exp_dir": "results/exp_stgcn",        "output_name": "stgcn_msl.onnx"},
}

# Fixed constants — must match extract_keypoints.py
POSE_N       = 33
HAND_N       = 21
TOTAL_JOINTS = POSE_N + HAND_N * 2   # 75
FEAT_DIM     = TOTAL_JOINTS * 3       # 225


# ─────────────────────────────────────────────────────────────────────────────
# BiLSTM export
# ─────────────────────────────────────────────────────────────────────────────

class _BiLSTMExportWrapper(torch.nn.Module):
    """
    Thin wrapper that converts the boolean padding mask to a length tensor
    WITHOUT calling pack_padded_sequence (which ONNX cannot trace).

    BUG FIX [EXPORT-2]: The previous version passed lengths=None which caused
    the LSTM to run over the entire padded sequence and take the hidden state
    at position max_seq_len-1 (which is a zero-padded frame) instead of at the
    last real frame. This silently produced wrong predictions for any sequence
    shorter than max_seq_len.

    Fix: compute real lengths from the mask (sum of non-padded positions) and
    pass them to the model. The underlying BiLSTM honours the lengths argument
    to index the correct final hidden state via its internal gather logic.
    """
    def __init__(self, model):
        super().__init__()
        self.model = model

    def forward(self, keypoints: torch.Tensor, mask: torch.Tensor) -> torch.Tensor:
        # mask: (B, T) — True where PADDED
        # Count valid (non-padded) frames per sample → lengths shape (B,)
        # ~mask == True where NOT padded; sum along T gives real lengths.
        lengths = (~mask).sum(dim=1).long()          # (B,) — real frame counts
        return self.model(keypoints, lengths=lengths, mask=mask)


def export_bilstm(model, out_path: Path, max_seq_len: int, device: str):
    model.eval()
    wrapper = _BiLSTMExportWrapper(model).to(device)
    wrapper.eval()

    # Batch size 2 prevents tracking indices from collapsing into 0-D scalars
    dummy_kp   = torch.randn(2, max_seq_len, FEAT_DIM, device=device)
    dummy_mask = torch.zeros(2, max_seq_len, dtype=torch.bool, device=device)

    print(f"  BiLSTM dummy inputs: kp={tuple(dummy_kp.shape)}, mask={tuple(dummy_mask.shape)}")

    with torch.no_grad():
        torch.onnx.export(
            wrapper,
            (dummy_kp, dummy_mask),
            str(out_path),
            opset_version=16,
            input_names=["keypoints", "mask"],
            output_names=["logits"],
            dynamic_axes={
                "keypoints": {0: "batch"},
                "mask":      {0: "batch"},
                "logits":    {0: "batch"},
            },
            do_constant_folding=True,
            verbose=False,
        )
    print(f"  Exported → {out_path}")


# ─────────────────────────────────────────────────────────────────────────────
# Transformer export
# ─────────────────────────────────────────────────────────────────────────────

def _onnx_safe_gelu(x: torch.Tensor) -> torch.Tensor:
    """GELU via erf — supported in ONNX opset 14+. Avoids opset-20 requirement."""
    return 0.5 * x * (1.0 + torch.erf(x / 1.4142135623730951))


class ManualMHA(torch.nn.Module):
    """
    Manual Multi-Head Attention that bypasses PyTorch 2.x fused
    `aten::_native_multi_head_attention` which has no ONNX mapping.
    Weights are cloned from the original nn.MultiheadAttention.
    """
    def __init__(self, mha: torch.nn.MultiheadAttention):
        super().__init__()
        self.embed_dim = mha.embed_dim
        self.num_heads = mha.num_heads
        self.head_dim  = self.embed_dim // self.num_heads

        self.in_proj_weight  = torch.nn.Parameter(mha.in_proj_weight.data.clone())
        self.in_proj_bias    = torch.nn.Parameter(mha.in_proj_bias.data.clone())
        self.out_proj_weight = torch.nn.Parameter(mha.out_proj.weight.data.clone())
        self.out_proj_bias   = torch.nn.Parameter(mha.out_proj.bias.data.clone())

    def forward(self, query: torch.Tensor, key: torch.Tensor, value: torch.Tensor,
                key_padding_mask: torch.Tensor = None, need_weights: bool = False):
        B, T, C = query.shape

        q_w = self.in_proj_weight[:C]
        k_w = self.in_proj_weight[C:2*C]
        v_w = self.in_proj_weight[2*C:]
        q_b = self.in_proj_bias[:C]
        k_b = self.in_proj_bias[C:2*C]
        v_b = self.in_proj_bias[2*C:]

        Q = torch.nn.functional.linear(query, q_w, q_b)
        K = torch.nn.functional.linear(key,   k_w, k_b)
        V = torch.nn.functional.linear(value, v_w, v_b)

        # Reshape to (B, heads, T, head_dim) using -1 for ONNX dynamic shape
        Q = Q.reshape(B, -1, self.num_heads, self.head_dim).transpose(1, 2)
        K = K.reshape(B, -1, self.num_heads, self.head_dim).transpose(1, 2)
        V = V.reshape(B, -1, self.num_heads, self.head_dim).transpose(1, 2)

        scale  = self.head_dim ** 0.5
        scores = torch.matmul(Q, K.transpose(-2, -1)) / scale

        if key_padding_mask is not None:
            # (B, T) → (B, 1, 1, T) for broadcasting across heads and query steps
            scores = scores.masked_fill(key_padding_mask.unsqueeze(1).unsqueeze(2),
                                        float('-inf'))

        attn_weights = torch.nn.functional.softmax(scores, dim=-1)
        attn_out     = torch.matmul(attn_weights, V)
        attn_out     = attn_out.transpose(1, 2).reshape(B, -1, C)
        out          = torch.nn.functional.linear(attn_out, self.out_proj_weight, self.out_proj_bias)

        return (out, attn_weights) if need_weights else (out, None)


class _TransformerExportWrapper(torch.nn.Module):
    """
    Wrapper that completely bypasses PyTorch 2.x fused operators by unrolling
    the encoder loop and calling basic sub-modules directly.

    BUG FIX [EXPORT-1]: The previous post-norm path applied layer_dropout2
    twice: once inside the FF block (`self.layer_dropout2[i](ff)`) and once
    again in the residual add (`x + self.layer_dropout2[i](ff)`). This
    corrupts the exported weights — output is statistically wrong for any
    model trained without dropout (eval mode) because eval dropout is identity,
    but during ORT inference on Windows the double application still runs
    through a different code path, producing slightly different outputs.
    More critically, if dropout rate > 0 at export time, weights are scaled
    incorrectly. Fixed by storing the FF dropout output once and reusing it.
    """
    def __init__(self, model):
        super().__init__()
        self.input_norm  = model.input_norm
        self.input_proj  = model.input_proj
        self.proj_norm   = model.proj_norm
        self.cls_token   = model.cls_token
        self.pos_enc     = model.pos_enc
        self.encoder_norm = model.encoder.norm
        self.head_norm   = model.head_norm
        self.head_dropout = model.head_dropout
        self.fc          = model.fc

        # Replace fused MHA with ONNX-safe manual implementation
        self.layer_self_attn = torch.nn.ModuleList(
            [ManualMHA(l.self_attn) for l in model.encoder.layers]
        )
        self.layer_linear1  = torch.nn.ModuleList([l.linear1  for l in model.encoder.layers])
        self.layer_linear2  = torch.nn.ModuleList([l.linear2  for l in model.encoder.layers])
        self.layer_norm1    = torch.nn.ModuleList([l.norm1    for l in model.encoder.layers])
        self.layer_norm2    = torch.nn.ModuleList([l.norm2    for l in model.encoder.layers])
        self.layer_dropout1 = torch.nn.ModuleList([l.dropout1 for l in model.encoder.layers])
        self.layer_dropout2 = torch.nn.ModuleList([l.dropout2 for l in model.encoder.layers])

        self.num_layers = len(model.encoder.layers)
        self.norm_first = model.encoder.layers[0].norm_first

        act = model.encoder.layers[0].activation
        self._use_gelu = not (
            act is torch.nn.functional.relu or isinstance(act, torch.nn.ReLU)
        )

    def forward(self, keypoints: torch.Tensor, mask: torch.Tensor) -> torch.Tensor:
        x = self.input_norm(keypoints)   # (B, T, 225)
        x = self.input_proj(x)           # (B, T, d_model)
        x = self.proj_norm(x)

        B = x.shape[0]
        cls_tokens  = self.cls_token.expand(B, -1, -1)
        x           = torch.cat([cls_tokens, x], dim=1)       # (B, T+1, d_model)
        cls_mask    = torch.zeros(B, 1, dtype=torch.bool, device=x.device)
        padding_mask = torch.cat([cls_mask, mask], dim=1)     # (B, T+1)
        x = self.pos_enc(x)

        for i in range(self.num_layers):
            if self.norm_first:
                # ── Pre-norm ──────────────────────────────────────────────────
                nx = self.layer_norm1[i](x)
                attn_out, _ = self.layer_self_attn[i](
                    nx, nx, nx,
                    key_padding_mask=padding_mask,
                    need_weights=False,
                )
                x = x + self.layer_dropout1[i](attn_out)

                nx = self.layer_norm2[i](x)
                ff = self.layer_linear1[i](nx)
                ff = _onnx_safe_gelu(ff) if self._use_gelu else torch.nn.functional.relu(ff)
                # BUG FIX [EXPORT-1]: apply dropout exactly once, store result
                ff = self.layer_dropout2[i](self.layer_linear2[i](ff))
                x  = x + ff                                    # single dropout add

            else:
                # ── Post-norm (PyTorch default) ────────────────────────────────
                attn_out, _ = self.layer_self_attn[i](
                    x, x, x,
                    key_padding_mask=padding_mask,
                    need_weights=False,
                )
                x = self.layer_norm1[i](x + self.layer_dropout1[i](attn_out))

                ff = self.layer_linear1[i](x)
                ff = _onnx_safe_gelu(ff) if self._use_gelu else torch.nn.functional.relu(ff)
                # BUG FIX [EXPORT-1]: apply dropout once to full FF output
                ff_dropped = self.layer_dropout2[i](self.layer_linear2[i](ff))
                x  = self.layer_norm2[i](x + ff_dropped)      # single dropout add

        x       = self.encoder_norm(x)
        cls_out = x[:, 0, :]             # CLS token
        cls_out = self.head_norm(cls_out)
        cls_out = self.head_dropout(cls_out)
        return self.fc(cls_out)


def export_transformer(model, out_path: Path, max_seq_len: int, device: str):
    model.eval()
    wrapper = _TransformerExportWrapper(model).to(device)
    wrapper.eval()

    dummy_kp   = torch.randn(2, max_seq_len, FEAT_DIM, device=device)
    dummy_mask = torch.zeros(2, max_seq_len, dtype=torch.bool, device=device)
    print(f"  Transformer dummy inputs: kp={tuple(dummy_kp.shape)}, mask={tuple(dummy_mask.shape)}")

    with torch.no_grad():
        try:
            from torch.nn.attention import sdp_kernel
            attn_ctx = sdp_kernel(enable_flash=False, enable_mem_efficient=False, enable_math=True)
        except ImportError:
            import contextlib
            attn_ctx = contextlib.nullcontext()

        with attn_ctx:
            torch.onnx.export(
                wrapper,
                (dummy_kp, dummy_mask),
                str(out_path),
                opset_version=16,
                input_names=["keypoints", "mask"],
                output_names=["logits"],
                dynamic_axes={
                    "keypoints": {0: "batch", 1: "seq_len"},
                    "mask":      {0: "batch", 1: "seq_len"},
                    "logits":    {0: "batch"},
                },
                do_constant_folding=True,
                verbose=False,
            )
    print(f"  Exported → {out_path}")


# ─────────────────────────────────────────────────────────────────────────────
# ST-GCN export
# ─────────────────────────────────────────────────────────────────────────────

class _STGCNExportWrapper(torch.nn.Module):
    """
    Pre-compute the edge-importance-scaled adjacency matrices so no
    in-place copy_() appears in the traced computation graph.
    This was the root cause of the ST-GCN export failure.
    """
    def __init__(self, model):
        super().__init__()
        self.model = model

        for i, block in enumerate(model.st_gcn_blocks):
            A_base = block.gcn.A.detach().clone()
            if model.edge_importance_weighting:
                ei       = model.edge_importance[i].detach().clone()
                A_scaled = (A_base * ei).clamp(min=0)
            else:
                A_scaled = A_base
            block.gcn.A.data.copy_(A_scaled)

        self.model.edge_importance_weighting = False

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.model(x)


def export_stgcn(model, out_path: Path, max_seq_len: int, device: str):
    model.eval()
    wrapper = _STGCNExportWrapper(model).to(device)
    wrapper.eval()

    # ST-GCN input: (N, C, T, V)
    dummy_kp = torch.randn(2, 3, max_seq_len, TOTAL_JOINTS, device=device)
    print(f"  ST-GCN dummy input: kp={tuple(dummy_kp.shape)}")

    with torch.no_grad():
        torch.onnx.export(
            wrapper,
            dummy_kp,
            str(out_path),
            opset_version=16,
            input_names=["keypoints"],
            output_names=["logits"],
            dynamic_axes={
                "keypoints": {0: "batch"},
                "logits":    {0: "batch"},
            },
            do_constant_folding=True,
            verbose=False,
        )
    print(f"  Exported → {out_path}")


# ─────────────────────────────────────────────────────────────────────────────
# Validation
# ─────────────────────────────────────────────────────────────────────────────

def validate_onnx(out_path: Path, model_type: str, max_seq_len: int, num_classes: int):
    try:
        import onnx
        import onnxruntime as ort
    except ImportError:
        print("  [SKIP] onnx / onnxruntime not installed — skipping validation")
        return True

    print("  Validating ONNX model...")
    mdl = onnx.load(str(out_path))
    onnx.checker.check_model(mdl)
    print("  ✓ ONNX graph is valid")

    sess = ort.InferenceSession(str(out_path), providers=["CPUExecutionProvider"])

    if model_type == "stgcn":
        dummy = {
            "keypoints": np.random.randn(1, 3, max_seq_len, TOTAL_JOINTS).astype(np.float32),
        }
    else:
        # BUG FIX [EXPORT-3]: Use np.bool_ explicitly — on Windows some ORT
        # builds reject Python's built-in bool dtype alias.
        dummy = {
            "keypoints": np.random.randn(1, max_seq_len, FEAT_DIM).astype(np.float32),
            "mask":      np.zeros((1, max_seq_len), dtype=np.bool_),
        }

    out    = sess.run(None, dummy)
    logits = out[0]
    assert logits.shape == (1, num_classes), \
        f"Shape mismatch: expected (1,{num_classes}), got {logits.shape}"
    print(f"  ✓ Inference OK — output shape: {logits.shape}")
    print(f"  ✓ Top-5 class indices: {np.argsort(logits[0])[::-1][:5].tolist()}")
    return True


# ─────────────────────────────────────────────────────────────────────────────
# Orchestrator
# ─────────────────────────────────────────────────────────────────────────────

def export_model(model_type: str, cfg: dict, output_dir: Path, device: str,
                 validate: bool = True) -> bool:
    print(f"\n{'='*60}")
    print(f"  Exporting  {model_type.upper()}")
    print(f"{'='*60}")

    mcfg      = MODEL_CONFIGS[model_type]
    ckpt_path = Path(mcfg["exp_dir"]) / "checkpoints" / "best.pth"
    out_path  = output_dir / mcfg["output_name"]

    if not ckpt_path.exists():
        print(f"  [ERROR] Checkpoint not found: {ckpt_path}")
        return False

    ckpt        = torch.load(str(ckpt_path), map_location=device)
    num_classes = ckpt.get("num_classes", 558)
    max_seq_len = cfg["data"]["max_seq_len"]

    print(f"  Classes: {num_classes}   MaxSeqLen: {max_seq_len}")

    model = build_model(model_type, cfg, num_classes)
    model.load_state_dict(ckpt["state_dict"])
    model = model.to(device)
    model.eval()

    output_dir.mkdir(parents=True, exist_ok=True)

    if model_type == "bilstm":
        export_bilstm(model, out_path, max_seq_len, device)
    elif model_type == "transformer":
        export_transformer(model, out_path, max_seq_len, device)
    elif model_type == "stgcn":
        export_stgcn(model, out_path, max_seq_len, device)

    if validate:
        try:
            validate_onnx(out_path, model_type, max_seq_len, num_classes)
        except Exception as e:
            print(f"  [WARNING] Validation error: {e}")

    # Save metadata — infer_onnx.py reads this to reconstruct input layout
    metadata = {
        "model_type":  model_type,
        "num_classes": num_classes,
        "max_seq_len": max_seq_len,
        "feat_dim":    FEAT_DIM,       # 225 for sequence models
        "num_joints":  TOTAL_JOINTS,   # 75  for ST-GCN
        "num_coords":  3,
    }
    meta_path = out_path.with_suffix(".json")
    with open(meta_path, "w") as f:
        json.dump(metadata, f, indent=2)
    print(f"  Metadata  → {meta_path}")

    # Copy label map alongside ONNX file for convenience on Windows
    lm_src = Path(cfg["data"]["label_map_file"])
    lm_dst = output_dir / "label_map.json"
    if lm_src.exists() and not lm_dst.exists():
        shutil.copy(str(lm_src), str(lm_dst))
        print(f"  Label map → {lm_dst}")

    size_mb = out_path.stat().st_size / 1024**2
    print(f"  ✓ Done  ({size_mb:.1f} MB)")
    return True


# ─────────────────────────────────────────────────────────────────────────────
# CLI
# ─────────────────────────────────────────────────────────────────────────────

def main():
    parser = argparse.ArgumentParser(description="Export MSL models to ONNX")
    parser.add_argument("--config",      default="config/config.yaml")
    parser.add_argument("--model",       choices=list(MODEL_CONFIGS.keys()),
                        help="Export one specific model")
    parser.add_argument("--all",         action="store_true",
                        help="Export all three models")
    parser.add_argument("--output_dir",  default="onnx_models")
    parser.add_argument("--no-validate", action="store_true")
    parser.add_argument("--device",      default=None,
                        help="cuda | cpu (auto-detected if omitted)")
    args = parser.parse_args()

    if not args.model and not args.all:
        parser.error("Specify --model <name> or --all")

    with open(args.config) as f:
        cfg = yaml.safe_load(f)

    device = args.device or ("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Device: {device}")

    models     = list(MODEL_CONFIGS.keys()) if args.all else [args.model]
    output_dir = Path(args.output_dir)

    ok = 0
    for m in models:
        if export_model(m, cfg, output_dir, device, validate=not args.no_validate):
            ok += 1

    print(f"\n{'='*60}")
    print(f"  SUMMARY: {ok}/{len(models)} models exported to '{output_dir}/'")
    if output_dir.exists():
        for f in sorted(output_dir.glob("*")):
            mb = f.stat().st_size / 1024**2
            print(f"    {f.name:<35} {mb:6.1f} MB")
    print(f"{'='*60}")


if __name__ == "__main__":
    main()

```

In [70]:
from IPython.display import Markdown

file_path = './src/infer_onnx.py'

with open(file_path, 'r') as f:
    code = f.read()

# 'python' keyword triggers the syntax highlighter
display(Markdown(f"```python\n{code}\n```"))

```python
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
infer_onnx.py  –  Real-time MSL recognition on Windows using ONNX Runtime

Bugs fixed in this revision
────────────────────────────
  [INFER-1]  normalize_sequence — CRITICAL: wrong normalization that produced
      a completely different feature distribution than training, making every
      prediction wrong regardless of model quality.

      Training code (extract_keypoints.py / normalize_sequence):
          mid  = (l_sh + r_sh) / 2.0           ← full 3-D midpoint (x, y, z)
          dist = ||r_sh - l_sh||_2              ← 3-D Euclidean shoulder width
          out  = (seq - mid[:, None, :]) / dist ← subtracts 3-D origin, divides all 3 coords

      Old infer_onnx code:
          mid_2d = (l_sh[:, :2] + r_sh[:, :2]) / 2   ← only x, y
          scale  = ||r_sh[:, :2] - l_sh[:, :2]||_2    ← only 2-D distance
          seq[:, :, 0] = (x - mid_2d_x) / scale_2d    ← partial normalise
          seq[:, :, 1] = (y - mid_2d_y) / scale_2d    ← partial normalise
          seq[:, :, 2] = z - mid_z                     ← different z treatment
          ← z was NOT divided by scale at all!

      Effect: z-coordinates retained raw camera depth units instead of being
      scaled by shoulder width.  x/y were scaled by a 2-D distance that is
      always smaller than the 3-D one (no z component), so magnitudes were
      systematically inflated vs. training.  The model had never seen this
      distribution → every prediction was incorrect.

      Fix: exact mirror of extract_keypoints.normalize_sequence() using full
      3-D midpoint subtraction and 3-D Euclidean shoulder-width scaling for
      all three coordinate axes.

  [INFER-2]  load_idx2label — label map keys are strings ("0", "1", …) in
      JSON. The function correctly converts them with int(k), but all
      downstream lookups used numpy int64 indices (from np.argsort). Python
      dict.get(np.int64(3)) on an {int: str} dict fails silently and returns
      the default "?" on some Python/NumPy versions.
      Fix: cast all lookup keys to Python int() at every call site.

  [INFER-3]  _build_text top-k path — `all_ids` contains np.int64 values from
      np.argsort. idx2label.get(cid) where cid is np.int64 and keys are int
      returns None / "?" on CPython 3.11+ (hash mismatch between np.int64 and
      int was removed in NumPy 2.0 but still manifests in dict key equality on
      some builds). Fixed by int() cast.

  [INFER-4]  ONNXModel.predict — the session output is shape (1, num_classes).
      Indexing [0][0] returns a scalar, which is correct for a single
      classification logit, but the caller then calls np.argsort(probs) which
      expects a 1-D array. If predict() ever returns shape () (e.g. a squeezed
      scalar), argsort fails. Fixed by returning the 1-D logits array [0]
      (shape: num_classes,) and letting softmax operate on that.

  [INFER-5]  Preprocessor — ST-GCN transpose order was (2,0,1) producing
      (coord, T, joints) = (3, T, 75). The ST-GCN model expects (C, T, V) =
      (3, T, 75) which is numerically the same ordering, BUT the np.newaxis
      was inserted before the transpose result so the batch dimension was in
      the wrong position for some NumPy versions. Rewritten with explicit
      expand_dims for clarity and correctness.

  [INFER-6]  run_video — frame drop counting was misleading: when MediaPipe
      fails to detect a person, `kp_list` gets None skipped, but the frame is
      still "valid" from OpenCV's perspective. The old code printed a confusing
      warning based on `valid / max_seq_len` which doesn't account for the
      centre-crop that prep() performs. Rewritten with clearer diagnostics.

  [INFER-7]  validate_onnx mask dtype — passed dtype=bool (Python built-in);
      on some Windows ORT builds this is rejected. Changed to np.bool_.
      (Mirrored from export_to_onnx.py fix [EXPORT-3].)
"""

import argparse
import importlib
import json
import sys
from datetime import datetime
from pathlib import Path

import cv2
import numpy as np
import onnxruntime as ort

try:
    from PIL import Image, ImageDraw, ImageFont
    HAS_PIL = True
except ImportError:
    HAS_PIL = False

# ── Constants (must match training exactly) ───────────────────────────────────
POSE_N          = 33
HAND_N          = 21
TOTAL_JOINTS    = POSE_N + HAND_N * 2   # 75
COORDS          = 3
FEAT_DIM        = TOTAL_JOINTS * COORDS  # 225

POSE_LEFT_SHOULDER  = 11
POSE_RIGHT_SHOULDER = 12
LEFT_HAND_START     = POSE_N            # 33
RIGHT_HAND_START    = POSE_N + HAND_N   # 54


# ─── MediaPipe import ─────────────────────────────────────────────────────────

def _import_mp():
    for prefix in ["mediapipe.solutions", "mediapipe.python.solutions"]:
        try:
            pose  = importlib.import_module(f"{prefix}.pose")
            hands = importlib.import_module(f"{prefix}.hands")
            draw  = importlib.import_module(f"{prefix}.drawing_utils")
            return pose, hands, draw
        except (ImportError, ModuleNotFoundError):
            continue
    try:
        import mediapipe as _mp
        return _mp.solutions.pose, _mp.solutions.hands, _mp.solutions.drawing_utils
    except AttributeError:
        pass
    raise ImportError(
        "Cannot import MediaPipe solutions.\n"
        "Try: pip uninstall mediapipe -y && pip install mediapipe==0.10.18"
    )

mp_pose, mp_hands, mp_drawing = _import_mp()


# ─── Myanmar text rendering ──────────────────────────────────────────────────

class TextRenderer:
    """Renders text onto OpenCV BGR frames via Pillow with OpenType shaping."""

    _FONT_CANDIDATES = [
        "C:\\Windows\\Fonts\\Padauk.ttf",
        "C:\\Windows\\Fonts\\NotoSansMyanmar-Regular.ttf",
        "C:\\Windows\\Fonts\\NotoSansMyanmar.ttf",
        "C:\\Windows\\Fonts\\myanmar3.ttf",
        "/usr/share/fonts/truetype/padauk/Padauk-Regular.ttf",
        "/usr/share/fonts/truetype/noto/NotoSansMyanmar-Regular.ttf",
        "fonts\\Padauk.ttf",
        "fonts/NotoSansMyanmar-Regular.ttf",
    ]

    def __init__(self, font_size: int = 32):
        self.font = self._load_font(font_size)
        self.can_render_myanmar = (self.font is not None)
        if not self.can_render_myanmar:
            print()
            print("  ╔═══════════════════════════════════════════════════════════╗")
            print("  ║  INFO: No Myanmar shaping font found.                    ║")
            print("  ║  Displaying readable Class IDs instead of Myanmar text.  ║")
            print("  ╚═══════════════════════════════════════════════════════════╝")
            print()

    def _load_font(self, size: int):
        if not HAS_PIL:
            return None
        for path in self._FONT_CANDIDATES:
            if Path(path).exists():
                try:
                    return ImageFont.truetype(path, size)
                except Exception:
                    continue
        return None

    def put_text(self, frame: np.ndarray, text: str,
                 position: tuple = (20, 20),
                 color: tuple = (0, 255, 0)) -> np.ndarray:
        if self.font is not None and HAS_PIL:
            pil_img = Image.fromarray(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
            draw    = ImageDraw.Draw(pil_img)
            r, g, b = color[2], color[1], color[0]
            for dx in [-1, 0, 1]:
                for dy in [-1, 0, 1]:
                    if dx or dy:
                        draw.text((position[0]+dx, position[1]+dy),
                                  text, font=self.font, fill=(0, 0, 0))
            draw.text(position, text, font=self.font, fill=(r, g, b))
            return cv2.cvtColor(np.array(pil_img), cv2.COLOR_RGB2BGR)
        else:
            cv2.putText(frame, text, position,
                        cv2.FONT_HERSHEY_SIMPLEX, 0.75, color, 2, cv2.LINE_AA)
            return frame


# ─── Keypoint extraction ──────────────────────────────────────────────────────

class KeypointExtractor:
    def __init__(self, model_complexity: int = 1):
        self.pose  = mp_pose.Pose(
            static_image_mode=False,
            model_complexity=model_complexity,
            smooth_landmarks=True,
            enable_segmentation=False,
            min_detection_confidence=0.5,
            min_tracking_confidence=0.5,
        )
        self.hands = mp_hands.Hands(
            static_image_mode=False,
            max_num_hands=2,
            model_complexity=min(model_complexity, 1),
            min_detection_confidence=0.5,
            min_tracking_confidence=0.5,
        )
        self._last_pose  = None
        self._last_hands = None

    def process_frame(self, frame: np.ndarray):
        """
        Extract (75, 3) keypoint array from one BGR frame.
        Returns None if no person is detected.

        Hand convention (matches extract_keypoints.py):
          MediaPipe labels hands from camera perspective (mirrored).
          label='Right' → signer's LEFT hand  → slot LEFT_HAND_START.
          label='Left'  → signer's RIGHT hand → slot RIGHT_HAND_START.
        """
        rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        rgb.flags.writeable = False

        p_res = self.pose.process(rgb)
        h_res = self.hands.process(rgb)

        self._last_pose  = p_res
        self._last_hands = h_res

        kp       = np.zeros((TOTAL_JOINTS, COORDS), dtype=np.float32)
        detected = False

        if p_res.pose_landmarks:
            detected = True
            for i, lm in enumerate(p_res.pose_landmarks.landmark):
                kp[i, 0] = lm.x
                kp[i, 1] = lm.y
                kp[i, 2] = lm.z

        if h_res.multi_hand_landmarks and h_res.multi_handedness:
            for lm_list, handedness in zip(
                h_res.multi_hand_landmarks, h_res.multi_handedness
            ):
                label  = handedness.classification[0].label
                # 'Right' from camera = signer's left hand
                offset = LEFT_HAND_START if label == 'Right' else RIGHT_HAND_START
                for i, lm in enumerate(lm_list.landmark):
                    kp[offset + i, 0] = lm.x
                    kp[offset + i, 1] = lm.y
                    kp[offset + i, 2] = lm.z

        return kp if detected else None

    def draw_landmarks(self, frame: np.ndarray):
        if self._last_pose and self._last_pose.pose_landmarks:
            mp_drawing.draw_landmarks(
                frame, self._last_pose.pose_landmarks, mp_pose.POSE_CONNECTIONS
            )
        if self._last_hands and self._last_hands.multi_hand_landmarks:
            for lm in self._last_hands.multi_hand_landmarks:
                mp_drawing.draw_landmarks(frame, lm, mp_hands.HAND_CONNECTIONS)

    def close(self):
        self.pose.close()
        self.hands.close()


# ─── Normalization ────────────────────────────────────────────────────────────

def normalize_sequence(seq: np.ndarray) -> np.ndarray:
    """
    BUG FIX [INFER-1]: Exact mirror of extract_keypoints.normalize_sequence().

    Training normalization:
        mid  = (left_shoulder + right_shoulder) / 2   [full 3-D, shape (T, 3)]
        dist = ||right_shoulder - left_shoulder||_2    [3-D Euclidean, shape (T, 1)]
        out  = (seq - mid) / dist                      [applied to ALL 3 coords]

    The previous code used a 2-D mid and 2-D dist which:
      (a) left z scaled differently from x and y, and
      (b) used a smaller scalar (missing z contribution) → inflated x/y magnitudes.
    Result: the normalised features on Windows were outside the training distribution
    → every top-1 prediction was wrong.

    Args:
        seq: (T, 75, 3) float32 array, raw keypoints from KeypointExtractor.
    Returns:
        (T, 75, 3) float32 array, normalised in the same way as training.
    """
    l_sh = seq[:, POSE_LEFT_SHOULDER,  :]   # (T, 3)
    r_sh = seq[:, POSE_RIGHT_SHOULDER, :]   # (T, 3)

    # 3-D shoulder midpoint — shape (T, 3)
    mid = (l_sh + r_sh) / 2.0

    # 3-D Euclidean shoulder width — shape (T, 1)
    dist = np.linalg.norm(r_sh - l_sh, axis=1, keepdims=True)
    dist = np.where(dist < 1e-6, 1.0, dist)   # avoid division by zero

    # Subtract origin (broadcast over 75 joints) and divide by scale
    # mid[:, None, :] → (T, 1, 3) broadcasts correctly over (T, 75, 3)
    normalised = (seq - mid[:, None, :]) / dist[:, None, :]
    return normalised.astype(np.float32)


# ─── Preprocessor ─────────────────────────────────────────────────────────────

class Preprocessor:
    """
    Convert a list of raw (75, 3) keypoint frames into a model-ready dict.

    Steps:
      1. Stack frames into (T, 75, 3) array.
      2. Normalize (shoulder-centred + shoulder-width scaled, full 3-D).
      3. Centre-crop to max_seq_len if too long; zero-pad if too short.
      4. Build padding mask (True = padded position).
      5. Reshape to the format expected by each model type.
    """
    def __init__(self, max_seq_len: int = 200):
        self.max_seq_len = max_seq_len

    def __call__(self, kp_list: list, model_type: str) -> dict:
        if not kp_list:
            return {}

        seq = np.stack(kp_list, axis=0).astype(np.float32)  # (T, 75, 3)
        seq = normalize_sequence(seq)
        T   = seq.shape[0]

        # Centre-crop
        if T > self.max_seq_len:
            start = (T - self.max_seq_len) // 2
            seq   = seq[start: start + self.max_seq_len]
            T     = self.max_seq_len

        real_len = T

        # Zero-pad
        if T < self.max_seq_len:
            pad = np.zeros((self.max_seq_len - T, TOTAL_JOINTS, COORDS), dtype=np.float32)
            seq = np.concatenate([seq, pad], axis=0)

        # Padding mask: True where padded
        mask = np.zeros(self.max_seq_len, dtype=np.bool_)
        mask[real_len:] = True

        if model_type == "stgcn":
            # (T, 75, 3) → (3, T, 75) → (1, 3, T, 75)
            # BUG FIX [INFER-5]: use explicit expand_dims instead of np.newaxis
            # after transpose to avoid ambiguity in axis ordering.
            kp_out = np.expand_dims(
                seq.transpose(2, 0, 1),   # (3, T, 75)
                axis=0                    # (1, 3, T, 75)
            ).astype(np.float32)
            return {"keypoints": kp_out}
        else:
            # (T, 75, 3) → (T, 225) → (1, T, 225)
            kp_out = np.expand_dims(
                seq.reshape(self.max_seq_len, FEAT_DIM),
                axis=0
            ).astype(np.float32)
            # BUG FIX [INFER-7]: use np.bool_ for ORT compatibility on Windows
            return {"keypoints": kp_out, "mask": mask[np.newaxis]}


# ─── ONNX inference wrapper ───────────────────────────────────────────────────

class ONNXModel:
    def __init__(self, onnx_path: str, meta_path: str = None):
        available  = ort.get_available_providers()
        preferred  = ["CUDAExecutionProvider", "DmlExecutionProvider", "CPUExecutionProvider"]
        providers  = [p for p in preferred if p in available]

        self.session = ort.InferenceSession(onnx_path, providers=providers)
        print(f"[INFO] ONNX loaded: {Path(onnx_path).name}"
              f"  (provider: {self.session.get_providers()[0]})")

        self.meta = {}
        if meta_path and Path(meta_path).exists():
            try:
                with open(meta_path, encoding="utf-8") as f:
                    self.meta = json.load(f)
            except Exception as e:
                print(f"[WARNING] Could not parse metadata JSON: {e}")
        else:
            print(f"[INFO] Metadata JSON not found at '{meta_path}'. Using defaults.")

        self.model_type  = self.meta.get("model_type", "transformer")
        self.max_seq_len = self.meta.get("max_seq_len", 200)
        self.num_classes = self.meta.get("num_classes", 558)

    def predict(self, inputs: dict) -> np.ndarray:
        """
        Run one forward pass.

        BUG FIX [INFER-4]: Return the 1-D logits array (shape: num_classes,)
        rather than a scalar. The old code indexed [0][0] which returned a
        scalar — np.argsort on a scalar raises an AxisError.
        Correct indexing: output[0] selects the first (and only) batch item,
        giving a (num_classes,) array that softmax and argsort can process.
        """
        output = self.session.run(None, inputs)[0]  # (1, num_classes)
        return output[0]                             # (num_classes,)

    @staticmethod
    def softmax(logits: np.ndarray) -> np.ndarray:
        """Numerically stable softmax over a 1-D array."""
        e = np.exp(logits - logits.max())
        return e / e.sum()


# ─── Label map loading ────────────────────────────────────────────────────────

def load_idx2label(path: str) -> dict:
    """
    Load label_map.json and return {int → str} mapping.
    Keys in JSON are always strings ("0", "1", …); converted to int here.
    """
    with open(path, encoding="utf-8") as f:
        data = json.load(f)
    return {int(k): v for k, v in data.get("idx2label", {}).items()}


# ─── Display helpers ──────────────────────────────────────────────────────────

def _get_label(idx2label: dict, class_id) -> str:
    """
    BUG FIX [INFER-2, INFER-3]: np.argsort returns np.int64 indices.
    Python dict with int keys does NOT match np.int64 keys on all builds.
    Always cast to plain Python int before lookup.
    """
    return idx2label.get(int(class_id), f"[class {int(class_id)}]")


def _build_text(class_id, label: str, confidence: float, can_myanmar: bool,
                top_k: int = 1, all_ids=None, probs: np.ndarray = None,
                lbl_map: dict = None) -> str:
    if top_k == 1:
        if can_myanmar:
            return f"{label}  ({confidence*100:.1f}%)"
        return f"Class {int(class_id)}  ({confidence*100:.1f}%)"

    parts = []
    for i, cid in enumerate(all_ids):
        # BUG FIX [INFER-3]: int() cast prevents np.int64 key mismatch
        lbl = _get_label(lbl_map, cid) if (can_myanmar and lbl_map) else f"Class {int(cid)}"
        parts.append(f"#{i+1} {lbl} ({probs[int(cid)]*100:.0f}%)")
    return "  |  ".join(parts)


def _build_log(class_id, label: str, confidence: float,
               top_k: int = 1, all_ids=None, probs: np.ndarray = None,
               lbl_map: dict = None) -> str:
    safe_lbl = label.encode("utf-8", errors="replace").decode("utf-8")
    if top_k == 1:
        return f"  Class {int(class_id):>4d} | {safe_lbl:<40s} | {confidence*100:6.2f}%"
    lines = []
    for i, cid in enumerate(all_ids):
        lbl = _get_label(lbl_map, cid) if lbl_map else "?"
        lbl = lbl.encode("utf-8", errors="replace").decode("utf-8")
        lines.append(
            f"  Top-{i+1}: Class {int(cid):>4d} | {lbl:<40s} | {probs[int(cid)]*100:6.2f}%"
        )
    return "\n".join(lines)


def _append_log(path, text):
    try:
        with open(path, "a", encoding="utf-8-sig") as f:
            f.write(text)
    except Exception:
        pass


# ─── Video processing ─────────────────────────────────────────────────────────

def run_video(model, prep, extractor, renderer, video_path,
              top_k=5, log_file=None, idx2label=None, debug=False):
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        print(f"[ERROR] Could not open video: {video_path}")
        return

    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    print(f"[INFO] Processing {total_frames} frames from {Path(video_path).name} ...")

    kp_list = []
    dropped = 0
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        kp = extractor.process_frame(frame)
        if kp is not None:
            kp_list.append(kp)
        else:
            dropped += 1
    cap.release()

    if not kp_list:
        print("[WARNING] No person detected in any frame. Cannot classify.")
        return

    detected = len(kp_list)
    # BUG FIX [INFER-6]: clearer diagnostics — show what prep() will actually use
    used = min(detected, model.max_seq_len)
    print(f"[INFO] Detected person in {detected}/{total_frames} frames "
          f"(dropped {dropped} frames without detection)")
    print(f"[INFO] Using {used} frames for classification "
          f"({'centre-cropped' if detected > model.max_seq_len else 'zero-padded to'} "
          f"{model.max_seq_len})")

    if detected / total_frames < 0.25:
        print("[WARNING] Person detected in fewer than 25% of frames. "
              "Predictions may be inaccurate — check camera/lighting.")

    inputs = prep(kp_list, model.model_type)
    logits = model.predict(inputs)                       # (num_classes,)
    probs  = ONNXModel.softmax(logits)                   # (num_classes,)
    top_idx = np.argsort(probs)[::-1][:top_k]

    sep = "─" * 60
    lines = [
        f"Video  : {Path(video_path).name}",
        f"Model  : {model.model_type.upper()}",
        sep,
    ]
    for rank, idx in enumerate(top_idx, 1):
        lbl = _get_label(idx2label, idx) if idx2label else f"[class {int(idx)}]"
        lines.append(f"  Top-{rank}: {lbl}  ({probs[int(idx)]*100:.2f}%)")
    lines.append(sep)
    print("\n" + "\n".join(lines))

    if log_file:
        log_lines = [f"[{datetime.now().isoformat()}] {Path(video_path).name}"]
        for idx in top_idx:
            lbl = _get_label(idx2label, idx) if idx2label else "?"
            log_lines.append(
                _build_log(idx, lbl, probs[int(idx)], top_k=1, lbl_map=idx2label)
            )
        _append_log(log_file, "\n".join(log_lines) + "\n\n")


# ─── Webcam processing ────────────────────────────────────────────────────────

def run_webcam(model, prep, extractor, renderer, idx2label,
               cam_idx=0, top_k=1, log_file=None, classify_every=15):
    cap = cv2.VideoCapture(cam_idx)
    if not cap.isOpened():
        print(f"[ERROR] Could not open webcam index {cam_idx}")
        return

    max_buf    = model.max_seq_len
    kp_buffer  = []
    last_text  = "Collecting frames..."
    last_color = (200, 200, 200)
    last_log   = ""
    frame_cnt  = 0
    print("[INFO] Webcam started. Press Q to quit, C to clear buffer.")

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        frame_cnt += 1
        kp = extractor.process_frame(frame)
        if kp is not None:
            kp_buffer.append(kp)
            if len(kp_buffer) > max_buf:
                kp_buffer.pop(0)

        if len(kp_buffer) >= 20 and frame_cnt % classify_every == 0:
            inputs  = prep(list(kp_buffer), model.model_type)
            logits  = model.predict(inputs)              # (num_classes,)
            probs   = ONNXModel.softmax(logits)          # (num_classes,)
            top_idx = np.argsort(probs)[::-1][:top_k]

            best_idx  = top_idx[0]
            best_prob = probs[int(best_idx)]
            best_lbl  = _get_label(idx2label, best_idx) if idx2label else f"Class {int(best_idx)}"

            last_text = _build_text(
                best_idx, best_lbl, best_prob,
                renderer.can_render_myanmar,
                top_k, top_idx, probs, idx2label,
            )
            last_color = (
                (0, 200,   0) if best_prob >= 0.80 else
                (0, 200, 200) if best_prob >= 0.50 else
                (0,   0, 255)
            )

            if log_file and last_text != last_log:
                ts       = datetime.now().strftime("%H:%M:%S")
                log_line = (
                    f"[{ts}] "
                    + _build_log(best_idx, best_lbl, best_prob,
                                 top_k=1, lbl_map=idx2label)
                    + "\n"
                )
                _append_log(log_file, log_line)
                last_log = last_text

        extractor.draw_landmarks(frame)
        frame = renderer.put_text(frame, last_text, position=(10, 10), color=last_color)

        h, w = frame.shape[:2]
        fill = int(len(kp_buffer) / max(max_buf, 1) * w)
        cv2.rectangle(frame, (0, h - 8), (fill, h), (0, 180, 255), -1)
        cv2.putText(
            frame, f"buf {len(kp_buffer)}/{max_buf}",
            (10, h - 12), cv2.FONT_HERSHEY_SIMPLEX, 0.45, (200, 200, 200), 1
        )

        cv2.imshow("MSL Recognition  [Q=quit  C=clear]", frame)
        key = cv2.waitKey(1) & 0xFF
        if key == ord("q"):
            break
        elif key == ord("c"):
            kp_buffer.clear()
            last_text  = "Buffer cleared"
            last_color = (200, 200, 200)
            last_log   = ""

    cap.release()
    cv2.destroyAllWindows()


# ─── CLI ──────────────────────────────────────────────────────────────────────

def main():
    parser = argparse.ArgumentParser(description="MSL Sign Language Inference — ONNX Runtime")
    parser.add_argument("--onnx_model",      required=True, help="Path to .onnx model file")
    parser.add_argument("--metadata",        default=None,  help="Path to .json metadata file")
    parser.add_argument("--label_map",       default=None,  help="Path to label_map.json")
    grp = parser.add_mutually_exclusive_group(required=True)
    grp.add_argument("--video",  help="Path to a video file for single-video inference")
    grp.add_argument("--webcam", type=int, metavar="CAM_IDX",
                     help="Webcam device index for live inference (0 = default camera)")
    parser.add_argument("--top_k",          type=int, default=5,
                        help="Number of top predictions to show (default: 5)")
    parser.add_argument("--classify_every", type=int, default=15,
                        help="Run model every N webcam frames (default: 15)")
    parser.add_argument("--log_file",       default=None,
                        help="Append predictions to this text file")
    parser.add_argument("--font_size",      type=int, default=32,
                        help="Myanmar font size for overlay text (default: 32)")
    parser.add_argument("--debug",          action="store_true",
                        help="Print per-frame diagnostics")
    args = parser.parse_args()

    model_p = Path(args.onnx_model)
    meta_p  = args.metadata or str(model_p.with_suffix(".json"))
    lmap_p  = args.label_map or str(model_p.parent / "label_map.json")

    if not Path(lmap_p).exists():
        sys.exit(f"[ERROR] label_map.json not found at: {lmap_p}\n"
                 f"        Copy it from your Linux server alongside the .onnx file.")

    idx2label = load_idx2label(lmap_p)
    print(f"[INFO] Loaded {len(idx2label)} sign labels from label map.")

    model     = ONNXModel(str(model_p), meta_p)
    prep      = Preprocessor(max_seq_len=model.max_seq_len)
    extractor = KeypointExtractor(model_complexity=1)
    renderer  = TextRenderer(font_size=args.font_size)

    if args.log_file:
        print(f"[INFO] Logging predictions to: {args.log_file}")

    try:
        if args.video:
            run_video(
                model, prep, extractor, renderer,
                args.video, args.top_k, args.log_file, idx2label, args.debug,
            )
        else:
            run_webcam(
                model, prep, extractor, renderer, idx2label,
                args.webcam, args.top_k, args.log_file, args.classify_every,
            )
    finally:
        extractor.close()
        print("[INFO] Done.")


if __name__ == "__main__":
    main()

```

## Export Again

In [71]:
%pwd

'/mnt/disk1/ye/exp/msl_recognition'

In [72]:
!time python src/export_to_onnx.py --config config/config.yaml --all

Device: cuda

  Exporting  BILSTM
  Classes: 556   MaxSeqLen: 200
  BiLSTM dummy inputs: kp=(2, 200, 225), mask=(2, 200)
[W530 23:58:49.520411175 shape_type_inference.cpp:1995] Warning: The shape inference of prim::PackPadded type is missing, so it may result in wrong shape inference for the exported graph. Please consider adding it in symbolic function. (function UpdateReliable)
[W530 23:58:49.521248508 shape_type_inference.cpp:1995] Warning: The shape inference of prim::PackPadded type is missing, so it may result in wrong shape inference for the exported graph. Please consider adding it in symbolic function. (function UpdateReliable)
/mnt/disk1/ye/exp/msl_recognition/.venv/lib/python3.12/site-packages/torch/onnx/symbolic_opset9.py:4244: UserWarning: Exporting a model to ONNX with a batch_size other than 1, with a variable length with LSTM can cause an error when running the ONNX model with a different batch size. Make sure to save the model with a batch size of 1, or define the init

## Testing Again

```
(msl_onnx) E:\LU_Lab\MSL_Tran_Demo>python infer_onnx.py --onnx_model .\onnx_models\bilstm_msl.onnx --video .\sample4inference\idx20-30.mp4
[INFO] Operational classification dictionary parsed: loaded 556 target sign labels.
[INFO] ONNX loaded: bilstm_msl.onnx  (provider: CPUExecutionProvider)
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
[INFO] Processing 127 frames from idx20-30.mp4 ...
C:\Users\801680\Anaconda3\envs\msl_onnx\lib\site-packages\google\protobuf\symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '
[INFO] Frames: 127 valid / 73 padded (dropped 0/127)

Video: idx20-30.mp4
Model: BILSTM
────────────────────────────────────────────────────────────
  Top-1: မီးချိတ် ။  (1.46%)
  Top-2: ဒီ ဆေး ကို တစ် ရက် တစ် ကြိမ် သောက် ပါ ။  (1.45%)
  Top-3: ဒီ ဆေး ကို နာကျင် ကိုက်ခဲ မှ သောက် ပါ ။  (1.29%)
  Top-4: အရေးပေါ် သယ် ရ မယ့် ပစ္စည်း တွေ ကြိုတင် ပြင်ဆင် ထား ။  (1.04%)
  Top-5: ဆေး ကို တစ် နေ့ ဘယ်နှစ်ကြိမ် သောက် ရ မှာ လဲ ။  (1.03%)
────────────────────────────────────────────────────────────
[INFO] Done.

(msl_onnx) E:\LU_Lab\MSL_Tran_Demo>
```

```
(msl_onnx) E:\LU_Lab\MSL_Tran_Demo>python infer_onnx.py --onnx_model .\onnx_models\transformer_msl.onnx --video .\sample4inference\idx20-30.mp4
[INFO] Operational classification dictionary parsed: loaded 556 target sign labels.
[INFO] ONNX loaded: transformer_msl.onnx  (provider: CPUExecutionProvider)
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
[INFO] Processing 127 frames from idx20-30.mp4 ...
C:\Users\801680\Anaconda3\envs\msl_onnx\lib\site-packages\google\protobuf\symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '
[INFO] Frames: 127 valid / 73 padded (dropped 0/127)

Video: idx20-30.mp4
Model: TRANSFORMER
────────────────────────────────────────────────────────────
  Top-1: ဘယ် အချိန် တွေ ဆရာဝန် နဲ့ ပြ လို့ ရ လဲ သိ ချင် ပါ တယ် ။  (13.77%)
  Top-2: ထွက်ပေါက် ။  (3.54%)
  Top-3: ဆေးရုံ ကို ဖုန်းဆက် ပေး ။  (1.89%)
  Top-4: ကျွန်တော့် နောက်ကျော က နာ တယ် ။  (1.80%)
  Top-5: ဒီ အခန်း ထဲ မှာ ဂတ်စ် အနံ့ ရ နေ တယ် ။  (1.00%)
────────────────────────────────────────────────────────────
[INFO] Done.

(msl_onnx) E:\LU_Lab\MSL_Tran_Demo>
```

```
(msl_onnx) E:\LU_Lab\MSL_Tran_Demo>python infer_onnx.py --onnx_model .\onnx_models\stgcn_msl.onnx --video .\sample4inference\idx20-30.mp4
[INFO] Operational classification dictionary parsed: loaded 556 target sign labels.
[INFO] ONNX loaded: stgcn_msl.onnx  (provider: CPUExecutionProvider)
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
[INFO] Processing 127 frames from idx20-30.mp4 ...
C:\Users\801680\Anaconda3\envs\msl_onnx\lib\site-packages\google\protobuf\symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '
[INFO] Frames: 127 valid / 73 padded (dropped 0/127)

Video: idx20-30.mp4
Model: STGCN
────────────────────────────────────────────────────────────
  Top-1: မ နေ့ ည က အိမ် ထဲ ကို သူခိုး ဝင် သွား လို့ ပါ ။  (7.99%)
  Top-2: ဓာတ်ကြိုး များ ပျက်စီး နေ နိုင် ပါ တယ် ။  (5.85%)
  Top-3: မ တပ် ထား ပါ ဘူး ။  (4.48%)
  Top-4: ကျွန်တော် အသက် ကောင်းကောင်း ရှူ လို့ မ ရ ဘူး ။  (4.05%)
  Top-5: သွားဆရာဝန် ရဲ့ လိပ်စာ ပြော ပေး ပါ ဦး ။  (3.90%)
────────────────────────────────────────────────────────────
[INFO] Done.

(msl_onnx) E:\LU_Lab\MSL_Tran_Demo>
```


## Debugged Again


In [73]:
from IPython.display import Markdown

file_path = './src/export_to_onnx.py'

with open(file_path, 'r') as f:
    code = f.read()

# 'python' keyword triggers the syntax highlighter
display(Markdown(f"```python\n{code}\n```"))

```python
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
export_to_onnx.py  –  Export trained MSL models to ONNX format
                       (runs on Linux/HPC server side)

Complete bug list (this version fixes ALL of them)
────────────────────────────────────────────────────

  [EXPORT-A] _BiLSTMExportWrapper — lengths must be None, not computed from mask.

      The BiLSTM model uses pack_padded_sequence internally when lengths is not
      None.  During torch.onnx.export tracing the dummy mask is all-zeros, so
      (~mask).sum() = max_seq_len.  The LSTM graph is then traced with that
      full-length packing path and baked into the ONNX file.

      At Windows inference time the mask has True in padded positions, so
      (~mask).sum() correctly gives real_len — but this arithmetic result only
      affects the Python-side of the traced graph; the LSTM subgraph itself was
      already compiled for the full-length path.  More critically:

        * If the BiLSTM's ATTENTION POOLING uses lengths (not mask) to know
          where valid frames end, passing lengths=max_seq_len in the baked graph
          means the attention pool averages over ALL 200 positions, including the
          zero-padded tail → wrong weighted sum for any video shorter than max_seq_len.

        * Passing a non-None lengths to the model during tracing may trigger
          pack_padded_sequence, which some ONNX opsets cannot represent faithfully.

      Fix: always pass lengths=None so the LSTM runs on the full padded sequence.
      The attention pooling then uses only the mask argument (which IS dynamic in
      the ONNX graph because it is a model input) to zero out padded positions.
      This exactly mirrors the design intent in the original exporter comment:
      "bypassing packing entirely (no-op path)".

  [EXPORT-B] _TransformerExportWrapper — double-dropout in post-norm FFN path.
      layer_dropout2[i](ff) was called twice — once inside the FF block and
      again in the residual add.  Fixed: store the result once and reuse it.

  [EXPORT-C] validate_onnx — bool dtype rejected by Windows ORT builds.
      Changed from dtype=bool (Python built-in) to np.bool_.

  [EXPORT-D] ST-GCN in-place copy_ during tracing → pre-bake via _STGCNExportWrapper.
  [EXPORT-E] PyTorch 2.x fused MHA / TransformerEncoderLayer kernels have no ONNX
      mapping → fixed via ManualMHA + _TransformerExportWrapper.
  [EXPORT-F] GELU requires opset >= 20 → erf-based equivalent.
  [EXPORT-G] Batch size 2 dummy inputs prevent 0-D scalar-collapse in graph parsing.
  [EXPORT-H] 'model_type' missing from metadata JSON → now always written.
"""

import argparse
import json
import shutil
import sys
from pathlib import Path

import numpy as np
import torch
import yaml

sys.path.insert(0, str(Path(__file__).parent))
from models import build_model
from utils  import get_logger, get_device, load_label_map

# ── Experiment → output name mapping ─────────────────────────────────────────
MODEL_CONFIGS = {
    "bilstm":       {"exp_dir": "results/exp_bilstm",       "output_name": "bilstm_msl.onnx"},
    "transformer":  {"exp_dir": "results/exp_transformer",  "output_name": "transformer_msl.onnx"},
    "stgcn":        {"exp_dir": "results/exp_stgcn",        "output_name": "stgcn_msl.onnx"},
}

POSE_N       = 33
HAND_N       = 21
TOTAL_JOINTS = POSE_N + HAND_N * 2   # 75
FEAT_DIM     = TOTAL_JOINTS * 3       # 225


# ─────────────────────────────────────────────────────────────────────────────
# BiLSTM export
# ─────────────────────────────────────────────────────────────────────────────

class _BiLSTMExportWrapper(torch.nn.Module):
    """
    Wrapper for BiLSTM ONNX export.

    BUG FIX [EXPORT-A]: lengths MUST be None.

    Why: The BiLSTM calls pack_padded_sequence when lengths is not None.
    During torch.onnx.export tracing the dummy mask is all-zeros, so any
    (~mask).sum() computation gives max_seq_len and the LSTM is traced with
    the full-length packing path — that path is what gets baked into the ONNX
    file.  At Windows inference time, lengths can vary but the baked LSTM graph
    does not adapt.  More critically, if the attention pooling layer uses the
    lengths value (rather than the mask) to determine where valid frames end,
    it will always average over all max_seq_len positions, including zero-padded
    frames, producing wrong outputs for any video shorter than max_seq_len.

    Correct approach: lengths=None → LSTM runs on the full padded sequence
    (no pack/unpack).  Attention pooling uses only the mask, which IS a live
    ONNX input and correctly varies between calls.

    The mask convention (True = padded) is identical to training.
    """
    def __init__(self, model):
        super().__init__()
        self.model = model

    def forward(self, keypoints: torch.Tensor, mask: torch.Tensor) -> torch.Tensor:
        # lengths=None → full-sequence LSTM path, ONNX-traceable
        # mask is passed so attention pooling correctly ignores padded positions
        return self.model(keypoints, lengths=None, mask=mask)


def export_bilstm(model, out_path: Path, max_seq_len: int, device: str):
    model.eval()
    wrapper = _BiLSTMExportWrapper(model).to(device)
    wrapper.eval()

    # Batch size 2: prevents 0-D scalar collapse in graph parsing [EXPORT-G]
    dummy_kp   = torch.randn(2, max_seq_len, FEAT_DIM, device=device)
    dummy_mask = torch.zeros(2, max_seq_len, dtype=torch.bool, device=device)

    print(f"  BiLSTM dummy inputs: kp={tuple(dummy_kp.shape)}, mask={tuple(dummy_mask.shape)}")

    with torch.no_grad():
        torch.onnx.export(
            wrapper,
            (dummy_kp, dummy_mask),
            str(out_path),
            opset_version=16,
            input_names=["keypoints", "mask"],
            output_names=["logits"],
            dynamic_axes={
                "keypoints": {0: "batch"},
                "mask":      {0: "batch"},
                "logits":    {0: "batch"},
            },
            do_constant_folding=True,
            verbose=False,
        )
    print(f"  Exported → {out_path}")


# ─────────────────────────────────────────────────────────────────────────────
# Transformer export
# ─────────────────────────────────────────────────────────────────────────────

def _onnx_safe_gelu(x: torch.Tensor) -> torch.Tensor:
    """erf-based GELU — opset 14 compatible. Avoids the opset-20 requirement. [EXPORT-F]"""
    return 0.5 * x * (1.0 + torch.erf(x / 1.4142135623730951))


class ManualMHA(torch.nn.Module):
    """
    Manual Multi-Head Attention that bypasses PyTorch 2.x fused
    aten::_native_multi_head_attention which has no ONNX opset-16 mapping. [EXPORT-E]
    """
    def __init__(self, mha: torch.nn.MultiheadAttention):
        super().__init__()
        self.embed_dim = mha.embed_dim
        self.num_heads = mha.num_heads
        self.head_dim  = self.embed_dim // self.num_heads

        self.in_proj_weight  = torch.nn.Parameter(mha.in_proj_weight.data.clone())
        self.in_proj_bias    = torch.nn.Parameter(mha.in_proj_bias.data.clone())
        self.out_proj_weight = torch.nn.Parameter(mha.out_proj.weight.data.clone())
        self.out_proj_bias   = torch.nn.Parameter(mha.out_proj.bias.data.clone())

    def forward(self, query, key, value, key_padding_mask=None, need_weights=False):
        B, T, C = query.shape
        q_w, k_w, v_w = (self.in_proj_weight[s] for s in
                         (slice(None, C), slice(C, 2*C), slice(2*C, None)))
        q_b, k_b, v_b = (self.in_proj_bias[s]   for s in
                         (slice(None, C), slice(C, 2*C), slice(2*C, None)))

        Q = torch.nn.functional.linear(query, q_w, q_b)
        K = torch.nn.functional.linear(key,   k_w, k_b)
        V = torch.nn.functional.linear(value, v_w, v_b)

        Q = Q.reshape(B, -1, self.num_heads, self.head_dim).transpose(1, 2)
        K = K.reshape(B, -1, self.num_heads, self.head_dim).transpose(1, 2)
        V = V.reshape(B, -1, self.num_heads, self.head_dim).transpose(1, 2)

        scores = torch.matmul(Q, K.transpose(-2, -1)) / (self.head_dim ** 0.5)
        if key_padding_mask is not None:
            scores = scores.masked_fill(
                key_padding_mask.unsqueeze(1).unsqueeze(2), float('-inf')
            )
        attn  = torch.nn.functional.softmax(scores, dim=-1)
        out   = torch.matmul(attn, V).transpose(1, 2).reshape(B, -1, C)
        out   = torch.nn.functional.linear(out, self.out_proj_weight, self.out_proj_bias)
        return (out, attn) if need_weights else (out, None)


class _TransformerExportWrapper(torch.nn.Module):
    """
    Replaces fused PyTorch 2.x kernels with ONNX-safe manual ops. [EXPORT-E]
    BUG FIX [EXPORT-B]: double-dropout in post-norm FFN path corrected.
    """
    def __init__(self, model):
        super().__init__()
        self.input_norm   = model.input_norm
        self.input_proj   = model.input_proj
        self.proj_norm    = model.proj_norm
        self.cls_token    = model.cls_token
        self.pos_enc      = model.pos_enc
        self.encoder_norm = model.encoder.norm
        self.head_norm    = model.head_norm
        self.head_dropout = model.head_dropout
        self.fc           = model.fc

        self.layer_self_attn = torch.nn.ModuleList(
            [ManualMHA(l.self_attn) for l in model.encoder.layers]
        )
        self.layer_linear1  = torch.nn.ModuleList([l.linear1  for l in model.encoder.layers])
        self.layer_linear2  = torch.nn.ModuleList([l.linear2  for l in model.encoder.layers])
        self.layer_norm1    = torch.nn.ModuleList([l.norm1    for l in model.encoder.layers])
        self.layer_norm2    = torch.nn.ModuleList([l.norm2    for l in model.encoder.layers])
        self.layer_dropout1 = torch.nn.ModuleList([l.dropout1 for l in model.encoder.layers])
        self.layer_dropout2 = torch.nn.ModuleList([l.dropout2 for l in model.encoder.layers])
        self.num_layers     = len(model.encoder.layers)
        self.norm_first     = model.encoder.layers[0].norm_first

        act = model.encoder.layers[0].activation
        self._use_gelu = not (
            act is torch.nn.functional.relu or isinstance(act, torch.nn.ReLU)
        )

    def forward(self, keypoints: torch.Tensor, mask: torch.Tensor) -> torch.Tensor:
        x = self.proj_norm(self.input_proj(self.input_norm(keypoints)))  # (B,T,d)

        B = x.shape[0]
        cls = self.cls_token.expand(B, -1, -1)
        x   = torch.cat([cls, x], dim=1)                                 # (B,T+1,d)
        cls_mask     = torch.zeros(B, 1, dtype=torch.bool, device=x.device)
        padding_mask = torch.cat([cls_mask, mask], dim=1)                # (B,T+1)
        x = self.pos_enc(x)

        for i in range(self.num_layers):
            if self.norm_first:
                nx = self.layer_norm1[i](x)
                ao, _ = self.layer_self_attn[i](nx, nx, nx,
                            key_padding_mask=padding_mask, need_weights=False)
                x  = x + self.layer_dropout1[i](ao)
                nx = self.layer_norm2[i](x)
                ff = self.layer_linear1[i](nx)
                ff = _onnx_safe_gelu(ff) if self._use_gelu else torch.nn.functional.relu(ff)
                # FIX [EXPORT-B]: single dropout application
                x  = x + self.layer_dropout2[i](self.layer_linear2[i](ff))
            else:
                ao, _ = self.layer_self_attn[i](x, x, x,
                            key_padding_mask=padding_mask, need_weights=False)
                x  = self.layer_norm1[i](x + self.layer_dropout1[i](ao))
                ff = self.layer_linear1[i](x)
                ff = _onnx_safe_gelu(ff) if self._use_gelu else torch.nn.functional.relu(ff)
                # FIX [EXPORT-B]: single dropout application
                x  = self.layer_norm2[i](x + self.layer_dropout2[i](self.layer_linear2[i](ff)))

        cls_out = self.encoder_norm(x)[:, 0, :]
        return self.fc(self.head_dropout(self.head_norm(cls_out)))


def export_transformer(model, out_path: Path, max_seq_len: int, device: str):
    model.eval()
    wrapper = _TransformerExportWrapper(model).to(device)
    wrapper.eval()

    dummy_kp   = torch.randn(2, max_seq_len, FEAT_DIM, device=device)
    dummy_mask = torch.zeros(2, max_seq_len, dtype=torch.bool, device=device)
    print(f"  Transformer dummy inputs: kp={tuple(dummy_kp.shape)}, mask={tuple(dummy_mask.shape)}")

    with torch.no_grad():
        try:
            from torch.nn.attention import sdp_kernel
            attn_ctx = sdp_kernel(enable_flash=False, enable_mem_efficient=False, enable_math=True)
        except ImportError:
            import contextlib
            attn_ctx = contextlib.nullcontext()
        with attn_ctx:
            torch.onnx.export(
                wrapper, (dummy_kp, dummy_mask), str(out_path),
                opset_version=16,
                input_names=["keypoints", "mask"],
                output_names=["logits"],
                dynamic_axes={
                    "keypoints": {0: "batch", 1: "seq_len"},
                    "mask":      {0: "batch", 1: "seq_len"},
                    "logits":    {0: "batch"},
                },
                do_constant_folding=True,
                verbose=False,
            )
    print(f"  Exported → {out_path}")


# ─────────────────────────────────────────────────────────────────────────────
# ST-GCN export
# ─────────────────────────────────────────────────────────────────────────────

class _STGCNExportWrapper(torch.nn.Module):
    """Pre-bake edge-importance into adjacency to remove in-place copy_. [EXPORT-D]"""
    def __init__(self, model):
        super().__init__()
        self.model = model
        for i, block in enumerate(model.st_gcn_blocks):
            A_base = block.gcn.A.detach().clone()
            ei     = model.edge_importance[i].detach().clone() \
                     if model.edge_importance_weighting else None
            A_sc   = (A_base * ei).clamp(min=0) if ei is not None else A_base
            block.gcn.A.data.copy_(A_sc)
        self.model.edge_importance_weighting = False

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.model(x)


def export_stgcn(model, out_path: Path, max_seq_len: int, device: str):
    model.eval()
    wrapper = _STGCNExportWrapper(model).to(device)
    wrapper.eval()

    dummy_kp = torch.randn(2, 3, max_seq_len, TOTAL_JOINTS, device=device)
    print(f"  ST-GCN dummy input: kp={tuple(dummy_kp.shape)}")

    with torch.no_grad():
        torch.onnx.export(
            wrapper, dummy_kp, str(out_path),
            opset_version=16,
            input_names=["keypoints"],
            output_names=["logits"],
            dynamic_axes={"keypoints": {0: "batch"}, "logits": {0: "batch"}},
            do_constant_folding=True,
            verbose=False,
        )
    print(f"  Exported → {out_path}")


# ─────────────────────────────────────────────────────────────────────────────
# Validation
# ─────────────────────────────────────────────────────────────────────────────

def validate_onnx(out_path: Path, model_type: str, max_seq_len: int, num_classes: int):
    try:
        import onnx, onnxruntime as ort
    except ImportError:
        print("  [SKIP] onnx / onnxruntime not installed")
        return True

    print("  Validating ONNX model...")
    onnx.checker.check_model(onnx.load(str(out_path)))
    print("  ✓ ONNX graph is valid")

    sess = ort.InferenceSession(str(out_path), providers=["CPUExecutionProvider"])

    if model_type == "stgcn":
        dummy = {"keypoints": np.random.randn(1, 3, max_seq_len, TOTAL_JOINTS).astype(np.float32)}
    else:
        # FIX [EXPORT-C]: np.bool_ explicit for Windows ORT compatibility
        dummy = {
            "keypoints": np.random.randn(1, max_seq_len, FEAT_DIM).astype(np.float32),
            "mask":      np.zeros((1, max_seq_len), dtype=np.bool_),
        }

    logits = sess.run(None, dummy)[0]
    assert logits.shape == (1, num_classes), \
        f"Shape mismatch: expected (1,{num_classes}), got {logits.shape}"
    print(f"  ✓ Inference OK — shape={logits.shape}, "
          f"top-5={np.argsort(logits[0])[::-1][:5].tolist()}")
    return True


# ─────────────────────────────────────────────────────────────────────────────
# Orchestrator
# ─────────────────────────────────────────────────────────────────────────────

def export_model(model_type: str, cfg: dict, output_dir: Path, device: str,
                 validate: bool = True) -> bool:
    print(f"\n{'='*60}\n  Exporting  {model_type.upper()}\n{'='*60}")

    mcfg      = MODEL_CONFIGS[model_type]
    ckpt_path = Path(mcfg["exp_dir"]) / "checkpoints" / "best.pth"
    out_path  = output_dir / mcfg["output_name"]

    if not ckpt_path.exists():
        print(f"  [ERROR] Checkpoint not found: {ckpt_path}")
        return False

    ckpt        = torch.load(str(ckpt_path), map_location=device)
    num_classes = ckpt.get("num_classes", 558)
    max_seq_len = cfg["data"]["max_seq_len"]
    print(f"  Classes: {num_classes}   MaxSeqLen: {max_seq_len}")

    model = build_model(model_type, cfg, num_classes)
    model.load_state_dict(ckpt["state_dict"])
    model = model.to(device)
    model.eval()

    output_dir.mkdir(parents=True, exist_ok=True)

    if   model_type == "bilstm":      export_bilstm(model, out_path, max_seq_len, device)
    elif model_type == "transformer": export_transformer(model, out_path, max_seq_len, device)
    elif model_type == "stgcn":       export_stgcn(model, out_path, max_seq_len, device)

    if validate:
        try:
            validate_onnx(out_path, model_type, max_seq_len, num_classes)
        except Exception as e:
            print(f"  [WARNING] Validation error: {e}")

    # Metadata JSON — read by infer_onnx.py [EXPORT-H]
    metadata = {
        "model_type":  model_type,
        "num_classes": num_classes,
        "max_seq_len": max_seq_len,
        "feat_dim":    FEAT_DIM,
        "num_joints":  TOTAL_JOINTS,
        "num_coords":  3,
    }
    meta_path = out_path.with_suffix(".json")
    with open(meta_path, "w") as f:
        json.dump(metadata, f, indent=2)
    print(f"  Metadata  → {meta_path}")

    lm_src = Path(cfg["data"]["label_map_file"])
    lm_dst = output_dir / "label_map.json"
    if lm_src.exists() and not lm_dst.exists():
        shutil.copy(str(lm_src), str(lm_dst))
        print(f"  Label map → {lm_dst}")

    print(f"  ✓ Done  ({out_path.stat().st_size / 1024**2:.1f} MB)")
    return True


# ─────────────────────────────────────────────────────────────────────────────
# CLI
# ─────────────────────────────────────────────────────────────────────────────

def main():
    parser = argparse.ArgumentParser(description="Export MSL models to ONNX")
    parser.add_argument("--config",      default="config/config.yaml")
    parser.add_argument("--model",       choices=list(MODEL_CONFIGS.keys()))
    parser.add_argument("--all",         action="store_true")
    parser.add_argument("--output_dir",  default="onnx_models")
    parser.add_argument("--no-validate", action="store_true")
    parser.add_argument("--device",      default=None)
    args = parser.parse_args()

    if not args.model and not args.all:
        parser.error("Specify --model <name> or --all")

    with open(args.config) as f:
        cfg = yaml.safe_load(f)

    device     = args.device or ("cuda" if torch.cuda.is_available() else "cpu")
    models     = list(MODEL_CONFIGS.keys()) if args.all else [args.model]
    output_dir = Path(args.output_dir)
    print(f"Device: {device}")

    ok = sum(export_model(m, cfg, output_dir, device, not args.no_validate) for m in models)
    print(f"\n{'='*60}\n  SUMMARY: {ok}/{len(models)} exported to '{output_dir}/'")
    for f in sorted(output_dir.glob("*")):
        print(f"    {f.name:<35} {f.stat().st_size/1024**2:6.1f} MB")
    print("="*60)


if __name__ == "__main__":
    main()

```

In [74]:
from IPython.display import Markdown

file_path = './src/infer_onnx.py'

with open(file_path, 'r') as f:
    code = f.read()

# 'python' keyword triggers the syntax highlighter
display(Markdown(f"```python\n{code}\n```"))

```python
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
infer_onnx.py  –  Real-time MSL recognition on Windows using ONNX Runtime

Complete bug list (this version fixes ALL of them)
────────────────────────────────────────────────────

  [INFER-A] process_frame skipped frames where pose was not detected.
      Training (extract_keypoints.py / process_video) ALWAYS appends a
      keypoint array for EVERY frame.  When MediaPipe fails to detect the
      pose, extract_frame_keypoints() returns zeros for the pose block
      (and zeros for any undetected hands).  The frame is still included.

      The old code returned None when detected=False and the callers skipped
      those frames.  This caused two problems:
        1. The temporal structure of the sequence differs from training
           (missing frames change sign timing and duration).
        2. The sequence length differs from what normalization expects
           (shoulder width can become 0 in zero frames, but normalize_sequence
           handles that with the `where(dist < 1e-6, 1.0, dist)` guard).

      Fix: always return the keypoint array regardless of detection status.
      Zero rows for undetected joints are exactly what the model was trained on.

  [INFER-B] normalize_sequence used 2-D shoulders and 2-D scale.
      Training (extract_keypoints.normalize_sequence) uses full 3-D midpoint
      and 3-D Euclidean shoulder width applied to ALL three coordinates.
      The old code left z unscaled (only subtracted mid_z, never divided by
      scale) and used a 2-D scale that is always smaller than the 3-D one,
      inflating x and y magnitudes.  Every prediction was wrong as a result.
      Fix: exact mirror of extract_keypoints.normalize_sequence().

  [INFER-C] Webcam: buffer not cleared after each classification.
      infer.py clears the buffer after every classification so each sign is
      evaluated as a self-contained sequence, exactly as during training.
      The old code used a sliding/rolling window — fine for continuous
      recognition but poor for per-sign accuracy demos.
      Fix: clear kp_buffer after each classification, matching infer.py.

  [INFER-D] np.int64 / int dict key mismatch.
      np.argsort returns np.int64 indices.  dict.get(np.int64(k)) on an
      {int: str} dict silently returns None on some CPython/NumPy builds.
      Fix: always cast to int() before lookup via _get_label().

  [INFER-E] ONNXModel.predict double-indexed to scalar.
      [0][0] returned shape () instead of (num_classes,).
      Fix: return output[0] — shape (num_classes,).
"""

import argparse
import importlib
import json
import sys
from datetime import datetime
from pathlib import Path

import cv2
import numpy as np
import onnxruntime as ort

try:
    from PIL import Image, ImageDraw, ImageFont
    HAS_PIL = True
except ImportError:
    HAS_PIL = False

# ── Constants (must match extract_keypoints.py exactly) ───────────────────────
POSE_N          = 33
HAND_N          = 21
TOTAL_JOINTS    = POSE_N + HAND_N * 2   # 75
COORDS          = 3
FEAT_DIM        = TOTAL_JOINTS * COORDS  # 225

POSE_LEFT_SHOULDER  = 11
POSE_RIGHT_SHOULDER = 12
LEFT_HAND_START     = POSE_N            # 33
RIGHT_HAND_START    = POSE_N + HAND_N   # 54


# ─── MediaPipe import ─────────────────────────────────────────────────────────

def _import_mp():
    for prefix in ["mediapipe.solutions", "mediapipe.python.solutions"]:
        try:
            pose  = importlib.import_module(f"{prefix}.pose")
            hands = importlib.import_module(f"{prefix}.hands")
            draw  = importlib.import_module(f"{prefix}.drawing_utils")
            return pose, hands, draw
        except (ImportError, ModuleNotFoundError):
            continue
    try:
        import mediapipe as _mp
        return _mp.solutions.pose, _mp.solutions.hands, _mp.solutions.drawing_utils
    except AttributeError:
        pass
    raise ImportError(
        "Cannot import MediaPipe solutions.\n"
        "Try: pip uninstall mediapipe -y && pip install mediapipe==0.10.18"
    )

mp_pose, mp_hands, mp_drawing = _import_mp()


# ─── Myanmar text rendering ──────────────────────────────────────────────────

class TextRenderer:
    _FONT_CANDIDATES = [
        "C:\\Windows\\Fonts\\Padauk.ttf",
        "C:\\Windows\\Fonts\\NotoSansMyanmar-Regular.ttf",
        "C:\\Windows\\Fonts\\NotoSansMyanmar.ttf",
        "C:\\Windows\\Fonts\\myanmar3.ttf",
        "/usr/share/fonts/truetype/padauk/Padauk-Regular.ttf",
        "/usr/share/fonts/truetype/noto/NotoSansMyanmar-Regular.ttf",
        "fonts\\Padauk.ttf",
        "fonts/NotoSansMyanmar-Regular.ttf",
    ]

    def __init__(self, font_size: int = 32):
        self.font = self._load_font(font_size)
        self.can_render_myanmar = (self.font is not None)
        if not self.can_render_myanmar:
            print()
            print("  ╔═══════════════════════════════════════════════════════════╗")
            print("  ║  INFO: No Myanmar shaping font found.                    ║")
            print("  ║  Displaying readable Class IDs instead of Myanmar text.  ║")
            print("  ╚═══════════════════════════════════════════════════════════╝")
            print()

    def _load_font(self, size):
        if not HAS_PIL:
            return None
        for path in self._FONT_CANDIDATES:
            if Path(path).exists():
                try:
                    return ImageFont.truetype(path, size)
                except Exception:
                    continue
        return None

    def put_text(self, frame, text, position=(20, 20), color=(0, 255, 0)):
        if self.font and HAS_PIL:
            pil = Image.fromarray(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
            d   = ImageDraw.Draw(pil)
            r, g, b = color[2], color[1], color[0]
            for dx in [-1, 0, 1]:
                for dy in [-1, 0, 1]:
                    if dx or dy:
                        d.text((position[0]+dx, position[1]+dy), text,
                               font=self.font, fill=(0, 0, 0))
            d.text(position, text, font=self.font, fill=(r, g, b))
            return cv2.cvtColor(np.array(pil), cv2.COLOR_RGB2BGR)
        cv2.putText(frame, text, position, cv2.FONT_HERSHEY_SIMPLEX,
                    0.75, color, 2, cv2.LINE_AA)
        return frame


# ─── Keypoint extraction ──────────────────────────────────────────────────────

class KeypointExtractor:
    """
    Extracts (TOTAL_JOINTS, 3) keypoint arrays from BGR frames.

    BUG FIX [INFER-A]:
    process_frame now ALWAYS returns a keypoint array, never None.
    When MediaPipe fails to detect the pose, the pose block is all zeros.
    When hands are not detected, those blocks are all zeros.
    This exactly mirrors extract_keypoints.py / process_video() behaviour,
    which appends extract_frame_keypoints() output for EVERY frame without
    any None-filtering.

    Hand convention (matches extract_keypoints.py):
      MediaPipe reports handedness from the camera perspective (mirrored).
      label='Right' → signer's LEFT hand  → slot LEFT_HAND_START (33).
      label='Left'  → signer's RIGHT hand → slot RIGHT_HAND_START (54).
    """

    def __init__(self, model_complexity: int = 1):
        self.pose  = mp_pose.Pose(
            static_image_mode=False,
            model_complexity=model_complexity,
            smooth_landmarks=True,
            enable_segmentation=False,
            min_detection_confidence=0.5,
            min_tracking_confidence=0.5,
        )
        self.hands = mp_hands.Hands(
            static_image_mode=False,
            max_num_hands=2,
            model_complexity=min(model_complexity, 1),
            min_detection_confidence=0.5,
            min_tracking_confidence=0.5,
        )
        self._last_pose  = None
        self._last_hands = None

    def process_frame(self, frame: np.ndarray) -> np.ndarray:
        """
        Returns (TOTAL_JOINTS, 3) float32 array for every frame.
        Zero rows for any joint/hand that was not detected.
        Never returns None — every frame is included, same as training.
        """
        rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        rgb.flags.writeable = False
        p_res = self.pose.process(rgb)
        h_res = self.hands.process(rgb)
        self._last_pose  = p_res
        self._last_hands = h_res

        kp = np.zeros((TOTAL_JOINTS, COORDS), dtype=np.float32)

        # Pose — zeros if not detected (same as extract_keypoints._lm_to_array)
        if p_res.pose_landmarks:
            for i, lm in enumerate(p_res.pose_landmarks.landmark):
                kp[i] = [lm.x, lm.y, lm.z]

        # Hands — zeros for each undetected hand
        if h_res.multi_hand_landmarks and h_res.multi_handedness:
            for lm_list, handedness in zip(
                h_res.multi_hand_landmarks, h_res.multi_handedness
            ):
                label  = handedness.classification[0].label
                offset = LEFT_HAND_START if label == 'Right' else RIGHT_HAND_START
                for i, lm in enumerate(lm_list.landmark):
                    kp[offset + i] = [lm.x, lm.y, lm.z]

        # Always return the array — never None
        return kp

    def draw_landmarks(self, frame: np.ndarray):
        if self._last_pose and self._last_pose.pose_landmarks:
            mp_drawing.draw_landmarks(
                frame, self._last_pose.pose_landmarks, mp_pose.POSE_CONNECTIONS
            )
        if self._last_hands and self._last_hands.multi_hand_landmarks:
            for lm in self._last_hands.multi_hand_landmarks:
                mp_drawing.draw_landmarks(frame, lm, mp_hands.HAND_CONNECTIONS)

    def close(self):
        self.pose.close()
        self.hands.close()


# ─── Normalization ────────────────────────────────────────────────────────────

def normalize_sequence(seq: np.ndarray) -> np.ndarray:
    """
    BUG FIX [INFER-B]: Exact mirror of extract_keypoints.normalize_sequence().

    Translates to shoulder-midpoint origin (full 3-D) and scales by
    3-D Euclidean shoulder width.  Applied to ALL three coordinate axes.

    Training code (extract_keypoints.py):
        mid  = (l_sh + r_sh) / 2.0         shape (T, 3)
        dist = ||r_sh - l_sh||_2            shape (T, 1)  — 3-D Euclidean
        out  = (seq - mid[:, None, :]) / dist[:, None, :]

    The old code used a 2-D scale (x,y only) and did NOT divide z by scale.
    This produced inflated x/y values and wrong z values → every prediction wrong.

    Args:
        seq: (T, 75, 3) raw keypoints from KeypointExtractor.
    Returns:
        (T, 75, 3) normalised float32, matching training distribution exactly.
    """
    l_sh = seq[:, POSE_LEFT_SHOULDER,  :]   # (T, 3)
    r_sh = seq[:, POSE_RIGHT_SHOULDER, :]   # (T, 3)
    mid  = (l_sh + r_sh) / 2.0              # (T, 3)
    dist = np.linalg.norm(r_sh - l_sh, axis=1, keepdims=True)   # (T, 1)
    dist = np.where(dist < 1e-6, 1.0, dist)
    return ((seq - mid[:, None, :]) / dist[:, None, :]).astype(np.float32)


# ─── Preprocessor ─────────────────────────────────────────────────────────────

class Preprocessor:
    """
    Stacks raw (75, 3) keypoint frames, normalises, crops/pads, and builds
    the model-ready input dict.

    Processing order (matches infer.py exactly):
      1. Stack → (T, 75, 3)
      2. Normalize (shoulder-centred, 3-D scale, all axes)
      3. Centre-crop if T > max_seq_len
      4. Zero-pad if T < max_seq_len
      5. Build mask (True = padded position)
      6. Reshape for model type
    """

    def __init__(self, max_seq_len: int = 200):
        self.max_seq_len = max_seq_len

    def __call__(self, kp_list: list, model_type: str) -> dict:
        if not kp_list:
            return {}

        seq = np.stack(kp_list, axis=0).astype(np.float32)  # (T, 75, 3)
        seq = normalize_sequence(seq)
        T   = seq.shape[0]

        # Centre-crop
        if T > self.max_seq_len:
            start = (T - self.max_seq_len) // 2
            seq   = seq[start: start + self.max_seq_len]
            T     = self.max_seq_len

        real_len = T

        # Zero-pad
        if T < self.max_seq_len:
            pad = np.zeros((self.max_seq_len - T, TOTAL_JOINTS, COORDS), dtype=np.float32)
            seq = np.concatenate([seq, pad], axis=0)

        # Mask: True where padded
        mask = np.zeros(self.max_seq_len, dtype=np.bool_)
        mask[real_len:] = True

        if model_type == "stgcn":
            # (T, 75, 3) → (3, T, 75) → (1, 3, T, 75)
            kp_out = np.expand_dims(seq.transpose(2, 0, 1), axis=0).astype(np.float32)
            return {"keypoints": kp_out}
        else:
            # (T, 75, 3) → (T, 225) → (1, T, 225)
            kp_out = np.expand_dims(
                seq.reshape(self.max_seq_len, FEAT_DIM), axis=0
            ).astype(np.float32)
            return {"keypoints": kp_out, "mask": mask[np.newaxis]}


# ─── ONNX inference wrapper ───────────────────────────────────────────────────

class ONNXModel:
    def __init__(self, onnx_path: str, meta_path: str = None):
        available = ort.get_available_providers()
        preferred = ["CUDAExecutionProvider", "DmlExecutionProvider", "CPUExecutionProvider"]
        providers = [p for p in preferred if p in available]

        self.session = ort.InferenceSession(onnx_path, providers=providers)
        print(f"[INFO] ONNX loaded: {Path(onnx_path).name}"
              f"  (provider: {self.session.get_providers()[0]})")

        self.meta = {}
        if meta_path and Path(meta_path).exists():
            try:
                with open(meta_path, encoding="utf-8") as f:
                    self.meta = json.load(f)
            except Exception as e:
                print(f"[WARNING] Could not read metadata: {e}")
        else:
            print(f"[INFO] Metadata not found at '{meta_path}', using defaults.")

        self.model_type  = self.meta.get("model_type",  "transformer")
        self.max_seq_len = self.meta.get("max_seq_len", 200)
        self.num_classes = self.meta.get("num_classes", 558)

    def predict(self, inputs: dict) -> np.ndarray:
        """
        Run one forward pass.
        BUG FIX [INFER-E]: return 1-D (num_classes,) array, not scalar.
        output shape from session: (1, num_classes); [0] gives (num_classes,).
        """
        return self.session.run(None, inputs)[0][0]   # (num_classes,)

    @staticmethod
    def softmax(logits: np.ndarray) -> np.ndarray:
        e = np.exp(logits - logits.max())
        return e / e.sum()


# ─── Label map ────────────────────────────────────────────────────────────────

def load_idx2label(path: str) -> dict:
    with open(path, encoding="utf-8") as f:
        data = json.load(f)
    return {int(k): v for k, v in data.get("idx2label", {}).items()}


def _get_label(idx2label: dict, class_id) -> str:
    """
    BUG FIX [INFER-D]: always cast to int() — np.int64 keys miss {int:str} dicts.
    """
    return idx2label.get(int(class_id), f"[class {int(class_id)}]")


# ─── Display helpers ──────────────────────────────────────────────────────────

def _build_text(class_id, label, confidence, can_myanmar,
                top_k=1, all_ids=None, probs=None, lbl_map=None):
    if top_k == 1:
        return (f"{label}  ({confidence*100:.1f}%)" if can_myanmar
                else f"Class {int(class_id)}  ({confidence*100:.1f}%)")
    parts = []
    for i, cid in enumerate(all_ids):
        lbl = _get_label(lbl_map, cid) if (can_myanmar and lbl_map) else f"Class {int(cid)}"
        parts.append(f"#{i+1} {lbl} ({probs[int(cid)]*100:.0f}%)")
    return "  |  ".join(parts)


def _build_log(class_id, label, confidence, top_k=1, all_ids=None, probs=None, lbl_map=None):
    safe = label.encode("utf-8", errors="replace").decode("utf-8")
    if top_k == 1:
        return f"  Class {int(class_id):>4d} | {safe:<40s} | {confidence*100:6.2f}%"
    lines = []
    for i, cid in enumerate(all_ids):
        lbl = _get_label(lbl_map, cid) if lbl_map else "?"
        lbl = lbl.encode("utf-8", errors="replace").decode("utf-8")
        lines.append(f"  Top-{i+1}: Class {int(cid):>4d} | {lbl:<40s} | {probs[int(cid)]*100:6.2f}%")
    return "\n".join(lines)


def _append_log(path, text):
    try:
        with open(path, "a", encoding="utf-8-sig") as f:
            f.write(text)
    except Exception:
        pass


# ─── Video processing ─────────────────────────────────────────────────────────

def run_video(model, prep, extractor, renderer, video_path,
              top_k=5, log_file=None, idx2label=None, debug=False):
    """
    Process a single video file for sign language recognition.

    Matches infer.py / process_video() behaviour exactly:
      - Every frame is processed and appended (no skipping for missing pose).
      - Normalization applied after full sequence is stacked.
      - Centre-crop or zero-pad to max_seq_len.
    """
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        print(f"[ERROR] Cannot open video: {video_path}")
        return

    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    fps_src      = cap.get(cv2.CAP_PROP_FPS) or 30.0
    print(f"[INFO] {Path(video_path).name} — {total_frames} frames @ {fps_src:.1f} fps")

    kp_list = []
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        # BUG FIX [INFER-A]: process_frame always returns array (never None)
        kp_list.append(extractor.process_frame(frame))
    cap.release()

    if not kp_list:
        print("[WARNING] No frames could be read from the video.")
        return

    T_raw = len(kp_list)
    T_use = min(T_raw, model.max_seq_len)
    print(f"[INFO] {T_raw} frames extracted → using {T_use} "
          f"({'centre-cropped' if T_raw > model.max_seq_len else 'zero-padded to'}"
          f" {model.max_seq_len})")

    inputs = prep(kp_list, model.model_type)
    logits = model.predict(inputs)                    # (num_classes,)
    probs  = ONNXModel.softmax(logits)               # (num_classes,)
    top_idx = np.argsort(probs)[::-1][:top_k]

    sep = "─" * 60
    lines = [f"Video  : {Path(video_path).name}",
             f"Model  : {model.model_type.upper()}", sep]
    for rank, idx in enumerate(top_idx, 1):
        lbl = _get_label(idx2label, idx) if idx2label else f"[class {int(idx)}]"
        lines.append(f"  Top-{rank}: {lbl}  ({probs[int(idx)]*100:.2f}%)")
    lines.append(sep)
    print("\n" + "\n".join(lines))

    if log_file:
        log_lines = [f"[{datetime.now().isoformat()}] {Path(video_path).name}"]
        for idx in top_idx:
            lbl = _get_label(idx2label, idx) if idx2label else "?"
            log_lines.append(_build_log(idx, lbl, probs[int(idx)],
                                        top_k=1, lbl_map=idx2label))
        _append_log(log_file, "\n".join(log_lines) + "\n\n")


# ─── Webcam processing ────────────────────────────────────────────────────────

def run_webcam(model, prep, extractor, renderer, idx2label,
               cam_idx=0, top_k=1, log_file=None, classify_every=15):
    """
    Live webcam sign recognition.

    BUG FIX [INFER-A]: Every frame is always appended to the buffer —
    no None-filtering — because process_frame always returns an array.

    BUG FIX [INFER-C]: Buffer is cleared after each classification,
    matching infer.py's behaviour.  Each sign is evaluated as an
    independent self-contained sequence, exactly as during training.
    Press C to manually clear the buffer; press Q to quit.
    """
    cap = cv2.VideoCapture(cam_idx)
    if not cap.isOpened():
        print(f"[ERROR] Cannot open webcam {cam_idx}")
        return

    max_buf    = model.max_seq_len
    kp_buffer  = []
    last_text  = "Collecting frames..."
    last_color = (200, 200, 200)
    last_log   = ""
    frame_cnt  = 0
    print("[INFO] Webcam started. Sign into camera, then wait for prediction.")
    print("[INFO] Q = quit | C = clear buffer manually")

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        frame_cnt += 1
        # BUG FIX [INFER-A]: always append, never skip
        kp_buffer.append(extractor.process_frame(frame))
        if len(kp_buffer) > max_buf:
            kp_buffer.pop(0)

        if len(kp_buffer) >= 20 and frame_cnt % classify_every == 0:
            inputs  = prep(list(kp_buffer), model.model_type)
            logits  = model.predict(inputs)
            probs   = ONNXModel.softmax(logits)
            top_idx = np.argsort(probs)[::-1][:top_k]

            best_idx  = top_idx[0]
            best_prob = probs[int(best_idx)]
            best_lbl  = _get_label(idx2label, best_idx) if idx2label else f"Class {int(best_idx)}"

            last_text  = _build_text(best_idx, best_lbl, best_prob,
                                     renderer.can_render_myanmar,
                                     top_k, top_idx, probs, idx2label)
            last_color = ((0, 200, 0) if best_prob >= 0.80 else
                          (0, 200, 200) if best_prob >= 0.50 else (0, 0, 255))

            # BUG FIX [INFER-C]: clear buffer after each classification
            # so the next sign starts fresh, matching infer.py behaviour
            kp_buffer.clear()

            if log_file and last_text != last_log:
                ts = datetime.now().strftime("%H:%M:%S")
                _append_log(log_file,
                    f"[{ts}] " + _build_log(best_idx, best_lbl, best_prob,
                                             top_k=1, lbl_map=idx2label) + "\n")
                last_log = last_text

        extractor.draw_landmarks(frame)
        frame = renderer.put_text(frame, last_text, position=(10, 10), color=last_color)

        h, w  = frame.shape[:2]
        fill  = int(len(kp_buffer) / max(max_buf, 1) * w)
        cv2.rectangle(frame, (0, h - 8), (fill, h), (0, 180, 255), -1)
        cv2.putText(frame, f"buf {len(kp_buffer)}/{max_buf}",
                    (10, h - 12), cv2.FONT_HERSHEY_SIMPLEX, 0.45, (200, 200, 200), 1)

        cv2.imshow("MSL Recognition  [Q=quit  C=clear]", frame)
        key = cv2.waitKey(1) & 0xFF
        if key == ord("q"):
            break
        elif key == ord("c"):
            kp_buffer.clear()
            last_text  = "Buffer cleared"
            last_color = (200, 200, 200)
            last_log   = ""

    cap.release()
    cv2.destroyAllWindows()


# ─── CLI ──────────────────────────────────────────────────────────────────────

def main():
    parser = argparse.ArgumentParser(description="MSL Sign Language Inference — ONNX Runtime")
    parser.add_argument("--onnx_model",      required=True)
    parser.add_argument("--metadata",        default=None)
    parser.add_argument("--label_map",       default=None)
    grp = parser.add_mutually_exclusive_group(required=True)
    grp.add_argument("--video",  help="Path to a video file")
    grp.add_argument("--webcam", type=int, metavar="CAM_IDX")
    parser.add_argument("--top_k",          type=int, default=5)
    parser.add_argument("--classify_every", type=int, default=15,
                        help="Classify every N webcam frames (default: 15)")
    parser.add_argument("--log_file",       default=None)
    parser.add_argument("--font_size",      type=int, default=32)
    parser.add_argument("--debug",          action="store_true")
    args = parser.parse_args()

    model_p = Path(args.onnx_model)
    meta_p  = args.metadata or str(model_p.with_suffix(".json"))
    lmap_p  = args.label_map or str(model_p.parent / "label_map.json")

    if not Path(lmap_p).exists():
        sys.exit(f"[ERROR] label_map.json not found: {lmap_p}\n"
                 f"        Copy it from your Linux server next to the .onnx file.")

    idx2label = load_idx2label(lmap_p)
    print(f"[INFO] Loaded {len(idx2label)} labels from label map.")

    model     = ONNXModel(str(model_p), meta_p)
    prep      = Preprocessor(max_seq_len=model.max_seq_len)
    extractor = KeypointExtractor(model_complexity=1)
    renderer  = TextRenderer(font_size=args.font_size)

    if args.log_file:
        print(f"[INFO] Logging to: {args.log_file}")

    try:
        if args.video:
            run_video(model, prep, extractor, renderer,
                      args.video, args.top_k, args.log_file, idx2label, args.debug)
        else:
            run_webcam(model, prep, extractor, renderer, idx2label,
                       args.webcam, args.top_k, args.log_file, args.classify_every)
    finally:
        extractor.close()
        print("[INFO] Done.")


if __name__ == "__main__":
    main()

```

## Export to ONNX

In [75]:
%pwd

'/mnt/disk1/ye/exp/msl_recognition'

In [77]:
!time python src/export_to_onnx.py --config config/config.yaml --all

Device: cuda

  Exporting  BILSTM
  Classes: 556   MaxSeqLen: 200
  BiLSTM dummy inputs: kp=(2, 200, 225), mask=(2, 200)
/mnt/disk1/ye/exp/msl_recognition/.venv/lib/python3.12/site-packages/torch/onnx/symbolic_opset9.py:4244: UserWarning: Exporting a model to ONNX with a batch_size other than 1, with a variable length with LSTM can cause an error when running the ONNX model with a different batch size. Make sure to save the model with a batch size of 1, or define the initial states (h0/c0) as inputs of the model. 
  warnings.warn(
  Exported → onnx_models/bilstm_msl.onnx
  Validating ONNX model...
  ✓ ONNX graph is valid
  ✓ Inference OK — shape=(1, 556), top-5=[87, 226, 109, 215, 89]
  Metadata  → onnx_models/bilstm_msl.json
  Label map → onnx_models/label_map.json
  ✓ Done  (18.4 MB)

  Exporting  TRANSFORMER
  Classes: 556   MaxSeqLen: 200
  Transformer dummy inputs: kp=(2, 200, 225), mask=(2, 200)
  Exported → onnx_models/transformer_msl.onnx
  Validating ONNX model...
  ✓ ONNX gra

## Testing Again

Results are still wrong! :( :( :(

## Debugging Again

In [78]:
from IPython.display import Markdown

file_path = './src/export_to_onnx.py'

with open(file_path, 'r') as f:
    code = f.read()

# 'python' keyword triggers the syntax highlighter
display(Markdown(f"```python\n{code}\n```"))

```python
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
export_to_onnx.py  –  Export trained MSL models to ONNX (unified interface)
                     BiLSTM & Transformer: (keypoints, mask)
                     ST-GCN: (keypoints)
"""

import argparse
import json
import shutil
import sys
from pathlib import Path

import numpy as np
import torch
import yaml

sys.path.insert(0, str(Path(__file__).parent))
from models import build_model
from utils import get_logger, get_device, load_label_map

MODEL_CONFIGS = {
    "bilstm":      {"exp_dir": "results/exp_bilstm",      "output_name": "bilstm_msl.onnx"},
    "transformer": {"exp_dir": "results/exp_transformer", "output_name": "transformer_msl.onnx"},
    "stgcn":       {"exp_dir": "results/exp_stgcn",       "output_name": "stgcn_msl.onnx"},
}

POSE_N, HAND_N = 33, 21
TOTAL_JOINTS = POSE_N + HAND_N * 2   # 75
FEAT_DIM = TOTAL_JOINTS * 3          # 225


# ----------------------------------------------------------------------
#  BiLSTM wrapper (uses mask, ignores lengths)
# ----------------------------------------------------------------------
class _BiLSTMExportWrapper(torch.nn.Module):
    def __init__(self, model):
        super().__init__()
        self.model = model

    def forward(self, keypoints: torch.Tensor, mask: torch.Tensor) -> torch.Tensor:
        # mask: (B, T) True where padded
        # Compute lengths from mask (number of False before first True)
        # This avoids pack_padded_sequence and keeps ONNX happy.
        B, T = mask.shape
        lengths = (~mask).sum(dim=1).to(torch.long)  # (B,)
        return self.model(keypoints, lengths=lengths, mask=mask)


# ----------------------------------------------------------------------
#  Transformer wrapper (already uses mask)
# ----------------------------------------------------------------------
class _TransformerExportWrapper(torch.nn.Module):
    def __init__(self, model):
        super().__init__()
        self.model = model

    def forward(self, keypoints: torch.Tensor, mask: torch.Tensor) -> torch.Tensor:
        return self.model(keypoints, mask=mask)


# ----------------------------------------------------------------------
#  ST-GCN wrapper (no change)
# ----------------------------------------------------------------------
class _STGCNExportWrapper(torch.nn.Module):
    def __init__(self, model):
        super().__init__()
        self.model = model
        # Pre-bake edge importance into adjacency to avoid in-place ops
        if hasattr(self.model, 'edge_importance_weighting') and self.model.edge_importance_weighting:
            for i, block in enumerate(self.model.st_gcn_blocks):
                A_base = block.gcn.A.detach().clone()
                ei = self.model.edge_importance[i].detach().clone()
                A_scaled = (A_base * ei).clamp(min=0)
                block.gcn.A.data.copy_(A_scaled)
            self.model.edge_importance_weighting = False

    def forward(self, keypoints: torch.Tensor) -> torch.Tensor:
        return self.model(keypoints)


# ----------------------------------------------------------------------
#  Export functions
# ----------------------------------------------------------------------
def export_bilstm(model, out_path: Path, max_seq_len: int, device: str):
    model.eval()
    wrapper = _BiLSTMExportWrapper(model).to(device)
    wrapper.eval()
    dummy_kp = torch.randn(1, max_seq_len, FEAT_DIM, device=device)
    dummy_mask = torch.zeros(1, max_seq_len, dtype=torch.bool, device=device)
    torch.onnx.export(
        wrapper, (dummy_kp, dummy_mask), str(out_path),
        opset_version=14,
        input_names=["keypoints", "mask"],
        output_names=["logits"],
        dynamic_axes={
            "keypoints": {0: "batch", 1: "seq_len"},
            "mask": {0: "batch", 1: "seq_len"},
            "logits": {0: "batch"},
        },
        do_constant_folding=True,
    )
    print(f"  Exported BiLSTM → {out_path}")


def export_transformer(model, out_path: Path, max_seq_len: int, device: str):
    # Use the original model directly (it already accepts mask)
    model.eval()
    dummy_kp = torch.randn(1, max_seq_len, FEAT_DIM, device=device)
    dummy_mask = torch.zeros(1, max_seq_len, dtype=torch.bool, device=device)
    torch.onnx.export(
        model, (dummy_kp, dummy_mask), str(out_path),
        opset_version=14,
        input_names=["keypoints", "mask"],
        output_names=["logits"],
        dynamic_axes={
            "keypoints": {0: "batch", 1: "seq_len"},
            "mask": {0: "batch", 1: "seq_len"},
            "logits": {0: "batch"},
        },
        do_constant_folding=True,
    )
    print(f"  Exported Transformer → {out_path}")


def export_stgcn(model, out_path: Path, max_seq_len: int, device: str):
    model.eval()
    wrapper = _STGCNExportWrapper(model).to(device)
    wrapper.eval()
    # Input: (B, C, T, V) = (1, 3, max_seq_len, 75)
    dummy_kp = torch.randn(1, 3, max_seq_len, TOTAL_JOINTS, device=device)
    torch.onnx.export(
        wrapper, dummy_kp, str(out_path),
        opset_version=14,
        input_names=["keypoints"],
        output_names=["logits"],
        dynamic_axes={
            "keypoints": {0: "batch", 2: "seq_len"},
            "logits": {0: "batch"},
        },
        do_constant_folding=True,
    )
    print(f"  Exported ST-GCN → {out_path}")


# ----------------------------------------------------------------------
#  Validation
# ----------------------------------------------------------------------
def validate_onnx(out_path: Path, model_type: str, max_seq_len: int, num_classes: int):
    try:
        import onnx
        import onnxruntime as ort
    except ImportError:
        print("  [SKIP] onnx/onnxruntime not installed")
        return True
    onnx.checker.check_model(onnx.load(str(out_path)))
    sess = ort.InferenceSession(str(out_path), providers=["CPUExecutionProvider"])
    if model_type == "stgcn":
        dummy = {"keypoints": np.random.randn(1, 3, max_seq_len, TOTAL_JOINTS).astype(np.float32)}
    else:
        dummy = {
            "keypoints": np.random.randn(1, max_seq_len, FEAT_DIM).astype(np.float32),
            "mask": np.zeros((1, max_seq_len), dtype=bool),
        }
    out = sess.run(None, dummy)
    assert out[0].shape == (1, num_classes), f"Shape mismatch: {out[0].shape}"
    print(f"  ✓ Validation passed")
    return True


# ----------------------------------------------------------------------
#  Main orchestrator
# ----------------------------------------------------------------------
def export_model(model_type: str, cfg: dict, output_dir: Path, device: str, validate: bool = True) -> bool:
    print(f"\n{'='*60}\n  Exporting {model_type.upper()}\n{'='*60}")
    mcfg = MODEL_CONFIGS[model_type]
    ckpt_path = Path(mcfg["exp_dir"]) / "checkpoints" / "best.pth"
    if not ckpt_path.exists():
        print(f"  [ERROR] Checkpoint not found: {ckpt_path}")
        return False
    ckpt = torch.load(str(ckpt_path), map_location=device)
    num_classes = ckpt.get("num_classes", 556)
    max_seq_len = cfg["data"]["max_seq_len"]
    print(f"  Classes: {num_classes}   MaxSeqLen: {max_seq_len}")
    model = build_model(model_type, cfg, num_classes).to(device)
    model.load_state_dict(ckpt["state_dict"])
    model.eval()
    out_path = output_dir / mcfg["output_name"]
    output_dir.mkdir(parents=True, exist_ok=True)

    if model_type == "bilstm":
        export_bilstm(model, out_path, max_seq_len, device)
    elif model_type == "transformer":
        export_transformer(model, out_path, max_seq_len, device)
    else:
        export_stgcn(model, out_path, max_seq_len, device)

    metadata = {
        "model_type": model_type,
        "num_classes": num_classes,
        "max_seq_len": max_seq_len,
        "feat_dim": FEAT_DIM,
        "num_joints": TOTAL_JOINTS,
        "num_coords": 3,
    }
    with open(out_path.with_suffix(".json"), "w") as f:
        json.dump(metadata, f, indent=2)

    # Copy label map
    lm_src = Path(cfg["data"]["label_map_file"])
    lm_dst = output_dir / "label_map.json"
    if lm_src.exists() and not lm_dst.exists():
        shutil.copy(str(lm_src), str(lm_dst))

    if validate:
        validate_onnx(out_path, model_type, max_seq_len, num_classes)
    print(f"  Done ({out_path.stat().st_size / 1024**2:.1f} MB)")
    return True


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--config", default="config/config.yaml")
    parser.add_argument("--model", choices=list(MODEL_CONFIGS.keys()), help="Export one model")
    parser.add_argument("--all", action="store_true", help="Export all three")
    parser.add_argument("--output_dir", default="onnx_models")
    parser.add_argument("--no-validate", action="store_true")
    parser.add_argument("--device", default=None)
    args = parser.parse_args()
    if not args.model and not args.all:
        parser.error("Specify --model <name> or --all")
    with open(args.config) as f:
        cfg = yaml.safe_load(f)
    device = args.device or ("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Device: {device}")
    models = list(MODEL_CONFIGS.keys()) if args.all else [args.model]
    output_dir = Path(args.output_dir)
    ok = 0
    for m in models:
        if export_model(m, cfg, output_dir, device, validate=not args.no_validate):
            ok += 1
    print(f"\nExported {ok}/{len(models)} models to {output_dir}/")


if __name__ == "__main__":
    main()


```

In [ ]:
from IPython.display import Markdown

file_path = './src/infer_onnx.py'

with open(file_path, 'r') as f:
    code = f.read()

# 'python' keyword triggers the syntax highlighter
display(Markdown(f"```python\n{code}\n```"))

## Exporting

In [79]:
%pwd

'/mnt/disk1/ye/exp/msl_recognition'

In [80]:
!rm -r onnx_models

In [81]:
!python src/export_to_onnx.py --config config/config.yaml --all

Device: cuda

  Exporting BILSTM
  Classes: 556   MaxSeqLen: 200
[W531 08:12:48.132776878 shape_type_inference.cpp:1995] Warning: The shape inference of prim::PackPadded type is missing, so it may result in wrong shape inference for the exported graph. Please consider adding it in symbolic function. (function UpdateReliable)
[W531 08:12:48.133584728 shape_type_inference.cpp:1995] Warning: The shape inference of prim::PackPadded type is missing, so it may result in wrong shape inference for the exported graph. Please consider adding it in symbolic function. (function UpdateReliable)
/mnt/disk1/ye/exp/msl_recognition/.venv/lib/python3.12/site-packages/torch/onnx/symbolic_opset9.py:4244: UserWarning: Exporting a model to ONNX with a batch_size other than 1, with a variable length with LSTM can cause an error when running the ONNX model with a different batch size. Make sure to save the model with a batch size of 1, or define the initial states (h0/c0) as inputs of the model. 
  warnings.w

## Inferring with Video

```
(msl_onnx) E:\LU_Lab\MSL_Tran_Demo>python infer_onnx.py --onnx_model .\onnx_models\bilstm_msl.onnx --video .\sample4inference\idx20-30.mp4
C:\Users\801680\Anaconda3\envs\msl_onnx\lib\site-packages\onnxruntime\capi\onnxruntime_inference_collection.py:123: UserWarning: Specified provider 'CUDAExecutionProvider' is not in available provider names.Available providers: 'AzureExecutionProvider, CPUExecutionProvider'
  warnings.warn(
[ONNX] Model type: bilstm, max_seq_len: 200
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
C:\Users\801680\Anaconda3\envs\msl_onnx\lib\site-packages\google\protobuf\symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '
[RESULT] မနက် ၉ နာရီ က နေ ည ၁၀ နာရီ ထိ ပါ ။ (4.3%) -> predictions_output\out_idx20-30.mp4

(msl_onnx) E:\LU_Lab\MSL_Tran_Demo>
```

```
(msl_onnx) E:\LU_Lab\MSL_Tran_Demo>python infer_onnx.py --onnx_model .\onnx_models\bilstm_msl.onnx --video .\sample4inference\idx20-86.mp4
C:\Users\801680\Anaconda3\envs\msl_onnx\lib\site-packages\onnxruntime\capi\onnxruntime_inference_collection.py:123: UserWarning: Specified provider 'CUDAExecutionProvider' is not in available provider names.Available providers: 'AzureExecutionProvider, CPUExecutionProvider'
  warnings.warn(
[ONNX] Model type: bilstm, max_seq_len: 200
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
C:\Users\801680\Anaconda3\envs\msl_onnx\lib\site-packages\google\protobuf\symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '
[RESULT] မိသားစုဝင် တယောက်ယောက် ကို အကြောင်းကြား ။ (5.1%) -> predictions_output\out_idx20-86.mp4

(msl_onnx) E:\LU_Lab\MSL_Tran_Demo>
```

```
(msl_onnx) E:\LU_Lab\MSL_Tran_Demo>python infer_onnx.py --onnx_model .\onnx_models\bilstm_msl.onnx --video .\sample4inference\idx20-244.mp4
C:\Users\801680\Anaconda3\envs\msl_onnx\lib\site-packages\onnxruntime\capi\onnxruntime_inference_collection.py:123: UserWarning: Specified provider 'CUDAExecutionProvider' is not in available provider names.Available providers: 'AzureExecutionProvider, CPUExecutionProvider'
  warnings.warn(
[ONNX] Model type: bilstm, max_seq_len: 200
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
C:\Users\801680\Anaconda3\envs\msl_onnx\lib\site-packages\google\protobuf\symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '
[RESULT] ငါ ဒီ ဆေး တွေ ကို ဆက် သောက် သင့် သ လား ။ (4.9%) -> predictions_output\out_idx20-244.mp4

(msl_onnx) E:\LU_Lab\MSL_Tran_Demo>
```

```
(msl_onnx) E:\LU_Lab\MSL_Tran_Demo>python infer_onnx.py --onnx_model .\onnx_models\bilstm_msl.onnx --video .\sample4inference\idx20-291.mp4
C:\Users\801680\Anaconda3\envs\msl_onnx\lib\site-packages\onnxruntime\capi\onnxruntime_inference_collection.py:123: UserWarning: Specified provider 'CUDAExecutionProvider' is not in available provider names.Available providers: 'AzureExecutionProvider, CPUExecutionProvider'
  warnings.warn(
[ONNX] Model type: bilstm, max_seq_len: 200
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
C:\Users\801680\Anaconda3\envs\msl_onnx\lib\site-packages\google\protobuf\symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '
[RESULT] မနက် ၉ နာရီ က နေ ည ၁၀ နာရီ ထိ ပါ ။ (9.0%) -> predictions_output\out_idx20-291.mp4

(msl_onnx) E:\LU_Lab\MSL_Tran_Demo>
```

```
(msl_onnx) E:\LU_Lab\MSL_Tran_Demo>python infer_onnx.py --onnx_model .\onnx_models\bilstm_msl.onnx --video .\sample4inference\idx20-440.mp4
C:\Users\801680\Anaconda3\envs\msl_onnx\lib\site-packages\onnxruntime\capi\onnxruntime_inference_collection.py:123: UserWarning: Specified provider 'CUDAExecutionProvider' is not in available provider names.Available providers: 'AzureExecutionProvider, CPUExecutionProvider'
  warnings.warn(
[ONNX] Model type: bilstm, max_seq_len: 200
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
C:\Users\801680\Anaconda3\envs\msl_onnx\lib\site-packages\google\protobuf\symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '
[RESULT] ကား က ဘာ အရောင် လဲ ။ (1.5%) -> predictions_output\out_idx20-440.mp4

(msl_onnx) E:\LU_Lab\MSL_Tran_Demo>
```

===================

```
(msl_onnx) E:\LU_Lab\MSL_Tran_Demo>python infer_onnx.py --onnx_model .\onnx_models\transformer_msl.onnx --video .\sample4inference\idx20-30.mp4
C:\Users\801680\Anaconda3\envs\msl_onnx\lib\site-packages\onnxruntime\capi\onnxruntime_inference_collection.py:123: UserWarning: Specified provider 'CUDAExecutionProvider' is not in available provider names.Available providers: 'AzureExecutionProvider, CPUExecutionProvider'
  warnings.warn(
[ONNX] Model type: transformer, max_seq_len: 200
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
C:\Users\801680\Anaconda3\envs\msl_onnx\lib\site-packages\google\protobuf\symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '
[RESULT] ခင်ဗျား ခံစား နေ ရ တာ ဘယ်လောက် ကြာ ပြီ လဲ ။ (4.5%) -> predictions_output\out_idx20-30.mp4

(msl_onnx) E:\LU_Lab\MSL_Tran_Demo>
```

```
(msl_onnx) E:\LU_Lab\MSL_Tran_Demo>python infer_onnx.py --onnx_model .\onnx_models\transformer_msl.onnx --video .\sample4inference\idx20-86.mp4
C:\Users\801680\Anaconda3\envs\msl_onnx\lib\site-packages\onnxruntime\capi\onnxruntime_inference_collection.py:123: UserWarning: Specified provider 'CUDAExecutionProvider' is not in available provider names.Available providers: 'AzureExecutionProvider, CPUExecutionProvider'
  warnings.warn(
[ONNX] Model type: transformer, max_seq_len: 200
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
C:\Users\801680\Anaconda3\envs\msl_onnx\lib\site-packages\google\protobuf\symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '
[RESULT] လူနာ က ကျွန်တော့် အမေ ပါ ။ (5.5%) -> predictions_output\out_idx20-86.mp4

(msl_onnx) E:\LU_Lab\MSL_Tran_Demo>
```

```
(msl_onnx) E:\LU_Lab\MSL_Tran_Demo>python infer_onnx.py --onnx_model .\onnx_models\transformer_msl.onnx --video .\sample4inference\idx20-244.mp4
C:\Users\801680\Anaconda3\envs\msl_onnx\lib\site-packages\onnxruntime\capi\onnxruntime_inference_collection.py:123: UserWarning: Specified provider 'CUDAExecutionProvider' is not in available provider names.Available providers: 'AzureExecutionProvider, CPUExecutionProvider'
  warnings.warn(
[ONNX] Model type: transformer, max_seq_len: 200
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
C:\Users\801680\Anaconda3\envs\msl_onnx\lib\site-packages\google\protobuf\symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '
[RESULT] လူနာ နဲ့ ဘယ်လို တော် တာ လဲ ။ (5.1%) -> predictions_output\out_idx20-244.mp4

(msl_onnx) E:\LU_Lab\MSL_Tran_Demo>
```

```
(msl_onnx) E:\LU_Lab\MSL_Tran_Demo>python infer_onnx.py --onnx_model .\onnx_models\transformer_msl.onnx --video .\sample4inference\idx20-291.mp4
C:\Users\801680\Anaconda3\envs\msl_onnx\lib\site-packages\onnxruntime\capi\onnxruntime_inference_collection.py:123: UserWarning: Specified provider 'CUDAExecutionProvider' is not in available provider names.Available providers: 'AzureExecutionProvider, CPUExecutionProvider'
  warnings.warn(
[ONNX] Model type: transformer, max_seq_len: 200
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
C:\Users\801680\Anaconda3\envs\msl_onnx\lib\site-packages\google\protobuf\symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '
[RESULT] သူ့ မိသားစု ကို ငါ တို့ ဖုန်းဆက် ရ မယ် ။ (8.3%) -> predictions_output\out_idx20-291.mp4

(msl_onnx) E:\LU_Lab\MSL_Tran_Demo>
```

```
(msl_onnx) E:\LU_Lab\MSL_Tran_Demo>python infer_onnx.py --onnx_model .\onnx_models\transformer_msl.onnx --video .\sample4inference\idx20-440.mp4
C:\Users\801680\Anaconda3\envs\msl_onnx\lib\site-packages\onnxruntime\capi\onnxruntime_inference_collection.py:123: UserWarning: Specified provider 'CUDAExecutionProvider' is not in available provider names.Available providers: 'AzureExecutionProvider, CPUExecutionProvider'
  warnings.warn(
[ONNX] Model type: transformer, max_seq_len: 200
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
C:\Users\801680\Anaconda3\envs\msl_onnx\lib\site-packages\google\protobuf\symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '
[RESULT] တိုက် သွား တဲ့ ကား ရဲ့ နံပါတ် နဲ့ အရောင် ကို ကောင်းကောင်း မှတ်မိ လား ။ (13.8%) -> predictions_output\out_idx20-440.mp4

(msl_onnx) E:\LU_Lab\MSL_Tran_Demo>
```

=================

```
(msl_onnx) E:\LU_Lab\MSL_Tran_Demo>python infer_onnx.py --onnx_model .\onnx_models\stgcn_msl.onnx --video .\sample4inference\idx20-30.mp4
C:\Users\801680\Anaconda3\envs\msl_onnx\lib\site-packages\onnxruntime\capi\onnxruntime_inference_collection.py:123: UserWarning: Specified provider 'CUDAExecutionProvider' is not in available provider names.Available providers: 'AzureExecutionProvider, CPUExecutionProvider'
  warnings.warn(
[ONNX] Model type: stgcn, max_seq_len: 200
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
C:\Users\801680\Anaconda3\envs\msl_onnx\lib\site-packages\google\protobuf\symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '
[RESULT] မြေနိမ့်ဒေသ ။ (8.5%) -> predictions_output\out_idx20-30.mp4

(msl_onnx) E:\LU_Lab\MSL_Tran_Demo>
```

```
(msl_onnx) E:\LU_Lab\MSL_Tran_Demo>python infer_onnx.py --onnx_model .\onnx_models\stgcn_msl.onnx --video .\sample4inference\idx20-86.mp4
C:\Users\801680\Anaconda3\envs\msl_onnx\lib\site-packages\onnxruntime\capi\onnxruntime_inference_collection.py:123: UserWarning: Specified provider 'CUDAExecutionProvider' is not in available provider names.Available providers: 'AzureExecutionProvider, CPUExecutionProvider'
  warnings.warn(
[ONNX] Model type: stgcn, max_seq_len: 200
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
C:\Users\801680\Anaconda3\envs\msl_onnx\lib\site-packages\google\protobuf\symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '
[RESULT] မနက် ၉ နာရီ က နေ ည ၁၀ နာရီ ထိ ပါ ။ (24.2%) -> predictions_output\out_idx20-86.mp4

(msl_onnx) E:\LU_Lab\MSL_Tran_Demo>
```

```
(msl_onnx) E:\LU_Lab\MSL_Tran_Demo>python infer_onnx.py --onnx_model .\onnx_models\stgcn_msl.onnx --video .\sample4inference\idx20-244.mp4
C:\Users\801680\Anaconda3\envs\msl_onnx\lib\site-packages\onnxruntime\capi\onnxruntime_inference_collection.py:123: UserWarning: Specified provider 'CUDAExecutionProvider' is not in available provider names.Available providers: 'AzureExecutionProvider, CPUExecutionProvider'
  warnings.warn(
[ONNX] Model type: stgcn, max_seq_len: 200
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
C:\Users\801680\Anaconda3\envs\msl_onnx\lib\site-packages\google\protobuf\symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '
[RESULT] မ တပ် ထား ပါ ဘူး ။ (25.6%) -> predictions_output\out_idx20-244.mp4

(msl_onnx) E:\LU_Lab\MSL_Tran_Demo>
```

```
(msl_onnx) E:\LU_Lab\MSL_Tran_Demo>python infer_onnx.py --onnx_model .\onnx_models\stgcn_msl.onnx --video .\sample4inference\idx20-291.mp4
C:\Users\801680\Anaconda3\envs\msl_onnx\lib\site-packages\onnxruntime\capi\onnxruntime_inference_collection.py:123: UserWarning: Specified provider 'CUDAExecutionProvider' is not in available provider names.Available providers: 'AzureExecutionProvider, CPUExecutionProvider'
  warnings.warn(
[ONNX] Model type: stgcn, max_seq_len: 200
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
C:\Users\801680\Anaconda3\envs\msl_onnx\lib\site-packages\google\protobuf\symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '
[RESULT] ညစ်ညမ်း နေ တဲ့ နေရာ များ ကို သန့်ရှင်းရေး လုပ် ပြီး ရောဂါပိုး သတ် ရ ပါ မယ် ။ (4.2%) -> predictions_output\out_idx20-291.mp4

(msl_onnx) E:\LU_Lab\MSL_Tran_Demo>
```

```
(msl_onnx) E:\LU_Lab\MSL_Tran_Demo>python infer_onnx.py --onnx_model .\onnx_models\stgcn_msl.onnx --video .\sample4inference\idx20-440.mp4
C:\Users\801680\Anaconda3\envs\msl_onnx\lib\site-packages\onnxruntime\capi\onnxruntime_inference_collection.py:123: UserWarning: Specified provider 'CUDAExecutionProvider' is not in available provider names.Available providers: 'AzureExecutionProvider, CPUExecutionProvider'
  warnings.warn(
[ONNX] Model type: stgcn, max_seq_len: 200
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
C:\Users\801680\Anaconda3\envs\msl_onnx\lib\site-packages\google\protobuf\symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '
[RESULT] ကား ထဲ မှာ လူ တစ်ယောက် ညှပ်မိ နေ တယ် ။ (10.9%) -> predictions_output\out_idx20-440.mp4

(msl_onnx) E:\LU_Lab\MSL_Tran_Demo>
```

အထက်မှာ မြင်ရတဲ့အတိုင်းပဲ ရလဒ်တွေက မှားနေတုန်းပဲ။  

## Updated Code and Export Again


In [82]:
%pwd

'/mnt/disk1/ye/exp/msl_recognition'

In [83]:
!rm -r onnx_models

In [84]:
!python src/export_to_onnx.py --config config/config.yaml --all

  File "/mnt/disk1/ye/exp/msl_recognition/src/export_to_onnx.py", line 305
    x = self.layer_norm2[i](x + self.layer_dropout2[i](self.layer_linear2[i](ff))
                           ^
SyntaxError: '(' was never closed


In [85]:
!python src/export_to_onnx.py --config config/config.yaml --all

Device: cuda

  Exporting  BILSTM
  Classes: 556   MaxSeqLen: 200
  BiLSTM dummy inputs: kp=(2, 200, 225), mask=(2, 200), lengths=(2,)
[W531 08:50:26.477049700 shape_type_inference.cpp:1995] Warning: The shape inference of prim::PackPadded type is missing, so it may result in wrong shape inference for the exported graph. Please consider adding it in symbolic function. (function UpdateReliable)
[W531 08:50:26.477864142 shape_type_inference.cpp:1995] Warning: The shape inference of prim::PackPadded type is missing, so it may result in wrong shape inference for the exported graph. Please consider adding it in symbolic function. (function UpdateReliable)
/mnt/disk1/ye/exp/msl_recognition/.venv/lib/python3.12/site-packages/torch/onnx/symbolic_opset9.py:4244: UserWarning: Exporting a model to ONNX with a batch_size other than 1, with a variable length with LSTM can cause an error when running the ONNX model with a different batch size. Make sure to save the model with a batch size of 1, or d

## Testing on Windows OS 

In [ ]:
(msl_onnx) E:\LU_Lab\MSL_Tran_Demo>python infer_onnx.py --onnx_model .\onnx_models\bilstm_msl.onnx --video .\sample4inference\idx20-30.mp4
[INFO] Loaded 556 labels from label map.
[INFO] ONNX loaded: bilstm_msl.onnx  (provider: CPUExecutionProvider)
[INFO] Model inputs: ['keypoints', 'mask', 'lengths']
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
[INFO] idx20-30.mp4 — 127 frames @ 30.0 fps
C:\Users\801680\Anaconda3\envs\msl_onnx\lib\site-packages\google\protobuf\symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '
[INFO] 127 frames extracted → using 127 (zero-padded to 200)

Video  : idx20-30.mp4
Model  : BILSTM
────────────────────────────────────────────────────────────
  Top-1: မနက် ၉ နာရီ က နေ ည ၁၀ နာရီ ထိ ပါ ။  (4.25%)
  Top-2: ဆိုင်ကယ် အချင်းချင်း တိုက်မိ လို့ ပါ ။  (2.66%)
  Top-3: ဆရာဝန် နဲ့ အမြန်ဆုံး ပြ ချင် ပါ တယ် ။  (2.54%)
  Top-4: ခင်ဗျား ဘယ် ဆရာဝန် နဲ့ ပြ နေ တာ လဲ ။  (2.27%)
  Top-5: အမျိုးသမီး လူနာစောင့် ပဲ ည အိပ် စောင့် ရ ပါ မယ် ။  (1.63%)
────────────────────────────────────────────────────────────
[INFO] Done.

(msl_onnx) E:\LU_Lab\MSL_Tran_Demo>

(msl_onnx) E:\LU_Lab\MSL_Tran_Demo>python infer_onnx.py --onnx_model .\onnx_models\bilstm_msl.onnx --video .\sample4inference\idx20-86.mp4
[INFO] Loaded 556 labels from label map.
[INFO] ONNX loaded: bilstm_msl.onnx  (provider: CPUExecutionProvider)
[INFO] Model inputs: ['keypoints', 'mask', 'lengths']
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
[INFO] idx20-86.mp4 — 169 frames @ 30.0 fps
C:\Users\801680\Anaconda3\envs\msl_onnx\lib\site-packages\google\protobuf\symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '
[INFO] 169 frames extracted → using 169 (zero-padded to 200)

Video  : idx20-86.mp4
Model  : BILSTM
────────────────────────────────────────────────────────────
  Top-1: မိသားစုဝင် တယောက်ယောက် ကို အကြောင်းကြား ။  (5.11%)
  Top-2: ဆရာဝန် ဖုန်း နံပါတ် လို ချင် လို့ ပါ ။  (4.28%)
  Top-3: သွားဆရာဝန် နဲ့ ဒီ နေ့ မနက် တွေ့ နိုင် မ လား ခင်ဗျာ ။  (3.11%)
  Top-4: ဆရာဝန် ဖုန်း နံပါတ် ခင်ဗျား မှာ ရှိ လား ။  (1.51%)
  Top-5: မနက်ဖြန် ဆေးခန်း လာ ဖို့ လို သေး လား ။  (1.03%)
────────────────────────────────────────────────────────────
[INFO] Done.

(msl_onnx) E:\LU_Lab\MSL_Tran_Demo>

(msl_onnx) E:\LU_Lab\MSL_Tran_Demo>python infer_onnx.py --onnx_model .\onnx_models\bilstm_msl.onnx --video .\sample4inference\idx20-244.mp4
[INFO] Loaded 556 labels from label map.
[INFO] ONNX loaded: bilstm_msl.onnx  (provider: CPUExecutionProvider)
[INFO] Model inputs: ['keypoints', 'mask', 'lengths']
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
[INFO] idx20-244.mp4 — 55 frames @ 30.0 fps
C:\Users\801680\Anaconda3\envs\msl_onnx\lib\site-packages\google\protobuf\symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '
[INFO] 55 frames extracted → using 55 (zero-padded to 200)

Video  : idx20-244.mp4
Model  : BILSTM
────────────────────────────────────────────────────────────
  Top-1: ငါ ဒီ ဆေး တွေ ကို ဆက် သောက် သင့် သ လား ။  (4.93%)
  Top-2: လမ်းလျှောက် ဖို့ လိုအပ် တယ် ။  (2.96%)
  Top-3: လူနာ နဲ့ ဘယ်လို တော် တာ လဲ ။  (2.13%)
  Top-4: ဘယ် နေရာ က နာ တာ လဲ ။  (1.69%)
  Top-5: ငါ့ ညီလေး အသက် မ ရှူ တော့ ဘူး ။  (0.84%)
────────────────────────────────────────────────────────────
[INFO] Done.

(msl_onnx) E:\LU_Lab\MSL_Tran_Demo>

(msl_onnx) E:\LU_Lab\MSL_Tran_Demo>python infer_onnx.py --onnx_model .\onnx_models\bilstm_msl.onnx --video .\sample4inference\idx20-291.mp4
[INFO] Loaded 556 labels from label map.
[INFO] ONNX loaded: bilstm_msl.onnx  (provider: CPUExecutionProvider)
[INFO] Model inputs: ['keypoints', 'mask', 'lengths']
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
[INFO] idx20-291.mp4 — 211 frames @ 30.0 fps
C:\Users\801680\Anaconda3\envs\msl_onnx\lib\site-packages\google\protobuf\symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '
[INFO] 211 frames extracted → using 200 (centre-cropped 200)

Video  : idx20-291.mp4
Model  : BILSTM
────────────────────────────────────────────────────────────
  Top-1: မနက် ၉ နာရီ က နေ ည ၁၀ နာရီ ထိ ပါ ။  (8.98%)
  Top-2: လူနာ တွေ့ ချိန် က ဘယ် အချိန် လဲ ။  (6.14%)
  Top-3: ကြမ်းပြင် တစ်ခု လုံး မှာ ရေ တွေ ကြည့် ပဲ ။  (1.78%)
  Top-4: လူနာစောင့် တစ်ယောက် ပဲ ည အိပ် စောင့် ခွင့် ရှိ ပါ မယ် ။  (1.03%)
  Top-5: ဆေး မ တည့် တာ တွေ ရှိ လား ။  (0.97%)
────────────────────────────────────────────────────────────
[INFO] Done.

(msl_onnx) E:\LU_Lab\MSL_Tran_Demo>

(msl_onnx) E:\LU_Lab\MSL_Tran_Demo>python infer_onnx.py --onnx_model .\onnx_models\bilstm_msl.onnx --video .\sample4inference\idx20-440.mp4
[INFO] Loaded 556 labels from label map.
[INFO] ONNX loaded: bilstm_msl.onnx  (provider: CPUExecutionProvider)
[INFO] Model inputs: ['keypoints', 'mask', 'lengths']
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
[INFO] idx20-440.mp4 — 248 frames @ 30.0 fps
C:\Users\801680\Anaconda3\envs\msl_onnx\lib\site-packages\google\protobuf\symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '
[INFO] 248 frames extracted → using 200 (centre-cropped 200)

Video  : idx20-440.mp4
Model  : BILSTM
────────────────────────────────────────────────────────────
  Top-1: ကား က ဘာ အရောင် လဲ ။  (1.50%)
  Top-2: ဆရာဝန် ဖုန်း နံပါတ် ခင်ဗျား မှာ ရှိ လား ။  (1.25%)
  Top-3: တိုက် သွား တဲ့ ကား ရဲ့ နံပါတ် နဲ့ အရောင် ကို ကောင်းကောင်း မှတ်မိ လား ။  (1.21%)
  Top-4: အနီးဆုံး ဆေးရုံ ကို ပို့ ပေး ပါ ။  (1.13%)
  Top-5: အရေပြား တွေ ခြောက် နေ တာ အတွက် ခရင်မ် လိမ်းဆေး လို ချင် ပါ တယ် ။  (1.08%)
────────────────────────────────────────────────────────────
[INFO] Done.

(msl_onnx) E:\LU_Lab\MSL_Tran_Demo>

==========================

(msl_onnx) E:\LU_Lab\MSL_Tran_Demo>python infer_onnx.py --onnx_model .\onnx_models\transformer_msl.onnx --video .\sample4inference\idx20-30.mp4
[INFO] Loaded 556 labels from label map.
[INFO] ONNX loaded: transformer_msl.onnx  (provider: CPUExecutionProvider)
[INFO] Model inputs: ['keypoints', 'mask']
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
[INFO] idx20-30.mp4 — 127 frames @ 30.0 fps
C:\Users\801680\Anaconda3\envs\msl_onnx\lib\site-packages\google\protobuf\symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '
[INFO] 127 frames extracted → using 127 (zero-padded to 200)

Video  : idx20-30.mp4
Model  : TRANSFORMER
────────────────────────────────────────────────────────────
  Top-1: ခင်ဗျား ခံစား နေ ရ တာ ဘယ်လောက် ကြာ ပြီ လဲ ။  (4.47%)
  Top-2: လမ်းလျှောက် ဖို့ လိုအပ် တယ် ။  (4.30%)
  Top-3: မနက် ၉ နာရီ က နေ ည ၁၀ နာရီ ထိ ပါ ။  (2.32%)
  Top-4: ဒဏ်ရာ ကို ရေခဲ ကပ် ပေး ပါ ။  (1.98%)
  Top-5: သူ့ မိသားစု ကို ငါ တို့ ဖုန်းဆက် ရ မယ် ။  (1.69%)
────────────────────────────────────────────────────────────
[INFO] Done.

(msl_onnx) E:\LU_Lab\MSL_Tran_Demo>

(msl_onnx) E:\LU_Lab\MSL_Tran_Demo>python infer_onnx.py --onnx_model .\onnx_models\transformer_msl.onnx --video .\sample4inference\idx20-86.mp4
[INFO] Loaded 556 labels from label map.
[INFO] ONNX loaded: transformer_msl.onnx  (provider: CPUExecutionProvider)
[INFO] Model inputs: ['keypoints', 'mask']
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
[INFO] idx20-86.mp4 — 169 frames @ 30.0 fps
C:\Users\801680\Anaconda3\envs\msl_onnx\lib\site-packages\google\protobuf\symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '
[INFO] 169 frames extracted → using 169 (zero-padded to 200)

Video  : idx20-86.mp4
Model  : TRANSFORMER
────────────────────────────────────────────────────────────
  Top-1: လူနာ က ကျွန်တော့် အမေ ပါ ။  (5.48%)
  Top-2: ကောလဟာလ သတင်း တွေ ကို မ ယုံ နဲ့ ။  (4.22%)
  Top-3: သူ့ မိသားစု ကို ငါ တို့ ဖုန်းဆက် ရ မယ် ။  (3.68%)
  Top-4: ကျွန်မ ကလေး မှန်ကြည်ဆေး တွေ သောက် ပစ် လိုက် လို့ ပါ ။  (3.32%)
  Top-5: လတ်ဆတ် တဲ့ ဟင်းသီးဟင်းရွက် တွေ ကို စား ပါ ။  (2.74%)
────────────────────────────────────────────────────────────
[INFO] Done.

(msl_onnx) E:\LU_Lab\MSL_Tran_Demo>

(msl_onnx) E:\LU_Lab\MSL_Tran_Demo>python infer_onnx.py --onnx_model .\onnx_models\transformer_msl.onnx --video .\sample4inference\idx20-244.mp4
[INFO] Loaded 556 labels from label map.
[INFO] ONNX loaded: transformer_msl.onnx  (provider: CPUExecutionProvider)
[INFO] Model inputs: ['keypoints', 'mask']
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
[INFO] idx20-244.mp4 — 55 frames @ 30.0 fps
C:\Users\801680\Anaconda3\envs\msl_onnx\lib\site-packages\google\protobuf\symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '
[INFO] 55 frames extracted → using 55 (zero-padded to 200)

Video  : idx20-244.mp4
Model  : TRANSFORMER
────────────────────────────────────────────────────────────
  Top-1: လူနာ နဲ့ ဘယ်လို တော် တာ လဲ ။  (5.14%)
  Top-2: လတ်ဆတ် တဲ့ ဟင်းသီးဟင်းရွက် တွေ ကို စား ပါ ။  (3.30%)
  Top-3: ငါ ဒီ ဆေး တွေ ကို ဆက် သောက် သင့် သ လား ။  (2.75%)
  Top-4: သူ ကျွန်တော့် ကို ထိုး တယ် ။  (2.68%)
  Top-5: အစာ စား ပြီး မှ သောက် ပါ ။  (2.42%)
────────────────────────────────────────────────────────────
[INFO] Done.

(msl_onnx) E:\LU_Lab\MSL_Tran_Demo>

(msl_onnx) E:\LU_Lab\MSL_Tran_Demo>python infer_onnx.py --onnx_model .\onnx_models\transformer_msl.onnx --video .\sample4inference\idx20-291.mp4
[INFO] Loaded 556 labels from label map.
[INFO] ONNX loaded: transformer_msl.onnx  (provider: CPUExecutionProvider)
[INFO] Model inputs: ['keypoints', 'mask']
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
[INFO] idx20-291.mp4 — 211 frames @ 30.0 fps
C:\Users\801680\Anaconda3\envs\msl_onnx\lib\site-packages\google\protobuf\symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '
[INFO] 211 frames extracted → using 200 (centre-cropped 200)

Video  : idx20-291.mp4
Model  : TRANSFORMER
────────────────────────────────────────────────────────────
  Top-1: သူ့ မိသားစု ကို ငါ တို့ ဖုန်းဆက် ရ မယ် ။  (8.28%)
  Top-2: လတ်ဆတ် တဲ့ ဟင်းသီးဟင်းရွက် တွေ ကို စား ပါ ။  (6.06%)
  Top-3: လူနာစောင့် တစ်ယောက် ပဲ ည အိပ် စောင့် ခွင့် ရှိ ပါ မယ် ။  (3.09%)
  Top-4: အမျိုးသမီး လူနာစောင့် ပဲ ည အိပ် စောင့် ရ ပါ မယ် ။  (1.93%)
  Top-5: ပျောက် သွား တဲ့ လူ ရဲ့ ပုံပန်းသဏ္ဌာန် ကို ပြောပြ နိုင် မ လား ။  (1.08%)
────────────────────────────────────────────────────────────
[INFO] Done.

(msl_onnx) E:\LU_Lab\MSL_Tran_Demo>

(msl_onnx) E:\LU_Lab\MSL_Tran_Demo>python infer_onnx.py --onnx_model .\onnx_models\transformer_msl.onnx --video .\sample4inference\idx20-440.mp4
[INFO] Loaded 556 labels from label map.
[INFO] ONNX loaded: transformer_msl.onnx  (provider: CPUExecutionProvider)
[INFO] Model inputs: ['keypoints', 'mask']
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
[INFO] idx20-440.mp4 — 248 frames @ 30.0 fps
C:\Users\801680\Anaconda3\envs\msl_onnx\lib\site-packages\google\protobuf\symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '
[INFO] 248 frames extracted → using 200 (centre-cropped 200)

Video  : idx20-440.mp4
Model  : TRANSFORMER
────────────────────────────────────────────────────────────
  Top-1: တိုက် သွား တဲ့ ကား ရဲ့ နံပါတ် နဲ့ အရောင် ကို ကောင်းကောင်း မှတ်မိ လား ။  (13.84%)
  Top-2: လတ်ဆတ် တဲ့ ဟင်းသီးဟင်းရွက် တွေ ကို စား ပါ ။  (6.91%)
  Top-3: သူ့ မိသားစု ကို ငါ တို့ ဖုန်းဆက် ရ မယ် ။  (2.18%)
  Top-4: မိသားစုဝင် တယောက်ယောက် ကို အကြောင်းကြား ။  (1.94%)
  Top-5: အရေပြား အထူးကု ဒေါက်တာ ဒေါ်ခင်မြလွင် နဲ့ ပြ ချင် လို့ ပါ ။  (1.24%)
────────────────────────────────────────────────────────────
[INFO] Done.

(msl_onnx) E:\LU_Lab\MSL_Tran_Demo>

==========================

(msl_onnx) E:\LU_Lab\MSL_Tran_Demo>python infer_onnx.py --onnx_model .\onnx_models\stgcn_msl.onnx --video .\sample4inference\idx20-30.mp4
[INFO] Loaded 556 labels from label map.
[INFO] ONNX loaded: stgcn_msl.onnx  (provider: CPUExecutionProvider)
[INFO] Model inputs: ['keypoints']
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
[INFO] idx20-30.mp4 — 127 frames @ 30.0 fps
C:\Users\801680\Anaconda3\envs\msl_onnx\lib\site-packages\google\protobuf\symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '
[INFO] 127 frames extracted → using 127 (zero-padded to 200)

Video  : idx20-30.mp4
Model  : STGCN
────────────────────────────────────────────────────────────
  Top-1: မြေနိမ့်ဒေသ ။  (8.49%)
  Top-2: တောမီး က အရမ်း အန္တရာယ် ကြီး တယ် ။  (2.76%)
  Top-3: ရေမြုပ် နေ တဲ့ နေရာ မှာ ကားမောင်း ဖို့ မ ကြိုးစား နဲ့ ။  (1.93%)
  Top-4: မင်း မီးလောင် တဲ့ အနံ့ မ ရ ဘူး လား ။  (1.91%)
  Top-5: တောင်စောင်း ။  (1.78%)
────────────────────────────────────────────────────────────
[INFO] Done.

(msl_onnx) E:\LU_Lab\MSL_Tran_Demo>

(msl_onnx) E:\LU_Lab\MSL_Tran_Demo>python infer_onnx.py --onnx_model .\onnx_models\stgcn_msl.onnx --video .\sample4inference\idx20-86.mp4
[INFO] Loaded 556 labels from label map.
[INFO] ONNX loaded: stgcn_msl.onnx  (provider: CPUExecutionProvider)
[INFO] Model inputs: ['keypoints']
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
[INFO] idx20-86.mp4 — 169 frames @ 30.0 fps
C:\Users\801680\Anaconda3\envs\msl_onnx\lib\site-packages\google\protobuf\symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '
[INFO] 169 frames extracted → using 169 (zero-padded to 200)

Video  : idx20-86.mp4
Model  : STGCN
────────────────────────────────────────────────────────────
  Top-1: မနက် ၉ နာရီ က နေ ည ၁၀ နာရီ ထိ ပါ ။  (24.19%)
  Top-2: အစာ မ စား ခင် နာရီ ဝက် အလို မှာ သောက် ပါ ။  (8.06%)
  Top-3: ခေါင်း တွေ ခဏခဏ ကိုက် နေ တယ် ။  (7.14%)
  Top-4: သွားဆရာဝန် နဲ့ ဒီ နေ့ မနက် တွေ့ နိုင် မ လား ခင်ဗျာ ။  (3.21%)
  Top-5: 199 ကို ဖုန်းဆက် ပေး ပါ ။  (3.03%)
────────────────────────────────────────────────────────────
[INFO] Done.

(msl_onnx) E:\LU_Lab\MSL_Tran_Demo>

(msl_onnx) E:\LU_Lab\MSL_Tran_Demo>python infer_onnx.py --onnx_model .\onnx_models\stgcn_msl.onnx --video .\sample4inference\idx20-244.mp4
[INFO] Loaded 556 labels from label map.
[INFO] ONNX loaded: stgcn_msl.onnx  (provider: CPUExecutionProvider)
[INFO] Model inputs: ['keypoints']
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
[INFO] idx20-244.mp4 — 55 frames @ 30.0 fps
C:\Users\801680\Anaconda3\envs\msl_onnx\lib\site-packages\google\protobuf\symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '
[INFO] 55 frames extracted → using 55 (zero-padded to 200)

Video  : idx20-244.mp4
Model  : STGCN
────────────────────────────────────────────────────────────
  Top-1: မ တပ် ထား ပါ ဘူး ။  (25.61%)
  Top-2: မြန်မြန် လုပ် ပါ ။  (7.39%)
  Top-3: ကား ကို ရပ် ထား ။  (3.51%)
  Top-4: ဂက်စ် မီး ။  (3.38%)
  Top-5: သူ ကျွန်တော့် ကို ထိုး တယ် ။  (2.96%)
────────────────────────────────────────────────────────────
[INFO] Done.

(msl_onnx) E:\LU_Lab\MSL_Tran_Demo>

(msl_onnx) E:\LU_Lab\MSL_Tran_Demo>python infer_onnx.py --onnx_model .\onnx_models\stgcn_msl.onnx --video .\sample4inference\idx20-291.mp4
[INFO] Loaded 556 labels from label map.
[INFO] ONNX loaded: stgcn_msl.onnx  (provider: CPUExecutionProvider)
[INFO] Model inputs: ['keypoints']
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
[INFO] idx20-291.mp4 — 211 frames @ 30.0 fps
C:\Users\801680\Anaconda3\envs\msl_onnx\lib\site-packages\google\protobuf\symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '
[INFO] 211 frames extracted → using 200 (centre-cropped 200)

Video  : idx20-291.mp4
Model  : STGCN
────────────────────────────────────────────────────────────
  Top-1: ညစ်ညမ်း နေ တဲ့ နေရာ များ ကို သန့်ရှင်းရေး လုပ် ပြီး ရောဂါပိုး သတ် ရ ပါ မယ် ။  (4.19%)
  Top-2: ၄ ရက် ကြာ မယ် ။  (3.24%)
  Top-3: မနက် ၆ နာရီ နောက် ပိုင်း အစား မ စား ပါ နဲ့ ။  (3.21%)
  Top-4: ဖန်ကွဲစ ရှ မိ ပြီး သွေး တွေ တအား ထွက် နေ တယ် ။  (3.17%)
  Top-5: ဆေးစစ်ချက် တွေ ဘယ် အချိန် ရ မ လဲ ။  (2.53%)
────────────────────────────────────────────────────────────
[INFO] Done.

(msl_onnx) E:\LU_Lab\MSL_Tran_Demo>

(msl_onnx) E:\LU_Lab\MSL_Tran_Demo>python infer_onnx.py --onnx_model .\onnx_models\stgcn_msl.onnx --video .\sample4inference\idx20-440.mp4
[INFO] Loaded 556 labels from label map.
[INFO] ONNX loaded: stgcn_msl.onnx  (provider: CPUExecutionProvider)
[INFO] Model inputs: ['keypoints']
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
[INFO] idx20-440.mp4 — 248 frames @ 30.0 fps
C:\Users\801680\Anaconda3\envs\msl_onnx\lib\site-packages\google\protobuf\symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '
[INFO] 248 frames extracted → using 200 (centre-cropped 200)

Video  : idx20-440.mp4
Model  : STGCN
────────────────────────────────────────────────────────────
  Top-1: ကား ထဲ မှာ လူ တစ်ယောက် ညှပ်မိ နေ တယ် ။  (10.89%)
  Top-2: ဘေး အိမ် ကို ဓားပြ ဝင် တိုက် နေ ပါ တယ် ။  (6.00%)
  Top-3: မိုးလေဝသ သတင်း များ ကို စဉ်ဆက်မပြတ် နားထောင် ပါ ။  (4.51%)
  Top-4: ပြိုကျ နိုင် တဲ့ အရာ များ နဲ့ ဝေးဝေး နေ ပါ ။  (4.19%)
  Top-5: သူ အခြေအနေ ကောင်း ပါ့ မ လား ။  (3.54%)
────────────────────────────────────────────────────────────
[INFO] Done.

(msl_onnx) E:\LU_Lab\MSL_Tran_Demo>



## Code Updated

In [86]:
from IPython.display import Markdown

file_path = './src/export_to_onnx.py'

with open(file_path, 'r') as f:
    code = f.read()

# 'python' keyword triggers the syntax highlighter
display(Markdown(f"```python\n{code}\n```"))

```python
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
export_to_onnx.py  –  Export trained MSL models to ONNX format
                       (runs on Linux/HPC server side)

Bug history / fixes
────────────────────
  [EXPORT-A] _BiLSTMExportWrapper — lengths must be None, NOT computed from mask.

      BiLSTM.forward calls pack_padded_sequence when lengths is not None.
      During torch.onnx.export tracing the dummy mask is all-zeros, so any
      (~mask).sum() gives max_seq_len and the LSTM is traced with the full-length
      packing path baked into the ONNX file.  At inference time the baked path
      doesn't adapt to shorter sequences; more critically, if the attention pooling
      uses a lengths value that is always max_seq_len it averages over the zero-
      padded tail → wrong context vector for any video shorter than max_seq_len.

      Fix: lengths=None → LSTM runs unpacked on the full padded sequence (the else
      branch in BiLSTM.forward).  AdditiveAttention receives the mask directly and
      uses masked_fill to exclude padded positions from softmax — the mask IS a live
      ONNX input and varies correctly at inference time.

  [EXPORT-B] _TransformerExportWrapper — missing inner FFN dropout.

      PyTorch's TransformerEncoderLayer has THREE dropout objects:
        self.dropout  — inner FFN dropout, placed between activation and linear2
        self.dropout1 — post-attention residual dropout
        self.dropout2 — post-FFN residual dropout

      The FFN path in PyTorch source:
        ff = linear2(dropout(activation(linear1(x))))  ← inner dropout
        x  = x + dropout2(ff)                          ← residual dropout

      Previous wrapper omitted the inner dropout entirely:
        ff = linear2(activation(linear1(x)))   ← wrong
        x  = x + dropout2(ff)

      This is a no-op during eval() export (all dropouts are identity), so it
      does not affect current inference results.  However it is architecturally
      incorrect and would silently produce wrong outputs if the model were ever
      exported in training mode.  Added self.layer_dropout_inner ModuleList.

  [EXPORT-C] _TransformerExportWrapper — double-dropout in post-norm FFN path.
      dropout2 was applied twice in the residual add. Fixed: apply once.
      (Retained from previous revision — still correct.)

  [EXPORT-D] validate_onnx — np.bool_ for Windows ORT compatibility.

  [EXPORT-E] ST-GCN in-place copy_ during tracing → pre-bake via _STGCNExportWrapper.
  [EXPORT-F] PyTorch 2.x fused MHA has no ONNX mapping → ManualMHA.
  [EXPORT-G] GELU requires opset >= 20 → erf-based equivalent.
  [EXPORT-H] Batch size 2 dummy inputs prevent 0-D scalar-collapse.
  [EXPORT-I] 'model_type' was missing from metadata JSON → now always written.

NOTE on the SUMMARY display "0.0 MB" for JSON files:
  The metadata JSON files are ~130 bytes.  Displayed with one decimal place
  in MB (bytes / 1024^2) this rounds to 0.0.  The files are correct and
  complete — infer_onnx.py reads them fine.
"""

import argparse
import json
import shutil
import sys
from pathlib import Path

import numpy as np
import torch
import yaml

sys.path.insert(0, str(Path(__file__).parent))
from models import build_model
from utils  import get_logger, get_device, load_label_map

MODEL_CONFIGS = {
    "bilstm":       {"exp_dir": "results/exp_bilstm",       "output_name": "bilstm_msl.onnx"},
    "transformer":  {"exp_dir": "results/exp_transformer",  "output_name": "transformer_msl.onnx"},
    "stgcn":        {"exp_dir": "results/exp_stgcn",        "output_name": "stgcn_msl.onnx"},
}

POSE_N       = 33
HAND_N       = 21
TOTAL_JOINTS = POSE_N + HAND_N * 2   # 75
FEAT_DIM     = TOTAL_JOINTS * 3       # 225


# ─────────────────────────────────────────────────────────────────────────────
# BiLSTM export
# ─────────────────────────────────────────────────────────────────────────────

class _BiLSTMExportWrapper(torch.nn.Module):
    """
    Wrapper for BiLSTM ONNX export.

    Fix [EXPORT-A]: lengths=None bypasses pack_padded_sequence entirely.

    BiLSTM.forward branch when lengths=None:
        lstm_out, _ = self.lstm(x)          # full padded sequence, no packing
        context = self.attention(lstm_out, mask)   # mask zeros out padded positions

    The mask (True = padded) is a live ONNX input, so attention correctly
    ignores padded frames at every inference call regardless of video length.
    """
    def __init__(self, model):
        super().__init__()
        self.model = model

    def forward(self, keypoints: torch.Tensor, mask: torch.Tensor) -> torch.Tensor:
        return self.model(keypoints, lengths=None, mask=mask)


def export_bilstm(model, out_path: Path, max_seq_len: int, device: str):
    model.eval()
    wrapper = _BiLSTMExportWrapper(model).to(device)
    wrapper.eval()

    dummy_kp   = torch.randn(2, max_seq_len, FEAT_DIM, device=device)
    dummy_mask = torch.zeros(2, max_seq_len, dtype=torch.bool, device=device)
    print(f"  BiLSTM dummy inputs: kp={tuple(dummy_kp.shape)}, mask={tuple(dummy_mask.shape)}")

    with torch.no_grad():
        torch.onnx.export(
            wrapper, (dummy_kp, dummy_mask), str(out_path),
            opset_version=16,
            input_names=["keypoints", "mask"],
            output_names=["logits"],
            dynamic_axes={"keypoints": {0: "batch"}, "mask": {0: "batch"}, "logits": {0: "batch"}},
            do_constant_folding=True,
            verbose=False,
        )
    print(f"  Exported → {out_path}")


# ─────────────────────────────────────────────────────────────────────────────
# Transformer export
# ─────────────────────────────────────────────────────────────────────────────

def _onnx_safe_gelu(x: torch.Tensor) -> torch.Tensor:
    """erf-based GELU — opset-14 compatible, avoids opset-20 requirement. [EXPORT-G]"""
    return 0.5 * x * (1.0 + torch.erf(x / 1.4142135623730951))


class ManualMHA(torch.nn.Module):
    """
    Manual Multi-Head Attention that bypasses PyTorch 2.x fused
    aten::_native_multi_head_attention (no ONNX opset-16 mapping). [EXPORT-F]
    Weights cloned from the original nn.MultiheadAttention.
    """
    def __init__(self, mha: torch.nn.MultiheadAttention):
        super().__init__()
        self.embed_dim = mha.embed_dim
        self.num_heads = mha.num_heads
        self.head_dim  = self.embed_dim // self.num_heads
        self.in_proj_weight  = torch.nn.Parameter(mha.in_proj_weight.data.clone())
        self.in_proj_bias    = torch.nn.Parameter(mha.in_proj_bias.data.clone())
        self.out_proj_weight = torch.nn.Parameter(mha.out_proj.weight.data.clone())
        self.out_proj_bias   = torch.nn.Parameter(mha.out_proj.bias.data.clone())

    def forward(self, query, key, value, key_padding_mask=None, need_weights=False):
        B, T, C = query.shape
        q_w = self.in_proj_weight[:C];         k_w = self.in_proj_weight[C:2*C];    v_w = self.in_proj_weight[2*C:]
        q_b = self.in_proj_bias[:C];           k_b = self.in_proj_bias[C:2*C];      v_b = self.in_proj_bias[2*C:]

        Q = torch.nn.functional.linear(query, q_w, q_b).reshape(B, -1, self.num_heads, self.head_dim).transpose(1, 2)
        K = torch.nn.functional.linear(key,   k_w, k_b).reshape(B, -1, self.num_heads, self.head_dim).transpose(1, 2)
        V = torch.nn.functional.linear(value, v_w, v_b).reshape(B, -1, self.num_heads, self.head_dim).transpose(1, 2)

        scores = torch.matmul(Q, K.transpose(-2, -1)) / (self.head_dim ** 0.5)
        if key_padding_mask is not None:
            scores = scores.masked_fill(key_padding_mask.unsqueeze(1).unsqueeze(2), float('-inf'))
        attn = torch.nn.functional.softmax(scores, dim=-1)
        out  = torch.matmul(attn, V).transpose(1, 2).reshape(B, -1, C)
        out  = torch.nn.functional.linear(out, self.out_proj_weight, self.out_proj_bias)
        return (out, attn) if need_weights else (out, None)


class _TransformerExportWrapper(torch.nn.Module):
    """
    Manually unrolls the TransformerEncoder to bypass PyTorch 2.x fused kernels.

    Fix [EXPORT-B]: Added missing inner FFN dropout (layer.dropout).
    Fix [EXPORT-C]: Residual dropout (dropout2) applied exactly once.

    PyTorch TransformerEncoderLayer has three dropout objects:
      layer.dropout  — inner FFN dropout: linear2(dropout(activation(linear1(x))))
      layer.dropout1 — post-attention residual: x + dropout1(attn_out)
      layer.dropout2 — post-FFN residual:      x + dropout2(ff_out)

    All are nn.Dropout instances, identity in eval() mode, so they do not affect
    current inference results. They are included for architectural correctness.

    The real model always uses norm_first=True (Pre-LN) so only that path runs.
    """
    def __init__(self, model):
        super().__init__()
        self.input_norm       = model.input_norm
        self.input_proj       = model.input_proj
        self.proj_norm        = model.proj_norm
        self.cls_token        = model.cls_token
        self.pos_enc          = model.pos_enc
        self.encoder_norm     = model.encoder.norm
        self.head_norm        = model.head_norm
        self.head_dropout     = model.head_dropout
        self.fc               = model.fc

        self.layer_self_attn       = torch.nn.ModuleList([ManualMHA(l.self_attn) for l in model.encoder.layers])
        self.layer_linear1         = torch.nn.ModuleList([l.linear1  for l in model.encoder.layers])
        self.layer_linear2         = torch.nn.ModuleList([l.linear2  for l in model.encoder.layers])
        self.layer_norm1           = torch.nn.ModuleList([l.norm1    for l in model.encoder.layers])
        self.layer_norm2           = torch.nn.ModuleList([l.norm2    for l in model.encoder.layers])
        self.layer_dropout1        = torch.nn.ModuleList([l.dropout1 for l in model.encoder.layers])
        # Fix [EXPORT-B]: inner FFN dropout (between activation and linear2)
        self.layer_dropout_inner   = torch.nn.ModuleList([l.dropout  for l in model.encoder.layers])
        self.layer_dropout2        = torch.nn.ModuleList([l.dropout2 for l in model.encoder.layers])

        self.num_layers = len(model.encoder.layers)
        self.norm_first = model.encoder.layers[0].norm_first  # always True

        act = model.encoder.layers[0].activation
        self._use_gelu = not (act is torch.nn.functional.relu or isinstance(act, torch.nn.ReLU))

    def forward(self, keypoints: torch.Tensor, mask: torch.Tensor) -> torch.Tensor:
        x = self.proj_norm(self.input_proj(self.input_norm(keypoints)))  # (B, T, d_model)

        B = x.shape[0]
        x = torch.cat([self.cls_token.expand(B, -1, -1), x], dim=1)     # (B, T+1, d_model)
        cls_mask     = torch.zeros(B, 1, dtype=torch.bool, device=x.device)
        padding_mask = torch.cat([cls_mask, mask], dim=1)                # (B, T+1)
        x = self.pos_enc(x)

        for i in range(self.num_layers):
            # Pre-LN (norm_first=True) — matches PyTorch TransformerEncoderLayer exactly
            nx = self.layer_norm1[i](x)
            ao, _ = self.layer_self_attn[i](nx, nx, nx,
                        key_padding_mask=padding_mask, need_weights=False)
            x = x + self.layer_dropout1[i](ao)

            nx = self.layer_norm2[i](x)
            ff = self.layer_linear1[i](nx)
            ff = _onnx_safe_gelu(ff) if self._use_gelu else torch.nn.functional.relu(ff)
            # Fix [EXPORT-B]: inner dropout (identity in eval, included for correctness)
            ff = self.layer_dropout_inner[i](ff)
            ff = self.layer_linear2[i](ff)
            # Fix [EXPORT-C]: residual dropout applied exactly once
            x  = x + self.layer_dropout2[i](ff)

        cls_out = self.encoder_norm(x)[:, 0, :]   # CLS token
        return self.fc(self.head_dropout(self.head_norm(cls_out)))


def export_transformer(model, out_path: Path, max_seq_len: int, device: str):
    model.eval()
    wrapper = _TransformerExportWrapper(model).to(device)
    wrapper.eval()

    dummy_kp   = torch.randn(2, max_seq_len, FEAT_DIM, device=device)
    dummy_mask = torch.zeros(2, max_seq_len, dtype=torch.bool, device=device)
    print(f"  Transformer dummy inputs: kp={tuple(dummy_kp.shape)}, mask={tuple(dummy_mask.shape)}")

    with torch.no_grad():
        try:
            from torch.nn.attention import sdp_kernel
            attn_ctx = sdp_kernel(enable_flash=False, enable_mem_efficient=False, enable_math=True)
        except ImportError:
            import contextlib
            attn_ctx = contextlib.nullcontext()
        with attn_ctx:
            torch.onnx.export(
                wrapper, (dummy_kp, dummy_mask), str(out_path),
                opset_version=16,
                input_names=["keypoints", "mask"],
                output_names=["logits"],
                dynamic_axes={
                    "keypoints": {0: "batch", 1: "seq_len"},
                    "mask":      {0: "batch", 1: "seq_len"},
                    "logits":    {0: "batch"},
                },
                do_constant_folding=True,
                verbose=False,
            )
    print(f"  Exported → {out_path}")


# ─────────────────────────────────────────────────────────────────────────────
# ST-GCN export
# ─────────────────────────────────────────────────────────────────────────────

class _STGCNExportWrapper(torch.nn.Module):
    """
    Pre-bakes edge-importance weights into the adjacency buffer of each block
    so no in-place copy_() appears in the traced graph. [EXPORT-E]

    After baking, sets edge_importance_weighting=False so the STGCN forward
    uses block.gcn.A directly (the pre-scaled buffer) and the
    edge_importance ParameterList is never accessed during tracing.
    """
    def __init__(self, model):
        super().__init__()
        self.model = model
        for i, block in enumerate(model.st_gcn_blocks):
            A_base = block.gcn.A.detach().clone()
            if model.edge_importance_weighting:
                ei     = model.edge_importance[i].detach().clone()
                A_sc   = (A_base * ei).clamp(min=0)
            else:
                A_sc = A_base
            block.gcn.A.data.copy_(A_sc)
        self.model.edge_importance_weighting = False

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.model(x)


def export_stgcn(model, out_path: Path, max_seq_len: int, device: str):
    model.eval()
    wrapper = _STGCNExportWrapper(model).to(device)
    wrapper.eval()

    dummy_kp = torch.randn(2, 3, max_seq_len, TOTAL_JOINTS, device=device)
    print(f"  ST-GCN dummy input: kp={tuple(dummy_kp.shape)}")

    with torch.no_grad():
        torch.onnx.export(
            wrapper, dummy_kp, str(out_path),
            opset_version=16,
            input_names=["keypoints"],
            output_names=["logits"],
            dynamic_axes={"keypoints": {0: "batch"}, "logits": {0: "batch"}},
            do_constant_folding=True,
            verbose=False,
        )
    print(f"  Exported → {out_path}")


# ─────────────────────────────────────────────────────────────────────────────
# Validation
# ─────────────────────────────────────────────────────────────────────────────

def validate_onnx(out_path: Path, model_type: str, max_seq_len: int, num_classes: int):
    try:
        import onnx, onnxruntime as ort
    except ImportError:
        print("  [SKIP] onnx / onnxruntime not installed")
        return True

    print("  Validating ONNX model...")
    onnx.checker.check_model(onnx.load(str(out_path)))
    print("  ✓ ONNX graph is valid")

    sess = ort.InferenceSession(str(out_path), providers=["CPUExecutionProvider"])

    if model_type == "stgcn":
        dummy = {"keypoints": np.random.randn(1, 3, max_seq_len, TOTAL_JOINTS).astype(np.float32)}
    else:
        # Fix [EXPORT-D]: np.bool_ explicit — Windows ORT rejects Python built-in bool
        dummy = {
            "keypoints": np.random.randn(1, max_seq_len, FEAT_DIM).astype(np.float32),
            "mask":      np.zeros((1, max_seq_len), dtype=np.bool_),
        }

    logits = sess.run(None, dummy)[0]
    assert logits.shape == (1, num_classes), \
        f"Shape mismatch: expected (1,{num_classes}), got {logits.shape}"
    print(f"  ✓ Inference OK — shape={logits.shape}, "
          f"top-5={np.argsort(logits[0])[::-1][:5].tolist()}")
    return True


# ─────────────────────────────────────────────────────────────────────────────
# Orchestrator
# ─────────────────────────────────────────────────────────────────────────────

def export_model(model_type: str, cfg: dict, output_dir: Path, device: str,
                 validate: bool = True) -> bool:
    print(f"\n{'='*60}\n  Exporting  {model_type.upper()}\n{'='*60}")

    mcfg      = MODEL_CONFIGS[model_type]
    ckpt_path = Path(mcfg["exp_dir"]) / "checkpoints" / "best.pth"
    out_path  = output_dir / mcfg["output_name"]

    if not ckpt_path.exists():
        print(f"  [ERROR] Checkpoint not found: {ckpt_path}")
        return False

    ckpt        = torch.load(str(ckpt_path), map_location=device)
    num_classes = ckpt.get("num_classes", 558)
    max_seq_len = cfg["data"]["max_seq_len"]
    print(f"  Classes: {num_classes}   MaxSeqLen: {max_seq_len}")

    model = build_model(model_type, cfg, num_classes)
    model.load_state_dict(ckpt["state_dict"])
    model = model.to(device)
    model.eval()

    output_dir.mkdir(parents=True, exist_ok=True)

    if   model_type == "bilstm":      export_bilstm(model, out_path, max_seq_len, device)
    elif model_type == "transformer": export_transformer(model, out_path, max_seq_len, device)
    elif model_type == "stgcn":       export_stgcn(model, out_path, max_seq_len, device)

    if validate:
        try:
            validate_onnx(out_path, model_type, max_seq_len, num_classes)
        except Exception as e:
            print(f"  [WARNING] Validation error: {e}")

    # Fix [EXPORT-I]: metadata always includes model_type
    metadata = {
        "model_type":  model_type,
        "num_classes": num_classes,
        "max_seq_len": max_seq_len,
        "feat_dim":    FEAT_DIM,
        "num_joints":  TOTAL_JOINTS,
        "num_coords":  3,
    }
    meta_path = out_path.with_suffix(".json")
    with open(meta_path, "w") as f:
        json.dump(metadata, f, indent=2)
    print(f"  Metadata  → {meta_path}  "
          f"(~{meta_path.stat().st_size} bytes — displays as 0.0 MB, this is normal)")

    lm_src = Path(cfg["data"]["label_map_file"])
    lm_dst = output_dir / "label_map.json"
    if lm_src.exists() and not lm_dst.exists():
        shutil.copy(str(lm_src), str(lm_dst))
        print(f"  Label map → {lm_dst}")

    print(f"  ✓ Done  ({out_path.stat().st_size / 1024**2:.1f} MB)")
    return True


# ─────────────────────────────────────────────────────────────────────────────
# CLI
# ─────────────────────────────────────────────────────────────────────────────

def main():
    parser = argparse.ArgumentParser(description="Export MSL models to ONNX")
    parser.add_argument("--config",      default="config/config.yaml")
    parser.add_argument("--model",       choices=list(MODEL_CONFIGS.keys()))
    parser.add_argument("--all",         action="store_true")
    parser.add_argument("--output_dir",  default="onnx_models")
    parser.add_argument("--no-validate", action="store_true")
    parser.add_argument("--device",      default=None)
    args = parser.parse_args()

    if not args.model and not args.all:
        parser.error("Specify --model <name> or --all")

    with open(args.config) as f:
        cfg = yaml.safe_load(f)

    device     = args.device or ("cuda" if torch.cuda.is_available() else "cpu")
    models     = list(MODEL_CONFIGS.keys()) if args.all else [args.model]
    output_dir = Path(args.output_dir)
    print(f"Device: {device}")

    ok = sum(export_model(m, cfg, output_dir, device, not args.no_validate) for m in models)

    print(f"\n{'='*60}\n  SUMMARY: {ok}/{len(models)} exported to '{output_dir}/'")
    for f in sorted(output_dir.glob("*")):
        size_mb = f.stat().st_size / 1024**2
        note = "  (metadata — tiny file, 0.0 MB is correct)" if f.suffix == ".json" and size_mb < 0.01 else ""
        print(f"    {f.name:<35} {size_mb:6.1f} MB{note}")
    print("="*60)


if __name__ == "__main__":
    main()

```

## Current infer_onnx.py Code

In [87]:
from IPython.display import Markdown

file_path = './src/infer_onnx.py'

with open(file_path, 'r') as f:
    code = f.read()

# 'python' keyword triggers the syntax highlighter
display(Markdown(f"```python\n{code}\n```"))

```python
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
infer_onnx.py  –  Real-time MSL recognition on Windows using ONNX Runtime

Complete bug list (this version fixes ALL of them)
────────────────────────────────────────────────────

  [INFER-A] process_frame skipped frames where pose was not detected.
      Training (extract_keypoints.py / process_video) ALWAYS appends a
      keypoint array for EVERY frame.  When MediaPipe fails to detect the
      pose, extract_frame_keypoints() returns zeros for the pose block
      (and zeros for any undetected hands).  The frame is still included.

      The old code returned None when detected=False and the callers skipped
      those frames.  This caused two problems:
        1. The temporal structure of the sequence differs from training
           (missing frames change sign timing and duration).
        2. The sequence length differs from what normalization expects
           (shoulder width can become 0 in zero frames, but normalize_sequence
           handles that with the `where(dist < 1e-6, 1.0, dist)` guard).

      Fix: always return the keypoint array regardless of detection status.
      Zero rows for undetected joints are exactly what the model was trained on.

  [INFER-B] normalize_sequence used 2-D shoulders and 2-D scale.
      Training (extract_keypoints.normalize_sequence) uses full 3-D midpoint
      and 3-D Euclidean shoulder width applied to ALL three coordinates.
      The old code left z unscaled (only subtracted mid_z, never divided by
      scale) and used a 2-D scale that is always smaller than the 3-D one,
      inflating x and y magnitudes.  Every prediction was wrong as a result.
      Fix: exact mirror of extract_keypoints.normalize_sequence().

  [INFER-C] Webcam: buffer not cleared after each classification.
      infer.py clears the buffer after every classification so each sign is
      evaluated as a self-contained sequence, exactly as during training.
      The old code used a sliding/rolling window — fine for continuous
      recognition but poor for per-sign accuracy demos.
      Fix: clear kp_buffer after each classification, matching infer.py.

  [INFER-D] np.int64 / int dict key mismatch.
      np.argsort returns np.int64 indices.  dict.get(np.int64(k)) on an
      {int: str} dict silently returns None on some CPython/NumPy builds.
      Fix: always cast to int() before lookup via _get_label().

  [INFER-E] ONNXModel.predict double-indexed to scalar.
      [0][0] returned shape () instead of (num_classes,).
      Fix: return output[0] — shape (num_classes,).
"""

import argparse
import importlib
import json
import sys
from datetime import datetime
from pathlib import Path

import cv2
import numpy as np
import onnxruntime as ort

try:
    from PIL import Image, ImageDraw, ImageFont
    HAS_PIL = True
except ImportError:
    HAS_PIL = False

# ── Constants (must match extract_keypoints.py exactly) ───────────────────────
POSE_N          = 33
HAND_N          = 21
TOTAL_JOINTS    = POSE_N + HAND_N * 2   # 75
COORDS          = 3
FEAT_DIM        = TOTAL_JOINTS * COORDS  # 225

POSE_LEFT_SHOULDER  = 11
POSE_RIGHT_SHOULDER = 12
LEFT_HAND_START     = POSE_N            # 33
RIGHT_HAND_START    = POSE_N + HAND_N   # 54


# ─── MediaPipe import ─────────────────────────────────────────────────────────

def _import_mp():
    for prefix in ["mediapipe.solutions", "mediapipe.python.solutions"]:
        try:
            pose  = importlib.import_module(f"{prefix}.pose")
            hands = importlib.import_module(f"{prefix}.hands")
            draw  = importlib.import_module(f"{prefix}.drawing_utils")
            return pose, hands, draw
        except (ImportError, ModuleNotFoundError):
            continue
    try:
        import mediapipe as _mp
        return _mp.solutions.pose, _mp.solutions.hands, _mp.solutions.drawing_utils
    except AttributeError:
        pass
    raise ImportError(
        "Cannot import MediaPipe solutions.\n"
        "Try: pip uninstall mediapipe -y && pip install mediapipe==0.10.18"
    )

mp_pose, mp_hands, mp_drawing = _import_mp()


# ─── Myanmar text rendering ──────────────────────────────────────────────────

class TextRenderer:
    _FONT_CANDIDATES = [
        "C:\\Windows\\Fonts\\Padauk.ttf",
        "C:\\Windows\\Fonts\\NotoSansMyanmar-Regular.ttf",
        "C:\\Windows\\Fonts\\NotoSansMyanmar.ttf",
        "C:\\Windows\\Fonts\\myanmar3.ttf",
        "/usr/share/fonts/truetype/padauk/Padauk-Regular.ttf",
        "/usr/share/fonts/truetype/noto/NotoSansMyanmar-Regular.ttf",
        "fonts\\Padauk.ttf",
        "fonts/NotoSansMyanmar-Regular.ttf",
    ]

    def __init__(self, font_size: int = 32):
        self.font = self._load_font(font_size)
        self.can_render_myanmar = (self.font is not None)
        if not self.can_render_myanmar:
            print()
            print("  ╔═══════════════════════════════════════════════════════════╗")
            print("  ║  INFO: No Myanmar shaping font found.                    ║")
            print("  ║  Displaying readable Class IDs instead of Myanmar text.  ║")
            print("  ╚═══════════════════════════════════════════════════════════╝")
            print()

    def _load_font(self, size):
        if not HAS_PIL:
            return None
        for path in self._FONT_CANDIDATES:
            if Path(path).exists():
                try:
                    return ImageFont.truetype(path, size)
                except Exception:
                    continue
        return None

    def put_text(self, frame, text, position=(20, 20), color=(0, 255, 0)):
        if self.font and HAS_PIL:
            pil = Image.fromarray(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
            d   = ImageDraw.Draw(pil)
            r, g, b = color[2], color[1], color[0]
            for dx in [-1, 0, 1]:
                for dy in [-1, 0, 1]:
                    if dx or dy:
                        d.text((position[0]+dx, position[1]+dy), text,
                               font=self.font, fill=(0, 0, 0))
            d.text(position, text, font=self.font, fill=(r, g, b))
            return cv2.cvtColor(np.array(pil), cv2.COLOR_RGB2BGR)
        cv2.putText(frame, text, position, cv2.FONT_HERSHEY_SIMPLEX,
                    0.75, color, 2, cv2.LINE_AA)
        return frame


# ─── Keypoint extraction ──────────────────────────────────────────────────────

class KeypointExtractor:
    """
    Extracts (TOTAL_JOINTS, 3) keypoint arrays from BGR frames.

    BUG FIX [INFER-A]:
    process_frame now ALWAYS returns a keypoint array, never None.
    When MediaPipe fails to detect the pose, the pose block is all zeros.
    When hands are not detected, those blocks are all zeros.
    This exactly mirrors extract_keypoints.py / process_video() behaviour,
    which appends extract_frame_keypoints() output for EVERY frame without
    any None-filtering.

    Hand convention (matches extract_keypoints.py):
      MediaPipe reports handedness from the camera perspective (mirrored).
      label='Right' → signer's LEFT hand  → slot LEFT_HAND_START (33).
      label='Left'  → signer's RIGHT hand → slot RIGHT_HAND_START (54).
    """

    def __init__(self, model_complexity: int = 1):
        self.pose  = mp_pose.Pose(
            static_image_mode=False,
            model_complexity=model_complexity,
            smooth_landmarks=True,
            enable_segmentation=False,
            min_detection_confidence=0.5,
            min_tracking_confidence=0.5,
        )
        self.hands = mp_hands.Hands(
            static_image_mode=False,
            max_num_hands=2,
            model_complexity=min(model_complexity, 1),
            min_detection_confidence=0.5,
            min_tracking_confidence=0.5,
        )
        self._last_pose  = None
        self._last_hands = None

    def process_frame(self, frame: np.ndarray) -> np.ndarray:
        """
        Returns (TOTAL_JOINTS, 3) float32 array for every frame.
        Zero rows for any joint/hand that was not detected.
        Never returns None — every frame is included, same as training.
        """
        rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        rgb.flags.writeable = False
        p_res = self.pose.process(rgb)
        h_res = self.hands.process(rgb)
        self._last_pose  = p_res
        self._last_hands = h_res

        kp = np.zeros((TOTAL_JOINTS, COORDS), dtype=np.float32)

        # Pose — zeros if not detected (same as extract_keypoints._lm_to_array)
        if p_res.pose_landmarks:
            for i, lm in enumerate(p_res.pose_landmarks.landmark):
                kp[i] = [lm.x, lm.y, lm.z]

        # Hands — zeros for each undetected hand
        if h_res.multi_hand_landmarks and h_res.multi_handedness:
            for lm_list, handedness in zip(
                h_res.multi_hand_landmarks, h_res.multi_handedness
            ):
                label  = handedness.classification[0].label
                offset = LEFT_HAND_START if label == 'Right' else RIGHT_HAND_START
                for i, lm in enumerate(lm_list.landmark):
                    kp[offset + i] = [lm.x, lm.y, lm.z]

        # Always return the array — never None
        return kp

    def draw_landmarks(self, frame: np.ndarray):
        if self._last_pose and self._last_pose.pose_landmarks:
            mp_drawing.draw_landmarks(
                frame, self._last_pose.pose_landmarks, mp_pose.POSE_CONNECTIONS
            )
        if self._last_hands and self._last_hands.multi_hand_landmarks:
            for lm in self._last_hands.multi_hand_landmarks:
                mp_drawing.draw_landmarks(frame, lm, mp_hands.HAND_CONNECTIONS)

    def close(self):
        self.pose.close()
        self.hands.close()


# ─── Normalization ────────────────────────────────────────────────────────────

def normalize_sequence(seq: np.ndarray) -> np.ndarray:
    """
    BUG FIX [INFER-B]: Exact mirror of extract_keypoints.normalize_sequence().

    Translates to shoulder-midpoint origin (full 3-D) and scales by
    3-D Euclidean shoulder width.  Applied to ALL three coordinate axes.

    Training code (extract_keypoints.py):
        mid  = (l_sh + r_sh) / 2.0         shape (T, 3)
        dist = ||r_sh - l_sh||_2            shape (T, 1)  — 3-D Euclidean
        out  = (seq - mid[:, None, :]) / dist[:, None, :]

    The old code used a 2-D scale (x,y only) and did NOT divide z by scale.
    This produced inflated x/y values and wrong z values → every prediction wrong.

    Args:
        seq: (T, 75, 3) raw keypoints from KeypointExtractor.
    Returns:
        (T, 75, 3) normalised float32, matching training distribution exactly.
    """
    l_sh = seq[:, POSE_LEFT_SHOULDER,  :]   # (T, 3)
    r_sh = seq[:, POSE_RIGHT_SHOULDER, :]   # (T, 3)
    mid  = (l_sh + r_sh) / 2.0              # (T, 3)
    dist = np.linalg.norm(r_sh - l_sh, axis=1, keepdims=True)   # (T, 1)
    dist = np.where(dist < 1e-6, 1.0, dist)
    return ((seq - mid[:, None, :]) / dist[:, None, :]).astype(np.float32)


# ─── Preprocessor ─────────────────────────────────────────────────────────────

class Preprocessor:
    """
    Stacks raw (75, 3) keypoint frames, normalises, crops/pads, and builds
    the model-ready input dict.

    Processing order (matches infer.py exactly):
      1. Stack → (T, 75, 3)
      2. Normalize (shoulder-centred, 3-D scale, all axes)
      3. Centre-crop if T > max_seq_len
      4. Zero-pad if T < max_seq_len
      5. Build mask (True = padded position)
      6. Reshape for model type
    """

    def __init__(self, max_seq_len: int = 200):
        self.max_seq_len = max_seq_len

    def __call__(self, kp_list: list, model_type: str) -> dict:
        if not kp_list:
            return {}

        seq = np.stack(kp_list, axis=0).astype(np.float32)  # (T, 75, 3)
        seq = normalize_sequence(seq)
        T   = seq.shape[0]

        # Centre-crop
        if T > self.max_seq_len:
            start = (T - self.max_seq_len) // 2
            seq   = seq[start: start + self.max_seq_len]
            T     = self.max_seq_len

        real_len = T

        # Zero-pad
        if T < self.max_seq_len:
            pad = np.zeros((self.max_seq_len - T, TOTAL_JOINTS, COORDS), dtype=np.float32)
            seq = np.concatenate([seq, pad], axis=0)

        # Mask: True where padded
        mask = np.zeros(self.max_seq_len, dtype=np.bool_)
        mask[real_len:] = True

        if model_type == "stgcn":
            # (T, 75, 3) → (3, T, 75) → (1, 3, T, 75)
            kp_out = np.expand_dims(seq.transpose(2, 0, 1), axis=0).astype(np.float32)
            return {"keypoints": kp_out}
        else:
            # (T, 75, 3) → (T, 225) → (1, T, 225)
            kp_out = np.expand_dims(
                seq.reshape(self.max_seq_len, FEAT_DIM), axis=0
            ).astype(np.float32)
            return {"keypoints": kp_out, "mask": mask[np.newaxis]}


# ─── ONNX inference wrapper ───────────────────────────────────────────────────

class ONNXModel:
    def __init__(self, onnx_path: str, meta_path: str = None):
        available = ort.get_available_providers()
        preferred = ["CUDAExecutionProvider", "DmlExecutionProvider", "CPUExecutionProvider"]
        providers = [p for p in preferred if p in available]

        self.session = ort.InferenceSession(onnx_path, providers=providers)
        print(f"[INFO] ONNX loaded: {Path(onnx_path).name}"
              f"  (provider: {self.session.get_providers()[0]})")

        self.meta = {}
        if meta_path and Path(meta_path).exists():
            try:
                with open(meta_path, encoding="utf-8") as f:
                    self.meta = json.load(f)
            except Exception as e:
                print(f"[WARNING] Could not read metadata: {e}")
        else:
            print(f"[INFO] Metadata not found at '{meta_path}', using defaults.")

        self.model_type  = self.meta.get("model_type",  "transformer")
        self.max_seq_len = self.meta.get("max_seq_len", 200)
        self.num_classes = self.meta.get("num_classes", 558)

    def predict(self, inputs: dict) -> np.ndarray:
        """
        Run one forward pass.
        BUG FIX [INFER-E]: return 1-D (num_classes,) array, not scalar.
        output shape from session: (1, num_classes); [0] gives (num_classes,).
        """
        return self.session.run(None, inputs)[0][0]   # (num_classes,)

    @staticmethod
    def softmax(logits: np.ndarray) -> np.ndarray:
        e = np.exp(logits - logits.max())
        return e / e.sum()


# ─── Label map ────────────────────────────────────────────────────────────────

def load_idx2label(path: str) -> dict:
    with open(path, encoding="utf-8") as f:
        data = json.load(f)
    return {int(k): v for k, v in data.get("idx2label", {}).items()}


def _get_label(idx2label: dict, class_id) -> str:
    """
    BUG FIX [INFER-D]: always cast to int() — np.int64 keys miss {int:str} dicts.
    """
    return idx2label.get(int(class_id), f"[class {int(class_id)}]")


# ─── Display helpers ──────────────────────────────────────────────────────────

def _build_text(class_id, label, confidence, can_myanmar,
                top_k=1, all_ids=None, probs=None, lbl_map=None):
    if top_k == 1:
        return (f"{label}  ({confidence*100:.1f}%)" if can_myanmar
                else f"Class {int(class_id)}  ({confidence*100:.1f}%)")
    parts = []
    for i, cid in enumerate(all_ids):
        lbl = _get_label(lbl_map, cid) if (can_myanmar and lbl_map) else f"Class {int(cid)}"
        parts.append(f"#{i+1} {lbl} ({probs[int(cid)]*100:.0f}%)")
    return "  |  ".join(parts)


def _build_log(class_id, label, confidence, top_k=1, all_ids=None, probs=None, lbl_map=None):
    safe = label.encode("utf-8", errors="replace").decode("utf-8")
    if top_k == 1:
        return f"  Class {int(class_id):>4d} | {safe:<40s} | {confidence*100:6.2f}%"
    lines = []
    for i, cid in enumerate(all_ids):
        lbl = _get_label(lbl_map, cid) if lbl_map else "?"
        lbl = lbl.encode("utf-8", errors="replace").decode("utf-8")
        lines.append(f"  Top-{i+1}: Class {int(cid):>4d} | {lbl:<40s} | {probs[int(cid)]*100:6.2f}%")
    return "\n".join(lines)


def _append_log(path, text):
    try:
        with open(path, "a", encoding="utf-8-sig") as f:
            f.write(text)
    except Exception:
        pass


# ─── Video processing ─────────────────────────────────────────────────────────

def run_video(model, prep, extractor, renderer, video_path,
              top_k=5, log_file=None, idx2label=None, debug=False):
    """
    Process a single video file for sign language recognition.

    Matches infer.py / process_video() behaviour exactly:
      - Every frame is processed and appended (no skipping for missing pose).
      - Normalization applied after full sequence is stacked.
      - Centre-crop or zero-pad to max_seq_len.
    """
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        print(f"[ERROR] Cannot open video: {video_path}")
        return

    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    fps_src      = cap.get(cv2.CAP_PROP_FPS) or 30.0
    print(f"[INFO] {Path(video_path).name} — {total_frames} frames @ {fps_src:.1f} fps")

    kp_list = []
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        # BUG FIX [INFER-A]: process_frame always returns array (never None)
        kp_list.append(extractor.process_frame(frame))
    cap.release()

    if not kp_list:
        print("[WARNING] No frames could be read from the video.")
        return

    T_raw = len(kp_list)
    T_use = min(T_raw, model.max_seq_len)
    print(f"[INFO] {T_raw} frames extracted → using {T_use} "
          f"({'centre-cropped' if T_raw > model.max_seq_len else 'zero-padded to'}"
          f" {model.max_seq_len})")

    inputs = prep(kp_list, model.model_type)
    logits = model.predict(inputs)                    # (num_classes,)
    probs  = ONNXModel.softmax(logits)               # (num_classes,)
    top_idx = np.argsort(probs)[::-1][:top_k]

    sep = "─" * 60
    lines = [f"Video  : {Path(video_path).name}",
             f"Model  : {model.model_type.upper()}", sep]
    for rank, idx in enumerate(top_idx, 1):
        lbl = _get_label(idx2label, idx) if idx2label else f"[class {int(idx)}]"
        lines.append(f"  Top-{rank}: {lbl}  ({probs[int(idx)]*100:.2f}%)")
    lines.append(sep)
    print("\n" + "\n".join(lines))

    if log_file:
        log_lines = [f"[{datetime.now().isoformat()}] {Path(video_path).name}"]
        for idx in top_idx:
            lbl = _get_label(idx2label, idx) if idx2label else "?"
            log_lines.append(_build_log(idx, lbl, probs[int(idx)],
                                        top_k=1, lbl_map=idx2label))
        _append_log(log_file, "\n".join(log_lines) + "\n\n")


# ─── Webcam processing ────────────────────────────────────────────────────────

def run_webcam(model, prep, extractor, renderer, idx2label,
               cam_idx=0, top_k=1, log_file=None, classify_every=15):
    """
    Live webcam sign recognition.

    BUG FIX [INFER-A]: Every frame is always appended to the buffer —
    no None-filtering — because process_frame always returns an array.

    BUG FIX [INFER-C]: Buffer is cleared after each classification,
    matching infer.py's behaviour.  Each sign is evaluated as an
    independent self-contained sequence, exactly as during training.
    Press C to manually clear the buffer; press Q to quit.
    """
    cap = cv2.VideoCapture(cam_idx)
    if not cap.isOpened():
        print(f"[ERROR] Cannot open webcam {cam_idx}")
        return

    max_buf    = model.max_seq_len
    kp_buffer  = []
    last_text  = "Collecting frames..."
    last_color = (200, 200, 200)
    last_log   = ""
    frame_cnt  = 0
    print("[INFO] Webcam started. Sign into camera, then wait for prediction.")
    print("[INFO] Q = quit | C = clear buffer manually")

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        frame_cnt += 1
        # BUG FIX [INFER-A]: always append, never skip
        kp_buffer.append(extractor.process_frame(frame))
        if len(kp_buffer) > max_buf:
            kp_buffer.pop(0)

        if len(kp_buffer) >= 20 and frame_cnt % classify_every == 0:
            inputs  = prep(list(kp_buffer), model.model_type)
            logits  = model.predict(inputs)
            probs   = ONNXModel.softmax(logits)
            top_idx = np.argsort(probs)[::-1][:top_k]

            best_idx  = top_idx[0]
            best_prob = probs[int(best_idx)]
            best_lbl  = _get_label(idx2label, best_idx) if idx2label else f"Class {int(best_idx)}"

            last_text  = _build_text(best_idx, best_lbl, best_prob,
                                     renderer.can_render_myanmar,
                                     top_k, top_idx, probs, idx2label)
            last_color = ((0, 200, 0) if best_prob >= 0.80 else
                          (0, 200, 200) if best_prob >= 0.50 else (0, 0, 255))

            # BUG FIX [INFER-C]: clear buffer after each classification
            # so the next sign starts fresh, matching infer.py behaviour
            kp_buffer.clear()

            if log_file and last_text != last_log:
                ts = datetime.now().strftime("%H:%M:%S")
                _append_log(log_file,
                    f"[{ts}] " + _build_log(best_idx, best_lbl, best_prob,
                                             top_k=1, lbl_map=idx2label) + "\n")
                last_log = last_text

        extractor.draw_landmarks(frame)
        frame = renderer.put_text(frame, last_text, position=(10, 10), color=last_color)

        h, w  = frame.shape[:2]
        fill  = int(len(kp_buffer) / max(max_buf, 1) * w)
        cv2.rectangle(frame, (0, h - 8), (fill, h), (0, 180, 255), -1)
        cv2.putText(frame, f"buf {len(kp_buffer)}/{max_buf}",
                    (10, h - 12), cv2.FONT_HERSHEY_SIMPLEX, 0.45, (200, 200, 200), 1)

        cv2.imshow("MSL Recognition  [Q=quit  C=clear]", frame)
        key = cv2.waitKey(1) & 0xFF
        if key == ord("q"):
            break
        elif key == ord("c"):
            kp_buffer.clear()
            last_text  = "Buffer cleared"
            last_color = (200, 200, 200)
            last_log   = ""

    cap.release()
    cv2.destroyAllWindows()


# ─── CLI ──────────────────────────────────────────────────────────────────────

def main():
    parser = argparse.ArgumentParser(description="MSL Sign Language Inference — ONNX Runtime")
    parser.add_argument("--onnx_model",      required=True)
    parser.add_argument("--metadata",        default=None)
    parser.add_argument("--label_map",       default=None)
    grp = parser.add_mutually_exclusive_group(required=True)
    grp.add_argument("--video",  help="Path to a video file")
    grp.add_argument("--webcam", type=int, metavar="CAM_IDX")
    parser.add_argument("--top_k",          type=int, default=5)
    parser.add_argument("--classify_every", type=int, default=15,
                        help="Classify every N webcam frames (default: 15)")
    parser.add_argument("--log_file",       default=None)
    parser.add_argument("--font_size",      type=int, default=32)
    parser.add_argument("--debug",          action="store_true")
    args = parser.parse_args()

    model_p = Path(args.onnx_model)
    meta_p  = args.metadata or str(model_p.with_suffix(".json"))
    lmap_p  = args.label_map or str(model_p.parent / "label_map.json")

    if not Path(lmap_p).exists():
        sys.exit(f"[ERROR] label_map.json not found: {lmap_p}\n"
                 f"        Copy it from your Linux server next to the .onnx file.")

    idx2label = load_idx2label(lmap_p)
    print(f"[INFO] Loaded {len(idx2label)} labels from label map.")

    model     = ONNXModel(str(model_p), meta_p)
    prep      = Preprocessor(max_seq_len=model.max_seq_len)
    extractor = KeypointExtractor(model_complexity=1)
    renderer  = TextRenderer(font_size=args.font_size)

    if args.log_file:
        print(f"[INFO] Logging to: {args.log_file}")

    try:
        if args.video:
            run_video(model, prep, extractor, renderer,
                      args.video, args.top_k, args.log_file, idx2label, args.debug)
        else:
            run_webcam(model, prep, extractor, renderer, idx2label,
                       args.webcam, args.top_k, args.log_file, args.classify_every)
    finally:
        extractor.close()
        print("[INFO] Done.")


if __name__ == "__main__":
    main()

```

## Export into ONNX

In [88]:
!ls

4paper					   plot.sh
bk					   README.md
config					   Real-Time-Inference.ipynb
cv.sh					   requirements.txt
data					   results
error_analysis.log			   results-folder-info.txt
exp-1					   run2.log
Export-to-ONNX-and-Infer-Experiment.ipynb  run_all.sh
fig					   run_inference.sh
inference.log				   run.log
launch_tensorboard.sh			   scripts
lookup.sh				   Single-Video-Inference.ipynb
MSL-Video-Translation-Tutorial.ipynb	   src
onnx_models				   src.zip
onnx_models.zip				   tmp
paper					   tools


In [89]:
!rm -r onnx_models

In [92]:
!time python src/export_to_onnx.py --config config/config.yaml --all

Device: cuda

  Exporting  BILSTM
  Classes: 556   MaxSeqLen: 200
  BiLSTM dummy inputs: kp=(2, 200, 225), mask=(2, 200)
/mnt/disk1/ye/exp/msl_recognition/.venv/lib/python3.12/site-packages/torch/onnx/symbolic_opset9.py:4244: UserWarning: Exporting a model to ONNX with a batch_size other than 1, with a variable length with LSTM can cause an error when running the ONNX model with a different batch size. Make sure to save the model with a batch size of 1, or define the initial states (h0/c0) as inputs of the model. 
  warnings.warn(
  Exported → onnx_models/bilstm_msl.onnx
  Validating ONNX model...
  ✓ ONNX graph is valid
  ✓ Inference OK — shape=(1, 556), top-5=[87, 256, 114, 112, 106]
  Metadata  → onnx_models/bilstm_msl.json  (~130 bytes — displays as 0.0 MB, this is normal)
  Label map → onnx_models/label_map.json
  ✓ Done  (18.4 MB)

  Exporting  TRANSFORMER
  Classes: 556   MaxSeqLen: 200
  Transformer dummy inputs: kp=(2, 200, 225), mask=(2, 200)
  Exported → onnx_models/transfor

In [93]:
!wc ./onnx_models/*

       7       14      130 ./onnx_models/bilstm_msl.json
   62902   359035 19266803 ./onnx_models/bilstm_msl.onnx
    1118     9352   116290 ./onnx_models/label_map.json
       7       14      129 ./onnx_models/stgcn_msl.json
   36700   206053 11143924 ./onnx_models/stgcn_msl.onnx
       7       14      135 ./onnx_models/transformer_msl.json
   45357   249033 13744518 ./onnx_models/transformer_msl.onnx
  146098   823515 44271929 total


## Infer on Windows OS  

မော်ဒယ်တွေနဲ့ updated two python ကုဒ်တွေကို Windows OS စက်ပေါ်ကို ကော်ပီကူးယူပြီး။ Infer testing ပြန်လုပ်။  

### BiLSTM  

(msl_onnx) E:\LU_Lab\MSL_Tran_Demo>python infer_onnx.py --onnx_model .\onnx_models\bilstm_msl.onnx --video .\sample4inference\idx20-30.mp4
[INFO] Loaded 556 labels from label map.
[INFO] ONNX loaded: bilstm_msl.onnx  (provider: CPUExecutionProvider)
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
[INFO] idx20-30.mp4 — 127 frames @ 30.0 fps
C:\Users\801680\Anaconda3\envs\msl_onnx\lib\site-packages\google\protobuf\symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '
[INFO] 127 frames extracted → using 127 (zero-padded to 200)

Video  : idx20-30.mp4
Model  : BILSTM
────────────────────────────────────────────────────────────
  Top-1: မနက် ၉ နာရီ က နေ ည ၁၀ နာရီ ထိ ပါ ။  (5.93%)
  Top-2: အမျိုးသမီး လူနာစောင့် ပဲ ည အိပ် စောင့် ရ ပါ မယ် ။  (4.63%)
  Top-3: ည ၁၀ နာရီ နောက် ပိုင်း အမျိုးသား နေ ခွင့် မ ရှိ ပါ ဘူး ။  (3.46%)
  Top-4: မိုးလေဝသ သတင်း များ ကို စဉ်ဆက်မပြတ် နားထောင် ပါ ။  (2.17%)
  Top-5: လူနာ တွေ့ ချိန် က ဘယ် အချိန် လဲ ။  (0.96%)
────────────────────────────────────────────────────────────
[INFO] Done.

(msl_onnx) E:\LU_Lab\MSL_Tran_Demo>

(msl_onnx) E:\LU_Lab\MSL_Tran_Demo>python infer_onnx.py --onnx_model .\onnx_models\bilstm_msl.onnx --video .\sample4inference\idx20-86.mp4
[INFO] Loaded 556 labels from label map.
[INFO] ONNX loaded: bilstm_msl.onnx  (provider: CPUExecutionProvider)
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
[INFO] idx20-86.mp4 — 169 frames @ 30.0 fps
C:\Users\801680\Anaconda3\envs\msl_onnx\lib\site-packages\google\protobuf\symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '
[INFO] 169 frames extracted → using 169 (zero-padded to 200)

Video  : idx20-86.mp4
Model  : BILSTM
────────────────────────────────────────────────────────────
  Top-1: အစားသောက် မ တည့် တာ တွေ ရှိ လား ။  (2.56%)
  Top-2: မိသားစုဝင် တယောက်ယောက် ကို အကြောင်းကြား ။  (2.47%)
  Top-3: ဆရာဝန် ဖုန်း နံပါတ် ခင်ဗျား မှာ ရှိ လား ။  (2.11%)
  Top-4: ည ၁၀ နာရီ နောက် ပိုင်း အမျိုးသား နေ ခွင့် မ ရှိ ပါ ဘူး ။  (1.76%)
  Top-5: သွားဆရာဝန် နဲ့ ဒီ နေ့ မနက် တွေ့ နိုင် မ လား ခင်ဗျာ ။  (1.27%)
────────────────────────────────────────────────────────────
[INFO] Done.

(msl_onnx) E:\LU_Lab\MSL_Tran_Demo>

(msl_onnx) E:\LU_Lab\MSL_Tran_Demo>python infer_onnx.py --onnx_model .\onnx_models\bilstm_msl.onnx --video .\sample4inference\idx20-244.mp4
[INFO] Loaded 556 labels from label map.
[INFO] ONNX loaded: bilstm_msl.onnx  (provider: CPUExecutionProvider)
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
[INFO] idx20-244.mp4 — 55 frames @ 30.0 fps
C:\Users\801680\Anaconda3\envs\msl_onnx\lib\site-packages\google\protobuf\symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '
[INFO] 55 frames extracted → using 55 (zero-padded to 200)

Video  : idx20-244.mp4
Model  : BILSTM
────────────────────────────────────────────────────────────
  Top-1: လူနာ နဲ့ ဘယ်လို တော် တာ လဲ ။  (6.74%)
  Top-2: ငါ ဒီ ဆေး တွေ ကို ဆက် သောက် သင့် သ လား ။  (1.35%)
  Top-3: ဘယ် နေရာ က နာ တာ လဲ ။  (1.01%)
  Top-4: ငါ့ ညီလေး အသက် မ ရှူ တော့ ဘူး ။  (0.92%)
  Top-5: ဘာ အကူညီ ပေး ရ မ လဲ ။  (0.81%)
────────────────────────────────────────────────────────────
[INFO] Done.

(msl_onnx) E:\LU_Lab\MSL_Tran_Demo>

(msl_onnx) E:\LU_Lab\MSL_Tran_Demo>python infer_onnx.py --onnx_model .\onnx_models\bilstm_msl.onnx --video .\sample4inference\idx20-291.mp4
[INFO] Loaded 556 labels from label map.
[INFO] ONNX loaded: bilstm_msl.onnx  (provider: CPUExecutionProvider)
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
[INFO] idx20-291.mp4 — 211 frames @ 30.0 fps
C:\Users\801680\Anaconda3\envs\msl_onnx\lib\site-packages\google\protobuf\symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '
[INFO] 211 frames extracted → using 200 (centre-cropped 200)

Video  : idx20-291.mp4
Model  : BILSTM
────────────────────────────────────────────────────────────
  Top-1: မနက် ၉ နာရီ က နေ ည ၁၀ နာရီ ထိ ပါ ။  (8.98%)
  Top-2: လူနာ တွေ့ ချိန် က ဘယ် အချိန် လဲ ။  (6.14%)
  Top-3: ကြမ်းပြင် တစ်ခု လုံး မှာ ရေ တွေ ကြည့် ပဲ ။  (1.78%)
  Top-4: လူနာစောင့် တစ်ယောက် ပဲ ည အိပ် စောင့် ခွင့် ရှိ ပါ မယ် ။  (1.03%)
  Top-5: ဆေး မ တည့် တာ တွေ ရှိ လား ။  (0.97%)
────────────────────────────────────────────────────────────
[INFO] Done.

(msl_onnx) E:\LU_Lab\MSL_Tran_Demo>

(msl_onnx) E:\LU_Lab\MSL_Tran_Demo>python infer_onnx.py --onnx_model .\onnx_models\bilstm_msl.onnx --video .\sample4inference\idx20-440.mp4
[INFO] Loaded 556 labels from label map.
[INFO] ONNX loaded: bilstm_msl.onnx  (provider: CPUExecutionProvider)
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
[INFO] idx20-440.mp4 — 248 frames @ 30.0 fps
C:\Users\801680\Anaconda3\envs\msl_onnx\lib\site-packages\google\protobuf\symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '
[INFO] 248 frames extracted → using 200 (centre-cropped 200)

Video  : idx20-440.mp4
Model  : BILSTM
────────────────────────────────────────────────────────────
  Top-1: ကား က ဘာ အရောင် လဲ ။  (1.50%)
  Top-2: ဆရာဝန် ဖုန်း နံပါတ် ခင်ဗျား မှာ ရှိ လား ။  (1.25%)
  Top-3: တိုက် သွား တဲ့ ကား ရဲ့ နံပါတ် နဲ့ အရောင် ကို ကောင်းကောင်း မှတ်မိ လား ။  (1.21%)
  Top-4: အနီးဆုံး ဆေးရုံ ကို ပို့ ပေး ပါ ။  (1.13%)
  Top-5: အရေပြား တွေ ခြောက် နေ တာ အတွက် ခရင်မ် လိမ်းဆေး လို ချင် ပါ တယ် ။  (1.08%)
────────────────────────────────────────────────────────────
[INFO] Done.

(msl_onnx) E:\LU_Lab\MSL_Tran_Demo>

## Transformer  

(msl_onnx) E:\LU_Lab\MSL_Tran_Demo>python infer_onnx.py --onnx_model .\onnx_models\transformer_msl.onnx --video .\sample4inference\idx20-30.mp4
[INFO] Loaded 556 labels from label map.
[INFO] ONNX loaded: transformer_msl.onnx  (provider: CPUExecutionProvider)
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
[INFO] idx20-30.mp4 — 127 frames @ 30.0 fps
C:\Users\801680\Anaconda3\envs\msl_onnx\lib\site-packages\google\protobuf\symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '
[INFO] 127 frames extracted → using 127 (zero-padded to 200)

Video  : idx20-30.mp4
Model  : TRANSFORMER
────────────────────────────────────────────────────────────
  Top-1: ခင်ဗျား ခံစား နေ ရ တာ ဘယ်လောက် ကြာ ပြီ လဲ ။  (4.47%)
  Top-2: လမ်းလျှောက် ဖို့ လိုအပ် တယ် ။  (4.30%)
  Top-3: မနက် ၉ နာရီ က နေ ည ၁၀ နာရီ ထိ ပါ ။  (2.32%)
  Top-4: ဒဏ်ရာ ကို ရေခဲ ကပ် ပေး ပါ ။  (1.98%)
  Top-5: သူ့ မိသားစု ကို ငါ တို့ ဖုန်းဆက် ရ မယ် ။  (1.69%)
────────────────────────────────────────────────────────────
[INFO] Done.

(msl_onnx) E:\LU_Lab\MSL_Tran_Demo>

(msl_onnx) E:\LU_Lab\MSL_Tran_Demo>python infer_onnx.py --onnx_model .\onnx_models\transformer_msl.onnx --video .\sample4inference\idx20-86.mp4
[INFO] Loaded 556 labels from label map.
[INFO] ONNX loaded: transformer_msl.onnx  (provider: CPUExecutionProvider)
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
[INFO] idx20-86.mp4 — 169 frames @ 30.0 fps
C:\Users\801680\Anaconda3\envs\msl_onnx\lib\site-packages\google\protobuf\symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '
[INFO] 169 frames extracted → using 169 (zero-padded to 200)

Video  : idx20-86.mp4
Model  : TRANSFORMER
────────────────────────────────────────────────────────────
  Top-1: လူနာ က ကျွန်တော့် အမေ ပါ ။  (5.48%)
  Top-2: ကောလဟာလ သတင်း တွေ ကို မ ယုံ နဲ့ ။  (4.22%)
  Top-3: သူ့ မိသားစု ကို ငါ တို့ ဖုန်းဆက် ရ မယ် ။  (3.68%)
  Top-4: ကျွန်မ ကလေး မှန်ကြည်ဆေး တွေ သောက် ပစ် လိုက် လို့ ပါ ။  (3.32%)
  Top-5: လတ်ဆတ် တဲ့ ဟင်းသီးဟင်းရွက် တွေ ကို စား ပါ ။  (2.74%)
────────────────────────────────────────────────────────────
[INFO] Done.

(msl_onnx) E:\LU_Lab\MSL_Tran_Demo>

(msl_onnx) E:\LU_Lab\MSL_Tran_Demo>python infer_onnx.py --onnx_model .\onnx_models\transformer_msl.onnx --video .\sample4inference\idx20-244.mp4
[INFO] Loaded 556 labels from label map.
[INFO] ONNX loaded: transformer_msl.onnx  (provider: CPUExecutionProvider)
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
[INFO] idx20-244.mp4 — 55 frames @ 30.0 fps
C:\Users\801680\Anaconda3\envs\msl_onnx\lib\site-packages\google\protobuf\symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '
[INFO] 55 frames extracted → using 55 (zero-padded to 200)

Video  : idx20-244.mp4
Model  : TRANSFORMER
────────────────────────────────────────────────────────────
  Top-1: လူနာ နဲ့ ဘယ်လို တော် တာ လဲ ။  (5.14%)
  Top-2: လတ်ဆတ် တဲ့ ဟင်းသီးဟင်းရွက် တွေ ကို စား ပါ ။  (3.30%)
  Top-3: ငါ ဒီ ဆေး တွေ ကို ဆက် သောက် သင့် သ လား ။  (2.75%)
  Top-4: သူ ကျွန်တော့် ကို ထိုး တယ် ။  (2.68%)
  Top-5: အစာ စား ပြီး မှ သောက် ပါ ။  (2.42%)
────────────────────────────────────────────────────────────
[INFO] Done.

(msl_onnx) E:\LU_Lab\MSL_Tran_Demo>

(msl_onnx) E:\LU_Lab\MSL_Tran_Demo>python infer_onnx.py --onnx_model .\onnx_models\transformer_msl.onnx --video .\sample4inference\idx20-291.mp4
[INFO] Loaded 556 labels from label map.
[INFO] ONNX loaded: transformer_msl.onnx  (provider: CPUExecutionProvider)
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
[INFO] idx20-291.mp4 — 211 frames @ 30.0 fps
C:\Users\801680\Anaconda3\envs\msl_onnx\lib\site-packages\google\protobuf\symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '
[INFO] 211 frames extracted → using 200 (centre-cropped 200)

Video  : idx20-291.mp4
Model  : TRANSFORMER
────────────────────────────────────────────────────────────
  Top-1: သူ့ မိသားစု ကို ငါ တို့ ဖုန်းဆက် ရ မယ် ။  (8.28%)
  Top-2: လတ်ဆတ် တဲ့ ဟင်းသီးဟင်းရွက် တွေ ကို စား ပါ ။  (6.06%)
  Top-3: လူနာစောင့် တစ်ယောက် ပဲ ည အိပ် စောင့် ခွင့် ရှိ ပါ မယ် ။  (3.09%)
  Top-4: အမျိုးသမီး လူနာစောင့် ပဲ ည အိပ် စောင့် ရ ပါ မယ် ။  (1.93%)
  Top-5: ပျောက် သွား တဲ့ လူ ရဲ့ ပုံပန်းသဏ္ဌာန် ကို ပြောပြ နိုင် မ လား ။  (1.08%)
────────────────────────────────────────────────────────────
[INFO] Done.

(msl_onnx) E:\LU_Lab\MSL_Tran_Demo>

(msl_onnx) E:\LU_Lab\MSL_Tran_Demo>python infer_onnx.py --onnx_model .\onnx_models\transformer_msl.onnx --video .\sample4inference\idx20-440.mp4
[INFO] Loaded 556 labels from label map.
[INFO] ONNX loaded: transformer_msl.onnx  (provider: CPUExecutionProvider)
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
[INFO] idx20-440.mp4 — 248 frames @ 30.0 fps
C:\Users\801680\Anaconda3\envs\msl_onnx\lib\site-packages\google\protobuf\symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '
[INFO] 248 frames extracted → using 200 (centre-cropped 200)

Video  : idx20-440.mp4
Model  : TRANSFORMER
────────────────────────────────────────────────────────────
  Top-1: တိုက် သွား တဲ့ ကား ရဲ့ နံပါတ် နဲ့ အရောင် ကို ကောင်းကောင်း မှတ်မိ လား ။  (13.84%)
  Top-2: လတ်ဆတ် တဲ့ ဟင်းသီးဟင်းရွက် တွေ ကို စား ပါ ။  (6.91%)
  Top-3: သူ့ မိသားစု ကို ငါ တို့ ဖုန်းဆက် ရ မယ် ။  (2.18%)
  Top-4: မိသားစုဝင် တယောက်ယောက် ကို အကြောင်းကြား ။  (1.94%)
  Top-5: အရေပြား အထူးကု ဒေါက်တာ ဒေါ်ခင်မြလွင် နဲ့ ပြ ချင် လို့ ပါ ။  (1.24%)
────────────────────────────────────────────────────────────
[INFO] Done.

(msl_onnx) E:\LU_Lab\MSL_Tran_Demo>

## STGCN

(msl_onnx) E:\LU_Lab\MSL_Tran_Demo>python infer_onnx.py --onnx_model .\onnx_models\stgcn_msl.onnx --video .\sample4inference\idx20-30.mp4
[INFO] Loaded 556 labels from label map.
[INFO] ONNX loaded: stgcn_msl.onnx  (provider: CPUExecutionProvider)
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
[INFO] idx20-30.mp4 — 127 frames @ 30.0 fps
C:\Users\801680\Anaconda3\envs\msl_onnx\lib\site-packages\google\protobuf\symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '
[INFO] 127 frames extracted → using 127 (zero-padded to 200)

Video  : idx20-30.mp4
Model  : STGCN
────────────────────────────────────────────────────────────
  Top-1: မြေနိမ့်ဒေသ ။  (8.49%)
  Top-2: တောမီး က အရမ်း အန္တရာယ် ကြီး တယ် ။  (2.76%)
  Top-3: ရေမြုပ် နေ တဲ့ နေရာ မှာ ကားမောင်း ဖို့ မ ကြိုးစား နဲ့ ။  (1.93%)
  Top-4: မင်း မီးလောင် တဲ့ အနံ့ မ ရ ဘူး လား ။  (1.91%)
  Top-5: တောင်စောင်း ။  (1.78%)
────────────────────────────────────────────────────────────
[INFO] Done.

(msl_onnx) E:\LU_Lab\MSL_Tran_Demo>

(msl_onnx) E:\LU_Lab\MSL_Tran_Demo>python infer_onnx.py --onnx_model .\onnx_models\stgcn_msl.onnx --video .\sample4inference\idx20-86.mp4
[INFO] Loaded 556 labels from label map.
[INFO] ONNX loaded: stgcn_msl.onnx  (provider: CPUExecutionProvider)
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
[INFO] idx20-86.mp4 — 169 frames @ 30.0 fps
C:\Users\801680\Anaconda3\envs\msl_onnx\lib\site-packages\google\protobuf\symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '
[INFO] 169 frames extracted → using 169 (zero-padded to 200)

Video  : idx20-86.mp4
Model  : STGCN
────────────────────────────────────────────────────────────
  Top-1: မနက် ၉ နာရီ က နေ ည ၁၀ နာရီ ထိ ပါ ။  (24.19%)
  Top-2: အစာ မ စား ခင် နာရီ ဝက် အလို မှာ သောက် ပါ ။  (8.06%)
  Top-3: ခေါင်း တွေ ခဏခဏ ကိုက် နေ တယ် ။  (7.14%)
  Top-4: သွားဆရာဝန် နဲ့ ဒီ နေ့ မနက် တွေ့ နိုင် မ လား ခင်ဗျာ ။  (3.21%)
  Top-5: 199 ကို ဖုန်းဆက် ပေး ပါ ။  (3.03%)
────────────────────────────────────────────────────────────
[INFO] Done.

(msl_onnx) E:\LU_Lab\MSL_Tran_Demo>

(msl_onnx) E:\LU_Lab\MSL_Tran_Demo>python infer_onnx.py --onnx_model .\onnx_models\stgcn_msl.onnx --video .\sample4inference\idx20-244.mp4
[INFO] Loaded 556 labels from label map.
[INFO] ONNX loaded: stgcn_msl.onnx  (provider: CPUExecutionProvider)
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
[INFO] idx20-244.mp4 — 55 frames @ 30.0 fps
C:\Users\801680\Anaconda3\envs\msl_onnx\lib\site-packages\google\protobuf\symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '
[INFO] 55 frames extracted → using 55 (zero-padded to 200)

Video  : idx20-244.mp4
Model  : STGCN
────────────────────────────────────────────────────────────
  Top-1: မ တပ် ထား ပါ ဘူး ။  (25.61%)
  Top-2: မြန်မြန် လုပ် ပါ ။  (7.39%)
  Top-3: ကား ကို ရပ် ထား ။  (3.51%)
  Top-4: ဂက်စ် မီး ။  (3.38%)
  Top-5: သူ ကျွန်တော့် ကို ထိုး တယ် ။  (2.96%)
────────────────────────────────────────────────────────────
[INFO] Done.

(msl_onnx) E:\LU_Lab\MSL_Tran_Demo>

(msl_onnx) E:\LU_Lab\MSL_Tran_Demo>python infer_onnx.py --onnx_model .\onnx_models\stgcn_msl.onnx --video .\sample4inference\idx20-291.mp4
[INFO] Loaded 556 labels from label map.
[INFO] ONNX loaded: stgcn_msl.onnx  (provider: CPUExecutionProvider)
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
[INFO] idx20-291.mp4 — 211 frames @ 30.0 fps
C:\Users\801680\Anaconda3\envs\msl_onnx\lib\site-packages\google\protobuf\symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '
[INFO] 211 frames extracted → using 200 (centre-cropped 200)

Video  : idx20-291.mp4
Model  : STGCN
────────────────────────────────────────────────────────────
  Top-1: ညစ်ညမ်း နေ တဲ့ နေရာ များ ကို သန့်ရှင်းရေး လုပ် ပြီး ရောဂါပိုး သတ် ရ ပါ မယ် ။  (4.19%)
  Top-2: ၄ ရက် ကြာ မယ် ။  (3.24%)
  Top-3: မနက် ၆ နာရီ နောက် ပိုင်း အစား မ စား ပါ နဲ့ ။  (3.21%)
  Top-4: ဖန်ကွဲစ ရှ မိ ပြီး သွေး တွေ တအား ထွက် နေ တယ် ။  (3.17%)
  Top-5: ဆေးစစ်ချက် တွေ ဘယ် အချိန် ရ မ လဲ ။  (2.53%)
────────────────────────────────────────────────────────────
[INFO] Done.

(msl_onnx) E:\LU_Lab\MSL_Tran_Demo>

(msl_onnx) E:\LU_Lab\MSL_Tran_Demo>python infer_onnx.py --onnx_model .\onnx_models\stgcn_msl.onnx --video .\sample4inference\idx20-440.mp4
[INFO] Loaded 556 labels from label map.
[INFO] ONNX loaded: stgcn_msl.onnx  (provider: CPUExecutionProvider)
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
[INFO] idx20-440.mp4 — 248 frames @ 30.0 fps
C:\Users\801680\Anaconda3\envs\msl_onnx\lib\site-packages\google\protobuf\symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '
[INFO] 248 frames extracted → using 200 (centre-cropped 200)

Video  : idx20-440.mp4
Model  : STGCN
────────────────────────────────────────────────────────────
  Top-1: ကား ထဲ မှာ လူ တစ်ယောက် ညှပ်မိ နေ တယ် ။  (10.89%)
  Top-2: ဘေး အိမ် ကို ဓားပြ ဝင် တိုက် နေ ပါ တယ် ။  (6.00%)
  Top-3: မိုးလေဝသ သတင်း များ ကို စဉ်ဆက်မပြတ် နားထောင် ပါ ။  (4.51%)
  Top-4: ပြိုကျ နိုင် တဲ့ အရာ များ နဲ့ ဝေးဝေး နေ ပါ ။  (4.19%)
  Top-5: သူ အခြေအနေ ကောင်း ပါ့ မ လား ။  (3.54%)
────────────────────────────────────────────────────────────
[INFO] Done.

(msl_onnx) E:\LU_Lab\MSL_Tran_Demo>



## Debugging with Claude

Let me read the notebook carefully to get the ground truth results for these exact videos.Now I have the ground truth. Let me build the complete comparison table:Confirmed. The root cause is clear: `model_complexity=1` everywhere in `infer_onnx.py` vs `model_complexity=2` for video/batch in `infer.py`. Now write the fixed file:## Root Cause Found

The problem is **`model_complexity=1` in `infer_onnx.py` vs `model_complexity=2` used during training and Linux inference.**

This was hiding in plain sight and the results themselves gave us the diagnostic clues to find it.

---

### The Evidence That Points Here

**Clue 1 — Confidence collapse (the smoking gun):**

| Video | Linux BiLSTM | Windows ONNX BiLSTM |
|---|---|---|
| idx20-30 | **91%** correct | 6% wrong |
| idx20-291 | **83%** correct | 9% wrong |
| idx20-440 | **93%** correct | 2% wrong |

A ~10× drop in confidence means the model is receiving inputs it has never seen before. The weights and computation are fine — the *features are alien to it*.

**Clue 2 — STGCN partially works (2/5 correct), BiLSTM/Transformer fail completely (0/5):**

STGCN's very first layer is `BatchNorm1d` which normalizes using running mean/variance baked in from training. It partially corrects a distribution shift. BiLSTM and Transformer start with `LayerNorm`, which normalizes using only the *current sample's* statistics — it cannot compensate for a distribution shift at all. This explains exactly why STGCN is more robust.

**Clue 3 — The complexity mismatch:**

```python
# Training (extract_keypoints.py default):
mp_pose.Pose(model_complexity=2, ...)   # full heavy model

# Linux infer.py:
_open_detectors(model_complexity=2)     # video ← complexity 2
_open_detectors(model_complexity=1)     # webcam ← complexity 1 (speed only)

# Windows infer_onnx.py (was wrong):
KeypointExtractor(model_complexity=1)   # EVERYTHING ← complexity 1 ← BUG
```

MediaPipe Pose complexity levels (0, 1, 2) are three entirely different neural network architectures producing measurably different landmark coordinates for the same frame. Using complexity=1 for inference when the model was trained on complexity=2 features is a guaranteed distribution mismatch.

---

### Fix Applied

- **Video mode**: `KeypointExtractor(model_complexity=2)` — matches training
- **Webcam mode**: `KeypointExtractor(model_complexity=1)` — matches `infer.py`'s choice for real-time speed
- **New `--model_complexity` CLI argument** — lets you override if your training used a different setting
- A startup info line now prints the complexity being used so this is never silent again

`export_to_onnx.py` is unchanged — the export itself was correct.

## Updated infer_onnx.py Code

In [95]:
from IPython.display import Markdown

file_path = './src/infer_onnx.py'

with open(file_path, 'r') as f:
    code = f.read()

# 'python' keyword triggers the syntax highlighter
display(Markdown(f"```python\n{code}\n```"))

```python
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
infer_onnx.py  –  Real-time MSL recognition on Windows using ONNX Runtime

Complete bug list (this version fixes ALL of them)
────────────────────────────────────────────────────

  [INFER-F] model_complexity=1 for video/batch, but training used complexity=2.
      *** THIS IS THE ROOT CAUSE OF ALL WRONG PREDICTIONS ON WINDOWS ***

      MediaPipe Pose has three complexity levels (0, 1, 2).  Each uses a
      different neural network with different architecture and weights, and
      they produce MEASURABLY DIFFERENT landmark coordinate values for the
      same input frame.  The differences are small visually but large enough
      to shift the normalised feature distribution away from what the model
      was trained on.

      Evidence from the notebook comparison:
        Linux infer.py  → correct predictions, 55–93% confidence
        Windows ONNX    → wrong predictions,    2–14% confidence (10× lower)

      The 10× confidence collapse is the diagnostic signature of a feature
      distribution mismatch: the model is running correctly but its inputs
      look like nothing it has ever seen.

      How training/Linux used complexity:
        extract_keypoints.py: default --complexity 2  → all .npy files use 2
        infer.py infer_video:  _open_detectors(model_complexity=2)       ← 2
        infer.py infer_batch:  _open_detectors(model_complexity=2)       ← 2
        infer.py infer_webcam: _open_detectors(model_complexity=1)       ← 1 (speed)

      How our infer_onnx.py was using complexity:
        KeypointExtractor(model_complexity=1) everywhere                  ← WRONG

      Fix: use model_complexity=2 for video and batch modes (matching
      training and infer.py).  Keep complexity=1 for webcam (same as
      infer.py, acceptable tradeoff for real-time speed).  Expose a
      --model_complexity CLI argument so users can override if needed.

      Note: MediaPipe Hands always caps at complexity=1 (MediaPipe limit),
      so min(complexity, 1) is applied to hands in both cases.

  [INFER-A] process_frame skipped frames where pose was not detected.
      Training (extract_keypoints.py / process_video) ALWAYS appends a
      keypoint array for EVERY frame.  When MediaPipe fails to detect the
      pose, extract_frame_keypoints() returns zeros for the pose block
      (and zeros for any undetected hands).  The frame is still included.

      The old code returned None when detected=False and the callers skipped
      those frames.  This caused two problems:
        1. The temporal structure of the sequence differs from training
           (missing frames change sign timing and duration).
        2. The sequence length differs from what normalization expects
           (shoulder width can become 0 in zero frames, but normalize_sequence
           handles that with the `where(dist < 1e-6, 1.0, dist)` guard).

      Fix: always return the keypoint array regardless of detection status.
      Zero rows for undetected joints are exactly what the model was trained on.

  [INFER-B] normalize_sequence used 2-D shoulders and 2-D scale.
      Training (extract_keypoints.normalize_sequence) uses full 3-D midpoint
      and 3-D Euclidean shoulder width applied to ALL three coordinates.
      The old code left z unscaled (only subtracted mid_z, never divided by
      scale) and used a 2-D scale that is always smaller than the 3-D one,
      inflating x and y magnitudes.  Every prediction was wrong as a result.
      Fix: exact mirror of extract_keypoints.normalize_sequence().

  [INFER-C] Webcam: buffer not cleared after each classification.
      infer.py clears the buffer after every classification so each sign is
      evaluated as a self-contained sequence, exactly as during training.
      The old code used a sliding/rolling window — fine for continuous
      recognition but poor for per-sign accuracy demos.
      Fix: clear kp_buffer after each classification, matching infer.py.

  [INFER-D] np.int64 / int dict key mismatch.
      np.argsort returns np.int64 indices.  dict.get(np.int64(k)) on an
      {int: str} dict silently returns None on some CPython/NumPy builds.
      Fix: always cast to int() before lookup via _get_label().

  [INFER-E] ONNXModel.predict double-indexed to scalar.
      [0][0] returned shape () instead of (num_classes,).
      Fix: return output[0] — shape (num_classes,).
"""

import argparse
import importlib
import json
import sys
from datetime import datetime
from pathlib import Path

import cv2
import numpy as np
import onnxruntime as ort

try:
    from PIL import Image, ImageDraw, ImageFont
    HAS_PIL = True
except ImportError:
    HAS_PIL = False

# ── Constants (must match extract_keypoints.py exactly) ───────────────────────
POSE_N          = 33
HAND_N          = 21
TOTAL_JOINTS    = POSE_N + HAND_N * 2   # 75
COORDS          = 3
FEAT_DIM        = TOTAL_JOINTS * COORDS  # 225

POSE_LEFT_SHOULDER  = 11
POSE_RIGHT_SHOULDER = 12
LEFT_HAND_START     = POSE_N            # 33
RIGHT_HAND_START    = POSE_N + HAND_N   # 54


# ─── MediaPipe import ─────────────────────────────────────────────────────────

def _import_mp():
    for prefix in ["mediapipe.solutions", "mediapipe.python.solutions"]:
        try:
            pose  = importlib.import_module(f"{prefix}.pose")
            hands = importlib.import_module(f"{prefix}.hands")
            draw  = importlib.import_module(f"{prefix}.drawing_utils")
            return pose, hands, draw
        except (ImportError, ModuleNotFoundError):
            continue
    try:
        import mediapipe as _mp
        return _mp.solutions.pose, _mp.solutions.hands, _mp.solutions.drawing_utils
    except AttributeError:
        pass
    raise ImportError(
        "Cannot import MediaPipe solutions.\n"
        "Try: pip uninstall mediapipe -y && pip install mediapipe==0.10.18"
    )

mp_pose, mp_hands, mp_drawing = _import_mp()


# ─── Myanmar text rendering ──────────────────────────────────────────────────

class TextRenderer:
    _FONT_CANDIDATES = [
        "C:\\Windows\\Fonts\\Padauk.ttf",
        "C:\\Windows\\Fonts\\NotoSansMyanmar-Regular.ttf",
        "C:\\Windows\\Fonts\\NotoSansMyanmar.ttf",
        "C:\\Windows\\Fonts\\myanmar3.ttf",
        "/usr/share/fonts/truetype/padauk/Padauk-Regular.ttf",
        "/usr/share/fonts/truetype/noto/NotoSansMyanmar-Regular.ttf",
        "fonts\\Padauk.ttf",
        "fonts/NotoSansMyanmar-Regular.ttf",
    ]

    def __init__(self, font_size: int = 32):
        self.font = self._load_font(font_size)
        self.can_render_myanmar = (self.font is not None)
        if not self.can_render_myanmar:
            print()
            print("  ╔═══════════════════════════════════════════════════════════╗")
            print("  ║  INFO: No Myanmar shaping font found.                    ║")
            print("  ║  Displaying readable Class IDs instead of Myanmar text.  ║")
            print("  ╚═══════════════════════════════════════════════════════════╝")
            print()

    def _load_font(self, size):
        if not HAS_PIL:
            return None
        for path in self._FONT_CANDIDATES:
            if Path(path).exists():
                try:
                    return ImageFont.truetype(path, size)
                except Exception:
                    continue
        return None

    def put_text(self, frame, text, position=(20, 20), color=(0, 255, 0)):
        if self.font and HAS_PIL:
            pil = Image.fromarray(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
            d   = ImageDraw.Draw(pil)
            r, g, b = color[2], color[1], color[0]
            for dx in [-1, 0, 1]:
                for dy in [-1, 0, 1]:
                    if dx or dy:
                        d.text((position[0]+dx, position[1]+dy), text,
                               font=self.font, fill=(0, 0, 0))
            d.text(position, text, font=self.font, fill=(r, g, b))
            return cv2.cvtColor(np.array(pil), cv2.COLOR_RGB2BGR)
        cv2.putText(frame, text, position, cv2.FONT_HERSHEY_SIMPLEX,
                    0.75, color, 2, cv2.LINE_AA)
        return frame


# ─── Keypoint extraction ──────────────────────────────────────────────────────

class KeypointExtractor:
    """
    Extracts (TOTAL_JOINTS, 3) keypoint arrays from BGR frames.

    BUG FIX [INFER-A]:
    process_frame now ALWAYS returns a keypoint array, never None.
    When MediaPipe fails to detect the pose, the pose block is all zeros.
    When hands are not detected, those blocks are all zeros.
    This exactly mirrors extract_keypoints.py / process_video() behaviour,
    which appends extract_frame_keypoints() output for EVERY frame without
    any None-filtering.

    Hand convention (matches extract_keypoints.py):
      MediaPipe reports handedness from the camera perspective (mirrored).
      label='Right' → signer's LEFT hand  → slot LEFT_HAND_START (33).
      label='Left'  → signer's RIGHT hand → slot RIGHT_HAND_START (54).
    """

    def __init__(self, model_complexity: int = 2):
        self.pose  = mp_pose.Pose(
            static_image_mode=False,
            model_complexity=model_complexity,
            smooth_landmarks=True,
            enable_segmentation=False,
            min_detection_confidence=0.5,
            min_tracking_confidence=0.5,
        )
        self.hands = mp_hands.Hands(
            static_image_mode=False,
            max_num_hands=2,
            model_complexity=min(model_complexity, 1),
            min_detection_confidence=0.5,
            min_tracking_confidence=0.5,
        )
        self._last_pose  = None
        self._last_hands = None

    def process_frame(self, frame: np.ndarray) -> np.ndarray:
        """
        Returns (TOTAL_JOINTS, 3) float32 array for every frame.
        Zero rows for any joint/hand that was not detected.
        Never returns None — every frame is included, same as training.
        """
        rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        rgb.flags.writeable = False
        p_res = self.pose.process(rgb)
        h_res = self.hands.process(rgb)
        self._last_pose  = p_res
        self._last_hands = h_res

        kp = np.zeros((TOTAL_JOINTS, COORDS), dtype=np.float32)

        # Pose — zeros if not detected (same as extract_keypoints._lm_to_array)
        if p_res.pose_landmarks:
            for i, lm in enumerate(p_res.pose_landmarks.landmark):
                kp[i] = [lm.x, lm.y, lm.z]

        # Hands — zeros for each undetected hand
        if h_res.multi_hand_landmarks and h_res.multi_handedness:
            for lm_list, handedness in zip(
                h_res.multi_hand_landmarks, h_res.multi_handedness
            ):
                label  = handedness.classification[0].label
                offset = LEFT_HAND_START if label == 'Right' else RIGHT_HAND_START
                for i, lm in enumerate(lm_list.landmark):
                    kp[offset + i] = [lm.x, lm.y, lm.z]

        # Always return the array — never None
        return kp

    def draw_landmarks(self, frame: np.ndarray):
        if self._last_pose and self._last_pose.pose_landmarks:
            mp_drawing.draw_landmarks(
                frame, self._last_pose.pose_landmarks, mp_pose.POSE_CONNECTIONS
            )
        if self._last_hands and self._last_hands.multi_hand_landmarks:
            for lm in self._last_hands.multi_hand_landmarks:
                mp_drawing.draw_landmarks(frame, lm, mp_hands.HAND_CONNECTIONS)

    def close(self):
        self.pose.close()
        self.hands.close()


# ─── Normalization ────────────────────────────────────────────────────────────

def normalize_sequence(seq: np.ndarray) -> np.ndarray:
    """
    BUG FIX [INFER-B]: Exact mirror of extract_keypoints.normalize_sequence().

    Translates to shoulder-midpoint origin (full 3-D) and scales by
    3-D Euclidean shoulder width.  Applied to ALL three coordinate axes.

    Training code (extract_keypoints.py):
        mid  = (l_sh + r_sh) / 2.0         shape (T, 3)
        dist = ||r_sh - l_sh||_2            shape (T, 1)  — 3-D Euclidean
        out  = (seq - mid[:, None, :]) / dist[:, None, :]

    The old code used a 2-D scale (x,y only) and did NOT divide z by scale.
    This produced inflated x/y values and wrong z values → every prediction wrong.

    Args:
        seq: (T, 75, 3) raw keypoints from KeypointExtractor.
    Returns:
        (T, 75, 3) normalised float32, matching training distribution exactly.
    """
    l_sh = seq[:, POSE_LEFT_SHOULDER,  :]   # (T, 3)
    r_sh = seq[:, POSE_RIGHT_SHOULDER, :]   # (T, 3)
    mid  = (l_sh + r_sh) / 2.0              # (T, 3)
    dist = np.linalg.norm(r_sh - l_sh, axis=1, keepdims=True)   # (T, 1)
    dist = np.where(dist < 1e-6, 1.0, dist)
    return ((seq - mid[:, None, :]) / dist[:, None, :]).astype(np.float32)


# ─── Preprocessor ─────────────────────────────────────────────────────────────

class Preprocessor:
    """
    Stacks raw (75, 3) keypoint frames, normalises, crops/pads, and builds
    the model-ready input dict.

    Processing order (matches infer.py exactly):
      1. Stack → (T, 75, 3)
      2. Normalize (shoulder-centred, 3-D scale, all axes)
      3. Centre-crop if T > max_seq_len
      4. Zero-pad if T < max_seq_len
      5. Build mask (True = padded position)
      6. Reshape for model type
    """

    def __init__(self, max_seq_len: int = 200):
        self.max_seq_len = max_seq_len

    def __call__(self, kp_list: list, model_type: str) -> dict:
        if not kp_list:
            return {}

        seq = np.stack(kp_list, axis=0).astype(np.float32)  # (T, 75, 3)
        seq = normalize_sequence(seq)
        T   = seq.shape[0]

        # Centre-crop
        if T > self.max_seq_len:
            start = (T - self.max_seq_len) // 2
            seq   = seq[start: start + self.max_seq_len]
            T     = self.max_seq_len

        real_len = T

        # Zero-pad
        if T < self.max_seq_len:
            pad = np.zeros((self.max_seq_len - T, TOTAL_JOINTS, COORDS), dtype=np.float32)
            seq = np.concatenate([seq, pad], axis=0)

        # Mask: True where padded
        mask = np.zeros(self.max_seq_len, dtype=np.bool_)
        mask[real_len:] = True

        if model_type == "stgcn":
            # (T, 75, 3) → (3, T, 75) → (1, 3, T, 75)
            kp_out = np.expand_dims(seq.transpose(2, 0, 1), axis=0).astype(np.float32)
            return {"keypoints": kp_out}
        else:
            # (T, 75, 3) → (T, 225) → (1, T, 225)
            kp_out = np.expand_dims(
                seq.reshape(self.max_seq_len, FEAT_DIM), axis=0
            ).astype(np.float32)
            return {"keypoints": kp_out, "mask": mask[np.newaxis]}


# ─── ONNX inference wrapper ───────────────────────────────────────────────────

class ONNXModel:
    def __init__(self, onnx_path: str, meta_path: str = None):
        available = ort.get_available_providers()
        preferred = ["CUDAExecutionProvider", "DmlExecutionProvider", "CPUExecutionProvider"]
        providers = [p for p in preferred if p in available]

        self.session = ort.InferenceSession(onnx_path, providers=providers)
        print(f"[INFO] ONNX loaded: {Path(onnx_path).name}"
              f"  (provider: {self.session.get_providers()[0]})")

        self.meta = {}
        if meta_path and Path(meta_path).exists():
            try:
                with open(meta_path, encoding="utf-8") as f:
                    self.meta = json.load(f)
            except Exception as e:
                print(f"[WARNING] Could not read metadata: {e}")
        else:
            print(f"[INFO] Metadata not found at '{meta_path}', using defaults.")

        self.model_type  = self.meta.get("model_type",  "transformer")
        self.max_seq_len = self.meta.get("max_seq_len", 200)
        self.num_classes = self.meta.get("num_classes", 558)

    def predict(self, inputs: dict) -> np.ndarray:
        """
        Run one forward pass.
        BUG FIX [INFER-E]: return 1-D (num_classes,) array, not scalar.
        output shape from session: (1, num_classes); [0] gives (num_classes,).
        """
        return self.session.run(None, inputs)[0][0]   # (num_classes,)

    @staticmethod
    def softmax(logits: np.ndarray) -> np.ndarray:
        e = np.exp(logits - logits.max())
        return e / e.sum()


# ─── Label map ────────────────────────────────────────────────────────────────

def load_idx2label(path: str) -> dict:
    with open(path, encoding="utf-8") as f:
        data = json.load(f)
    return {int(k): v for k, v in data.get("idx2label", {}).items()}


def _get_label(idx2label: dict, class_id) -> str:
    """
    BUG FIX [INFER-D]: always cast to int() — np.int64 keys miss {int:str} dicts.
    """
    return idx2label.get(int(class_id), f"[class {int(class_id)}]")


# ─── Display helpers ──────────────────────────────────────────────────────────

def _build_text(class_id, label, confidence, can_myanmar,
                top_k=1, all_ids=None, probs=None, lbl_map=None):
    if top_k == 1:
        return (f"{label}  ({confidence*100:.1f}%)" if can_myanmar
                else f"Class {int(class_id)}  ({confidence*100:.1f}%)")
    parts = []
    for i, cid in enumerate(all_ids):
        lbl = _get_label(lbl_map, cid) if (can_myanmar and lbl_map) else f"Class {int(cid)}"
        parts.append(f"#{i+1} {lbl} ({probs[int(cid)]*100:.0f}%)")
    return "  |  ".join(parts)


def _build_log(class_id, label, confidence, top_k=1, all_ids=None, probs=None, lbl_map=None):
    safe = label.encode("utf-8", errors="replace").decode("utf-8")
    if top_k == 1:
        return f"  Class {int(class_id):>4d} | {safe:<40s} | {confidence*100:6.2f}%"
    lines = []
    for i, cid in enumerate(all_ids):
        lbl = _get_label(lbl_map, cid) if lbl_map else "?"
        lbl = lbl.encode("utf-8", errors="replace").decode("utf-8")
        lines.append(f"  Top-{i+1}: Class {int(cid):>4d} | {lbl:<40s} | {probs[int(cid)]*100:6.2f}%")
    return "\n".join(lines)


def _append_log(path, text):
    try:
        with open(path, "a", encoding="utf-8-sig") as f:
            f.write(text)
    except Exception:
        pass


# ─── Video processing ─────────────────────────────────────────────────────────

def run_video(model, prep, extractor, renderer, video_path,
              top_k=5, log_file=None, idx2label=None, debug=False):
    """
    Process a single video file for sign language recognition.

    Matches infer.py / process_video() behaviour exactly:
      - Every frame is processed and appended (no skipping for missing pose).
      - Normalization applied after full sequence is stacked.
      - Centre-crop or zero-pad to max_seq_len.
    """
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        print(f"[ERROR] Cannot open video: {video_path}")
        return

    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    fps_src      = cap.get(cv2.CAP_PROP_FPS) or 30.0
    print(f"[INFO] {Path(video_path).name} — {total_frames} frames @ {fps_src:.1f} fps")
    print(f"[INFO] MediaPipe pose complexity: {extractor.pose.model_complexity}"
          f"  (must match training, default=2)")

    kp_list = []
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        # BUG FIX [INFER-A]: process_frame always returns array (never None)
        kp_list.append(extractor.process_frame(frame))
    cap.release()

    if not kp_list:
        print("[WARNING] No frames could be read from the video.")
        return

    T_raw = len(kp_list)
    T_use = min(T_raw, model.max_seq_len)
    print(f"[INFO] {T_raw} frames extracted → using {T_use} "
          f"({'centre-cropped' if T_raw > model.max_seq_len else 'zero-padded to'}"
          f" {model.max_seq_len})")

    inputs = prep(kp_list, model.model_type)
    logits = model.predict(inputs)                    # (num_classes,)
    probs  = ONNXModel.softmax(logits)               # (num_classes,)
    top_idx = np.argsort(probs)[::-1][:top_k]

    sep = "─" * 60
    lines = [f"Video  : {Path(video_path).name}",
             f"Model  : {model.model_type.upper()}", sep]
    for rank, idx in enumerate(top_idx, 1):
        lbl = _get_label(idx2label, idx) if idx2label else f"[class {int(idx)}]"
        lines.append(f"  Top-{rank}: {lbl}  ({probs[int(idx)]*100:.2f}%)")
    lines.append(sep)
    print("\n" + "\n".join(lines))

    if log_file:
        log_lines = [f"[{datetime.now().isoformat()}] {Path(video_path).name}"]
        for idx in top_idx:
            lbl = _get_label(idx2label, idx) if idx2label else "?"
            log_lines.append(_build_log(idx, lbl, probs[int(idx)],
                                        top_k=1, lbl_map=idx2label))
        _append_log(log_file, "\n".join(log_lines) + "\n\n")


# ─── Webcam processing ────────────────────────────────────────────────────────

def run_webcam(model, prep, extractor, renderer, idx2label,
               cam_idx=0, top_k=1, log_file=None, classify_every=15):
    """
    Live webcam sign recognition.

    BUG FIX [INFER-A]: Every frame is always appended to the buffer —
    no None-filtering — because process_frame always returns an array.

    BUG FIX [INFER-C]: Buffer is cleared after each classification,
    matching infer.py's behaviour.  Each sign is evaluated as an
    independent self-contained sequence, exactly as during training.
    Press C to manually clear the buffer; press Q to quit.
    """
    cap = cv2.VideoCapture(cam_idx)
    if not cap.isOpened():
        print(f"[ERROR] Cannot open webcam {cam_idx}")
        return

    max_buf    = model.max_seq_len
    kp_buffer  = []
    last_text  = "Collecting frames..."
    last_color = (200, 200, 200)
    last_log   = ""
    frame_cnt  = 0
    print("[INFO] Webcam started. Sign into camera, then wait for prediction.")
    print("[INFO] Q = quit | C = clear buffer manually")

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        frame_cnt += 1
        # BUG FIX [INFER-A]: always append, never skip
        kp_buffer.append(extractor.process_frame(frame))
        if len(kp_buffer) > max_buf:
            kp_buffer.pop(0)

        if len(kp_buffer) >= 20 and frame_cnt % classify_every == 0:
            inputs  = prep(list(kp_buffer), model.model_type)
            logits  = model.predict(inputs)
            probs   = ONNXModel.softmax(logits)
            top_idx = np.argsort(probs)[::-1][:top_k]

            best_idx  = top_idx[0]
            best_prob = probs[int(best_idx)]
            best_lbl  = _get_label(idx2label, best_idx) if idx2label else f"Class {int(best_idx)}"

            last_text  = _build_text(best_idx, best_lbl, best_prob,
                                     renderer.can_render_myanmar,
                                     top_k, top_idx, probs, idx2label)
            last_color = ((0, 200, 0) if best_prob >= 0.80 else
                          (0, 200, 200) if best_prob >= 0.50 else (0, 0, 255))

            # BUG FIX [INFER-C]: clear buffer after each classification
            # so the next sign starts fresh, matching infer.py behaviour
            kp_buffer.clear()

            if log_file and last_text != last_log:
                ts = datetime.now().strftime("%H:%M:%S")
                _append_log(log_file,
                    f"[{ts}] " + _build_log(best_idx, best_lbl, best_prob,
                                             top_k=1, lbl_map=idx2label) + "\n")
                last_log = last_text

        extractor.draw_landmarks(frame)
        frame = renderer.put_text(frame, last_text, position=(10, 10), color=last_color)

        h, w  = frame.shape[:2]
        fill  = int(len(kp_buffer) / max(max_buf, 1) * w)
        cv2.rectangle(frame, (0, h - 8), (fill, h), (0, 180, 255), -1)
        cv2.putText(frame, f"buf {len(kp_buffer)}/{max_buf}",
                    (10, h - 12), cv2.FONT_HERSHEY_SIMPLEX, 0.45, (200, 200, 200), 1)

        cv2.imshow("MSL Recognition  [Q=quit  C=clear]", frame)
        key = cv2.waitKey(1) & 0xFF
        if key == ord("q"):
            break
        elif key == ord("c"):
            kp_buffer.clear()
            last_text  = "Buffer cleared"
            last_color = (200, 200, 200)
            last_log   = ""

    cap.release()
    cv2.destroyAllWindows()


# ─── CLI ──────────────────────────────────────────────────────────────────────

def main():
    parser = argparse.ArgumentParser(description="MSL Sign Language Inference — ONNX Runtime")
    parser.add_argument("--onnx_model",      required=True)
    parser.add_argument("--metadata",        default=None)
    parser.add_argument("--label_map",       default=None)
    grp = parser.add_mutually_exclusive_group(required=True)
    grp.add_argument("--video",  help="Path to a video file")
    grp.add_argument("--webcam", type=int, metavar="CAM_IDX")
    parser.add_argument("--top_k",          type=int, default=5)
    parser.add_argument("--classify_every", type=int, default=15,
                        help="Classify every N webcam frames (default: 15)")
    parser.add_argument("--log_file",       default=None)
    parser.add_argument("--font_size",      type=int, default=32)
    parser.add_argument("--model_complexity", type=int, default=None,
                        help="MediaPipe pose model complexity: 0, 1, or 2. "
                             "Default: 2 for video (matches training), 1 for webcam (speed). "
                             "MUST match what was used during extract_keypoints.py.")
    parser.add_argument("--debug",          action="store_true")
    args = parser.parse_args()

    model_p = Path(args.onnx_model)
    meta_p  = args.metadata or str(model_p.with_suffix(".json"))
    lmap_p  = args.label_map or str(model_p.parent / "label_map.json")

    if not Path(lmap_p).exists():
        sys.exit(f"[ERROR] label_map.json not found: {lmap_p}\n"
                 f"        Copy it from your Linux server next to the .onnx file.")

    idx2label = load_idx2label(lmap_p)
    print(f"[INFO] Loaded {len(idx2label)} labels from label map.")

    model  = ONNXModel(str(model_p), meta_p)
    prep   = Preprocessor(max_seq_len=model.max_seq_len)

    # Fix [INFER-F]: complexity must match training (=2 for video, =1 for webcam).
    # Training used extract_keypoints.py --complexity 2 (the default).
    # infer.py uses complexity=2 for video/batch, complexity=1 for webcam.
    if args.model_complexity is not None:
        video_complexity  = args.model_complexity
        webcam_complexity = args.model_complexity
    else:
        video_complexity  = 2   # matches training and infer.py infer_video()
        webcam_complexity = 1   # matches infer.py infer_webcam() — speed tradeoff

    renderer = TextRenderer(font_size=args.font_size)

    if args.log_file:
        print(f"[INFO] Logging to: {args.log_file}")

    try:
        if args.video:
            extractor = KeypointExtractor(model_complexity=video_complexity)
            try:
                run_video(model, prep, extractor, renderer,
                          args.video, args.top_k, args.log_file, idx2label, args.debug)
            finally:
                extractor.close()
        else:
            extractor = KeypointExtractor(model_complexity=webcam_complexity)
            try:
                run_webcam(model, prep, extractor, renderer, idx2label,
                           args.webcam, args.top_k, args.log_file, args.classify_every)
            finally:
                extractor.close()
    finally:
        print("[INFO] Done.")


if __name__ == "__main__":
    main()

```

## Inferring a Pretrained Model

ဒီတစ်ခါ နောက်ဆုံး Infer ထင်ပါတယ် ♫⋆｡ ♪ ₊˚♬ 

```
(msl_onnx) E:\LU_Lab\MSL_Tran_Demo>python infer_onnx.py --onnx_model .\onnx_models\bilstm_msl.onnx --video .\sample4inference\idx20-30.mp4
[INFO] Loaded 556 labels from label map.
[INFO] ONNX loaded: bilstm_msl.onnx  (provider: CPUExecutionProvider)
Downloading model to C:\Users\801680\Anaconda3\envs\msl_onnx\lib\site-packages\mediapipe/modules/pose_landmark/pose_landmark_heavy.tflite
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
[INFO] idx20-30.mp4 — 127 frames @ 30.0 fps
[INFO] Done.
Traceback (most recent call last):
  File "E:\LU_Lab\MSL_Tran_Demo\infer_onnx.py", line 680, in <module>
    main()
  File "E:\LU_Lab\MSL_Tran_Demo\infer_onnx.py", line 664, in main
    run_video(model, prep, extractor, renderer,
  File "E:\LU_Lab\MSL_Tran_Demo\infer_onnx.py", line 474, in run_video
    print(f"[INFO] MediaPipe pose complexity: {extractor.pose.model_complexity}"
AttributeError: 'Pose' object has no attribute 'model_complexity'

(msl_onnx) E:\LU_Lab\MSL_Tran_Demo>
```

## Debugging Again

Let me read everything carefully and do a deep systematic analysis.Good. The crash is a single bad attribute access. Now let me fix it properly and do a complete final clean version:## What went wrong and the full picture

### The immediate crash

```
AttributeError: 'Pose' object has no attribute 'model_complexity'
```

The MediaPipe `Pose` context manager object on Windows does **not** expose a public `.model_complexity` attribute in mediapipe 0.10.x builds. We added that diagnostic print line last round but referenced the wrong object. The fix is two lines — store complexity as `self.model_complexity` on `KeypointExtractor`, then read `extractor.model_complexity` instead of `extractor.pose.model_complexity`.

### Full deep analysis summary

I re-read every file in the pipeline — `extract_keypoints.py`, `augment.py`, `dataset.py`, `train.py`, `infer.py`, `bilstm.py`, `transformer_model.py`, `stgcn.py`, `export_to_onnx.py`, and `infer_onnx.py` — and cross-checked each stage:

**Training pipeline:** `extract_keypoints.py` uses complexity=2, saves normalized (T,75,3) `.npy`. `augment.py` loads these and saves augmented copies (still normalized). `MSLDataset` crops/pads and masks. The BiLSTM during training calls `pack_padded_sequence` via `lengths`. The Transformer uses `src_key_padding_mask`. ✓

**BiLSTM lengths=None correctness:** With `lengths=None` in the ONNX wrapper the LSTM runs on the full zero-padded sequence. I verified this is numerically equivalent: the forward LSTM is causal so `h_0..h_{T-1}` are identical regardless of what follows. The backward LSTM processes zeros from zero initial state — with forget bias initialized to 1.0 the cell state stays near-zero through all-zero inputs, so `h_backward` at real positions is effectively the same as if packing stopped there. The attention mask then correctly excludes padded positions. The ONNX export is numerically sound. ✓

**Transformer wrapper:** Every step matches the real `SignTransformer.forward` exactly — input norm → proj → proj norm → CLS prepend → mask update → pos enc → manual encoder layers (pre-LN path, ManualMHA, inner FFN dropout included) → encoder norm → CLS token → head norm → head dropout → FC. ✓

**STGCN wrapper:** Edge importance pre-baked into adjacency buffers, `edge_importance_weighting=False` set so no in-place ops during tracing. ✓

**`infer_onnx.py`:** After the crash fix, every component is correct — normalize_sequence uses full 3D (matching training), all frames included, complexity=2 for video matching training, buffer cleared after webcam classification, `int()` cast for dict lookups, predict returns 1D array. ✓

**`export_to_onnx.py`:** No changes needed. All three model exports are correct.

The crash was the only remaining issue. `export_to_onnx.py` does not need to be re-exported — your existing `.onnx` files are fine.

## Updated Code

In [96]:
from IPython.display import Markdown

file_path = './src/infer_onnx.py'

with open(file_path, 'r') as f:
    code = f.read()

# 'python' keyword triggers the syntax highlighter
display(Markdown(f"```python\n{code}\n```"))

```python
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
infer_onnx.py  –  Real-time MSL recognition on Windows using ONNX Runtime

Complete bug list (this version fixes ALL of them)
────────────────────────────────────────────────────

  [INFER-F] model_complexity=1 for video/batch, but training used complexity=2.
      *** THIS IS THE ROOT CAUSE OF ALL WRONG PREDICTIONS ON WINDOWS ***

      MediaPipe Pose has three complexity levels (0, 1, 2).  Each uses a
      different neural network with different architecture and weights, and
      they produce MEASURABLY DIFFERENT landmark coordinate values for the
      same input frame.  The differences are small visually but large enough
      to shift the normalised feature distribution away from what the model
      was trained on.

      Evidence from the notebook comparison:
        Linux infer.py  → correct predictions, 55–93% confidence
        Windows ONNX    → wrong predictions,    2–14% confidence (10× lower)

      The 10× confidence collapse is the diagnostic signature of a feature
      distribution mismatch: the model is running correctly but its inputs
      look like nothing it has ever seen.

      How training/Linux used complexity:
        extract_keypoints.py: default --complexity 2  → all .npy files use 2
        infer.py infer_video:  _open_detectors(model_complexity=2)       ← 2
        infer.py infer_batch:  _open_detectors(model_complexity=2)       ← 2
        infer.py infer_webcam: _open_detectors(model_complexity=1)       ← 1 (speed)

      How our infer_onnx.py was using complexity:
        KeypointExtractor(model_complexity=1) everywhere                  ← WRONG

      Fix: use model_complexity=2 for video and batch modes (matching
      training and infer.py).  Keep complexity=1 for webcam (same as
      infer.py, acceptable tradeoff for real-time speed).  Expose a
      --model_complexity CLI argument so users can override if needed.

      Note: MediaPipe Hands always caps at complexity=1 (MediaPipe limit),
      so min(complexity, 1) is applied to hands in both cases.

  [INFER-A] process_frame skipped frames where pose was not detected.
      Training (extract_keypoints.py / process_video) ALWAYS appends a
      keypoint array for EVERY frame.  When MediaPipe fails to detect the
      pose, extract_frame_keypoints() returns zeros for the pose block
      (and zeros for any undetected hands).  The frame is still included.

      The old code returned None when detected=False and the callers skipped
      those frames.  This caused two problems:
        1. The temporal structure of the sequence differs from training
           (missing frames change sign timing and duration).
        2. The sequence length differs from what normalization expects
           (shoulder width can become 0 in zero frames, but normalize_sequence
           handles that with the `where(dist < 1e-6, 1.0, dist)` guard).

      Fix: always return the keypoint array regardless of detection status.
      Zero rows for undetected joints are exactly what the model was trained on.

  [INFER-B] normalize_sequence used 2-D shoulders and 2-D scale.
      Training (extract_keypoints.normalize_sequence) uses full 3-D midpoint
      and 3-D Euclidean shoulder width applied to ALL three coordinates.
      The old code left z unscaled (only subtracted mid_z, never divided by
      scale) and used a 2-D scale that is always smaller than the 3-D one,
      inflating x and y magnitudes.  Every prediction was wrong as a result.
      Fix: exact mirror of extract_keypoints.normalize_sequence().

  [INFER-C] Webcam: buffer not cleared after each classification.
      infer.py clears the buffer after every classification so each sign is
      evaluated as a self-contained sequence, exactly as during training.
      The old code used a sliding/rolling window — fine for continuous
      recognition but poor for per-sign accuracy demos.
      Fix: clear kp_buffer after each classification, matching infer.py.

  [INFER-D] np.int64 / int dict key mismatch.
      np.argsort returns np.int64 indices.  dict.get(np.int64(k)) on an
      {int: str} dict silently returns None on some CPython/NumPy builds.
      Fix: always cast to int() before lookup via _get_label().

  [INFER-E] ONNXModel.predict double-indexed to scalar.
      [0][0] returned shape () instead of (num_classes,).
      Fix: return output[0] — shape (num_classes,).
"""

import argparse
import importlib
import json
import sys
from datetime import datetime
from pathlib import Path

import cv2
import numpy as np
import onnxruntime as ort

try:
    from PIL import Image, ImageDraw, ImageFont
    HAS_PIL = True
except ImportError:
    HAS_PIL = False

# ── Constants (must match extract_keypoints.py exactly) ───────────────────────
POSE_N          = 33
HAND_N          = 21
TOTAL_JOINTS    = POSE_N + HAND_N * 2   # 75
COORDS          = 3
FEAT_DIM        = TOTAL_JOINTS * COORDS  # 225

POSE_LEFT_SHOULDER  = 11
POSE_RIGHT_SHOULDER = 12
LEFT_HAND_START     = POSE_N            # 33
RIGHT_HAND_START    = POSE_N + HAND_N   # 54


# ─── MediaPipe import ─────────────────────────────────────────────────────────

def _import_mp():
    for prefix in ["mediapipe.solutions", "mediapipe.python.solutions"]:
        try:
            pose  = importlib.import_module(f"{prefix}.pose")
            hands = importlib.import_module(f"{prefix}.hands")
            draw  = importlib.import_module(f"{prefix}.drawing_utils")
            return pose, hands, draw
        except (ImportError, ModuleNotFoundError):
            continue
    try:
        import mediapipe as _mp
        return _mp.solutions.pose, _mp.solutions.hands, _mp.solutions.drawing_utils
    except AttributeError:
        pass
    raise ImportError(
        "Cannot import MediaPipe solutions.\n"
        "Try: pip uninstall mediapipe -y && pip install mediapipe==0.10.18"
    )

mp_pose, mp_hands, mp_drawing = _import_mp()


# ─── Myanmar text rendering ──────────────────────────────────────────────────

class TextRenderer:
    _FONT_CANDIDATES = [
        "C:\\Windows\\Fonts\\Padauk.ttf",
        "C:\\Windows\\Fonts\\NotoSansMyanmar-Regular.ttf",
        "C:\\Windows\\Fonts\\NotoSansMyanmar.ttf",
        "C:\\Windows\\Fonts\\myanmar3.ttf",
        "/usr/share/fonts/truetype/padauk/Padauk-Regular.ttf",
        "/usr/share/fonts/truetype/noto/NotoSansMyanmar-Regular.ttf",
        "fonts\\Padauk.ttf",
        "fonts/NotoSansMyanmar-Regular.ttf",
    ]

    def __init__(self, font_size: int = 32):
        self.font = self._load_font(font_size)
        self.can_render_myanmar = (self.font is not None)
        if not self.can_render_myanmar:
            print()
            print("  ╔═══════════════════════════════════════════════════════════╗")
            print("  ║  INFO: No Myanmar shaping font found.                    ║")
            print("  ║  Displaying readable Class IDs instead of Myanmar text.  ║")
            print("  ╚═══════════════════════════════════════════════════════════╝")
            print()

    def _load_font(self, size):
        if not HAS_PIL:
            return None
        for path in self._FONT_CANDIDATES:
            if Path(path).exists():
                try:
                    return ImageFont.truetype(path, size)
                except Exception:
                    continue
        return None

    def put_text(self, frame, text, position=(20, 20), color=(0, 255, 0)):
        if self.font and HAS_PIL:
            pil = Image.fromarray(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
            d   = ImageDraw.Draw(pil)
            r, g, b = color[2], color[1], color[0]
            for dx in [-1, 0, 1]:
                for dy in [-1, 0, 1]:
                    if dx or dy:
                        d.text((position[0]+dx, position[1]+dy), text,
                               font=self.font, fill=(0, 0, 0))
            d.text(position, text, font=self.font, fill=(r, g, b))
            return cv2.cvtColor(np.array(pil), cv2.COLOR_RGB2BGR)
        cv2.putText(frame, text, position, cv2.FONT_HERSHEY_SIMPLEX,
                    0.75, color, 2, cv2.LINE_AA)
        return frame


# ─── Keypoint extraction ──────────────────────────────────────────────────────

class KeypointExtractor:
    """
    Extracts (TOTAL_JOINTS, 3) keypoint arrays from BGR frames.

    BUG FIX [INFER-A]:
    process_frame now ALWAYS returns a keypoint array, never None.
    When MediaPipe fails to detect the pose, the pose block is all zeros.
    When hands are not detected, those blocks are all zeros.
    This exactly mirrors extract_keypoints.py / process_video() behaviour,
    which appends extract_frame_keypoints() output for EVERY frame without
    any None-filtering.

    Hand convention (matches extract_keypoints.py):
      MediaPipe reports handedness from the camera perspective (mirrored).
      label='Right' → signer's LEFT hand  → slot LEFT_HAND_START (33).
      label='Left'  → signer's RIGHT hand → slot RIGHT_HAND_START (54).
    """

    def __init__(self, model_complexity: int = 2):
        # Store complexity ourselves — mediapipe's Pose object does NOT expose
        # a public .model_complexity attribute in all versions (crashes on
        # mediapipe 0.10.x Windows builds with AttributeError).
        self.model_complexity = model_complexity

        self.pose  = mp_pose.Pose(
            static_image_mode=False,
            model_complexity=model_complexity,
            smooth_landmarks=True,
            enable_segmentation=False,
            min_detection_confidence=0.5,
            min_tracking_confidence=0.5,
        )
        self.hands = mp_hands.Hands(
            static_image_mode=False,
            max_num_hands=2,
            model_complexity=min(model_complexity, 1),
            min_detection_confidence=0.5,
            min_tracking_confidence=0.5,
        )
        self._last_pose  = None
        self._last_hands = None

    def process_frame(self, frame: np.ndarray) -> np.ndarray:
        """
        Returns (TOTAL_JOINTS, 3) float32 array for every frame.
        Zero rows for any joint/hand that was not detected.
        Never returns None — every frame is included, same as training.
        """
        rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        rgb.flags.writeable = False
        p_res = self.pose.process(rgb)
        h_res = self.hands.process(rgb)
        self._last_pose  = p_res
        self._last_hands = h_res

        kp = np.zeros((TOTAL_JOINTS, COORDS), dtype=np.float32)

        # Pose — zeros if not detected (same as extract_keypoints._lm_to_array)
        if p_res.pose_landmarks:
            for i, lm in enumerate(p_res.pose_landmarks.landmark):
                kp[i] = [lm.x, lm.y, lm.z]

        # Hands — zeros for each undetected hand
        if h_res.multi_hand_landmarks and h_res.multi_handedness:
            for lm_list, handedness in zip(
                h_res.multi_hand_landmarks, h_res.multi_handedness
            ):
                label  = handedness.classification[0].label
                offset = LEFT_HAND_START if label == 'Right' else RIGHT_HAND_START
                for i, lm in enumerate(lm_list.landmark):
                    kp[offset + i] = [lm.x, lm.y, lm.z]

        # Always return the array — never None
        return kp

    def draw_landmarks(self, frame: np.ndarray):
        if self._last_pose and self._last_pose.pose_landmarks:
            mp_drawing.draw_landmarks(
                frame, self._last_pose.pose_landmarks, mp_pose.POSE_CONNECTIONS
            )
        if self._last_hands and self._last_hands.multi_hand_landmarks:
            for lm in self._last_hands.multi_hand_landmarks:
                mp_drawing.draw_landmarks(frame, lm, mp_hands.HAND_CONNECTIONS)

    def close(self):
        self.pose.close()
        self.hands.close()


# ─── Normalization ────────────────────────────────────────────────────────────

def normalize_sequence(seq: np.ndarray) -> np.ndarray:
    """
    BUG FIX [INFER-B]: Exact mirror of extract_keypoints.normalize_sequence().

    Translates to shoulder-midpoint origin (full 3-D) and scales by
    3-D Euclidean shoulder width.  Applied to ALL three coordinate axes.

    Training code (extract_keypoints.py):
        mid  = (l_sh + r_sh) / 2.0         shape (T, 3)
        dist = ||r_sh - l_sh||_2            shape (T, 1)  — 3-D Euclidean
        out  = (seq - mid[:, None, :]) / dist[:, None, :]

    The old code used a 2-D scale (x,y only) and did NOT divide z by scale.
    This produced inflated x/y values and wrong z values → every prediction wrong.

    Args:
        seq: (T, 75, 3) raw keypoints from KeypointExtractor.
    Returns:
        (T, 75, 3) normalised float32, matching training distribution exactly.
    """
    l_sh = seq[:, POSE_LEFT_SHOULDER,  :]   # (T, 3)
    r_sh = seq[:, POSE_RIGHT_SHOULDER, :]   # (T, 3)
    mid  = (l_sh + r_sh) / 2.0              # (T, 3)
    dist = np.linalg.norm(r_sh - l_sh, axis=1, keepdims=True)   # (T, 1)
    dist = np.where(dist < 1e-6, 1.0, dist)
    return ((seq - mid[:, None, :]) / dist[:, None, :]).astype(np.float32)


# ─── Preprocessor ─────────────────────────────────────────────────────────────

class Preprocessor:
    """
    Stacks raw (75, 3) keypoint frames, normalises, crops/pads, and builds
    the model-ready input dict.

    Processing order (matches infer.py exactly):
      1. Stack → (T, 75, 3)
      2. Normalize (shoulder-centred, 3-D scale, all axes)
      3. Centre-crop if T > max_seq_len
      4. Zero-pad if T < max_seq_len
      5. Build mask (True = padded position)
      6. Reshape for model type
    """

    def __init__(self, max_seq_len: int = 200):
        self.max_seq_len = max_seq_len

    def __call__(self, kp_list: list, model_type: str) -> dict:
        if not kp_list:
            return {}

        seq = np.stack(kp_list, axis=0).astype(np.float32)  # (T, 75, 3)
        seq = normalize_sequence(seq)
        T   = seq.shape[0]

        # Centre-crop
        if T > self.max_seq_len:
            start = (T - self.max_seq_len) // 2
            seq   = seq[start: start + self.max_seq_len]
            T     = self.max_seq_len

        real_len = T

        # Zero-pad
        if T < self.max_seq_len:
            pad = np.zeros((self.max_seq_len - T, TOTAL_JOINTS, COORDS), dtype=np.float32)
            seq = np.concatenate([seq, pad], axis=0)

        # Mask: True where padded
        mask = np.zeros(self.max_seq_len, dtype=np.bool_)
        mask[real_len:] = True

        if model_type == "stgcn":
            # (T, 75, 3) → (3, T, 75) → (1, 3, T, 75)
            kp_out = np.expand_dims(seq.transpose(2, 0, 1), axis=0).astype(np.float32)
            return {"keypoints": kp_out}
        else:
            # (T, 75, 3) → (T, 225) → (1, T, 225)
            kp_out = np.expand_dims(
                seq.reshape(self.max_seq_len, FEAT_DIM), axis=0
            ).astype(np.float32)
            return {"keypoints": kp_out, "mask": mask[np.newaxis]}


# ─── ONNX inference wrapper ───────────────────────────────────────────────────

class ONNXModel:
    def __init__(self, onnx_path: str, meta_path: str = None):
        available = ort.get_available_providers()
        preferred = ["CUDAExecutionProvider", "DmlExecutionProvider", "CPUExecutionProvider"]
        providers = [p for p in preferred if p in available]

        self.session = ort.InferenceSession(onnx_path, providers=providers)
        print(f"[INFO] ONNX loaded: {Path(onnx_path).name}"
              f"  (provider: {self.session.get_providers()[0]})")

        self.meta = {}
        if meta_path and Path(meta_path).exists():
            try:
                with open(meta_path, encoding="utf-8") as f:
                    self.meta = json.load(f)
            except Exception as e:
                print(f"[WARNING] Could not read metadata: {e}")
        else:
            print(f"[INFO] Metadata not found at '{meta_path}', using defaults.")

        self.model_type  = self.meta.get("model_type",  "transformer")
        self.max_seq_len = self.meta.get("max_seq_len", 200)
        self.num_classes = self.meta.get("num_classes", 558)

    def predict(self, inputs: dict) -> np.ndarray:
        """
        Run one forward pass.
        BUG FIX [INFER-E]: return 1-D (num_classes,) array, not scalar.
        output shape from session: (1, num_classes); [0] gives (num_classes,).
        """
        return self.session.run(None, inputs)[0][0]   # (num_classes,)

    @staticmethod
    def softmax(logits: np.ndarray) -> np.ndarray:
        e = np.exp(logits - logits.max())
        return e / e.sum()


# ─── Label map ────────────────────────────────────────────────────────────────

def load_idx2label(path: str) -> dict:
    with open(path, encoding="utf-8") as f:
        data = json.load(f)
    return {int(k): v for k, v in data.get("idx2label", {}).items()}


def _get_label(idx2label: dict, class_id) -> str:
    """
    BUG FIX [INFER-D]: always cast to int() — np.int64 keys miss {int:str} dicts.
    """
    return idx2label.get(int(class_id), f"[class {int(class_id)}]")


# ─── Display helpers ──────────────────────────────────────────────────────────

def _build_text(class_id, label, confidence, can_myanmar,
                top_k=1, all_ids=None, probs=None, lbl_map=None):
    if top_k == 1:
        return (f"{label}  ({confidence*100:.1f}%)" if can_myanmar
                else f"Class {int(class_id)}  ({confidence*100:.1f}%)")
    parts = []
    for i, cid in enumerate(all_ids):
        lbl = _get_label(lbl_map, cid) if (can_myanmar and lbl_map) else f"Class {int(cid)}"
        parts.append(f"#{i+1} {lbl} ({probs[int(cid)]*100:.0f}%)")
    return "  |  ".join(parts)


def _build_log(class_id, label, confidence, top_k=1, all_ids=None, probs=None, lbl_map=None):
    safe = label.encode("utf-8", errors="replace").decode("utf-8")
    if top_k == 1:
        return f"  Class {int(class_id):>4d} | {safe:<40s} | {confidence*100:6.2f}%"
    lines = []
    for i, cid in enumerate(all_ids):
        lbl = _get_label(lbl_map, cid) if lbl_map else "?"
        lbl = lbl.encode("utf-8", errors="replace").decode("utf-8")
        lines.append(f"  Top-{i+1}: Class {int(cid):>4d} | {lbl:<40s} | {probs[int(cid)]*100:6.2f}%")
    return "\n".join(lines)


def _append_log(path, text):
    try:
        with open(path, "a", encoding="utf-8-sig") as f:
            f.write(text)
    except Exception:
        pass


# ─── Video processing ─────────────────────────────────────────────────────────

def run_video(model, prep, extractor, renderer, video_path,
              top_k=5, log_file=None, idx2label=None, debug=False):
    """
    Process a single video file for sign language recognition.

    Matches infer.py / process_video() behaviour exactly:
      - Every frame is processed and appended (no skipping for missing pose).
      - Normalization applied after full sequence is stacked.
      - Centre-crop or zero-pad to max_seq_len.
    """
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        print(f"[ERROR] Cannot open video: {video_path}")
        return

    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    fps_src      = cap.get(cv2.CAP_PROP_FPS) or 30.0
    print(f"[INFO] {Path(video_path).name} — {total_frames} frames @ {fps_src:.1f} fps")
    print(f"[INFO] MediaPipe pose complexity: {extractor.model_complexity}"
          f"  (must match training; training default=2)")

    kp_list = []
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        # BUG FIX [INFER-A]: process_frame always returns array (never None)
        kp_list.append(extractor.process_frame(frame))
    cap.release()

    if not kp_list:
        print("[WARNING] No frames could be read from the video.")
        return

    T_raw = len(kp_list)
    T_use = min(T_raw, model.max_seq_len)
    print(f"[INFO] {T_raw} frames extracted → using {T_use} "
          f"({'centre-cropped' if T_raw > model.max_seq_len else 'zero-padded to'}"
          f" {model.max_seq_len})")

    inputs = prep(kp_list, model.model_type)
    logits = model.predict(inputs)                    # (num_classes,)
    probs  = ONNXModel.softmax(logits)               # (num_classes,)
    top_idx = np.argsort(probs)[::-1][:top_k]

    sep = "─" * 60
    lines = [f"Video  : {Path(video_path).name}",
             f"Model  : {model.model_type.upper()}", sep]
    for rank, idx in enumerate(top_idx, 1):
        lbl = _get_label(idx2label, idx) if idx2label else f"[class {int(idx)}]"
        lines.append(f"  Top-{rank}: {lbl}  ({probs[int(idx)]*100:.2f}%)")
    lines.append(sep)
    print("\n" + "\n".join(lines))

    if log_file:
        log_lines = [f"[{datetime.now().isoformat()}] {Path(video_path).name}"]
        for idx in top_idx:
            lbl = _get_label(idx2label, idx) if idx2label else "?"
            log_lines.append(_build_log(idx, lbl, probs[int(idx)],
                                        top_k=1, lbl_map=idx2label))
        _append_log(log_file, "\n".join(log_lines) + "\n\n")


# ─── Webcam processing ────────────────────────────────────────────────────────

def run_webcam(model, prep, extractor, renderer, idx2label,
               cam_idx=0, top_k=1, log_file=None, classify_every=15):
    """
    Live webcam sign recognition.

    BUG FIX [INFER-A]: Every frame is always appended to the buffer —
    no None-filtering — because process_frame always returns an array.

    BUG FIX [INFER-C]: Buffer is cleared after each classification,
    matching infer.py's behaviour.  Each sign is evaluated as an
    independent self-contained sequence, exactly as during training.
    Press C to manually clear the buffer; press Q to quit.
    """
    cap = cv2.VideoCapture(cam_idx)
    if not cap.isOpened():
        print(f"[ERROR] Cannot open webcam {cam_idx}")
        return

    max_buf    = model.max_seq_len
    kp_buffer  = []
    last_text  = "Collecting frames..."
    last_color = (200, 200, 200)
    last_log   = ""
    frame_cnt  = 0
    print("[INFO] Webcam started. Sign into camera, then wait for prediction.")
    print("[INFO] Q = quit | C = clear buffer manually")

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        frame_cnt += 1
        # BUG FIX [INFER-A]: always append, never skip
        kp_buffer.append(extractor.process_frame(frame))
        if len(kp_buffer) > max_buf:
            kp_buffer.pop(0)

        if len(kp_buffer) >= 20 and frame_cnt % classify_every == 0:
            inputs  = prep(list(kp_buffer), model.model_type)
            logits  = model.predict(inputs)
            probs   = ONNXModel.softmax(logits)
            top_idx = np.argsort(probs)[::-1][:top_k]

            best_idx  = top_idx[0]
            best_prob = probs[int(best_idx)]
            best_lbl  = _get_label(idx2label, best_idx) if idx2label else f"Class {int(best_idx)}"

            last_text  = _build_text(best_idx, best_lbl, best_prob,
                                     renderer.can_render_myanmar,
                                     top_k, top_idx, probs, idx2label)
            last_color = ((0, 200, 0) if best_prob >= 0.80 else
                          (0, 200, 200) if best_prob >= 0.50 else (0, 0, 255))

            # BUG FIX [INFER-C]: clear buffer after each classification
            # so the next sign starts fresh, matching infer.py behaviour
            kp_buffer.clear()

            if log_file and last_text != last_log:
                ts = datetime.now().strftime("%H:%M:%S")
                _append_log(log_file,
                    f"[{ts}] " + _build_log(best_idx, best_lbl, best_prob,
                                             top_k=1, lbl_map=idx2label) + "\n")
                last_log = last_text

        extractor.draw_landmarks(frame)
        frame = renderer.put_text(frame, last_text, position=(10, 10), color=last_color)

        h, w  = frame.shape[:2]
        fill  = int(len(kp_buffer) / max(max_buf, 1) * w)
        cv2.rectangle(frame, (0, h - 8), (fill, h), (0, 180, 255), -1)
        cv2.putText(frame, f"buf {len(kp_buffer)}/{max_buf}",
                    (10, h - 12), cv2.FONT_HERSHEY_SIMPLEX, 0.45, (200, 200, 200), 1)

        cv2.imshow("MSL Recognition  [Q=quit  C=clear]", frame)
        key = cv2.waitKey(1) & 0xFF
        if key == ord("q"):
            break
        elif key == ord("c"):
            kp_buffer.clear()
            last_text  = "Buffer cleared"
            last_color = (200, 200, 200)
            last_log   = ""

    cap.release()
    cv2.destroyAllWindows()


# ─── CLI ──────────────────────────────────────────────────────────────────────

def main():
    parser = argparse.ArgumentParser(description="MSL Sign Language Inference — ONNX Runtime")
    parser.add_argument("--onnx_model",      required=True)
    parser.add_argument("--metadata",        default=None)
    parser.add_argument("--label_map",       default=None)
    grp = parser.add_mutually_exclusive_group(required=True)
    grp.add_argument("--video",  help="Path to a video file")
    grp.add_argument("--webcam", type=int, metavar="CAM_IDX")
    parser.add_argument("--top_k",          type=int, default=5)
    parser.add_argument("--classify_every", type=int, default=15,
                        help="Classify every N webcam frames (default: 15)")
    parser.add_argument("--log_file",       default=None)
    parser.add_argument("--font_size",      type=int, default=32)
    parser.add_argument("--model_complexity", type=int, default=None,
                        help="MediaPipe pose model complexity: 0, 1, or 2. "
                             "Default: 2 for video (matches training), 1 for webcam (speed). "
                             "MUST match what was used during extract_keypoints.py.")
    parser.add_argument("--debug",          action="store_true")
    args = parser.parse_args()

    model_p = Path(args.onnx_model)
    meta_p  = args.metadata or str(model_p.with_suffix(".json"))
    lmap_p  = args.label_map or str(model_p.parent / "label_map.json")

    if not Path(lmap_p).exists():
        sys.exit(f"[ERROR] label_map.json not found: {lmap_p}\n"
                 f"        Copy it from your Linux server next to the .onnx file.")

    idx2label = load_idx2label(lmap_p)
    print(f"[INFO] Loaded {len(idx2label)} labels from label map.")

    model  = ONNXModel(str(model_p), meta_p)
    prep   = Preprocessor(max_seq_len=model.max_seq_len)

    # Fix [INFER-F]: complexity must match training (=2 for video, =1 for webcam).
    # Training used extract_keypoints.py --complexity 2 (the default).
    # infer.py uses complexity=2 for video/batch, complexity=1 for webcam.
    if args.model_complexity is not None:
        video_complexity  = args.model_complexity
        webcam_complexity = args.model_complexity
    else:
        video_complexity  = 2   # matches training and infer.py infer_video()
        webcam_complexity = 1   # matches infer.py infer_webcam() — speed tradeoff

    renderer = TextRenderer(font_size=args.font_size)

    if args.log_file:
        print(f"[INFO] Logging to: {args.log_file}")

    try:
        if args.video:
            extractor = KeypointExtractor(model_complexity=video_complexity)
            try:
                run_video(model, prep, extractor, renderer,
                          args.video, args.top_k, args.log_file, idx2label, args.debug)
            finally:
                extractor.close()
        else:
            extractor = KeypointExtractor(model_complexity=webcam_complexity)
            try:
                run_webcam(model, prep, extractor, renderer, idx2label,
                           args.webcam, args.top_k, args.log_file, args.classify_every)
            finally:
                extractor.close()
    finally:
        print("[INFO] Done.")


if __name__ == "__main__":
    main()

```

## Inferring a Pretrained Model

ဒီတစ်ခါ နောက်ဆုံး Infer ထင်ပါတယ် ♫⋆｡ ♪ ₊˚♬ 

```
(msl_onnx) E:\LU_Lab\MSL_Tran_Demo>python infer_onnx.py --onnx_model .\onnx_models\bilstm_msl.onnx --video .\sample4inference\idx20-30.mp4
[INFO] Loaded 556 labels from label map.
[INFO] ONNX loaded: bilstm_msl.onnx  (provider: CPUExecutionProvider)
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
[INFO] idx20-30.mp4 — 127 frames @ 30.0 fps
[INFO] MediaPipe pose complexity: 2  (must match training; training default=2)
C:\Users\801680\Anaconda3\envs\msl_onnx\lib\site-packages\google\protobuf\symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '
[INFO] 127 frames extracted → using 127 (zero-padded to 200)

Video  : idx20-30.mp4
Model  : BILSTM
────────────────────────────────────────────────────────────
  Top-1: မြေနိမ့်ဒေသ ။  (80.73%)
  Top-2: ချိုင့် ။  (0.16%)
  Top-3: ဘေး အိမ် ကို ဓားပြ ဝင် တိုက် နေ ပါ တယ် ။  (0.13%)
  Top-4: ဓာတ်မှန် ရိုက် တဲ့ နေရာ က ဘယ်နား မှာ လဲ သိ ချင် လို့ ပါ ။  (0.12%)
  Top-5: လျှပ်စစ် ဝါယာ ရှော့ ဖြစ် ရင် မီးသတ်ဆေးဘူး အသုံးပြု ပါ ။  (0.12%)
────────────────────────────────────────────────────────────
[INFO] Done.

(msl_onnx) E:\LU_Lab\MSL_Tran_Demo>
```

```
(msl_onnx) E:\LU_Lab\MSL_Tran_Demo>python infer_onnx.py --onnx_model .\onnx_models\bilstm_msl.onnx --video .\sample4inference\idx20-86.mp4
[INFO] Loaded 556 labels from label map.
[INFO] ONNX loaded: bilstm_msl.onnx  (provider: CPUExecutionProvider)
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
[INFO] idx20-86.mp4 — 169 frames @ 30.0 fps
[INFO] MediaPipe pose complexity: 2  (must match training; training default=2)
C:\Users\801680\Anaconda3\envs\msl_onnx\lib\site-packages\google\protobuf\symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '
[INFO] 169 frames extracted → using 169 (zero-padded to 200)

Video  : idx20-86.mp4
Model  : BILSTM
────────────────────────────────────────────────────────────
  Top-1: ပတ်ဝန်းကျင် ကို အကူအညီ တောင်း ။  (65.91%)
  Top-2: ထွက်ပေါက် ။  (0.45%)
  Top-3: သူ က ဘယ် လူနာဆောင် ထဲ မှာ လဲ ။  (0.31%)
  Top-4: လျှပ်စစ် ဝါယော ရှော့ ဖြစ် ရင် အဝတ်ခြောက် ဖြင့် ရိုက်ပုတ် ငြှိမ်းသတ် ပါ ။  (0.29%)
  Top-5: မ သန့်ရှင်း သော ရေ အရင်းမြစ် များ ကို မ သုံး ပါ နဲ့ ။  (0.25%)
────────────────────────────────────────────────────────────
[INFO] Done.

(msl_onnx) E:\LU_Lab\MSL_Tran_Demo>
```

```
(msl_onnx) E:\LU_Lab\MSL_Tran_Demo>python infer_onnx.py --onnx_model .\onnx_models\bilstm_msl.onnx --video .\sample4inference\idx20-244.mp4
[INFO] Loaded 556 labels from label map.
[INFO] ONNX loaded: bilstm_msl.onnx  (provider: CPUExecutionProvider)
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
[INFO] idx20-244.mp4 — 55 frames @ 30.0 fps
[INFO] MediaPipe pose complexity: 2  (must match training; training default=2)
C:\Users\801680\Anaconda3\envs\msl_onnx\lib\site-packages\google\protobuf\symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '
[INFO] 55 frames extracted → using 55 (zero-padded to 200)

Video  : idx20-244.mp4
Model  : BILSTM
────────────────────────────────────────────────────────────
  Top-1: မ တပ် ထား ပါ ဘူး ။  (10.26%)
  Top-2: ရေကူး ခြင်း ။  (2.74%)
  Top-3: ခင်ဗျား ကို ဒီ ဆေး ဘယ်သူ ပေး တာ လဲ ။  (1.34%)
  Top-4: ကျိုချက် ထား သော ရေ ကို သာ သောက် သုံး ပါ ။  (0.74%)
  Top-5: ရေ စို ပြီး အနံ့အသက် မ ကောင်း တဲ့ အစာ များ ကို မ စား ပါ နှင့် ။  (0.73%)
────────────────────────────────────────────────────────────
[INFO] Done.

(msl_onnx) E:\LU_Lab\MSL_Tran_Demo>
```

```
(msl_onnx) E:\LU_Lab\MSL_Tran_Demo>python infer_onnx.py --onnx_model .\onnx_models\bilstm_msl.onnx --video .\sample4inference\idx20-291.mp4
[INFO] Loaded 556 labels from label map.
[INFO] ONNX loaded: bilstm_msl.onnx  (provider: CPUExecutionProvider)
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
[INFO] idx20-291.mp4 — 211 frames @ 30.0 fps
[INFO] MediaPipe pose complexity: 2  (must match training; training default=2)
C:\Users\801680\Anaconda3\envs\msl_onnx\lib\site-packages\google\protobuf\symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '
[INFO] 211 frames extracted → using 200 (centre-cropped 200)

Video  : idx20-291.mp4
Model  : BILSTM
────────────────────────────────────────────────────────────
  Top-1: ပျောက် သွား တဲ့ လူ ရဲ့ ပုံပန်းသဏ္ဌာန် ကို ပြောပြ နိုင် မ လား ။  (89.00%)
  Top-2: ပွဲကောက် ရေတံခွန် မှာ ပါ ။  (0.10%)
  Top-3: မနက် ၉ နာရီ က နေ ည ၁၀ နာရီ ထိ ပါ ။  (0.08%)
  Top-4: ကလေး ကို တစ် နာရီ အတွင်း ဆေး တိုက် လိုက် ပါ ။  (0.06%)
  Top-5: သူ အခြေအနေ ကောင်း ပါ့ မ လား ။  (0.06%)
────────────────────────────────────────────────────────────
[INFO] Done.

(msl_onnx) E:\LU_Lab\MSL_Tran_Demo>
```

```
(msl_onnx) E:\LU_Lab\MSL_Tran_Demo>python infer_onnx.py --onnx_model .\onnx_models\bilstm_msl.onnx --video .\sample4inference\idx20-440.mp4
[INFO] Loaded 556 labels from label map.
[INFO] ONNX loaded: bilstm_msl.onnx  (provider: CPUExecutionProvider)
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
[INFO] idx20-440.mp4 — 248 frames @ 30.0 fps
[INFO] MediaPipe pose complexity: 2  (must match training; training default=2)
C:\Users\801680\Anaconda3\envs\msl_onnx\lib\site-packages\google\protobuf\symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '
[INFO] 248 frames extracted → using 200 (centre-cropped 200)

Video  : idx20-440.mp4
Model  : BILSTM
────────────────────────────────────────────────────────────
  Top-1: ခင်ဗျား နောက်ဆုံး ဘယ် ဆရာဝန် နဲ့ ပြ ခဲ့ တာ လဲ ။  (93.32%)
  Top-2: ကျွန်တော့် ရည်းစား လှေကား ပေါ် က ချော်ကျ သွား လို့ ပါ ။  (0.05%)
  Top-3: မီးလောင် နေ တဲ့ အဆောက်အအုံ ထဲ မ ဝင် နဲ့ ။  (0.04%)
  Top-4: ဆရာဝန် နဲ့ ပြ ဖို့ ရက်ချိန်း ယူ ချင် လို့ ပါ ။  (0.04%)
  Top-5: ကျွန်တော် ကိုယ်အပူချိန် တက် နေ တယ် ။  (0.03%)
────────────────────────────────────────────────────────────
[INFO] Done.

(msl_onnx) E:\LU_Lab\MSL_Tran_Demo>
```

```
(msl_onnx) E:\LU_Lab\MSL_Tran_Demo>python infer_onnx.py --onnx_model .\onnx_models\transformer_msl.onnx --video .\sample4inference\idx20-30.mp4
[INFO] Loaded 556 labels from label map.
[INFO] ONNX loaded: transformer_msl.onnx  (provider: CPUExecutionProvider)
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
[INFO] idx20-30.mp4 — 127 frames @ 30.0 fps
[INFO] MediaPipe pose complexity: 2  (must match training; training default=2)
C:\Users\801680\Anaconda3\envs\msl_onnx\lib\site-packages\google\protobuf\symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '
[INFO] 127 frames extracted → using 127 (zero-padded to 200)

Video  : idx20-30.mp4
Model  : TRANSFORMER
────────────────────────────────────────────────────────────
  Top-1: မြေနိမ့်ဒေသ ။  (80.68%)
  Top-2: ချိုင့် ။  (0.30%)
  Top-3: မီးခလုပ် များ ကို ပိတ် ပါ ။  (0.28%)
  Top-4: ဂရုစိုက် ။  (0.25%)
  Top-5: လေမုန်တိုင်း ကျ မယ် ။  (0.15%)
────────────────────────────────────────────────────────────
[INFO] Done.

(msl_onnx) E:\LU_Lab\MSL_Tran_Demo>
```

```
(msl_onnx) E:\LU_Lab\MSL_Tran_Demo>python infer_onnx.py --onnx_model .\onnx_models\transformer_msl.onnx --video .\sample4inference\idx20-86.mp4
[INFO] Loaded 556 labels from label map.
[INFO] ONNX loaded: transformer_msl.onnx  (provider: CPUExecutionProvider)
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
[INFO] idx20-86.mp4 — 169 frames @ 30.0 fps
[INFO] MediaPipe pose complexity: 2  (must match training; training default=2)
C:\Users\801680\Anaconda3\envs\msl_onnx\lib\site-packages\google\protobuf\symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '
[INFO] 169 frames extracted → using 169 (zero-padded to 200)

Video  : idx20-86.mp4
Model  : TRANSFORMER
────────────────────────────────────────────────────────────
  Top-1: ပတ်ဝန်းကျင် ကို အကူအညီ တောင်း ။  (25.39%)
  Top-2: ကား ထဲ မှာ လူ တစ်ယောက် ညှပ်မိ နေ တယ် ။  (1.29%)
  Top-3: လေ ။  (1.13%)
  Top-4: ကျိုးကျ နိုင် တဲ့ သစ်ပင် သစ်ကိုင်း တွေ ကို ရှင်းလင်း ထား ။  (0.99%)
  Top-5: အိမ် အပြင် မှာ ဆိုရင် အဆောက်ဦး တွေ နဲ့ ဝေးဝေး ကို သွား ။  (0.99%)
────────────────────────────────────────────────────────────
[INFO] Done.

(msl_onnx) E:\LU_Lab\MSL_Tran_Demo>
```

```
(msl_onnx) E:\LU_Lab\MSL_Tran_Demo>python infer_onnx.py --onnx_model .\onnx_models\transformer_msl.onnx --video .\sample4inference\idx20-244.mp4
[INFO] Loaded 556 labels from label map.
[INFO] ONNX loaded: transformer_msl.onnx  (provider: CPUExecutionProvider)
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
[INFO] idx20-244.mp4 — 55 frames @ 30.0 fps
[INFO] MediaPipe pose complexity: 2  (must match training; training default=2)
C:\Users\801680\Anaconda3\envs\msl_onnx\lib\site-packages\google\protobuf\symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '
[INFO] 55 frames extracted → using 55 (zero-padded to 200)

Video  : idx20-244.mp4
Model  : TRANSFORMER
────────────────────────────────────────────────────────────
  Top-1: မ တပ် ထား ပါ ဘူး ။  (12.22%)
  Top-2: မြန်မြန် လုပ် ပါ ။  (4.74%)
  Top-3: ဘာ များ အကူအညီ ပေး ရ မ လဲ ။  (2.06%)
  Top-4: သူ့ သမီး ဘာ ဖြစ် တာ လဲ ။  (1.96%)
  Top-5: အသက်ရှူ နေ သေး လား ။  (1.84%)
────────────────────────────────────────────────────────────
[INFO] Done.

(msl_onnx) E:\LU_Lab\MSL_Tran_Demo>
```

```
(msl_onnx) E:\LU_Lab\MSL_Tran_Demo>python infer_onnx.py --onnx_model .\onnx_models\transformer_msl.onnx --video .\sample4inference\idx20-291.mp4
[INFO] Loaded 556 labels from label map.
[INFO] ONNX loaded: transformer_msl.onnx  (provider: CPUExecutionProvider)
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
[INFO] idx20-291.mp4 — 211 frames @ 30.0 fps
[INFO] MediaPipe pose complexity: 2  (must match training; training default=2)
C:\Users\801680\Anaconda3\envs\msl_onnx\lib\site-packages\google\protobuf\symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '
[INFO] 211 frames extracted → using 200 (centre-cropped 200)

Video  : idx20-291.mp4
Model  : TRANSFORMER
────────────────────────────────────────────────────────────
  Top-1: ပျောက် သွား တဲ့ လူ ရဲ့ ပုံပန်းသဏ္ဌာန် ကို ပြောပြ နိုင် မ လား ။  (85.17%)
  Top-2: ဒီ နေ့ မနက် ကျွန်တော့် ဆိုင်ကယ် ပျောက် သွား တယ် ။  (0.23%)
  Top-3: ဓာတ်မှန် ရိုက် တဲ့ နေရာ က ဘယ် မှာ လဲ ။  (0.18%)
  Top-4: လူ တစ်ယောက် ရေ နစ် သွား လို့ ရှာ ဖို့ အကူညီ လွတ် ပေး ပါ ။  (0.17%)
  Top-5: မနက် ၆ နာရီ နောက် ပိုင်း အစား မ စား ပါ နဲ့ ။  (0.13%)
────────────────────────────────────────────────────────────
[INFO] Done.

(msl_onnx) E:\LU_Lab\MSL_Tran_Demo>
```

```
(msl_onnx) E:\LU_Lab\MSL_Tran_Demo>python infer_onnx.py --onnx_model .\onnx_models\transformer_msl.onnx --video .\sample4inference\idx20-440.mp4
[INFO] Loaded 556 labels from label map.
[INFO] ONNX loaded: transformer_msl.onnx  (provider: CPUExecutionProvider)
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
[INFO] idx20-440.mp4 — 248 frames @ 30.0 fps
[INFO] MediaPipe pose complexity: 2  (must match training; training default=2)
C:\Users\801680\Anaconda3\envs\msl_onnx\lib\site-packages\google\protobuf\symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '
[INFO] 248 frames extracted → using 200 (centre-cropped 200)

Video  : idx20-440.mp4
Model  : TRANSFORMER
────────────────────────────────────────────────────────────
  Top-1: ခင်ဗျား နောက်ဆုံး ဘယ် ဆရာဝန် နဲ့ ပြ ခဲ့ တာ လဲ ။  (93.87%)
  Top-2: ရေဒီယို ၊ ရုပ်မြင်သံကြား က သတင်း တွေ ကို သေချာ ကြည့် ။  (0.07%)
  Top-3: ကျွန်တော့် လက်ခေျာင်း တွေ လှုပ် လို့ မ ရ ဘူး ။  (0.05%)
  Top-4: မုန်တိုင်း ။  (0.04%)
  Top-5: ညစ်ညမ်း နေ တဲ့ နေရာ များ ကို သန့်ရှင်းရေး လုပ် ပြီး ရောဂါပိုး သတ် ရ ပါ မယ် ။  (0.04%)
────────────────────────────────────────────────────────────
[INFO] Done.

(msl_onnx) E:\LU_Lab\MSL_Tran_Demo>
```

```
(msl_onnx) E:\LU_Lab\MSL_Tran_Demo>python infer_onnx.py --onnx_model .\onnx_models\stgcn_msl.onnx --video .\sample4inference\idx20-30.mp4
[INFO] Loaded 556 labels from label map.
[INFO] ONNX loaded: stgcn_msl.onnx  (provider: CPUExecutionProvider)
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
[INFO] idx20-30.mp4 — 127 frames @ 30.0 fps
[INFO] MediaPipe pose complexity: 2  (must match training; training default=2)
C:\Users\801680\Anaconda3\envs\msl_onnx\lib\site-packages\google\protobuf\symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '
[INFO] 127 frames extracted → using 127 (zero-padded to 200)

Video  : idx20-30.mp4
Model  : STGCN
────────────────────────────────────────────────────────────
  Top-1: မြေနိမ့်ဒေသ ။  (82.02%)
  Top-2: မီးသတ်သမား ။  (0.68%)
  Top-3: အထူး သတိပြု ။  (0.63%)
  Top-4: မြစ်ခေျာင်း ။  (0.51%)
  Top-5: စားပွဲခုံ အောက် မှာ ဝင်နေ ။  (0.34%)
────────────────────────────────────────────────────────────
[INFO] Done.

(msl_onnx) E:\LU_Lab\MSL_Tran_Demo>
```

```
(msl_onnx) E:\LU_Lab\MSL_Tran_Demo>python infer_onnx.py --onnx_model .\onnx_models\stgcn_msl.onnx --video .\sample4inference\idx20-86.mp4
[INFO] Loaded 556 labels from label map.
[INFO] ONNX loaded: stgcn_msl.onnx  (provider: CPUExecutionProvider)
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
[INFO] idx20-86.mp4 — 169 frames @ 30.0 fps
[INFO] MediaPipe pose complexity: 2  (must match training; training default=2)
C:\Users\801680\Anaconda3\envs\msl_onnx\lib\site-packages\google\protobuf\symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '
[INFO] 169 frames extracted → using 169 (zero-padded to 200)

Video  : idx20-86.mp4
Model  : STGCN
────────────────────────────────────────────────────────────
  Top-1: ပတ်ဝန်းကျင် ကို အကူအညီ တောင်း ။  (66.45%)
  Top-2: သူ အခြေအနေ ကောင်း ပါ့ မ လား ။  (1.82%)
  Top-3: ဘေးအန္တရာယ် ကင်း သော နေရာ တွင် ဝပ်နေ ပါ ။  (1.11%)
  Top-4: မီးချိတ် ။  (1.07%)
  Top-5: သွေးတက် ပြီး ခေါင်း အရမ်း မူး နေ တယ် ။  (0.77%)
────────────────────────────────────────────────────────────
[INFO] Done.

(msl_onnx) E:\LU_Lab\MSL_Tran_Demo>
```

```
(msl_onnx) E:\LU_Lab\MSL_Tran_Demo>python infer_onnx.py --onnx_model .\onnx_models\stgcn_msl.onnx --video .\sample4inference\idx20-244.mp4
[INFO] Loaded 556 labels from label map.
[INFO] ONNX loaded: stgcn_msl.onnx  (provider: CPUExecutionProvider)
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
[INFO] idx20-244.mp4 — 55 frames @ 30.0 fps
[INFO] MediaPipe pose complexity: 2  (must match training; training default=2)
C:\Users\801680\Anaconda3\envs\msl_onnx\lib\site-packages\google\protobuf\symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '
[INFO] 55 frames extracted → using 55 (zero-padded to 200)

Video  : idx20-244.mp4
Model  : STGCN
────────────────────────────────────────────────────────────
  Top-1: မ တပ် ထား ပါ ဘူး ။  (73.95%)
  Top-2: အစာ စား ပြီး မှ သောက် ပါ ။  (1.26%)
  Top-3: ကား ကို ရပ် ထား ။  (1.21%)
  Top-4: cpr ဘယ်သူ လုပ်တတ် လဲ ။  (1.03%)
  Top-5: ဘယ်သူ တွေ ပါ သေး လဲ ။  (0.80%)
────────────────────────────────────────────────────────────
[INFO] Done.

(msl_onnx) E:\LU_Lab\MSL_Tran_Demo>
```

```
(msl_onnx) E:\LU_Lab\MSL_Tran_Demo>python infer_onnx.py --onnx_model .\onnx_models\stgcn_msl.onnx --video .\sample4inference\idx20-291.mp4
[INFO] Loaded 556 labels from label map.
[INFO] ONNX loaded: stgcn_msl.onnx  (provider: CPUExecutionProvider)
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
[INFO] idx20-291.mp4 — 211 frames @ 30.0 fps
[INFO] MediaPipe pose complexity: 2  (must match training; training default=2)
C:\Users\801680\Anaconda3\envs\msl_onnx\lib\site-packages\google\protobuf\symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '
[INFO] 211 frames extracted → using 200 (centre-cropped 200)

Video  : idx20-291.mp4
Model  : STGCN
────────────────────────────────────────────────────────────
  Top-1: ပျောက် သွား တဲ့ လူ ရဲ့ ပုံပန်းသဏ္ဌာန် ကို ပြောပြ နိုင် မ လား ။  (77.08%)
  Top-2: အထူး ကြပ်မတ်ဆောင် မှာ တစ် ရက် နေ ရ မယ် ။  (1.60%)
  Top-3: လူနာစောင့် က ဘယ် မှာ လဲ ။  (0.80%)
  Top-4: ကျွန်တော့ ကလေး အရက်ပျံ တွေ သောက် မိ သွား လို့ ပါ ။  (0.78%)
  Top-5: ခင်ဗျား ဆရာဝန် နာမည် က ဘာ လဲ ။  (0.68%)
────────────────────────────────────────────────────────────
[INFO] Done.

(msl_onnx) E:\LU_Lab\MSL_Tran_Demo>
```

```
(msl_onnx) E:\LU_Lab\MSL_Tran_Demo>python infer_onnx.py --onnx_model .\onnx_models\stgcn_msl.onnx --video .\sample4inference\idx20-440.mp4
[INFO] Loaded 556 labels from label map.
[INFO] ONNX loaded: stgcn_msl.onnx  (provider: CPUExecutionProvider)
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
[INFO] idx20-440.mp4 — 248 frames @ 30.0 fps
[INFO] MediaPipe pose complexity: 2  (must match training; training default=2)
C:\Users\801680\Anaconda3\envs\msl_onnx\lib\site-packages\google\protobuf\symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '
[INFO] 248 frames extracted → using 200 (centre-cropped 200)

Video  : idx20-440.mp4
Model  : STGCN
────────────────────────────────────────────────────────────
  Top-1: ခင်ဗျား နောက်ဆုံး ဘယ် ဆရာဝန် နဲ့ ပြ ခဲ့ တာ လဲ ။  (92.39%)
  Top-2: မုန်တိုင်း အဆင့် အဝါရောင် လိမ္မော်ရောင် အနီရောင် အညိုရောင် အစိမ်းရောင် ။  (0.42%)
  Top-3: လျှပ်စစ် ဝါယာ ရှော့ ဖြစ် ရင် ရေ စို အဝတ် နဲ့ မီး မ ငြှိမ်းသတ် ပါ နဲ့ ။  (0.29%)
  Top-4: ငါ့ အမေ ပျား တုပ် ခံ ရ လို့ ဆေးရုံ တင် လိုက် ရ တယ် ။  (0.27%)
  Top-5: အကောင်းဆုံး ဆေးခန်း က ဘယ် ဆေးခန်း လဲ ။  (0.20%)
────────────────────────────────────────────────────────────
[INFO] Done.

(msl_onnx) E:\LU_Lab\MSL_Tran_Demo>
```

ဒီတစ်ခါတော့ ရလဒ် အားလုံး ကောင်းသွားပြီ။  

## To Do

- Online Video Recognition Testing with Updated Code!
- Documentation/Logging

## Debugging (especially on Online-Recognition)  

Updated infer_onnx.py code is as follows:  

In [97]:
from IPython.display import Markdown

file_path = './src/infer_onnx.py'

with open(file_path, 'r') as f:
    code = f.read()

# 'python' keyword triggers the syntax highlighter
display(Markdown(f"```python\n{code}\n```"))

```python
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
infer_onnx.py  –  Real-time MSL recognition on Windows using ONNX Runtime

Bugs fixed in this version
───────────────────────────
  [INFER-A] process_frame skipped frames without pose detection.
      Training always includes every frame (zeros for undetected joints).
      Fix: process_frame always returns the keypoint array, never None.

  [INFER-B] normalize_sequence used 2-D scale and left z unscaled.
      Fix: exact mirror of extract_keypoints.normalize_sequence() —
      full 3-D midpoint and 3-D Euclidean shoulder width, all axes.

  [INFER-C] *** ROOT CAUSE OF WEBCAM FAILURE ***
      Webcam classified every classify_every=15 frames = 0.5 seconds.
      Signs take 1–4 seconds; feeding 15 frames to a model trained on
      full-length signs produces garbage predictions.

      infer.py uses a TIME-BASED buffer: accumulate buffer_sec=2.5 sec
      of frames (~75 at 30 fps), then classify and clear.  This gives
      the model a realistic-duration sign sequence matching training.

      Fix: replaced the frame-count trigger with a time-based trigger
      using --buffer_sec (default 2.5), identical to infer.py.

  [INFER-D] np.int64 / int dict key mismatch.
      Fix: always cast to int() before idx2label lookup.

  [INFER-E] ONNXModel.predict returned a scalar instead of 1-D array.
      Fix: return output[0] — shape (num_classes,).

  [INFER-F] model_complexity=1 everywhere; training used complexity=2.
      Fix: complexity=2 for video (matching training), 1 for webcam
      (speed tradeoff, same as infer.py).

  [INFER-G] extractor.pose.model_complexity AttributeError on Windows.
      MediaPipe's Pose object does not expose model_complexity publicly.
      Fix: stored as self.model_complexity on KeypointExtractor.

New features
────────────
  • Video saving — two output files per inference:
      {name}_clean.mp4  —  label/confidence overlay only
      {name}_debug.mp4  —  label overlay + MediaPipe skeleton lines
    Works for both --video and --webcam modes.
    Saved to --output_dir (default: predictions_output/).

  • Webcam log — every classification event writes top-k results to the
    log file with timestamp and separator, matching the video log format.
"""

import argparse
import importlib
import json
import sys
from datetime import datetime
from pathlib import Path

import cv2
import numpy as np
import onnxruntime as ort

try:
    from PIL import Image, ImageDraw, ImageFont
    HAS_PIL = True
except ImportError:
    HAS_PIL = False

# ── Constants (must match extract_keypoints.py exactly) ───────────────────────
POSE_N          = 33
HAND_N          = 21
TOTAL_JOINTS    = POSE_N + HAND_N * 2   # 75
COORDS          = 3
FEAT_DIM        = TOTAL_JOINTS * COORDS  # 225

POSE_LEFT_SHOULDER  = 11
POSE_RIGHT_SHOULDER = 12
LEFT_HAND_START     = POSE_N            # 33
RIGHT_HAND_START    = POSE_N + HAND_N   # 54


# ─── MediaPipe import ─────────────────────────────────────────────────────────

def _import_mp():
    for prefix in ["mediapipe.solutions", "mediapipe.python.solutions"]:
        try:
            pose  = importlib.import_module(f"{prefix}.pose")
            hands = importlib.import_module(f"{prefix}.hands")
            draw  = importlib.import_module(f"{prefix}.drawing_utils")
            return pose, hands, draw
        except (ImportError, ModuleNotFoundError):
            continue
    try:
        import mediapipe as _mp
        return _mp.solutions.pose, _mp.solutions.hands, _mp.solutions.drawing_utils
    except AttributeError:
        pass
    raise ImportError(
        "Cannot import MediaPipe solutions.\n"
        "Try: pip uninstall mediapipe -y && pip install mediapipe==0.10.18"
    )

mp_pose, mp_hands, mp_drawing = _import_mp()


# ─── Myanmar text rendering ──────────────────────────────────────────────────

class TextRenderer:
    _FONT_CANDIDATES = [
        "C:\\Windows\\Fonts\\Padauk.ttf",
        "C:\\Windows\\Fonts\\NotoSansMyanmar-Regular.ttf",
        "C:\\Windows\\Fonts\\NotoSansMyanmar.ttf",
        "C:\\Windows\\Fonts\\myanmar3.ttf",
        "/usr/share/fonts/truetype/padauk/Padauk-Regular.ttf",
        "/usr/share/fonts/truetype/noto/NotoSansMyanmar-Regular.ttf",
        "fonts\\Padauk.ttf",
        "fonts/NotoSansMyanmar-Regular.ttf",
    ]

    def __init__(self, font_size: int = 32):
        self.font = self._load_font(font_size)
        self.can_render_myanmar = (self.font is not None)
        if not self.can_render_myanmar:
            print()
            print("  ╔═══════════════════════════════════════════════════════════╗")
            print("  ║  INFO: No Myanmar shaping font found.                    ║")
            print("  ║  Displaying readable Class IDs instead of Myanmar text.  ║")
            print("  ╚═══════════════════════════════════════════════════════════╝")
            print()

    def _load_font(self, size):
        if not HAS_PIL:
            return None
        for path in self._FONT_CANDIDATES:
            if Path(path).exists():
                try:
                    return ImageFont.truetype(path, size)
                except Exception:
                    continue
        return None

    def put_text(self, frame, text, position=(20, 20), color=(0, 255, 0)):
        if self.font and HAS_PIL:
            pil = Image.fromarray(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
            d   = ImageDraw.Draw(pil)
            r, g, b = color[2], color[1], color[0]
            for dx in [-1, 0, 1]:
                for dy in [-1, 0, 1]:
                    if dx or dy:
                        d.text((position[0]+dx, position[1]+dy), text,
                               font=self.font, fill=(0, 0, 0))
            d.text(position, text, font=self.font, fill=(r, g, b))
            return cv2.cvtColor(np.array(pil), cv2.COLOR_RGB2BGR)
        cv2.putText(frame, text, position, cv2.FONT_HERSHEY_SIMPLEX,
                    0.75, color, 2, cv2.LINE_AA)
        return frame


# ─── Keypoint extraction ──────────────────────────────────────────────────────

class KeypointExtractor:
    """
    Extracts (TOTAL_JOINTS, 3) keypoint arrays from BGR frames.

    Fix [INFER-G]: model_complexity is stored as self.model_complexity
    because mediapipe's Pose object does not expose it as a public attribute
    on all Windows builds (AttributeError otherwise).

    Fix [INFER-A]: process_frame always returns an array, never None.
    Matches extract_keypoints.py which includes every frame, using zeros
    for any joint that MediaPipe failed to detect.

    Hand convention (matches extract_keypoints.py):
      MediaPipe reports handedness from the camera perspective (mirrored).
      label='Right' → signer's LEFT  hand → slot LEFT_HAND_START  (33).
      label='Left'  → signer's RIGHT hand → slot RIGHT_HAND_START (54).
    """

    def __init__(self, model_complexity: int = 2):
        # Fix [INFER-G]: store complexity ourselves — mediapipe's Pose object
        # does NOT expose a public .model_complexity on all Windows builds.
        self.model_complexity = model_complexity

        self.pose  = mp_pose.Pose(
            static_image_mode=False,
            model_complexity=model_complexity,
            smooth_landmarks=True,
            enable_segmentation=False,
            min_detection_confidence=0.5,
            min_tracking_confidence=0.5,
        )
        self.hands = mp_hands.Hands(
            static_image_mode=False,
            max_num_hands=2,
            model_complexity=min(model_complexity, 1),  # Hands caps at 1
            min_detection_confidence=0.5,
            min_tracking_confidence=0.5,
        )
        self._last_pose  = None
        self._last_hands = None

    def process_frame(self, frame: np.ndarray) -> np.ndarray:
        """
        Returns (TOTAL_JOINTS, 3) float32 for every frame.
        Zeros for any undetected joint/hand — never returns None.
        """
        rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        rgb.flags.writeable = False
        p_res = self.pose.process(rgb)
        h_res = self.hands.process(rgb)
        self._last_pose  = p_res
        self._last_hands = h_res

        kp = np.zeros((TOTAL_JOINTS, COORDS), dtype=np.float32)

        if p_res.pose_landmarks:
            for i, lm in enumerate(p_res.pose_landmarks.landmark):
                kp[i] = [lm.x, lm.y, lm.z]

        if h_res.multi_hand_landmarks and h_res.multi_handedness:
            for lm_list, handedness in zip(
                h_res.multi_hand_landmarks, h_res.multi_handedness
            ):
                label  = handedness.classification[0].label
                offset = LEFT_HAND_START if label == 'Right' else RIGHT_HAND_START
                for i, lm in enumerate(lm_list.landmark):
                    kp[offset + i] = [lm.x, lm.y, lm.z]

        return kp  # always an array, never None

    def draw_landmarks(self, frame: np.ndarray) -> np.ndarray:
        """Draw MediaPipe skeleton onto frame IN-PLACE and return it."""
        if self._last_pose and self._last_pose.pose_landmarks:
            mp_drawing.draw_landmarks(
                frame, self._last_pose.pose_landmarks, mp_pose.POSE_CONNECTIONS
            )
        if self._last_hands and self._last_hands.multi_hand_landmarks:
            for lm in self._last_hands.multi_hand_landmarks:
                mp_drawing.draw_landmarks(frame, lm, mp_hands.HAND_CONNECTIONS)
        return frame

    def close(self):
        self.pose.close()
        self.hands.close()


# ─── Normalization ────────────────────────────────────────────────────────────

def normalize_sequence(seq: np.ndarray) -> np.ndarray:
    """
    Fix [INFER-B]: exact mirror of extract_keypoints.normalize_sequence().
    Full 3-D shoulder-midpoint subtraction and 3-D shoulder-width scaling,
    applied to all three coordinate axes.
    """
    l_sh = seq[:, POSE_LEFT_SHOULDER,  :]   # (T, 3)
    r_sh = seq[:, POSE_RIGHT_SHOULDER, :]   # (T, 3)
    mid  = (l_sh + r_sh) / 2.0
    dist = np.linalg.norm(r_sh - l_sh, axis=1, keepdims=True)
    dist = np.where(dist < 1e-6, 1.0, dist)
    return ((seq - mid[:, None, :]) / dist[:, None, :]).astype(np.float32)


# ─── Preprocessor ─────────────────────────────────────────────────────────────

class Preprocessor:
    """
    Stack raw (75, 3) keypoint frames → normalize → crop/pad → model input.

    Processing order matches infer.py exactly:
      1. Stack   → (T, 75, 3)
      2. Normalize (3-D shoulder-centred, full 3-D scale)
      3. Centre-crop  if T > max_seq_len
      4. Zero-pad     if T < max_seq_len
      5. Build mask   (True = padded position)
      6. Reshape for model type
    """

    def __init__(self, max_seq_len: int = 200):
        self.max_seq_len = max_seq_len

    def __call__(self, kp_list: list, model_type: str) -> dict:
        if not kp_list:
            return {}

        seq = np.stack(kp_list, axis=0).astype(np.float32)  # (T, 75, 3)
        seq = normalize_sequence(seq)
        T   = seq.shape[0]

        if T > self.max_seq_len:
            start = (T - self.max_seq_len) // 2
            seq   = seq[start: start + self.max_seq_len]
            T     = self.max_seq_len

        real_len = T

        if T < self.max_seq_len:
            pad = np.zeros((self.max_seq_len - T, TOTAL_JOINTS, COORDS), dtype=np.float32)
            seq = np.concatenate([seq, pad], axis=0)

        mask = np.zeros(self.max_seq_len, dtype=np.bool_)
        mask[real_len:] = True

        if model_type == "stgcn":
            kp_out = np.expand_dims(seq.transpose(2, 0, 1), axis=0).astype(np.float32)
            return {"keypoints": kp_out}
        else:
            kp_out = np.expand_dims(
                seq.reshape(self.max_seq_len, FEAT_DIM), axis=0
            ).astype(np.float32)
            return {"keypoints": kp_out, "mask": mask[np.newaxis]}


# ─── ONNX model wrapper ───────────────────────────────────────────────────────

class ONNXModel:
    def __init__(self, onnx_path: str, meta_path: str = None):
        available = ort.get_available_providers()
        preferred = ["CUDAExecutionProvider", "DmlExecutionProvider", "CPUExecutionProvider"]
        providers = [p for p in preferred if p in available]

        self.session = ort.InferenceSession(onnx_path, providers=providers)
        print(f"[INFO] ONNX loaded: {Path(onnx_path).name}"
              f"  (provider: {self.session.get_providers()[0]})")

        self.meta = {}
        if meta_path and Path(meta_path).exists():
            try:
                with open(meta_path, encoding="utf-8") as f:
                    self.meta = json.load(f)
            except Exception as e:
                print(f"[WARNING] Could not read metadata: {e}")
        else:
            print(f"[INFO] Metadata not found at '{meta_path}', using defaults.")

        self.model_type  = self.meta.get("model_type",  "transformer")
        self.max_seq_len = self.meta.get("max_seq_len", 200)
        self.num_classes = self.meta.get("num_classes", 558)

    def predict(self, inputs: dict) -> np.ndarray:
        """Fix [INFER-E]: return (num_classes,) not scalar."""
        return self.session.run(None, inputs)[0][0]

    @staticmethod
    def softmax(logits: np.ndarray) -> np.ndarray:
        e = np.exp(logits - logits.max())
        return e / e.sum()


# ─── Label map ────────────────────────────────────────────────────────────────

def load_idx2label(path: str) -> dict:
    with open(path, encoding="utf-8") as f:
        data = json.load(f)
    return {int(k): v for k, v in data.get("idx2label", {}).items()}


def _get_label(idx2label: dict, class_id) -> str:
    """Fix [INFER-D]: always cast to int() before dict lookup."""
    return idx2label.get(int(class_id), f"[class {int(class_id)}]")


# ─── Helpers ──────────────────────────────────────────────────────────────────

def _overlay_text(renderer, frame, text, color):
    """Put prediction text at top-left of frame."""
    return renderer.put_text(frame, text, position=(10, 10), color=color)


def _pred_color(confidence: float) -> tuple:
    """Green ≥80%, yellow ≥50%, red otherwise."""
    if confidence >= 0.80: return (0, 200,   0)
    if confidence >= 0.50: return (0, 200, 200)
    return (0, 0, 255)


def _build_overlay_text(top_idx, probs, idx2label, renderer, top_k):
    """Single-line text for the video overlay (top-1 label + confidence)."""
    best = top_idx[0]
    conf = probs[int(best)]
    lbl  = _get_label(idx2label, best) if idx2label else f"Class {int(best)}"
    if renderer.can_render_myanmar:
        return f"{lbl}  ({conf*100:.1f}%)", conf
    return f"Class {int(best)}  ({conf*100:.1f}%)", conf


def _build_console_block(top_idx, probs, idx2label, model_type, source_name, top_k):
    """Multi-line console output block, same format as run_video."""
    sep   = "─" * 60
    lines = [f"Source : {source_name}", f"Model  : {model_type.upper()}", sep]
    for rank, idx in enumerate(top_idx, 1):
        lbl = _get_label(idx2label, idx) if idx2label else f"[class {int(idx)}]"
        lines.append(f"  Top-{rank}: {lbl}  ({probs[int(idx)]*100:.2f}%)")
    lines.append(sep)
    return "\n".join(lines)


def _build_log_block(top_idx, probs, idx2label, source_name, top_k):
    """Log block with timestamp and top-k results."""
    ts    = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    sep   = "=" * 60
    lines = [sep, f"[{ts}]  {source_name}"]
    for rank, idx in enumerate(top_idx, 1):
        lbl = _get_label(idx2label, idx) if idx2label else "?"
        lbl = lbl.encode("utf-8", errors="replace").decode("utf-8")
        lines.append(f"  Top-{rank:>2}: Class {int(idx):>4d} | "
                     f"{lbl:<45s} | {probs[int(idx)]*100:6.2f}%")
    lines.append("")
    return "\n".join(lines) + "\n"


def _append_log(path, text):
    try:
        with open(path, "a", encoding="utf-8-sig") as f:
            f.write(text)
    except Exception as e:
        print(f"[WARNING] Could not write log: {e}")


def _make_video_writer(output_path: Path, fps: float, width: int, height: int):
    """Create a cv2.VideoWriter with MP4V codec."""
    fourcc = cv2.VideoWriter_fourcc(*"mp4v")
    return cv2.VideoWriter(str(output_path), fourcc, fps, (width, height))


# ─── Video processing ─────────────────────────────────────────────────────────

def run_video(model, prep, extractor, renderer, video_path,
              top_k=5, log_file=None, idx2label=None,
              output_dir=None, debug=False):
    """
    Process a single video file.  Saves two output videos:
      {stem}_clean.mp4  —  label overlay only
      {stem}_debug.mp4  —  label overlay + MediaPipe skeleton lines

    Matches infer.py / process_video() exactly:
      every frame included (zeros for undetected joints),
      normalize after stacking, centre-crop or zero-pad.
    """
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        print(f"[ERROR] Cannot open video: {video_path}")
        return

    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    fps_src      = cap.get(cv2.CAP_PROP_FPS) or 30.0
    w            = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    h            = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    stem         = Path(video_path).stem

    print(f"[INFO] {Path(video_path).name} — {total_frames} frames @ {fps_src:.1f} fps")
    print(f"[INFO] MediaPipe pose complexity: {extractor.model_complexity}"
          f"  (must match training; training default=2)")

    # --- Pass 1: extract all keypoints + collect frames for output video ---
    kp_list    = []
    raw_frames = []   # store for writing output videos after prediction
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        kp_list.append(extractor.process_frame(frame))  # always appends
        raw_frames.append(frame.copy())
    cap.release()

    if not kp_list:
        print("[WARNING] No frames could be read from the video.")
        return

    T_raw = len(kp_list)
    T_use = min(T_raw, model.max_seq_len)
    print(f"[INFO] {T_raw} frames extracted → using {T_use} "
          f"({'centre-cropped' if T_raw > model.max_seq_len else 'zero-padded to'}"
          f" {model.max_seq_len})")

    # --- Inference ---
    inputs  = prep(kp_list, model.model_type)
    logits  = model.predict(inputs)
    probs   = ONNXModel.softmax(logits)
    top_idx = np.argsort(probs)[::-1][:top_k]

    # --- Console output ---
    print("\n" + _build_console_block(top_idx, probs, idx2label,
                                      model.model_type, Path(video_path).name, top_k))

    # --- Log file ---
    if log_file:
        _append_log(log_file,
                    _build_log_block(top_idx, probs, idx2label,
                                     Path(video_path).name, top_k))

    # --- Save output videos ---
    if output_dir:
        out_dir = Path(output_dir)
        out_dir.mkdir(parents=True, exist_ok=True)

        overlay_text, conf = _build_overlay_text(top_idx, probs, idx2label, renderer, top_k)
        color              = _pred_color(conf)

        clean_path = out_dir / f"{stem}_clean.mp4"
        debug_path = out_dir / f"{stem}_debug.mp4"
        vw_clean   = _make_video_writer(clean_path, fps_src, w, h)
        vw_debug   = _make_video_writer(debug_path, fps_src, w, h)

        for i, raw in enumerate(raw_frames):
            # Clean: label overlay only
            clean_frame = renderer.put_text(raw.copy(), overlay_text,
                                            position=(10, 10), color=color)
            vw_clean.write(clean_frame)

            # Debug: re-run MediaPipe on this frame to get landmarks,
            # then draw skeleton + label
            debug_frame = raw.copy()
            extractor.process_frame(debug_frame)       # updates _last_pose/_last_hands
            extractor.draw_landmarks(debug_frame)       # draws skeleton in-place
            debug_frame = renderer.put_text(debug_frame, overlay_text,
                                            position=(10, 10), color=color)
            vw_debug.write(debug_frame)

        vw_clean.release()
        vw_debug.release()
        print(f"[INFO] Saved  clean  → {clean_path}")
        print(f"[INFO] Saved  debug  → {debug_path}")


# ─── Webcam processing ────────────────────────────────────────────────────────

def run_webcam(model, prep, extractor, renderer, idx2label,
               cam_idx=0, top_k=5, log_file=None,
               buffer_sec=2.5, output_dir=None):
    """
    Live webcam sign recognition.

    Fix [INFER-C] — root cause of webcam failure:
      The old code classified every classify_every=15 frames ≈ 0.5 sec.
      Signs take 1–4 seconds; 15 frames is far too short and produces
      garbage predictions regardless of model quality.

      Now uses a TIME-BASED buffer matching infer.py exactly:
        buf_frames = int(fps * buffer_sec)   (default 2.5 sec ≈ 75 frames)
        Classify and clear when the buffer reaches buf_frames.
        Progress bar fills towards buf_frames, giving clear visual feedback.

    Saves two output videos (if --output_dir given):
      webcam_{timestamp}_clean.mp4  — label overlay only
      webcam_{timestamp}_debug.mp4  — label overlay + skeleton

    Every classification event is logged to --log_file with top-k results.

    Keys:  Q = quit  |  C = clear buffer manually
    """
    cap = cv2.VideoCapture(cam_idx)
    if not cap.isOpened():
        print(f"[ERROR] Cannot open webcam {cam_idx}")
        return

    fps        = cap.get(cv2.CAP_PROP_FPS) or 30.0
    w          = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    h          = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    # Fix [INFER-C]: time-based buffer, matching infer.py
    buf_frames = max(20, int(fps * buffer_sec))

    print(f"[INFO] Webcam  {w}×{h} @ {fps:.1f} fps")
    print(f"[INFO] Buffer  {buf_frames} frames ({buffer_sec:.1f} sec) — sign within this window")
    print(f"[INFO] MediaPipe pose complexity: {extractor.model_complexity}")
    print(f"[INFO] Q = quit | C = clear buffer manually")

    # ── Optional video writers ──
    ts_str     = datetime.now().strftime("%Y%m%d_%H%M%S")
    vw_clean   = vw_debug = None
    if output_dir:
        out_dir = Path(output_dir)
        out_dir.mkdir(parents=True, exist_ok=True)
        clean_path = out_dir / f"webcam_{ts_str}_clean.mp4"
        debug_path = out_dir / f"webcam_{ts_str}_debug.mp4"
        vw_clean   = _make_video_writer(clean_path, fps, w, h)
        vw_debug   = _make_video_writer(debug_path, fps, w, h)
        print(f"[INFO] Recording clean → {clean_path}")
        print(f"[INFO] Recording debug → {debug_path}")

    kp_buffer  = []
    last_text  = "Perform a sign — buffer filling..."
    last_color = (200, 200, 200)
    classify_count = 0

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        # Extract keypoints — always append (Fix [INFER-A])
        kp = extractor.process_frame(frame)
        kp_buffer.append(kp)

        # Fix [INFER-C]: classify when buffer reaches buf_frames, then clear
        if len(kp_buffer) >= buf_frames:
            inputs  = prep(list(kp_buffer), model.model_type)
            logits  = model.predict(inputs)
            probs   = ONNXModel.softmax(logits)
            top_idx = np.argsort(probs)[::-1][:top_k]

            overlay_text, conf = _build_overlay_text(
                top_idx, probs, idx2label, renderer, top_k
            )
            last_text  = overlay_text
            last_color = _pred_color(conf)
            classify_count += 1

            # Console output
            print(f"\n[Prediction #{classify_count}]")
            print(_build_console_block(top_idx, probs, idx2label,
                                       model.model_type, f"webcam frame", top_k))

            # Log file — top_k results
            if log_file:
                _append_log(log_file,
                            _build_log_block(top_idx, probs, idx2label,
                                             f"webcam #{classify_count}", top_k))

            kp_buffer.clear()   # start fresh for next sign

        # ── Build display frames ──
        # Progress bar towards buf_frames (not max_seq_len)
        fill   = int(len(kp_buffer) / buf_frames * w)
        pct    = len(kp_buffer) / buf_frames * 100

        # Clean frame: label + progress bar only
        clean_frame = renderer.put_text(frame.copy(), last_text,
                                        position=(10, 10), color=last_color)
        cv2.rectangle(clean_frame, (0, h - 12), (fill, h), (0, 180, 255), -1)
        cv2.putText(clean_frame, f"buf {len(kp_buffer)}/{buf_frames}  ({pct:.0f}%)",
                    (10, h - 15), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (220, 220, 220), 1)

        # Debug frame: skeleton + label + progress bar
        debug_frame = frame.copy()
        extractor.draw_landmarks(debug_frame)
        debug_frame = renderer.put_text(debug_frame, last_text,
                                        position=(10, 10), color=last_color)
        cv2.rectangle(debug_frame, (0, h - 12), (fill, h), (0, 180, 255), -1)
        cv2.putText(debug_frame, f"buf {len(kp_buffer)}/{buf_frames}  ({pct:.0f}%)",
                    (10, h - 15), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (220, 220, 220), 1)

        # Write to video files
        if vw_clean: vw_clean.write(clean_frame)
        if vw_debug: vw_debug.write(debug_frame)

        # Show the debug view on screen (most useful for real-time feedback)
        cv2.imshow("MSL Recognition  [Q=quit  C=clear]", debug_frame)

        key = cv2.waitKey(1) & 0xFF
        if key == ord("q"):
            break
        elif key == ord("c"):
            kp_buffer.clear()
            last_text  = "Buffer cleared — perform a sign"
            last_color = (200, 200, 200)
            print("[INFO] Buffer manually cleared.")

    cap.release()
    if vw_clean: vw_clean.release()
    if vw_debug: vw_debug.release()
    cv2.destroyAllWindows()

    if output_dir:
        print(f"[INFO] Saved  clean  → {clean_path}")
        print(f"[INFO] Saved  debug  → {debug_path}")
    print(f"[INFO] Total classifications: {classify_count}")


# ─── CLI ──────────────────────────────────────────────────────────────────────

def main():
    parser = argparse.ArgumentParser(
        description="MSL Sign Language Inference — ONNX Runtime",
        formatter_class=argparse.RawDescriptionHelpFormatter,
        epilog="""
Examples:
  # Single video, save both clean and debug outputs
  python infer_onnx.py --onnx_model onnx_models/bilstm_msl.onnx \\
      --video sample.mp4 --top_k 5 --output_dir predictions_output

  # Webcam, 3-second buffer, save recordings + log
  python infer_onnx.py --onnx_model onnx_models/transformer_msl.onnx \\
      --webcam 0 --buffer_sec 3.0 --top_k 5 \\
      --output_dir predictions_output --log_file webcam_log.txt
""")

    parser.add_argument("--onnx_model",   required=True,
                        help="Path to .onnx model file (NOT the .json metadata)")
    parser.add_argument("--metadata",     default=None,
                        help="Path to .json metadata (default: same stem as .onnx)")
    parser.add_argument("--label_map",    default=None,
                        help="Path to label_map.json (default: next to .onnx)")
    parser.add_argument("--output_dir",   default="predictions_output",
                        help="Directory for saved videos (default: predictions_output/)")
    parser.add_argument("--log_file",     default=None,
                        help="Append top-k results to this text file")
    parser.add_argument("--top_k",        type=int, default=5,
                        help="Number of top predictions to show/log (default: 5)")
    parser.add_argument("--font_size",    type=int, default=32,
                        help="Myanmar font size for overlay (default: 32)")
    parser.add_argument("--model_complexity", type=int, default=None,
                        help="MediaPipe pose complexity: 0/1/2. "
                             "Default: 2 for video (matches training), 1 for webcam.")

    grp = parser.add_mutually_exclusive_group(required=True)
    grp.add_argument("--video",  help="Path to a video file")
    grp.add_argument("--webcam", type=int, metavar="CAM_IDX",
                     help="Webcam device index (0 = default camera)")

    parser.add_argument("--buffer_sec", type=float, default=2.5,
                        help="[Webcam] Seconds of frames to accumulate before "
                             "each classification (default: 2.5, same as infer.py)")
    parser.add_argument("--debug",       action="store_true",
                        help="Print extra diagnostic information")
    args = parser.parse_args()

    # ── Validate ONNX path ────────────────────────────────────────────────────
    model_p = Path(args.onnx_model)
    if model_p.suffix.lower() == ".json":
        sys.exit(
            f"[ERROR] --onnx_model must be a .onnx file, not a .json file.\n"
            f"        You passed : {model_p}\n"
            f"        Did you mean: {model_p.with_suffix('.onnx')}"
        )
    if not model_p.exists():
        sys.exit(f"[ERROR] ONNX model not found: {model_p}")

    meta_p = args.metadata or str(model_p.with_suffix(".json"))
    lmap_p = args.label_map or str(model_p.parent / "label_map.json")

    if not Path(lmap_p).exists():
        sys.exit(f"[ERROR] label_map.json not found: {lmap_p}\n"
                 f"        Copy it from your Linux server alongside the .onnx file.")

    # ── Load resources ────────────────────────────────────────────────────────
    idx2label = load_idx2label(lmap_p)
    print(f"[INFO] Loaded {len(idx2label)} labels from label map.")

    model    = ONNXModel(str(model_p), meta_p)
    prep     = Preprocessor(max_seq_len=model.max_seq_len)
    renderer = TextRenderer(font_size=args.font_size)

    if args.log_file:
        print(f"[INFO] Logging top-{args.top_k} results to: {args.log_file}")
    if args.output_dir:
        print(f"[INFO] Saving output videos to: {args.output_dir}/")

    # Fix [INFER-F]: complexity=2 for video (matches training), 1 for webcam (speed)
    if args.model_complexity is not None:
        video_complexity  = args.model_complexity
        webcam_complexity = args.model_complexity
    else:
        video_complexity  = 2
        webcam_complexity = 1

    try:
        if args.video:
            extractor = KeypointExtractor(model_complexity=video_complexity)
            try:
                run_video(model, prep, extractor, renderer, args.video,
                          top_k      = args.top_k,
                          log_file   = args.log_file,
                          idx2label  = idx2label,
                          output_dir = args.output_dir,
                          debug      = args.debug)
            finally:
                extractor.close()
        else:
            extractor = KeypointExtractor(model_complexity=webcam_complexity)
            try:
                run_webcam(model, prep, extractor, renderer, idx2label,
                           cam_idx    = args.webcam,
                           top_k      = args.top_k,
                           log_file   = args.log_file,
                           buffer_sec = args.buffer_sec,
                           output_dir = args.output_dir)
            finally:
                extractor.close()
    finally:
        print("[INFO] Done.")


if __name__ == "__main__":
    main()


```

## Video Testing on Windows OS

```
(msl_onnx) E:\LU_Lab\MSL_Tran_Demo>python infer_onnx.py --onnx_model onnx_models\bilstm_msl.onnx   --video sample4inference\idx20-30.mp4  --top_k 5 --output_dir predictions_output --log_file results.txt
[INFO] Loaded 556 labels from label map.
[INFO] ONNX loaded: bilstm_msl.onnx  (provider: CPUExecutionProvider)
[INFO] Logging top-5 results to: results.txt
[INFO] Saving output videos to: predictions_output/
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
[INFO] idx20-30.mp4 — 127 frames @ 30.0 fps
[INFO] MediaPipe pose complexity: 2  (must match training; training default=2)
C:\Users\801680\Anaconda3\envs\msl_onnx\lib\site-packages\google\protobuf\symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '
[INFO] 127 frames extracted → using 127 (zero-padded to 200)

Source : idx20-30.mp4
Model  : BILSTM
────────────────────────────────────────────────────────────
  Top-1: မြေနိမ့်ဒေသ ။  (80.73%)
  Top-2: ချိုင့် ။  (0.16%)
  Top-3: ဘေး အိမ် ကို ဓားပြ ဝင် တိုက် နေ ပါ တယ် ။  (0.13%)
  Top-4: ဓာတ်မှန် ရိုက် တဲ့ နေရာ က ဘယ်နား မှာ လဲ သိ ချင် လို့ ပါ ။  (0.12%)
  Top-5: လျှပ်စစ် ဝါယာ ရှော့ ဖြစ် ရင် မီးသတ်ဆေးဘူး အသုံးပြု ပါ ။  (0.12%)
────────────────────────────────────────────────────────────
[INFO] Saved  clean  → predictions_output\idx20-30_clean.mp4
[INFO] Saved  debug  → predictions_output\idx20-30_debug.mp4
[INFO] Done.

(msl_onnx) E:\LU_Lab\MSL_Tran_Demo>
```

```
(msl_onnx) E:\LU_Lab\MSL_Tran_Demo>python infer_onnx.py --onnx_model onnx_models\transformer_msl.onnx   --video sample4inference\idx20-30.mp4  --top_k 5 --output_dir predictions_output --log_file results.txt
[INFO] Loaded 556 labels from label map.
[INFO] ONNX loaded: transformer_msl.onnx  (provider: CPUExecutionProvider)
[INFO] Logging top-5 results to: results.txt
[INFO] Saving output videos to: predictions_output/
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
[INFO] idx20-30.mp4 — 127 frames @ 30.0 fps
[INFO] MediaPipe pose complexity: 2  (must match training; training default=2)
C:\Users\801680\Anaconda3\envs\msl_onnx\lib\site-packages\google\protobuf\symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '
[INFO] 127 frames extracted → using 127 (zero-padded to 200)

Source : idx20-30.mp4
Model  : TRANSFORMER
────────────────────────────────────────────────────────────
  Top-1: မြေနိမ့်ဒေသ ။  (80.68%)
  Top-2: ချိုင့် ။  (0.30%)
  Top-3: မီးခလုပ် များ ကို ပိတ် ပါ ။  (0.28%)
  Top-4: ဂရုစိုက် ။  (0.25%)
  Top-5: လေမုန်တိုင်း ကျ မယ် ။  (0.15%)
────────────────────────────────────────────────────────────
[INFO] Saved  clean  → predictions_output\idx20-30_clean.mp4
[INFO] Saved  debug  → predictions_output\idx20-30_debug.mp4
[INFO] Done.

(msl_onnx) E:\LU_Lab\MSL_Tran_Demo>
```

```
(msl_onnx) E:\LU_Lab\MSL_Tran_Demo>python infer_onnx.py --onnx_model onnx_models\stgcn_msl.onnx   --video sample4inference\idx20-30.mp4  --top_k 5 --output_dir predictions_output --log_file results.txt
[INFO] Loaded 556 labels from label map.
[INFO] ONNX loaded: stgcn_msl.onnx  (provider: CPUExecutionProvider)
[INFO] Logging top-5 results to: results.txt
[INFO] Saving output videos to: predictions_output/
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
[INFO] idx20-30.mp4 — 127 frames @ 30.0 fps
[INFO] MediaPipe pose complexity: 2  (must match training; training default=2)
C:\Users\801680\Anaconda3\envs\msl_onnx\lib\site-packages\google\protobuf\symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '
[INFO] 127 frames extracted → using 127 (zero-padded to 200)

Source : idx20-30.mp4
Model  : STGCN
────────────────────────────────────────────────────────────
  Top-1: မြေနိမ့်ဒေသ ။  (82.02%)
  Top-2: မီးသတ်သမား ။  (0.68%)
  Top-3: အထူး သတိပြု ။  (0.63%)
  Top-4: မြစ်ခေျာင်း ။  (0.51%)
  Top-5: စားပွဲခုံ အောက် မှာ ဝင်နေ ။  (0.34%)
────────────────────────────────────────────────────────────
[INFO] Saved  clean  → predictions_output\idx20-30_clean.mp4
[INFO] Saved  debug  → predictions_output\idx20-30_debug.mp4
[INFO] Done.

(msl_onnx) E:\LU_Lab\MSL_Tran_Demo>
```


## Online-Testing Errors

စမ်းနေတာက "မြေနိမ့်ဒေသ"  

```
(msl_onnx) E:\LU_Lab\MSL_Tran_Demo>python infer_onnx.py --onnx_model onnx_models\bilstm_msl.onnx    --webcam 0 --buffer_sec 3.0 --top_k 5   --output_dir predictions_output --log_file webcam_log.txt
```

## Log1 Result

```
============================================================
[2026-06-04 10:01:07]  webcam #1
  Top- 1: Class  490 | မနက်ဖြန် ဆေးခန်း လာ ဖို့ လို သေး လား ။        |   5.44%
  Top- 2: Class  484 | မနက် ၉ နာရီ က နေ ည ၁၀ နာရီ ထိ ပါ ။            |   5.43%
  Top- 3: Class  485 | ည ၁၀ နာရီ နောက် ပိုင်း အမျိုးသား နေ ခွင့် မ ရှိ ပါ ဘူး ။ |   1.90%
  Top- 4: Class  552 | ခေါင်း မှာ သွေး တွေ ထွက် နေ တယ် ။             |   1.77%
  Top- 5: Class  528 | အစားသောက် မ တည့် တာ တွေ ရှိ လား ။             |   1.08%

============================================================
[2026-06-04 10:01:15]  webcam #2
  Top- 1: Class  485 | ည ၁၀ နာရီ နောက် ပိုင်း အမျိုးသား နေ ခွင့် မ ရှိ ပါ ဘူး ။ |   7.29%
  Top- 2: Class  484 | မနက် ၉ နာရီ က နေ ည ၁၀ နာရီ ထိ ပါ ။            |   2.96%
  Top- 3: Class  490 | မနက်ဖြန် ဆေးခန်း လာ ဖို့ လို သေး လား ။        |   1.70%
  Top- 4: Class  500 | သွားဆရာဝန် နဲ့ ဒီ နေ့ မနက် တွေ့ နိုင် မ လား ခင်ဗျာ ။ |   1.46%
  Top- 5: Class  114 | မိုးလေဝသ သတင်း များ ကို စဉ်ဆက်မပြတ် နားထောင် ပါ ။ |   1.18%

============================================================
[2026-06-04 10:01:24]  webcam #3
  Top- 1: Class  295 | သူ ဆိုင်ကယ် ပေါ် က ပြုတ်ကျ ပြီး ခေါင်း ပေါက် သွား တယ် ။ |   3.49%
  Top- 2: Class  291 | ငါ့ မောင်လေး ကို သူ တို့ ဝိုင်းရိုက် ကြ တယ် ။ |   1.72%
  Top- 3: Class  298 | ဖန်ကွဲစ ရှ မိ ပြီး သွေး တွေ တအား ထွက် နေ တယ် ။ |   1.45%
  Top- 4: Class  465 | နှစ် ပတ် ကြာ မယ် ။                            |   1.22%
  Top- 5: Class  250 | အရေးပေါ်သူနာပြု အဖွဲ့ ခေါ် ပေး ပါ ။           |   0.86%

============================================================
[2026-06-04 10:01:33]  webcam #4
  Top- 1: Class  256 | လူ တစ်ယောက် ရေ နစ် သွား လို့ ရှာ ဖို့ အကူညီ လွတ် ပေး ပါ ။ |   6.55%
  Top- 2: Class  468 | ဆရာဝန် နဲ့ အမြန်ဆုံး ပြ ချင် ပါ တယ် ။         |   1.91%
  Top- 3: Class  513 | မျက်စဉ်းခတ်ဆေး လို ချင် တယ် ။                 |   1.75%
  Top- 4: Class  255 | ကူညီ ပါ ဦး ဟို မှာ လူ တစ်ယောက် ရေ နစ် နေ တယ် ။ |   1.25%
  Top- 5: Class  470 | ခင်ဗျား ဆရာဝန် နာမည် က ဘာ လဲ ။                |   1.00%

============================================================
[2026-06-04 10:01:42]  webcam #5
  Top- 1: Class  484 | မနက် ၉ နာရီ က နေ ည ၁၀ နာရီ ထိ ပါ ။            |   8.24%
  Top- 2: Class  490 | မနက်ဖြန် ဆေးခန်း လာ ဖို့ လို သေး လား ။        |   4.05%
  Top- 3: Class  514 | သူ့ ခြေတောက် ခွဲစိတ် ရ မယ် ။                  |   1.29%
  Top- 4: Class  527 | ဆေး မ တည့် တာ တွေ ရှိ လား ။                   |   0.86%
  Top- 5: Class  488 | ဘယ် အချိန် ဆေးရုံ ကို အရောက် လာ ရ မှာ လဲ ။    |   0.85%

============================================================
[2026-06-04 10:01:51]  webcam #6
  Top- 1: Class  484 | မနက် ၉ နာရီ က နေ ည ၁၀ နာရီ ထိ ပါ ။            |   5.14%
  Top- 2: Class  498 | ပလာစတာ တစ် ခု လောက် ရ မ လား ။                 |   4.06%
  Top- 3: Class  485 | ည ၁၀ နာရီ နောက် ပိုင်း အမျိုးသား နေ ခွင့် မ ရှိ ပါ ဘူး ။ |   1.91%
  Top- 4: Class  488 | ဘယ် အချိန် ဆေးရုံ ကို အရောက် လာ ရ မှာ လဲ ။    |   1.63%
  Top- 5: Class  522 | လတ်ဆတ် တဲ့ ဟင်းသီးဟင်းရွက် တွေ ကို စား ပါ ။   |   1.35%

============================================================
[2026-06-04 10:02:00]  webcam #7
  Top- 1: Class  484 | မနက် ၉ နာရီ က နေ ည ၁၀ နာရီ ထိ ပါ ။            |  11.01%
  Top- 2: Class  114 | မိုးလေဝသ သတင်း များ ကို စဉ်ဆက်မပြတ် နားထောင် ပါ ။ |   4.66%
  Top- 3: Class  490 | မနက်ဖြန် ဆေးခန်း လာ ဖို့ လို သေး လား ။        |   3.76%
  Top- 4: Class  368 | ရေဒီယို ၊ ရုပ်မြင်သံကြား က သတင်း တွေ ကို သေချာ ကြည့် ။ |   0.96%
  Top- 5: Class  527 | ဆေး မ တည့် တာ တွေ ရှိ လား ။                   |   0.88%

============================================================
[2026-06-04 10:03:49]  webcam #1
  Top- 1: Class  488 | ဘယ် အချိန် ဆေးရုံ ကို အရောက် လာ ရ မှာ လဲ ။    |   6.02%
  Top- 2: Class  484 | မနက် ၉ နာရီ က နေ ည ၁၀ နာရီ ထိ ပါ ။            |   3.85%
  Top- 3: Class  498 | ပလာစတာ တစ် ခု လောက် ရ မ လား ။                 |   1.72%
  Top- 4: Class  114 | မိုးလေဝသ သတင်း များ ကို စဉ်ဆက်မပြတ် နားထောင် ပါ ။ |   1.50%
  Top- 5: Class  516 | ခင်ဗျား မိသားစု ကို ဆက်သွယ် ဖို့ လို တယ် ။    |   1.47%

============================================================
[2026-06-04 10:03:54]  webcam #2
  Top- 1: Class  114 | မိုးလေဝသ သတင်း များ ကို စဉ်ဆက်မပြတ် နားထောင် ပါ ။ |   6.66%
  Top- 2: Class  484 | မနက် ၉ နာရီ က နေ ည ၁၀ နာရီ ထိ ပါ ။            |   2.36%
  Top- 3: Class  141 | ကွင်းပြင် ရှိ တဲ့ နေရာ ကို ပြေး ။             |   1.27%
  Top- 4: Class  485 | ည ၁၀ နာရီ နောက် ပိုင်း အမျိုးသား နေ ခွင့် မ ရှိ ပါ ဘူး ။ |   1.26%
  Top- 5: Class   35 | လျှပ်တစ်ပြက် ရေကြီး ခြင်း ။                   |   0.92%

============================================================
[2026-06-04 10:03:59]  webcam #3
  Top- 1: Class  484 | မနက် ၉ နာရီ က နေ ည ၁၀ နာရီ ထိ ပါ ။            |   3.53%
  Top- 2: Class  485 | ည ၁၀ နာရီ နောက် ပိုင်း အမျိုးသား နေ ခွင့် မ ရှိ ပါ ဘူး ။ |   3.07%
  Top- 3: Class  114 | မိုးလေဝသ သတင်း များ ကို စဉ်ဆက်မပြတ် နားထောင် ပါ ။ |   2.17%
  Top- 4: Class  516 | ခင်ဗျား မိသားစု ကို ဆက်သွယ် ဖို့ လို တယ် ။    |   1.59%
  Top- 5: Class  488 | ဘယ် အချိန် ဆေးရုံ ကို အရောက် လာ ရ မှာ လဲ ။    |   1.28%

============================================================
[2026-06-04 10:04:04]  webcam #4
  Top- 1: Class  515 | သူ့ မိသားစု ကို ငါ တို့ ဖုန်းဆက် ရ မယ် ။      |   2.72%
  Top- 2: Class  484 | မနက် ၉ နာရီ က နေ ည ၁၀ နာရီ ထိ ပါ ။            |   2.39%
  Top- 3: Class  114 | မိုးလေဝသ သတင်း များ ကို စဉ်ဆက်မပြတ် နားထောင် ပါ ။ |   1.67%
  Top- 4: Class  514 | သူ့ ခြေတောက် ခွဲစိတ် ရ မယ် ။                  |   1.63%
  Top- 5: Class  516 | ခင်ဗျား မိသားစု ကို ဆက်သွယ် ဖို့ လို တယ် ။    |   1.43%

============================================================
[2026-06-04 10:04:08]  webcam #5
  Top- 1: Class  516 | ခင်ဗျား မိသားစု ကို ဆက်သွယ် ဖို့ လို တယ် ။    |   2.70%
  Top- 2: Class  124 | ကျိုးကျ နိုင် တဲ့ သစ်ပင် သစ်ကိုင်း တွေ ကို ရှင်းလင်း ထား ။ |   2.00%
  Top- 3: Class  114 | မိုးလေဝသ သတင်း များ ကို စဉ်ဆက်မပြတ် နားထောင် ပါ ။ |   1.98%
  Top- 4: Class  484 | မနက် ၉ နာရီ က နေ ည ၁၀ နာရီ ထိ ပါ ။            |   1.77%
  Top- 5: Class  485 | ည ၁၀ နာရီ နောက် ပိုင်း အမျိုးသား နေ ခွင့် မ ရှိ ပါ ဘူး ။ |   1.55%

============================================================
[2026-06-04 10:04:13]  webcam #6
  Top- 1: Class  484 | မနက် ၉ နာရီ က နေ ည ၁၀ နာရီ ထိ ပါ ။            |   2.37%
  Top- 2: Class  485 | ည ၁၀ နာရီ နောက် ပိုင်း အမျိုးသား နေ ခွင့် မ ရှိ ပါ ဘူး ။ |   1.78%
  Top- 3: Class  124 | ကျိုးကျ နိုင် တဲ့ သစ်ပင် သစ်ကိုင်း တွေ ကို ရှင်းလင်း ထား ။ |   1.21%
  Top- 4: Class  500 | သွားဆရာဝန် နဲ့ ဒီ နေ့ မနက် တွေ့ နိုင် မ လား ခင်ဗျာ ။ |   1.17%
  Top- 5: Class  373 | တောမီး က အရမ်း အန္တရာယ် ကြီး တယ် ။            |   1.12%

============================================================
[2026-06-04 10:04:18]  webcam #7
  Top- 1: Class  141 | ကွင်းပြင် ရှိ တဲ့ နေရာ ကို ပြေး ။             |   4.46%
  Top- 2: Class  485 | ည ၁၀ နာရီ နောက် ပိုင်း အမျိုးသား နေ ခွင့် မ ရှိ ပါ ဘူး ။ |   2.76%
  Top- 3: Class  514 | သူ့ ခြေတောက် ခွဲစိတ် ရ မယ် ။                  |   2.36%
  Top- 4: Class  484 | မနက် ၉ နာရီ က နေ ည ၁၀ နာရီ ထိ ပါ ။            |   2.14%
  Top- 5: Class  124 | ကျိုးကျ နိုင် တဲ့ သစ်ပင် သစ်ကိုင်း တွေ ကို ရှင်းလင်း ထား ။ |   1.68%

============================================================
[2026-06-04 10:04:22]  webcam #8
  Top- 1: Class  514 | သူ့ ခြေတောက် ခွဲစိတ် ရ မယ် ။                  |   5.73%
  Top- 2: Class  298 | ဖန်ကွဲစ ရှ မိ ပြီး သွေး တွေ တအား ထွက် နေ တယ် ။ |   1.39%
  Top- 3: Class  114 | မိုးလေဝသ သတင်း များ ကို စဉ်ဆက်မပြတ် နားထောင် ပါ ။ |   1.04%
  Top- 4: Class  488 | ဘယ် အချိန် ဆေးရုံ ကို အရောက် လာ ရ မှာ လဲ ။    |   0.89%
  Top- 5: Class  288 | ပျောက် သွား တဲ့ လူ ရဲ့ ပုံပန်းသဏ္ဌာန် ကို ပြောပြ နိုင် မ လား ။ |   0.85%

============================================================
[2026-06-04 10:04:27]  webcam #9
  Top- 1: Class  488 | ဘယ် အချိန် ဆေးရုံ ကို အရောက် လာ ရ မှာ လဲ ။    |   5.40%
  Top- 2: Class  465 | နှစ် ပတ် ကြာ မယ် ။                            |   2.02%
  Top- 3: Class  298 | ဖန်ကွဲစ ရှ မိ ပြီး သွေး တွေ တအား ထွက် နေ တယ် ။ |   1.86%
  Top- 4: Class  514 | သူ့ ခြေတောက် ခွဲစိတ် ရ မယ် ။                  |   1.74%
  Top- 5: Class  484 | မနက် ၉ နာရီ က နေ ည ၁၀ နာရီ ထိ ပါ ။            |   1.34%

============================================================
[2026-06-04 10:04:32]  webcam #10
  Top- 1: Class  514 | သူ့ ခြေတောက် ခွဲစိတ် ရ မယ် ။                  |   6.06%
  Top- 2: Class  298 | ဖန်ကွဲစ ရှ မိ ပြီး သွေး တွေ တအား ထွက် နေ တယ် ။ |   2.19%
  Top- 3: Class  114 | မိုးလေဝသ သတင်း များ ကို စဉ်ဆက်မပြတ် နားထောင် ပါ ။ |   1.09%
  Top- 4: Class  288 | ပျောက် သွား တဲ့ လူ ရဲ့ ပုံပန်းသဏ္ဌာန် ကို ပြောပြ နိုင် မ လား ။ |   1.05%
  Top- 5: Class  484 | မနက် ၉ နာရီ က နေ ည ၁၀ နာရီ ထိ ပါ ။            |   0.84%

============================================================
[2026-06-04 10:04:37]  webcam #11
  Top- 1: Class  514 | သူ့ ခြေတောက် ခွဲစိတ် ရ မယ် ။                  |   1.97%
  Top- 2: Class  298 | ဖန်ကွဲစ ရှ မိ ပြီး သွေး တွေ တအား ထွက် နေ တယ် ။ |   1.50%
  Top- 3: Class  485 | ည ၁၀ နာရီ နောက် ပိုင်း အမျိုးသား နေ ခွင့် မ ရှိ ပါ ဘူး ။ |   1.15%
  Top- 4: Class  488 | ဘယ် အချိန် ဆေးရုံ ကို အရောက် လာ ရ မှာ လဲ ။    |   1.02%
  Top- 5: Class  114 | မိုးလေဝသ သတင်း များ ကို စဉ်ဆက်မပြတ် နားထောင် ပါ ။ |   0.99%

============================================================
[2026-06-04 10:04:42]  webcam #12
  Top- 1: Class  485 | ည ၁၀ နာရီ နောက် ပိုင်း အမျိုးသား နေ ခွင့် မ ရှိ ပါ ဘူး ။ |   1.68%
  Top- 2: Class  484 | မနက် ၉ နာရီ က နေ ည ၁၀ နာရီ ထိ ပါ ။            |   1.65%
  Top- 3: Class  515 | သူ့ မိသားစု ကို ငါ တို့ ဖုန်းဆက် ရ မယ် ။      |   1.31%
  Top- 4: Class  498 | ပလာစတာ တစ် ခု လောက် ရ မ လား ။                 |   1.09%
  Top- 5: Class  298 | ဖန်ကွဲစ ရှ မိ ပြီး သွေး တွေ တအား ထွက် နေ တယ် ။ |   1.05%

============================================================
[2026-06-04 10:04:48]  webcam #13
  Top- 1: Class  484 | မနက် ၉ နာရီ က နေ ည ၁၀ နာရီ ထိ ပါ ။            |   2.65%
  Top- 2: Class  485 | ည ၁၀ နာရီ နောက် ပိုင်း အမျိုးသား နေ ခွင့် မ ရှိ ပါ ဘူး ။ |   2.60%
  Top- 3: Class  298 | ဖန်ကွဲစ ရှ မိ ပြီး သွေး တွေ တအား ထွက် နေ တယ် ။ |   2.34%
  Top- 4: Class  509 | ကျွန်မ အမျိုးသမီး ဆရာဝန် နဲ့ ပဲ ပြ ချင် တယ် ။ |   1.14%
  Top- 5: Class  515 | သူ့ မိသားစု ကို ငါ တို့ ဖုန်းဆက် ရ မယ် ။      |   1.14%

============================================================
[2026-06-04 10:04:53]  webcam #14
  Top- 1: Class  484 | မနက် ၉ နာရီ က နေ ည ၁၀ နာရီ ထိ ပါ ။            |  14.48%
  Top- 2: Class  490 | မနက်ဖြန် ဆေးခန်း လာ ဖို့ လို သေး လား ။        |   2.24%
  Top- 3: Class  515 | သူ့ မိသားစု ကို ငါ တို့ ဖုန်းဆက် ရ မယ် ။      |   1.83%
  Top- 4: Class  516 | ခင်ဗျား မိသားစု ကို ဆက်သွယ် ဖို့ လို တယ် ။    |   1.45%
  Top- 5: Class  485 | ည ၁၀ နာရီ နောက် ပိုင်း အမျိုးသား နေ ခွင့် မ ရှိ ပါ ဘူး ။ |   1.19%
```

## Log2 Result

```
============================================================
[2026-06-04 10:17:19]  webcam #1
  Top- 1: Class  485 | ည ၁၀ နာရီ နောက် ပိုင်း အမျိုးသား နေ ခွင့် မ ရှိ ပါ ဘူး ။ |   6.00%
  Top- 2: Class  487 | အမျိုးသမီး လူနာစောင့် ပဲ ည အိပ် စောင့် ရ ပါ မယ် ။ |   3.45%
  Top- 3: Class  484 | မနက် ၉ နာရီ က နေ ည ၁၀ နာရီ ထိ ပါ ။            |   2.70%
  Top- 4: Class   35 | လျှပ်တစ်ပြက် ရေကြီး ခြင်း ။                   |   1.16%
  Top- 5: Class  114 | မိုးလေဝသ သတင်း များ ကို စဉ်ဆက်မပြတ် နားထောင် ပါ ။ |   1.00%

============================================================
[2026-06-04 10:17:25]  webcam #2
  Top- 1: Class  114 | မိုးလေဝသ သတင်း များ ကို စဉ်ဆက်မပြတ် နားထောင် ပါ ။ |   8.82%
  Top- 2: Class  484 | မနက် ၉ နာရီ က နေ ည ၁၀ နာရီ ထိ ပါ ။            |   3.19%
  Top- 3: Class  485 | ည ၁၀ နာရီ နောက် ပိုင်း အမျိုးသား နေ ခွင့် မ ရှိ ပါ ဘူး ။ |   1.82%
  Top- 4: Class  490 | မနက်ဖြန် ဆေးခန်း လာ ဖို့ လို သေး လား ။        |   1.66%
  Top- 5: Class  498 | ပလာစတာ တစ် ခု လောက် ရ မ လား ။                 |   1.30%

============================================================
[2026-06-04 10:17:30]  webcam #3
  Top- 1: Class  114 | မိုးလေဝသ သတင်း များ ကို စဉ်ဆက်မပြတ် နားထောင် ပါ ။ |  11.36%
  Top- 2: Class  485 | ည ၁၀ နာရီ နောက် ပိုင်း အမျိုးသား နေ ခွင့် မ ရှိ ပါ ဘူး ။ |   4.55%
  Top- 3: Class  464 | ဆေးရုံ ဘယ်လောက် ကြာ အောင် တက် ရ မှာ လဲ ။      |   1.21%
  Top- 4: Class  484 | မနက် ၉ နာရီ က နေ ည ၁၀ နာရီ ထိ ပါ ။            |   1.12%
  Top- 5: Class  516 | ခင်ဗျား မိသားစု ကို ဆက်သွယ် ဖို့ လို တယ် ။    |   1.00%

============================================================
[2026-06-04 10:17:35]  webcam #4
  Top- 1: Class  485 | ည ၁၀ နာရီ နောက် ပိုင်း အမျိုးသား နေ ခွင့် မ ရှိ ပါ ဘူး ။ |   4.21%
  Top- 2: Class  515 | သူ့ မိသားစု ကို ငါ တို့ ဖုန်းဆက် ရ မယ် ။      |   3.57%
  Top- 3: Class  516 | ခင်ဗျား မိသားစု ကို ဆက်သွယ် ဖို့ လို တယ် ။    |   1.37%
  Top- 4: Class  114 | မိုးလေဝသ သတင်း များ ကို စဉ်ဆက်မပြတ် နားထောင် ပါ ။ |   1.34%
  Top- 5: Class  298 | ဖန်ကွဲစ ရှ မိ ပြီး သွေး တွေ တအား ထွက် နေ တယ် ။ |   1.02%

============================================================
[2026-06-04 10:17:41]  webcam #5
  Top- 1: Class  484 | မနက် ၉ နာရီ က နေ ည ၁၀ နာရီ ထိ ပါ ။            |   2.34%
  Top- 2: Class  485 | ည ၁၀ နာရီ နောက် ပိုင်း အမျိုးသား နေ ခွင့် မ ရှိ ပါ ဘူး ။ |   1.66%
  Top- 3: Class  515 | သူ့ မိသားစု ကို ငါ တို့ ဖုန်းဆက် ရ မယ် ။      |   1.42%
  Top- 4: Class  516 | ခင်ဗျား မိသားစု ကို ဆက်သွယ် ဖို့ လို တယ် ။    |   1.19%
  Top- 5: Class  498 | ပလာစတာ တစ် ခု လောက် ရ မ လား ။                 |   1.18%

============================================================
[2026-06-04 10:17:46]  webcam #6
  Top- 1: Class  485 | ည ၁၀ နာရီ နောက် ပိုင်း အမျိုးသား နေ ခွင့် မ ရှိ ပါ ဘူး ။ |   2.16%
  Top- 2: Class   67 | မုန်တိုင်း ။                                  |   1.72%
  Top- 3: Class  515 | သူ့ မိသားစု ကို ငါ တို့ ဖုန်းဆက် ရ မယ် ။      |   0.92%
  Top- 4: Class  484 | မနက် ၉ နာရီ က နေ ည ၁၀ နာရီ ထိ ပါ ။            |   0.81%
  Top- 5: Class  298 | ဖန်ကွဲစ ရှ မိ ပြီး သွေး တွေ တအား ထွက် နေ တယ် ။ |   0.75%

```

Online-testing က မရသေးဘူး။ :(  


## Commands of Current infer_onnx.py Code  

# Video with debug output and log
```
python infer_onnx.py --onnx_model onnx_models\bilstm_msl.onnx \
    --video sample4inference\idx20-30.mp4 \
    --top_k 5 --output_dir predictions_output --log_file results.txt
```

# Webcam with 3-second signing window, saves recordings + log
```
python infer_onnx.py --onnx_model onnx_models\transformer_msl.onnx \
    --webcam 0 --buffer_sec 3.0 --top_k 5 \
    --output_dir predictions_output --log_file webcam_log.txt
```

## Debugging with Claude  

Let me watch both videos carefully before doing anything.Now apply all four changes surgically to the current output file:Now update the webcam extractor creation to use complexity=2 and lower confidence, and the video extractor to match training exactly:Now add a live pose detection status indicator to the webcam display:Also print detection rate summary at the end of webcam and log a warning if it's low:It's there — the check failed because the f-string has quotes inside quotes. All good.## Root cause found — confirmed by the debug videos

The answer came directly from comparing the two debug videos frame-by-frame.

### What the video analysis showed

| | `idx20-30_debug.mp4` (video file) | `webcam_20260604_101715_debug.mp4` |
|---|---|---|
| Skeleton drawn | **100% of frames** (26/26 sampled) | **0% of frames** (0/115 sampled) |
| Colored pixels per frame | ~780,000 | ~30,000 |
| Pose complexity used | **2** (heavy model) | **1** (lite model) |
| Max confidence | 80.73% | 11.36% |

Zero skeleton pixels in 575 webcam frames while the person fills 95% of the frame height means **MediaPipe Pose completely failed to detect the person** in every single frame. The model was running but returning `pose_landmarks = None` every time, so `draw_landmarks()` drew nothing, and `process_frame()` returned all-zeros keypoints for every frame. The sign language model then received 75 frames of pure zeros and guessed randomly at 1–11% confidence.

### Why complexity=1 failed but complexity=2 worked

The log from the first video run showed `Downloading model to ...pose_landmark_heavy.tflite` — that is the complexity=2 heavy model downloading. Once downloaded, it detected pose landmarks in 100% of frames on the video file.

Webcam used complexity=1 (the lite model) by design — originally a speed tradeoff from `infer.py`. The lite model is substantially less robust: lower quality webcam sensors, variable indoor lighting, and the fact that the signer is a live person rather than a studio-recorded sign language video all push the input below its detection threshold.

### Changes made

**1. `webcam_complexity = 2`** — the most important fix. Both video and webcam now use the heavy model, which is the one that actually detects pose landmarks reliably.

**2. `min_detection_confidence=0.3` for webcam** (was 0.5) — live webcam conditions are inherently harder than pre-recorded video; a lower threshold catches poses the stricter setting would miss.

**3. Live "Pose: OK / NOT DETECTED" status** drawn on the debug frame in green/red — you can now see in real time exactly whether MediaPipe is tracking you, making future debugging immediate.

**4. "Detect rate: X%" overlay** — running percentage of frames where pose was detected, so you can instantly see if you need to move closer, change the lighting, or adjust your position.

**5. End-of-session summary** printed to console with total frames, detection rate, and a warning with tips if detection was below 30%.

## Updated infer_onnx.py Code

နောက်ပိုင်း ပြန်လေ့လာလို့ ရအောင်လို့ ...  

In [101]:
from IPython.display import Markdown

file_path = './src/infer_onnx.py'

with open(file_path, 'r') as f:
    code = f.read()

# 'python' keyword triggers the syntax highlighter
display(Markdown(f"```python\n{code}\n```"))

```python
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
infer_onnx.py  –  Real-time MSL recognition on Windows using ONNX Runtime

Bugs fixed in this version
───────────────────────────
  [INFER-A] process_frame skipped frames without pose detection.
      Training always includes every frame (zeros for undetected joints).
      Fix: process_frame always returns the keypoint array, never None.

  [INFER-B] normalize_sequence used 2-D scale and left z unscaled.
      Fix: exact mirror of extract_keypoints.normalize_sequence() —
      full 3-D midpoint and 3-D Euclidean shoulder width, all axes.

  [INFER-C] *** ROOT CAUSE OF WEBCAM FAILURE ***
      Webcam classified every classify_every=15 frames = 0.5 seconds.
      Signs take 1–4 seconds; feeding 15 frames to a model trained on
      full-length signs produces garbage predictions.

      infer.py uses a TIME-BASED buffer: accumulate buffer_sec=2.5 sec
      of frames (~75 at 30 fps), then classify and clear.  This gives
      the model a realistic-duration sign sequence matching training.

      Fix: replaced the frame-count trigger with a time-based trigger
      using --buffer_sec (default 2.5), identical to infer.py.

  [INFER-D] np.int64 / int dict key mismatch.
      Fix: always cast to int() before idx2label lookup.

  [INFER-E] ONNXModel.predict returned a scalar instead of 1-D array.
      Fix: return output[0] — shape (num_classes,).

  [INFER-F] model_complexity=1 everywhere; training used complexity=2.
      Fix: complexity=2 for video (matching training), 1 for webcam
      (speed tradeoff, same as infer.py).

  [INFER-G] extractor.pose.model_complexity AttributeError on Windows.
      MediaPipe's Pose object does not expose model_complexity publicly.
      Fix: stored as self.model_complexity on KeypointExtractor.

New features
────────────
  • Video saving — two output files per inference:
      {name}_clean.mp4  —  label/confidence overlay only
      {name}_debug.mp4  —  label overlay + MediaPipe skeleton lines
    Works for both --video and --webcam modes.
    Saved to --output_dir (default: predictions_output/).

  • Webcam log — every classification event writes top-k results to the
    log file with timestamp and separator, matching the video log format.
"""

import argparse
import importlib
import json
import sys
from datetime import datetime
from pathlib import Path

import cv2
import numpy as np
import onnxruntime as ort

try:
    from PIL import Image, ImageDraw, ImageFont
    HAS_PIL = True
except ImportError:
    HAS_PIL = False

# ── Constants (must match extract_keypoints.py exactly) ───────────────────────
POSE_N          = 33
HAND_N          = 21
TOTAL_JOINTS    = POSE_N + HAND_N * 2   # 75
COORDS          = 3
FEAT_DIM        = TOTAL_JOINTS * COORDS  # 225

POSE_LEFT_SHOULDER  = 11
POSE_RIGHT_SHOULDER = 12
LEFT_HAND_START     = POSE_N            # 33
RIGHT_HAND_START    = POSE_N + HAND_N   # 54


# ─── MediaPipe import ─────────────────────────────────────────────────────────

def _import_mp():
    for prefix in ["mediapipe.solutions", "mediapipe.python.solutions"]:
        try:
            pose  = importlib.import_module(f"{prefix}.pose")
            hands = importlib.import_module(f"{prefix}.hands")
            draw  = importlib.import_module(f"{prefix}.drawing_utils")
            return pose, hands, draw
        except (ImportError, ModuleNotFoundError):
            continue
    try:
        import mediapipe as _mp
        return _mp.solutions.pose, _mp.solutions.hands, _mp.solutions.drawing_utils
    except AttributeError:
        pass
    raise ImportError(
        "Cannot import MediaPipe solutions.\n"
        "Try: pip uninstall mediapipe -y && pip install mediapipe==0.10.18"
    )

mp_pose, mp_hands, mp_drawing = _import_mp()


# ─── Myanmar text rendering ──────────────────────────────────────────────────

class TextRenderer:
    _FONT_CANDIDATES = [
        "C:\\Windows\\Fonts\\Padauk.ttf",
        "C:\\Windows\\Fonts\\NotoSansMyanmar-Regular.ttf",
        "C:\\Windows\\Fonts\\NotoSansMyanmar.ttf",
        "C:\\Windows\\Fonts\\myanmar3.ttf",
        "/usr/share/fonts/truetype/padauk/Padauk-Regular.ttf",
        "/usr/share/fonts/truetype/noto/NotoSansMyanmar-Regular.ttf",
        "fonts\\Padauk.ttf",
        "fonts/NotoSansMyanmar-Regular.ttf",
    ]

    def __init__(self, font_size: int = 32):
        self.font = self._load_font(font_size)
        self.can_render_myanmar = (self.font is not None)
        if not self.can_render_myanmar:
            print()
            print("  ╔═══════════════════════════════════════════════════════════╗")
            print("  ║  INFO: No Myanmar shaping font found.                    ║")
            print("  ║  Displaying readable Class IDs instead of Myanmar text.  ║")
            print("  ╚═══════════════════════════════════════════════════════════╝")
            print()

    def _load_font(self, size):
        if not HAS_PIL:
            return None
        for path in self._FONT_CANDIDATES:
            if Path(path).exists():
                try:
                    return ImageFont.truetype(path, size)
                except Exception:
                    continue
        return None

    def put_text(self, frame, text, position=(20, 20), color=(0, 255, 0)):
        if self.font and HAS_PIL:
            pil = Image.fromarray(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
            d   = ImageDraw.Draw(pil)
            r, g, b = color[2], color[1], color[0]
            for dx in [-1, 0, 1]:
                for dy in [-1, 0, 1]:
                    if dx or dy:
                        d.text((position[0]+dx, position[1]+dy), text,
                               font=self.font, fill=(0, 0, 0))
            d.text(position, text, font=self.font, fill=(r, g, b))
            return cv2.cvtColor(np.array(pil), cv2.COLOR_RGB2BGR)
        cv2.putText(frame, text, position, cv2.FONT_HERSHEY_SIMPLEX,
                    0.75, color, 2, cv2.LINE_AA)
        return frame


# ─── Keypoint extraction ──────────────────────────────────────────────────────

class KeypointExtractor:
    """
    Extracts (TOTAL_JOINTS, 3) keypoint arrays from BGR frames.

    Fix [INFER-G]: model_complexity is stored as self.model_complexity
    because mediapipe's Pose object does not expose it as a public attribute
    on all Windows builds (AttributeError otherwise).

    Fix [INFER-A]: process_frame always returns an array, never None.
    Matches extract_keypoints.py which includes every frame, using zeros
    for any joint that MediaPipe failed to detect.

    Hand convention (matches extract_keypoints.py):
      MediaPipe reports handedness from the camera perspective (mirrored).
      label='Right' → signer's LEFT  hand → slot LEFT_HAND_START  (33).
      label='Left'  → signer's RIGHT hand → slot RIGHT_HAND_START (54).
    """

    def __init__(self, model_complexity: int = 2,
                 min_detection_confidence: float = 0.5,
                 min_tracking_confidence:  float = 0.5):
        # Fix [INFER-G]: store complexity ourselves — mediapipe's Pose object
        # does NOT expose a public .model_complexity on all Windows builds.
        self.model_complexity = model_complexity

        self.pose  = mp_pose.Pose(
            static_image_mode=False,
            model_complexity=model_complexity,
            smooth_landmarks=True,
            enable_segmentation=False,
            min_detection_confidence=min_detection_confidence,
            min_tracking_confidence=min_tracking_confidence,
        )
        self.hands = mp_hands.Hands(
            static_image_mode=False,
            max_num_hands=2,
            model_complexity=min(model_complexity, 1),  # Hands caps at 1
            min_detection_confidence=min_detection_confidence,
            min_tracking_confidence=min_tracking_confidence,
        )
        self._last_pose  = None
        self._last_hands = None

    def process_frame(self, frame: np.ndarray) -> np.ndarray:
        """
        Returns (TOTAL_JOINTS, 3) float32 for every frame.
        Zeros for any undetected joint/hand — never returns None.
        """
        rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        rgb.flags.writeable = False
        p_res = self.pose.process(rgb)
        h_res = self.hands.process(rgb)
        self._last_pose  = p_res
        self._last_hands = h_res

        kp = np.zeros((TOTAL_JOINTS, COORDS), dtype=np.float32)

        if p_res.pose_landmarks:
            for i, lm in enumerate(p_res.pose_landmarks.landmark):
                kp[i] = [lm.x, lm.y, lm.z]

        if h_res.multi_hand_landmarks and h_res.multi_handedness:
            for lm_list, handedness in zip(
                h_res.multi_hand_landmarks, h_res.multi_handedness
            ):
                label  = handedness.classification[0].label
                offset = LEFT_HAND_START if label == 'Right' else RIGHT_HAND_START
                for i, lm in enumerate(lm_list.landmark):
                    kp[offset + i] = [lm.x, lm.y, lm.z]

        return kp  # always an array, never None

    def draw_landmarks(self, frame: np.ndarray) -> np.ndarray:
        """Draw MediaPipe skeleton onto frame IN-PLACE and return it."""
        if self._last_pose and self._last_pose.pose_landmarks:
            mp_drawing.draw_landmarks(
                frame, self._last_pose.pose_landmarks, mp_pose.POSE_CONNECTIONS
            )
        if self._last_hands and self._last_hands.multi_hand_landmarks:
            for lm in self._last_hands.multi_hand_landmarks:
                mp_drawing.draw_landmarks(frame, lm, mp_hands.HAND_CONNECTIONS)
        return frame

    def close(self):
        self.pose.close()
        self.hands.close()


# ─── Normalization ────────────────────────────────────────────────────────────

def normalize_sequence(seq: np.ndarray) -> np.ndarray:
    """
    Fix [INFER-B]: exact mirror of extract_keypoints.normalize_sequence().
    Full 3-D shoulder-midpoint subtraction and 3-D shoulder-width scaling,
    applied to all three coordinate axes.
    """
    l_sh = seq[:, POSE_LEFT_SHOULDER,  :]   # (T, 3)
    r_sh = seq[:, POSE_RIGHT_SHOULDER, :]   # (T, 3)
    mid  = (l_sh + r_sh) / 2.0
    dist = np.linalg.norm(r_sh - l_sh, axis=1, keepdims=True)
    dist = np.where(dist < 1e-6, 1.0, dist)
    return ((seq - mid[:, None, :]) / dist[:, None, :]).astype(np.float32)


# ─── Preprocessor ─────────────────────────────────────────────────────────────

class Preprocessor:
    """
    Stack raw (75, 3) keypoint frames → normalize → crop/pad → model input.

    Processing order matches infer.py exactly:
      1. Stack   → (T, 75, 3)
      2. Normalize (3-D shoulder-centred, full 3-D scale)
      3. Centre-crop  if T > max_seq_len
      4. Zero-pad     if T < max_seq_len
      5. Build mask   (True = padded position)
      6. Reshape for model type
    """

    def __init__(self, max_seq_len: int = 200):
        self.max_seq_len = max_seq_len

    def __call__(self, kp_list: list, model_type: str) -> dict:
        if not kp_list:
            return {}

        seq = np.stack(kp_list, axis=0).astype(np.float32)  # (T, 75, 3)
        seq = normalize_sequence(seq)
        T   = seq.shape[0]

        if T > self.max_seq_len:
            start = (T - self.max_seq_len) // 2
            seq   = seq[start: start + self.max_seq_len]
            T     = self.max_seq_len

        real_len = T

        if T < self.max_seq_len:
            pad = np.zeros((self.max_seq_len - T, TOTAL_JOINTS, COORDS), dtype=np.float32)
            seq = np.concatenate([seq, pad], axis=0)

        mask = np.zeros(self.max_seq_len, dtype=np.bool_)
        mask[real_len:] = True

        if model_type == "stgcn":
            kp_out = np.expand_dims(seq.transpose(2, 0, 1), axis=0).astype(np.float32)
            return {"keypoints": kp_out}
        else:
            kp_out = np.expand_dims(
                seq.reshape(self.max_seq_len, FEAT_DIM), axis=0
            ).astype(np.float32)
            return {"keypoints": kp_out, "mask": mask[np.newaxis]}


# ─── ONNX model wrapper ───────────────────────────────────────────────────────

class ONNXModel:
    def __init__(self, onnx_path: str, meta_path: str = None):
        available = ort.get_available_providers()
        preferred = ["CUDAExecutionProvider", "DmlExecutionProvider", "CPUExecutionProvider"]
        providers = [p for p in preferred if p in available]

        self.session = ort.InferenceSession(onnx_path, providers=providers)
        print(f"[INFO] ONNX loaded: {Path(onnx_path).name}"
              f"  (provider: {self.session.get_providers()[0]})")

        self.meta = {}
        if meta_path and Path(meta_path).exists():
            try:
                with open(meta_path, encoding="utf-8") as f:
                    self.meta = json.load(f)
            except Exception as e:
                print(f"[WARNING] Could not read metadata: {e}")
        else:
            print(f"[INFO] Metadata not found at '{meta_path}', using defaults.")

        self.model_type  = self.meta.get("model_type",  "transformer")
        self.max_seq_len = self.meta.get("max_seq_len", 200)
        self.num_classes = self.meta.get("num_classes", 558)

    def predict(self, inputs: dict) -> np.ndarray:
        """Fix [INFER-E]: return (num_classes,) not scalar."""
        return self.session.run(None, inputs)[0][0]

    @staticmethod
    def softmax(logits: np.ndarray) -> np.ndarray:
        e = np.exp(logits - logits.max())
        return e / e.sum()


# ─── Label map ────────────────────────────────────────────────────────────────

def load_idx2label(path: str) -> dict:
    with open(path, encoding="utf-8") as f:
        data = json.load(f)
    return {int(k): v for k, v in data.get("idx2label", {}).items()}


def _get_label(idx2label: dict, class_id) -> str:
    """Fix [INFER-D]: always cast to int() before dict lookup."""
    return idx2label.get(int(class_id), f"[class {int(class_id)}]")


# ─── Helpers ──────────────────────────────────────────────────────────────────

def _overlay_text(renderer, frame, text, color):
    """Put prediction text at top-left of frame."""
    return renderer.put_text(frame, text, position=(10, 10), color=color)


def _pred_color(confidence: float) -> tuple:
    """Green ≥80%, yellow ≥50%, red otherwise."""
    if confidence >= 0.80: return (0, 200,   0)
    if confidence >= 0.50: return (0, 200, 200)
    return (0, 0, 255)


def _build_overlay_text(top_idx, probs, idx2label, renderer, top_k):
    """Single-line text for the video overlay (top-1 label + confidence)."""
    best = top_idx[0]
    conf = probs[int(best)]
    lbl  = _get_label(idx2label, best) if idx2label else f"Class {int(best)}"
    if renderer.can_render_myanmar:
        return f"{lbl}  ({conf*100:.1f}%)", conf
    return f"Class {int(best)}  ({conf*100:.1f}%)", conf


def _build_console_block(top_idx, probs, idx2label, model_type, source_name, top_k):
    """Multi-line console output block, same format as run_video."""
    sep   = "─" * 60
    lines = [f"Source : {source_name}", f"Model  : {model_type.upper()}", sep]
    for rank, idx in enumerate(top_idx, 1):
        lbl = _get_label(idx2label, idx) if idx2label else f"[class {int(idx)}]"
        lines.append(f"  Top-{rank}: {lbl}  ({probs[int(idx)]*100:.2f}%)")
    lines.append(sep)
    return "\n".join(lines)


def _build_log_block(top_idx, probs, idx2label, source_name, top_k):
    """Log block with timestamp and top-k results."""
    ts    = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    sep   = "=" * 60
    lines = [sep, f"[{ts}]  {source_name}"]
    for rank, idx in enumerate(top_idx, 1):
        lbl = _get_label(idx2label, idx) if idx2label else "?"
        lbl = lbl.encode("utf-8", errors="replace").decode("utf-8")
        lines.append(f"  Top-{rank:>2}: Class {int(idx):>4d} | "
                     f"{lbl:<45s} | {probs[int(idx)]*100:6.2f}%")
    lines.append("")
    return "\n".join(lines) + "\n"


def _append_log(path, text):
    try:
        with open(path, "a", encoding="utf-8-sig") as f:
            f.write(text)
    except Exception as e:
        print(f"[WARNING] Could not write log: {e}")


def _make_video_writer(output_path: Path, fps: float, width: int, height: int):
    """Create a cv2.VideoWriter with MP4V codec."""
    fourcc = cv2.VideoWriter_fourcc(*"mp4v")
    return cv2.VideoWriter(str(output_path), fourcc, fps, (width, height))


# ─── Video processing ─────────────────────────────────────────────────────────

def run_video(model, prep, extractor, renderer, video_path,
              top_k=5, log_file=None, idx2label=None,
              output_dir=None, debug=False):
    """
    Process a single video file.  Saves two output videos:
      {stem}_clean.mp4  —  label overlay only
      {stem}_debug.mp4  —  label overlay + MediaPipe skeleton lines

    Matches infer.py / process_video() exactly:
      every frame included (zeros for undetected joints),
      normalize after stacking, centre-crop or zero-pad.
    """
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        print(f"[ERROR] Cannot open video: {video_path}")
        return

    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    fps_src      = cap.get(cv2.CAP_PROP_FPS) or 30.0
    w            = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    h            = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    stem         = Path(video_path).stem

    print(f"[INFO] {Path(video_path).name} — {total_frames} frames @ {fps_src:.1f} fps")
    print(f"[INFO] MediaPipe pose complexity: {extractor.model_complexity}"
          f"  (must match training; training default=2)")

    # --- Pass 1: extract all keypoints + collect frames for output video ---
    kp_list    = []
    raw_frames = []   # store for writing output videos after prediction
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        kp_list.append(extractor.process_frame(frame))  # always appends
        raw_frames.append(frame.copy())
    cap.release()

    if not kp_list:
        print("[WARNING] No frames could be read from the video.")
        return

    T_raw = len(kp_list)
    T_use = min(T_raw, model.max_seq_len)
    print(f"[INFO] {T_raw} frames extracted → using {T_use} "
          f"({'centre-cropped' if T_raw > model.max_seq_len else 'zero-padded to'}"
          f" {model.max_seq_len})")

    # --- Inference ---
    inputs  = prep(kp_list, model.model_type)
    logits  = model.predict(inputs)
    probs   = ONNXModel.softmax(logits)
    top_idx = np.argsort(probs)[::-1][:top_k]

    # --- Console output ---
    print("\n" + _build_console_block(top_idx, probs, idx2label,
                                      model.model_type, Path(video_path).name, top_k))

    # --- Log file ---
    if log_file:
        _append_log(log_file,
                    _build_log_block(top_idx, probs, idx2label,
                                     Path(video_path).name, top_k))

    # --- Save output videos ---
    if output_dir:
        out_dir = Path(output_dir)
        out_dir.mkdir(parents=True, exist_ok=True)

        overlay_text, conf = _build_overlay_text(top_idx, probs, idx2label, renderer, top_k)
        color              = _pred_color(conf)

        clean_path = out_dir / f"{stem}_clean.mp4"
        debug_path = out_dir / f"{stem}_debug.mp4"
        vw_clean   = _make_video_writer(clean_path, fps_src, w, h)
        vw_debug   = _make_video_writer(debug_path, fps_src, w, h)

        for i, raw in enumerate(raw_frames):
            # Clean: label overlay only
            clean_frame = renderer.put_text(raw.copy(), overlay_text,
                                            position=(10, 10), color=color)
            vw_clean.write(clean_frame)

            # Debug: re-run MediaPipe on this frame to get landmarks,
            # then draw skeleton + label
            debug_frame = raw.copy()
            extractor.process_frame(debug_frame)       # updates _last_pose/_last_hands
            extractor.draw_landmarks(debug_frame)       # draws skeleton in-place
            debug_frame = renderer.put_text(debug_frame, overlay_text,
                                            position=(10, 10), color=color)
            vw_debug.write(debug_frame)

        vw_clean.release()
        vw_debug.release()
        print(f"[INFO] Saved  clean  → {clean_path}")
        print(f"[INFO] Saved  debug  → {debug_path}")


# ─── Webcam processing ────────────────────────────────────────────────────────

def run_webcam(model, prep, extractor, renderer, idx2label,
               cam_idx=0, top_k=5, log_file=None,
               buffer_sec=2.5, output_dir=None):
    """
    Live webcam sign recognition.

    Fix [INFER-C] — root cause of webcam failure:
      The old code classified every classify_every=15 frames ≈ 0.5 sec.
      Signs take 1–4 seconds; 15 frames is far too short and produces
      garbage predictions regardless of model quality.

      Now uses a TIME-BASED buffer matching infer.py exactly:
        buf_frames = int(fps * buffer_sec)   (default 2.5 sec ≈ 75 frames)
        Classify and clear when the buffer reaches buf_frames.
        Progress bar fills towards buf_frames, giving clear visual feedback.

    Saves two output videos (if --output_dir given):
      webcam_{timestamp}_clean.mp4  — label overlay only
      webcam_{timestamp}_debug.mp4  — label overlay + skeleton

    Every classification event is logged to --log_file with top-k results.

    Keys:  Q = quit  |  C = clear buffer manually
    """
    cap = cv2.VideoCapture(cam_idx)
    if not cap.isOpened():
        print(f"[ERROR] Cannot open webcam {cam_idx}")
        return

    fps        = cap.get(cv2.CAP_PROP_FPS) or 30.0
    w          = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    h          = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    # Fix [INFER-C]: time-based buffer, matching infer.py
    buf_frames = max(20, int(fps * buffer_sec))

    print(f"[INFO] Webcam  {w}×{h} @ {fps:.1f} fps")
    print(f"[INFO] Buffer  {buf_frames} frames ({buffer_sec:.1f} sec) — sign within this window")
    print(f"[INFO] MediaPipe pose complexity: {extractor.model_complexity}")
    print(f"[INFO] Q = quit | C = clear buffer manually")

    # ── Optional video writers ──
    ts_str     = datetime.now().strftime("%Y%m%d_%H%M%S")
    vw_clean   = vw_debug = None
    if output_dir:
        out_dir = Path(output_dir)
        out_dir.mkdir(parents=True, exist_ok=True)
        clean_path = out_dir / f"webcam_{ts_str}_clean.mp4"
        debug_path = out_dir / f"webcam_{ts_str}_debug.mp4"
        vw_clean   = _make_video_writer(clean_path, fps, w, h)
        vw_debug   = _make_video_writer(debug_path, fps, w, h)
        print(f"[INFO] Recording clean → {clean_path}")
        print(f"[INFO] Recording debug → {debug_path}")

    kp_buffer  = []
    last_text  = "Perform a sign — buffer filling..."
    last_color = (200, 200, 200)
    classify_count = 0
    total_frames_seen  = 0
    pose_detected_count = 0

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        # Extract keypoints — always append (Fix [INFER-A])
        kp = extractor.process_frame(frame)
        kp_buffer.append(kp)
        total_frames_seen += 1

        # Track whether pose was detected this frame (for status display)
        pose_this_frame = (
            extractor._last_pose is not None and
            extractor._last_pose.pose_landmarks is not None
        )
        if pose_this_frame:
            pose_detected_count += 1
        detect_rate = pose_detected_count / total_frames_seen * 100

        # Fix [INFER-C]: classify when buffer reaches buf_frames, then clear
        if len(kp_buffer) >= buf_frames:
            inputs  = prep(list(kp_buffer), model.model_type)
            logits  = model.predict(inputs)
            probs   = ONNXModel.softmax(logits)
            top_idx = np.argsort(probs)[::-1][:top_k]

            overlay_text, conf = _build_overlay_text(
                top_idx, probs, idx2label, renderer, top_k
            )
            last_text  = overlay_text
            last_color = _pred_color(conf)
            classify_count += 1

            # Console output
            print(f"\n[Prediction #{classify_count}]")
            print(_build_console_block(top_idx, probs, idx2label,
                                       model.model_type, f"webcam frame", top_k))

            # Log file — top_k results
            if log_file:
                _append_log(log_file,
                            _build_log_block(top_idx, probs, idx2label,
                                             f"webcam #{classify_count}", top_k))

            kp_buffer.clear()   # start fresh for next sign

        # ── Build display frames ──
        # Progress bar towards buf_frames (not max_seq_len)
        fill   = int(len(kp_buffer) / buf_frames * w)
        pct    = len(kp_buffer) / buf_frames * 100

        # Pose detection status: green=detected, red=not detected
        pose_status_text  = f"Pose: {'OK' if pose_this_frame else 'NOT DETECTED'}"
        pose_status_color = (0, 200, 0) if pose_this_frame else (0, 0, 255)
        detect_rate_text  = f"Detect rate: {detect_rate:.0f}%"

        # Clean frame: label + progress bar + pose status
        clean_frame = renderer.put_text(frame.copy(), last_text,
                                        position=(10, 10), color=last_color)
        cv2.putText(clean_frame, pose_status_text,
                    (10, 55), cv2.FONT_HERSHEY_SIMPLEX, 0.6, pose_status_color, 2)
        cv2.putText(clean_frame, detect_rate_text,
                    (10, 80), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (180, 180, 180), 1)
        cv2.rectangle(clean_frame, (0, h - 12), (fill, h), (0, 180, 255), -1)
        cv2.putText(clean_frame, f"buf {len(kp_buffer)}/{buf_frames}  ({pct:.0f}%)",
                    (10, h - 15), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (220, 220, 220), 1)

        # Debug frame: skeleton + label + pose status + progress bar
        debug_frame = frame.copy()
        extractor.draw_landmarks(debug_frame)
        debug_frame = renderer.put_text(debug_frame, last_text,
                                        position=(10, 10), color=last_color)
        cv2.putText(debug_frame, pose_status_text,
                    (10, 55), cv2.FONT_HERSHEY_SIMPLEX, 0.6, pose_status_color, 2)
        cv2.putText(debug_frame, detect_rate_text,
                    (10, 80), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (180, 180, 180), 1)
        cv2.rectangle(debug_frame, (0, h - 12), (fill, h), (0, 180, 255), -1)
        cv2.putText(debug_frame, f"buf {len(kp_buffer)}/{buf_frames}  ({pct:.0f}%)",
                    (10, h - 15), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (220, 220, 220), 1)

        # Write to video files
        if vw_clean: vw_clean.write(clean_frame)
        if vw_debug: vw_debug.write(debug_frame)

        # Show the debug view on screen (most useful for real-time feedback)
        cv2.imshow("MSL Recognition  [Q=quit  C=clear]", debug_frame)

        key = cv2.waitKey(1) & 0xFF
        if key == ord("q"):
            break
        elif key == ord("c"):
            kp_buffer.clear()
            last_text  = "Buffer cleared — perform a sign"
            last_color = (200, 200, 200)
            print("[INFO] Buffer manually cleared.")

    cap.release()
    if vw_clean: vw_clean.release()
    if vw_debug: vw_debug.release()
    cv2.destroyAllWindows()

    print(f"\n[INFO] Webcam session summary:")
    print(f"  Total frames     : {total_frames_seen}")
    print(f"  Pose detected    : {pose_detected_count}  ({pose_detected_count/max(total_frames_seen,1)*100:.0f}%)")
    print(f"  Classifications  : {classify_count}")
    if total_frames_seen > 0 and pose_detected_count / total_frames_seen < 0.3:
        print(f"  [WARNING] Low detection rate ({pose_detected_count/total_frames_seen*100:.0f}%).")
        print(f"            Tips: move closer, improve lighting, ensure upper body is visible.")
    if output_dir:
        print(f"  Saved  clean  → {clean_path}")
        print(f"  Saved  debug  → {debug_path}")


# ─── CLI ──────────────────────────────────────────────────────────────────────

def main():
    parser = argparse.ArgumentParser(
        description="MSL Sign Language Inference — ONNX Runtime",
        formatter_class=argparse.RawDescriptionHelpFormatter,
        epilog="""
Examples:
  # Single video, save both clean and debug outputs
  python infer_onnx.py --onnx_model onnx_models/bilstm_msl.onnx \\
      --video sample.mp4 --top_k 5 --output_dir predictions_output

  # Webcam, 3-second buffer, save recordings + log
  python infer_onnx.py --onnx_model onnx_models/transformer_msl.onnx \\
      --webcam 0 --buffer_sec 3.0 --top_k 5 \\
      --output_dir predictions_output --log_file webcam_log.txt
""")

    parser.add_argument("--onnx_model",   required=True,
                        help="Path to .onnx model file (NOT the .json metadata)")
    parser.add_argument("--metadata",     default=None,
                        help="Path to .json metadata (default: same stem as .onnx)")
    parser.add_argument("--label_map",    default=None,
                        help="Path to label_map.json (default: next to .onnx)")
    parser.add_argument("--output_dir",   default="predictions_output",
                        help="Directory for saved videos (default: predictions_output/)")
    parser.add_argument("--log_file",     default=None,
                        help="Append top-k results to this text file")
    parser.add_argument("--top_k",        type=int, default=5,
                        help="Number of top predictions to show/log (default: 5)")
    parser.add_argument("--font_size",    type=int, default=32,
                        help="Myanmar font size for overlay (default: 32)")
    parser.add_argument("--model_complexity", type=int, default=None,
                        help="MediaPipe pose complexity: 0/1/2. "
                             "Default: 2 for video (matches training), 1 for webcam.")

    grp = parser.add_mutually_exclusive_group(required=True)
    grp.add_argument("--video",  help="Path to a video file")
    grp.add_argument("--webcam", type=int, metavar="CAM_IDX",
                     help="Webcam device index (0 = default camera)")

    parser.add_argument("--buffer_sec", type=float, default=2.5,
                        help="[Webcam] Seconds of frames to accumulate before "
                             "each classification (default: 2.5, same as infer.py)")
    parser.add_argument("--debug",       action="store_true",
                        help="Print extra diagnostic information")
    args = parser.parse_args()

    # ── Validate ONNX path ────────────────────────────────────────────────────
    model_p = Path(args.onnx_model)
    if model_p.suffix.lower() == ".json":
        sys.exit(
            f"[ERROR] --onnx_model must be a .onnx file, not a .json file.\n"
            f"        You passed : {model_p}\n"
            f"        Did you mean: {model_p.with_suffix('.onnx')}"
        )
    if not model_p.exists():
        sys.exit(f"[ERROR] ONNX model not found: {model_p}")

    meta_p = args.metadata or str(model_p.with_suffix(".json"))
    lmap_p = args.label_map or str(model_p.parent / "label_map.json")

    if not Path(lmap_p).exists():
        sys.exit(f"[ERROR] label_map.json not found: {lmap_p}\n"
                 f"        Copy it from your Linux server alongside the .onnx file.")

    # ── Load resources ────────────────────────────────────────────────────────
    idx2label = load_idx2label(lmap_p)
    print(f"[INFO] Loaded {len(idx2label)} labels from label map.")

    model    = ONNXModel(str(model_p), meta_p)
    prep     = Preprocessor(max_seq_len=model.max_seq_len)
    renderer = TextRenderer(font_size=args.font_size)

    if args.log_file:
        print(f"[INFO] Logging top-{args.top_k} results to: {args.log_file}")
    if args.output_dir:
        print(f"[INFO] Saving output videos to: {args.output_dir}/")

    # Fix [INFER-F] + webcam detection fix:
    # complexity=2 (heavy model) for BOTH video and webcam.
    # Webcam needs the heavy model — the lite model (complexity=1) failed to detect
    # any pose landmarks in real webcam conditions (0% detection rate confirmed
    # by comparing debug videos: skeleton drawn 100% on video file, 0% on webcam).
    # The heavy model is more robust to webcam image quality, lighting variation,
    # and different distances. Slower, but actually detects the signer.
    #
    # min_detection_confidence lowered to 0.3 for webcam: live conditions are
    # harder than pre-recorded videos, so a more permissive threshold is needed.
    # Video keeps 0.5 to match training (extract_keypoints.py default).
    if args.model_complexity is not None:
        video_complexity  = args.model_complexity
        webcam_complexity = args.model_complexity
    else:
        video_complexity  = 2   # matches training (extract_keypoints.py --complexity 2)
        webcam_complexity = 2   # FIXED: was 1 (lite), now 2 (heavy) — see note above

    try:
        if args.video:
            extractor = KeypointExtractor(
                model_complexity=video_complexity,
                min_detection_confidence=0.5,   # matches training
                min_tracking_confidence=0.5,
            )
            try:
                run_video(model, prep, extractor, renderer, args.video,
                          top_k      = args.top_k,
                          log_file   = args.log_file,
                          idx2label  = idx2label,
                          output_dir = args.output_dir,
                          debug      = args.debug)
            finally:
                extractor.close()
        else:
            extractor = KeypointExtractor(
                model_complexity=webcam_complexity,
                min_detection_confidence=0.3,   # lower threshold for live webcam conditions
                min_tracking_confidence=0.3,
            )
            try:
                run_webcam(model, prep, extractor, renderer, idx2label,
                           cam_idx    = args.webcam,
                           top_k      = args.top_k,
                           log_file   = args.log_file,
                           buffer_sec = args.buffer_sec,
                           output_dir = args.output_dir)
            finally:
                extractor.close()
    finally:
        print("[INFO] Done.")


if __name__ == "__main__":
    main()


```

## Real-Time MSL Recognition with Updated Code

Updated လုပ်ထားတဲ့ code နဲ့ real-time MSL recognition လုပ်ကြည့်ခဲ့...  

## BiLSTM Online Result

```
============================================================
[2026-06-04 16:13:29]  webcam #1
  Top- 1: Class  141 | ကွင်းပြင် ရှိ တဲ့ နေရာ ကို ပြေး ။             |   3.92%
  Top- 2: Class  214 | အဆောက်အဦး မီးလောင် တာ များ လား ။              |   1.00%
  Top- 3: Class  433 | ဆေးရုံ လိပ်စာ လေး သိ ချင် လို့ ပါ ။           |   0.96%
  Top- 4: Class   69 | လုံခြုံ သော နေရာ ။                            |   0.92%
  Top- 5: Class  256 | လူ တစ်ယောက် ရေ နစ် သွား လို့ ရှာ ဖို့ အကူညီ လွတ် ပေး ပါ ။ |   0.92%

============================================================
[2026-06-04 16:13:37]  webcam #2
  Top- 1: Class  274 | စောင် တွေ ယူလာ ပေး ပါ ။                       |   3.22%
  Top- 2: Class  141 | ကွင်းပြင် ရှိ တဲ့ နေရာ ကို ပြေး ။             |   2.96%
  Top- 3: Class  214 | အဆောက်အဦး မီးလောင် တာ များ လား ။              |   2.75%
  Top- 4: Class  514 | သူ့ ခြေတောက် ခွဲစိတ် ရ မယ် ။                  |   1.35%
  Top- 5: Class  433 | ဆေးရုံ လိပ်စာ လေး သိ ချင် လို့ ပါ ။           |   1.18%

============================================================
[2026-06-04 16:13:45]  webcam #3
  Top- 1: Class  274 | စောင် တွေ ယူလာ ပေး ပါ ။                       |   3.15%
  Top- 2: Class    8 | မီးလောင် လွယ် သော ပစ္စည်း များ ။              |   1.98%
  Top- 3: Class  141 | ကွင်းပြင် ရှိ တဲ့ နေရာ ကို ပြေး ။             |   1.03%
  Top- 4: Class  109 | သောက် သုံး ရေ များ ကြိုတင် စုဆောင်း ပါ ။      |   1.00%
  Top- 5: Class  433 | ဆေးရုံ လိပ်စာ လေး သိ ချင် လို့ ပါ ။           |   0.96%

============================================================
[2026-06-04 16:13:52]  webcam #4
  Top- 1: Class  369 | မိုး ကြီး ရင် ကုန်းမြေအနိမ့်ပိုင်း တွေ မှာ ချက်ခြင်း ရေ ကြီး နိုင် တယ် ။ |   4.58%
  Top- 2: Class  141 | ကွင်းပြင် ရှိ တဲ့ နေရာ ကို ပြေး ။             |   1.98%
  Top- 3: Class  305 | ဆရာဝန် မြန်မြန် ခေါ် လိုက် လေ ။               |   0.99%
  Top- 4: Class  161 | အရေးပေါ်ကား အမြန် လွတ်ပေး စေ ချင် ပါ တယ် ။    |   0.75%
  Top- 5: Class  240 | မ နေ့ ည က အိမ် ထဲ ကို သူခိုး ဝင် သွား လို့ ပါ ။ |   0.74%
```

## Transformer Online Result

```
============================================================
[2026-06-04 16:18:16]  webcam #1
  Top- 1: Class   61 | တောင်စောင်း ။                                 |   4.60%
  Top- 2: Class  485 | ည ၁၀ နာရီ နောက် ပိုင်း အမျိုးသား နေ ခွင့် မ ရှိ ပါ ဘူး ။ |   4.15%
  Top- 3: Class  348 | အိမ် အပြင် မှာ ဆိုရင် အဆောက်ဦး တွေ နဲ့ ဝေးဝေး ကို သွား ။ |   4.02%
  Top- 4: Class  463 | ဒဏ်ရာ ကို ရေခဲ ကပ် ပေး ပါ ။                   |   1.65%
  Top- 5: Class  160 | အဝေးပြေးကား ပေါ် က ခရီးသည် တွေ တော်တော်များများ ထိခိုက် ထား တယ် ။ |   1.51%

============================================================
[2026-06-04 16:18:23]  webcam #2
  Top- 1: Class   59 | ကမ်းခြေ ။                                     |   3.63%
  Top- 2: Class  214 | အဆောက်အဦး မီးလောင် တာ များ လား ။              |   2.17%
  Top- 3: Class  348 | အိမ် အပြင် မှာ ဆိုရင် အဆောက်ဦး တွေ နဲ့ ဝေးဝေး ကို သွား ။ |   1.41%
  Top- 4: Class  264 | ကလေး က ဆေး တွေ ကို ချိုချဉ် ထင် ပြီး စား ပစ် လိုက် တယ် ။ |   1.40%
  Top- 5: Class  141 | ကွင်းပြင် ရှိ တဲ့ နေရာ ကို ပြေး ။             |   1.38%

============================================================
[2026-06-04 16:18:30]  webcam #3
  Top- 1: Class  213 | ခုချက်ခြင်း မီးသတ်ကား လွတ် လိုက် ပါ မယ် ။     |   2.84%
  Top- 2: Class  214 | အဆောက်အဦး မီးလောင် တာ များ လား ။              |   1.73%
  Top- 3: Class  508 | အနီးဆုံး ဆေးရုံ ကို ပို့ ပေး ပါ ။             |   1.51%
  Top- 4: Class  396 | သူ့ ခြေတောက် က နာ နေ တယ် ။                    |   1.44%
  Top- 5: Class   46 | မျှော်စင် ။                                   |   1.38%

============================================================
[2026-06-04 16:18:38]  webcam #4
  Top- 1: Class  248 | ကြမ်းပြင် တစ်ခု လုံး မှာ ရေ တွေ ကြည့် ပဲ ။    |   3.47%
  Top- 2: Class  305 | ဆရာဝန် မြန်မြန် ခေါ် လိုက် လေ ။               |   2.33%
  Top- 3: Class  396 | သူ့ ခြေတောက် က နာ နေ တယ် ။                    |   2.23%
  Top- 4: Class  390 | ညစ်ညမ်း နေ တဲ့ နေရာ များ ကို သန့်ရှင်းရေး လုပ် ပြီး ရောဂါပိုး သတ် ရ ပါ မယ် ။ |   2.20%
  Top- 5: Class  214 | အဆောက်အဦး မီးလောင် တာ များ လား ။              |   1.93%

============================================================
[2026-06-04 16:18:46]  webcam #5
  Top- 1: Class  305 | ဆရာဝန် မြန်မြန် ခေါ် လိုက် လေ ။               |   3.60%
  Top- 2: Class  213 | ခုချက်ခြင်း မီးသတ်ကား လွတ် လိုက် ပါ မယ် ။     |   3.35%
  Top- 3: Class  458 | ဒီ ဆေး ကို နာကျင် ကိုက်ခဲ မှ သောက် ပါ ။       |   3.10%
  Top- 4: Class  162 | လူနာတင်ကား နဲ့ မီးသတ်ကား ချက်ခြင်း လွတ်လိုက် မယ် ။ |   2.68%
  Top- 5: Class  214 | အဆောက်အဦး မီးလောင် တာ များ လား ။              |   2.21%

============================================================
[2026-06-04 16:18:55]  webcam #6
  Top- 1: Class  508 | အနီးဆုံး ဆေးရုံ ကို ပို့ ပေး ပါ ။             |   3.03%
  Top- 2: Class  248 | ကြမ်းပြင် တစ်ခု လုံး မှာ ရေ တွေ ကြည့် ပဲ ။    |   2.48%
  Top- 3: Class  345 | အိမ် ထဲ မှာ ဆိုရင် နီး တဲ့ တံခါးပေါက်ဝ မှာ ရပ် နေ ပါ ။ |   2.08%
  Top- 4: Class  274 | စောင် တွေ ယူလာ ပေး ပါ ။                       |   1.46%
  Top- 5: Class   73 | မုန်တိုင်း တိုက် ။                            |   1.21%

============================================================
[2026-06-04 16:19:04]  webcam #7
  Top- 1: Class  248 | ကြမ်းပြင် တစ်ခု လုံး မှာ ရေ တွေ ကြည့် ပဲ ။    |   3.48%
  Top- 2: Class  305 | ဆရာဝန် မြန်မြန် ခေါ် လိုက် လေ ။               |   3.10%
  Top- 3: Class  214 | အဆောက်အဦး မီးလောင် တာ များ လား ။              |   2.81%
  Top- 4: Class  213 | ခုချက်ခြင်း မီးသတ်ကား လွတ် လိုက် ပါ မယ် ။     |   2.41%
  Top- 5: Class  390 | ညစ်ညမ်း နေ တဲ့ နေရာ များ ကို သန့်ရှင်းရေး လုပ် ပြီး ရောဂါပိုး သတ် ရ ပါ မယ် ။ |   2.21%

============================================================
[2026-06-04 16:19:13]  webcam #8
  Top- 1: Class  160 | အဝေးပြေးကား ပေါ် က ခရီးသည် တွေ တော်တော်များများ ထိခိုက် ထား တယ် ။ |   3.95%
  Top- 2: Class  119 | ပင်လယ် သမုဒ္ဒရာ ကြမ်းပြင် အောက် ငလျင် လှုပ် ရင် ဆူနာမီလှိုင်း များ ဖြစ် နိုင် တယ် ။ |   2.11%
  Top- 3: Class  157 | အဝေးပြေးကား တစ်စီး နဲ့ အိမ်စီးကား တစ်စီး ကားတိုက် မှု ဖြစ် လို့ ပါ ။ |   1.55%
  Top- 4: Class  115 | စားပွဲခုံ အောက် မှာ ဝင်နေ ။                   |   1.54%
  Top- 5: Class  147 | ကားတိုက် မှု ဖြစ် လို့ ပါ ။                   |   1.42%

============================================================
[2026-06-04 16:19:23]  webcam #9
  Top- 1: Class  214 | အဆောက်အဦး မီးလောင် တာ များ လား ။              |   2.55%
  Top- 2: Class  463 | ဒဏ်ရာ ကို ရေခဲ ကပ် ပေး ပါ ။                   |   1.94%
  Top- 3: Class   59 | ကမ်းခြေ ။                                     |   1.84%
  Top- 4: Class  274 | စောင် တွေ ယူလာ ပေး ပါ ။                       |   1.76%
  Top- 5: Class  141 | ကွင်းပြင် ရှိ တဲ့ နေရာ ကို ပြေး ။             |   1.37%
```

### STGCN Online Result

```
============================================================
[2026-06-04 16:21:36]  webcam #1
  Top- 1: Class  147 | ကားတိုက် မှု ဖြစ် လို့ ပါ ။                   |  19.09%
  Top- 2: Class  223 | မီးလောင် နေ တဲ့ အိမ် ထဲ မှာ လူ ရှိ သေး တယ် ။  |   9.80%
  Top- 3: Class  160 | အဝေးပြေးကား ပေါ် က ခရီးသည် တွေ တော်တော်များများ ထိခိုက် ထား တယ် ။ |   7.74%
  Top- 4: Class  218 | ငါ တစ်ခုခု မီးလောင် နေ တဲ့ အနံ့ ရ တယ် ။       |   3.67%
  Top- 5: Class   59 | ကမ်းခြေ ။                                     |   2.78%

============================================================
[2026-06-04 16:21:44]  webcam #2
  Top- 1: Class  214 | အဆောက်အဦး မီးလောင် တာ များ လား ။              |  15.70%
  Top- 2: Class  106 | လျှပ်စစ် ဝါယာ ရှော့ ဖြစ် ရင် ရေ စို အဝတ် နဲ့ မီး မ ငြှိမ်းသတ် ပါ နဲ့ ။ |   5.75%
  Top- 3: Class  545 | ခဏ ပါ မ ကြာ ပါ ဘူး ။                          |   4.28%
  Top- 4: Class  329 | သေချာ ဂရုစိုက် ။                              |   2.54%
  Top- 5: Class  305 | ဆရာဝန် မြန်မြန် ခေါ် လိုက် လေ ။               |   2.43%

============================================================
[2026-06-04 16:21:52]  webcam #3
  Top- 1: Class  147 | ကားတိုက် မှု ဖြစ် လို့ ပါ ။                   |  13.08%
  Top- 2: Class  106 | လျှပ်စစ် ဝါယာ ရှော့ ဖြစ် ရင် ရေ စို အဝတ် နဲ့ မီး မ ငြှိမ်းသတ် ပါ နဲ့ ။ |   7.73%
  Top- 3: Class  214 | အဆောက်အဦး မီးလောင် တာ များ လား ။              |   4.37%
  Top- 4: Class  219 | တစ်ခုခု မီးလောင် နေ တဲ့ ညှော်နံ့ ရ တယ် ။      |   4.22%
  Top- 5: Class  545 | ခဏ ပါ မ ကြာ ပါ ဘူး ။                          |   3.69%

============================================================
[2026-06-04 16:22:00]  webcam #4
  Top- 1: Class  147 | ကားတိုက် မှု ဖြစ် လို့ ပါ ။                   |  29.02%
  Top- 2: Class  106 | လျှပ်စစ် ဝါယာ ရှော့ ဖြစ် ရင် ရေ စို အဝတ် နဲ့ မီး မ ငြှိမ်းသတ် ပါ နဲ့ ။ |  12.70%
  Top- 3: Class  160 | အဝေးပြေးကား ပေါ် က ခရီးသည် တွေ တော်တော်များများ ထိခိုက် ထား တယ် ။ |   4.83%
  Top- 4: Class  214 | အဆောက်အဦး မီးလောင် တာ များ လား ။              |   3.02%
  Top- 5: Class  218 | ငါ တစ်ခုခု မီးလောင် နေ တဲ့ အနံ့ ရ တယ် ။       |   2.58%

============================================================
[2026-06-04 16:22:08]  webcam #5
  Top- 1: Class  147 | ကားတိုက် မှု ဖြစ် လို့ ပါ ။                   |  12.13%
  Top- 2: Class  214 | အဆောက်အဦး မီးလောင် တာ များ လား ။              |   5.78%
  Top- 3: Class  160 | အဝေးပြေးကား ပေါ် က ခရီးသည် တွေ တော်တော်များများ ထိခိုက် ထား တယ် ။ |   3.10%
  Top- 4: Class  381 | ဓာတ်ကြိုး များ ပျက်စီး နေ နိုင် ပါ တယ် ။      |   3.00%
  Top- 5: Class  545 | ခဏ ပါ မ ကြာ ပါ ဘူး ။                          |   2.90%

============================================================
[2026-06-04 16:22:17]  webcam #6
  Top- 1: Class  147 | ကားတိုက် မှု ဖြစ် လို့ ပါ ။                   |  10.89%
  Top- 2: Class  214 | အဆောက်အဦး မီးလောင် တာ များ လား ။              |   7.69%
  Top- 3: Class  160 | အဝေးပြေးကား ပေါ် က ခရီးသည် တွေ တော်တော်များများ ထိခိုက် ထား တယ် ။ |   5.39%
  Top- 4: Class   53 | ရေကူး ခြင်း ။                                 |   4.76%
  Top- 5: Class  155 | ဘယ်နား မှာ ကားတိုက် မှု ဖြစ် တာ လဲ ။          |   4.20%

============================================================
[2026-06-04 16:22:26]  webcam #7
  Top- 1: Class  155 | ဘယ်နား မှာ ကားတိုက် မှု ဖြစ် တာ လဲ ။          |   8.28%
  Top- 2: Class  214 | အဆောက်အဦး မီးလောင် တာ များ လား ။              |   7.37%
  Top- 3: Class  480 | ညအိပ်ယာဝင် သောက် ပါ ။                         |   5.08%
  Top- 4: Class  205 | ဟို မှာ ကားတိုက် မှု ဖြစ် နေ လို့ ပါ ။        |   4.24%
  Top- 5: Class  160 | အဝေးပြေးကား ပေါ် က ခရီးသည် တွေ တော်တော်များများ ထိခိုက် ထား တယ် ။ |   3.81%
```

အကုန်မှားနေတယ်။ :(  

## Testing with Input Video  

## BilSTM

```
(msl_onnx) E:\LU_Lab\MSL_Tran_Demo>python infer_onnx.py --onnx_model onnx_models\bilstm_msl.onnx   --video sample4inference\idx20-30.mp4   --top_k 5 --output_dir predictions_output --log_file results.txt
[INFO] Loaded 556 labels from label map.
[INFO] ONNX loaded: bilstm_msl.onnx  (provider: CPUExecutionProvider)
[INFO] Logging top-5 results to: results.txt
[INFO] Saving output videos to: predictions_output/
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
[INFO] idx20-30.mp4 — 127 frames @ 30.0 fps
[INFO] MediaPipe pose complexity: 2  (must match training; training default=2)
C:\Users\801680\Anaconda3\envs\msl_onnx\lib\site-packages\google\protobuf\symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '
[INFO] 127 frames extracted → using 127 (zero-padded to 200)

Source : idx20-30.mp4
Model  : BILSTM
────────────────────────────────────────────────────────────
  Top-1: မြေနိမ့်ဒေသ ။  (80.73%)
  Top-2: ချိုင့် ။  (0.16%)
  Top-3: ဘေး အိမ် ကို ဓားပြ ဝင် တိုက် နေ ပါ တယ် ။  (0.13%)
  Top-4: ဓာတ်မှန် ရိုက် တဲ့ နေရာ က ဘယ်နား မှာ လဲ သိ ချင် လို့ ပါ ။  (0.12%)
  Top-5: လျှပ်စစ် ဝါယာ ရှော့ ဖြစ် ရင် မီးသတ်ဆေးဘူး အသုံးပြု ပါ ။  (0.12%)
────────────────────────────────────────────────────────────
[INFO] Saved  clean  → predictions_output\idx20-30_clean.mp4
[INFO] Saved  debug  → predictions_output\idx20-30_debug.mp4
[INFO] Done.

(msl_onnx) E:\LU_Lab\MSL_Tran_Demo>
```
    
## Transformer

```
(msl_onnx) E:\LU_Lab\MSL_Tran_Demo>python infer_onnx.py --onnx_model onnx_models\transformer_msl.onnx   --video sample4inference\idx20-30.mp4   --top_k 5 --output_dir predictions_output --log_file results.txt
[INFO] Loaded 556 labels from label map.
[INFO] ONNX loaded: transformer_msl.onnx  (provider: CPUExecutionProvider)
[INFO] Logging top-5 results to: results.txt
[INFO] Saving output videos to: predictions_output/
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
[INFO] idx20-30.mp4 — 127 frames @ 30.0 fps
[INFO] MediaPipe pose complexity: 2  (must match training; training default=2)
C:\Users\801680\Anaconda3\envs\msl_onnx\lib\site-packages\google\protobuf\symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '
[INFO] 127 frames extracted → using 127 (zero-padded to 200)

Source : idx20-30.mp4
Model  : TRANSFORMER
────────────────────────────────────────────────────────────
  Top-1: မြေနိမ့်ဒေသ ။  (80.68%)
  Top-2: ချိုင့် ။  (0.30%)
  Top-3: မီးခလုပ် များ ကို ပိတ် ပါ ။  (0.28%)
  Top-4: ဂရုစိုက် ။  (0.25%)
  Top-5: လေမုန်တိုင်း ကျ မယ် ။  (0.15%)
────────────────────────────────────────────────────────────
[INFO] Saved  clean  → predictions_output\idx20-30_clean.mp4
[INFO] Saved  debug  → predictions_output\idx20-30_debug.mp4
[INFO] Done.

(msl_onnx) E:\LU_Lab\MSL_Tran_Demo>
```
      
```
(msl_onnx) E:\LU_Lab\MSL_Tran_Demo>python infer_onnx.py --onnx_model onnx_models\stgcn_msl.onnx   --video sample4inference\idx20-30.mp4   --top_k 5 --output_dir predictions_output --log_file results.txt
[INFO] Loaded 556 labels from label map.
[INFO] ONNX loaded: stgcn_msl.onnx  (provider: CPUExecutionProvider)
[INFO] Logging top-5 results to: results.txt
[INFO] Saving output videos to: predictions_output/
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
[INFO] idx20-30.mp4 — 127 frames @ 30.0 fps
[INFO] MediaPipe pose complexity: 2  (must match training; training default=2)
C:\Users\801680\Anaconda3\envs\msl_onnx\lib\site-packages\google\protobuf\symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '
[INFO] 127 frames extracted → using 127 (zero-padded to 200)

Source : idx20-30.mp4
Model  : STGCN
────────────────────────────────────────────────────────────
  Top-1: မြေနိမ့်ဒေသ ။  (82.02%)
  Top-2: မီးသတ်သမား ။  (0.68%)
  Top-3: အထူး သတိပြု ။  (0.63%)
  Top-4: မြစ်ခေျာင်း ။  (0.51%)
  Top-5: စားပွဲခုံ အောက် မှာ ဝင်နေ ။  (0.34%)
────────────────────────────────────────────────────────────
[INFO] Saved  clean  → predictions_output\idx20-30_clean.mp4
[INFO] Saved  debug  → predictions_output\idx20-30_debug.mp4
[INFO] Done.

(msl_onnx) E:\LU_Lab\MSL_Tran_Demo>
```
                

## Updated Code  

In [102]:
from IPython.display import Markdown

file_path = './src/infer_onnx.py'

with open(file_path, 'r') as f:
    code = f.read()

# 'python' keyword triggers the syntax highlighter
display(Markdown(f"```python\n{code}\n```"))

```python
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
infer_onnx.py  –  Real-time MSL recognition on Windows using ONNX Runtime

Bugs fixed in this version
───────────────────────────
  [INFER-A] process_frame skipped frames without pose detection.
      Training always includes every frame (zeros for undetected joints).
      Fix: process_frame always returns the keypoint array, never None.

  [INFER-B] normalize_sequence used 2-D scale and left z unscaled.
      Fix: exact mirror of extract_keypoints.normalize_sequence() —
      full 3-D midpoint and 3-D Euclidean shoulder width, all axes.

  [INFER-C] *** ROOT CAUSE OF WEBCAM FAILURE ***
      Webcam classified every classify_every=15 frames = 0.5 seconds.
      Signs take 1–4 seconds; feeding 15 frames to a model trained on
      full-length signs produces garbage predictions.

      infer.py uses a TIME-BASED buffer: accumulate buffer_sec=2.5 sec
      of frames (~75 at 30 fps), then classify and clear.  This gives
      the model a realistic-duration sign sequence matching training.

      Fix: replaced the frame-count trigger with a time-based trigger
      using --buffer_sec (default 2.5), identical to infer.py.

  [INFER-D] np.int64 / int dict key mismatch.
      Fix: always cast to int() before idx2label lookup.

  [INFER-E] ONNXModel.predict returned a scalar instead of 1-D array.
      Fix: return output[0] — shape (num_classes,).

  [INFER-F] model_complexity=1 everywhere; training used complexity=2.
      Fix: complexity=2 for video (matching training), 1 for webcam
      (speed tradeoff, same as infer.py).

  [INFER-G] extractor.pose.model_complexity AttributeError on Windows.
      MediaPipe's Pose object does not expose model_complexity publicly.
      Fix: stored as self.model_complexity on KeypointExtractor.

New features
────────────
  • Video saving — two output files per inference:
      {name}_clean.mp4  —  label/confidence overlay only
      {name}_debug.mp4  —  label overlay + MediaPipe skeleton lines
    Works for both --video and --webcam modes.
    Saved to --output_dir (default: predictions_output/).

  • Webcam log — every classification event writes top-k results to the
    log file with timestamp and separator, matching the video log format.
"""

import argparse
import importlib
import json
import sys
from datetime import datetime
from pathlib import Path

import cv2
import numpy as np
import onnxruntime as ort

try:
    from PIL import Image, ImageDraw, ImageFont
    HAS_PIL = True
except ImportError:
    HAS_PIL = False

# ── Constants (must match extract_keypoints.py exactly) ───────────────────────
POSE_N          = 33
HAND_N          = 21
TOTAL_JOINTS    = POSE_N + HAND_N * 2   # 75
COORDS          = 3
FEAT_DIM        = TOTAL_JOINTS * COORDS  # 225

POSE_LEFT_SHOULDER  = 11
POSE_RIGHT_SHOULDER = 12
LEFT_HAND_START     = POSE_N            # 33
RIGHT_HAND_START    = POSE_N + HAND_N   # 54


# ─── MediaPipe import ─────────────────────────────────────────────────────────

def _import_mp():
    for prefix in ["mediapipe.solutions", "mediapipe.python.solutions"]:
        try:
            pose  = importlib.import_module(f"{prefix}.pose")
            hands = importlib.import_module(f"{prefix}.hands")
            draw  = importlib.import_module(f"{prefix}.drawing_utils")
            return pose, hands, draw
        except (ImportError, ModuleNotFoundError):
            continue
    try:
        import mediapipe as _mp
        return _mp.solutions.pose, _mp.solutions.hands, _mp.solutions.drawing_utils
    except AttributeError:
        pass
    raise ImportError(
        "Cannot import MediaPipe solutions.\n"
        "Try: pip uninstall mediapipe -y && pip install mediapipe==0.10.18"
    )

mp_pose, mp_hands, mp_drawing = _import_mp()


# ─── Myanmar text rendering ──────────────────────────────────────────────────

class TextRenderer:
    _FONT_CANDIDATES = [
        "C:\\Windows\\Fonts\\Padauk.ttf",
        "C:\\Windows\\Fonts\\NotoSansMyanmar-Regular.ttf",
        "C:\\Windows\\Fonts\\NotoSansMyanmar.ttf",
        "C:\\Windows\\Fonts\\myanmar3.ttf",
        "/usr/share/fonts/truetype/padauk/Padauk-Regular.ttf",
        "/usr/share/fonts/truetype/noto/NotoSansMyanmar-Regular.ttf",
        "fonts\\Padauk.ttf",
        "fonts/NotoSansMyanmar-Regular.ttf",
    ]

    def __init__(self, font_size: int = 32):
        self.font = self._load_font(font_size)
        self.can_render_myanmar = (self.font is not None)
        if not self.can_render_myanmar:
            print()
            print("  ╔═══════════════════════════════════════════════════════════╗")
            print("  ║  INFO: No Myanmar shaping font found.                    ║")
            print("  ║  Displaying readable Class IDs instead of Myanmar text.  ║")
            print("  ╚═══════════════════════════════════════════════════════════╝")
            print()

    def _load_font(self, size):
        if not HAS_PIL:
            return None
        for path in self._FONT_CANDIDATES:
            if Path(path).exists():
                try:
                    return ImageFont.truetype(path, size)
                except Exception:
                    continue
        return None

    def put_text(self, frame, text, position=(20, 20), color=(0, 255, 0)):
        if self.font and HAS_PIL:
            pil = Image.fromarray(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
            d   = ImageDraw.Draw(pil)
            r, g, b = color[2], color[1], color[0]
            for dx in [-1, 0, 1]:
                for dy in [-1, 0, 1]:
                    if dx or dy:
                        d.text((position[0]+dx, position[1]+dy), text,
                               font=self.font, fill=(0, 0, 0))
            d.text(position, text, font=self.font, fill=(r, g, b))
            return cv2.cvtColor(np.array(pil), cv2.COLOR_RGB2BGR)
        cv2.putText(frame, text, position, cv2.FONT_HERSHEY_SIMPLEX,
                    0.75, color, 2, cv2.LINE_AA)
        return frame


# ─── Keypoint extraction ──────────────────────────────────────────────────────

class KeypointExtractor:
    """
    Extracts (TOTAL_JOINTS, 3) keypoint arrays from BGR frames.

    Fix [INFER-G]: model_complexity is stored as self.model_complexity
    because mediapipe's Pose object does not expose it as a public attribute
    on all Windows builds (AttributeError otherwise).

    Fix [INFER-A]: process_frame always returns an array, never None.
    Matches extract_keypoints.py which includes every frame, using zeros
    for any joint that MediaPipe failed to detect.

    Hand convention (matches extract_keypoints.py):
      MediaPipe reports handedness from the camera perspective (mirrored).
      label='Right' → signer's LEFT  hand → slot LEFT_HAND_START  (33).
      label='Left'  → signer's RIGHT hand → slot RIGHT_HAND_START (54).
    """

    def __init__(self, model_complexity: int = 2,
                 min_detection_confidence: float = 0.5,
                 min_tracking_confidence:  float = 0.5):
        # Fix [INFER-G]: store complexity ourselves — mediapipe's Pose object
        # does NOT expose a public .model_complexity on all Windows builds.
        self.model_complexity = model_complexity

        self.pose  = mp_pose.Pose(
            static_image_mode=False,
            model_complexity=model_complexity,
            smooth_landmarks=True,
            enable_segmentation=False,
            min_detection_confidence=min_detection_confidence,
            min_tracking_confidence=min_tracking_confidence,
        )
        self.hands = mp_hands.Hands(
            static_image_mode=False,
            max_num_hands=2,
            model_complexity=min(model_complexity, 1),  # Hands caps at 1
            min_detection_confidence=min_detection_confidence,
            min_tracking_confidence=min_tracking_confidence,
        )
        self._last_pose  = None
        self._last_hands = None

    def process_frame(self, frame: np.ndarray,
                      mirror: bool = False) -> np.ndarray:
        """
        Returns (TOTAL_JOINTS, 3) float32 for every frame.
        Zeros for any undetected joint/hand — never returns None.

        Args:
            frame:  BGR frame from cv2.VideoCapture.
            mirror: If True, flip the frame horizontally before processing.
                    Many laptop/webcam cameras output a mirrored (selfie-view)
                    image.  Training videos were likely recorded un-mirrored
                    (normal camera view).  Setting mirror=True corrects this so
                    the normalised coordinates match the training distribution.
                    The DISPLAYED frame is NOT flipped — only the frame sent to
                    MediaPipe is flipped, so landmark positions are in the
                    coordinate space of the un-mirrored view.
        """
        if mirror:
            frame = cv2.flip(frame, 1)   # horizontal flip only for MediaPipe
        rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        rgb.flags.writeable = False
        p_res = self.pose.process(rgb)
        h_res = self.hands.process(rgb)
        self._last_pose  = p_res
        self._last_hands = h_res

        kp = np.zeros((TOTAL_JOINTS, COORDS), dtype=np.float32)

        if p_res.pose_landmarks:
            for i, lm in enumerate(p_res.pose_landmarks.landmark):
                kp[i] = [lm.x, lm.y, lm.z]

        if h_res.multi_hand_landmarks and h_res.multi_handedness:
            for lm_list, handedness in zip(
                h_res.multi_hand_landmarks, h_res.multi_handedness
            ):
                label  = handedness.classification[0].label
                offset = LEFT_HAND_START if label == 'Right' else RIGHT_HAND_START
                for i, lm in enumerate(lm_list.landmark):
                    kp[offset + i] = [lm.x, lm.y, lm.z]

        return kp  # always an array, never None

    def draw_landmarks(self, frame: np.ndarray) -> np.ndarray:
        """Draw MediaPipe skeleton onto frame IN-PLACE and return it."""
        if self._last_pose and self._last_pose.pose_landmarks:
            mp_drawing.draw_landmarks(
                frame, self._last_pose.pose_landmarks, mp_pose.POSE_CONNECTIONS
            )
        if self._last_hands and self._last_hands.multi_hand_landmarks:
            for lm in self._last_hands.multi_hand_landmarks:
                mp_drawing.draw_landmarks(frame, lm, mp_hands.HAND_CONNECTIONS)
        return frame

    def close(self):
        self.pose.close()
        self.hands.close()


# ─── Normalization ────────────────────────────────────────────────────────────

def normalize_sequence(seq: np.ndarray) -> np.ndarray:
    """
    Fix [INFER-B]: exact mirror of extract_keypoints.normalize_sequence().
    Full 3-D shoulder-midpoint subtraction and 3-D shoulder-width scaling,
    applied to all three coordinate axes.
    """
    l_sh = seq[:, POSE_LEFT_SHOULDER,  :]   # (T, 3)
    r_sh = seq[:, POSE_RIGHT_SHOULDER, :]   # (T, 3)
    mid  = (l_sh + r_sh) / 2.0
    dist = np.linalg.norm(r_sh - l_sh, axis=1, keepdims=True)
    dist = np.where(dist < 1e-6, 1.0, dist)
    return ((seq - mid[:, None, :]) / dist[:, None, :]).astype(np.float32)


# ─── Preprocessor ─────────────────────────────────────────────────────────────

class Preprocessor:
    """
    Stack raw (75, 3) keypoint frames → normalize → crop/pad → model input.

    Processing order matches infer.py exactly:
      1. Stack   → (T, 75, 3)
      2. Normalize (3-D shoulder-centred, full 3-D scale)
      3. Centre-crop  if T > max_seq_len
      4. Zero-pad     if T < max_seq_len
      5. Build mask   (True = padded position)
      6. Reshape for model type
    """

    def __init__(self, max_seq_len: int = 200):
        self.max_seq_len = max_seq_len

    def __call__(self, kp_list: list, model_type: str) -> dict:
        if not kp_list:
            return {}

        seq = np.stack(kp_list, axis=0).astype(np.float32)  # (T, 75, 3)
        seq = normalize_sequence(seq)
        T   = seq.shape[0]

        if T > self.max_seq_len:
            start = (T - self.max_seq_len) // 2
            seq   = seq[start: start + self.max_seq_len]
            T     = self.max_seq_len

        real_len = T

        if T < self.max_seq_len:
            pad = np.zeros((self.max_seq_len - T, TOTAL_JOINTS, COORDS), dtype=np.float32)
            seq = np.concatenate([seq, pad], axis=0)

        mask = np.zeros(self.max_seq_len, dtype=np.bool_)
        mask[real_len:] = True

        if model_type == "stgcn":
            kp_out = np.expand_dims(seq.transpose(2, 0, 1), axis=0).astype(np.float32)
            return {"keypoints": kp_out}
        else:
            kp_out = np.expand_dims(
                seq.reshape(self.max_seq_len, FEAT_DIM), axis=0
            ).astype(np.float32)
            return {"keypoints": kp_out, "mask": mask[np.newaxis]}


# ─── ONNX model wrapper ───────────────────────────────────────────────────────

class ONNXModel:
    def __init__(self, onnx_path: str, meta_path: str = None):
        available = ort.get_available_providers()
        preferred = ["CUDAExecutionProvider", "DmlExecutionProvider", "CPUExecutionProvider"]
        providers = [p for p in preferred if p in available]

        self.session = ort.InferenceSession(onnx_path, providers=providers)
        print(f"[INFO] ONNX loaded: {Path(onnx_path).name}"
              f"  (provider: {self.session.get_providers()[0]})")

        self.meta = {}
        if meta_path and Path(meta_path).exists():
            try:
                with open(meta_path, encoding="utf-8") as f:
                    self.meta = json.load(f)
            except Exception as e:
                print(f"[WARNING] Could not read metadata: {e}")
        else:
            print(f"[INFO] Metadata not found at '{meta_path}', using defaults.")

        self.model_type  = self.meta.get("model_type",  "transformer")
        self.max_seq_len = self.meta.get("max_seq_len", 200)
        self.num_classes = self.meta.get("num_classes", 558)

    def predict(self, inputs: dict) -> np.ndarray:
        """Fix [INFER-E]: return (num_classes,) not scalar."""
        return self.session.run(None, inputs)[0][0]

    @staticmethod
    def softmax(logits: np.ndarray) -> np.ndarray:
        e = np.exp(logits - logits.max())
        return e / e.sum()


# ─── Label map ────────────────────────────────────────────────────────────────

def load_idx2label(path: str) -> dict:
    with open(path, encoding="utf-8") as f:
        data = json.load(f)
    return {int(k): v for k, v in data.get("idx2label", {}).items()}


def _get_label(idx2label: dict, class_id) -> str:
    """Fix [INFER-D]: always cast to int() before dict lookup."""
    return idx2label.get(int(class_id), f"[class {int(class_id)}]")


# ─── Helpers ──────────────────────────────────────────────────────────────────

def _overlay_text(renderer, frame, text, color):
    """Put prediction text at top-left of frame."""
    return renderer.put_text(frame, text, position=(10, 10), color=color)


def _pred_color(confidence: float) -> tuple:
    """Green ≥80%, yellow ≥50%, red otherwise."""
    if confidence >= 0.80: return (0, 200,   0)
    if confidence >= 0.50: return (0, 200, 200)
    return (0, 0, 255)


def _build_overlay_text(top_idx, probs, idx2label, renderer, top_k):
    """Single-line text for the video overlay (top-1 label + confidence)."""
    best = top_idx[0]
    conf = probs[int(best)]
    lbl  = _get_label(idx2label, best) if idx2label else f"Class {int(best)}"
    if renderer.can_render_myanmar:
        return f"{lbl}  ({conf*100:.1f}%)", conf
    return f"Class {int(best)}  ({conf*100:.1f}%)", conf


def _build_console_block(top_idx, probs, idx2label, model_type, source_name, top_k):
    """Multi-line console output block, same format as run_video."""
    sep   = "─" * 60
    lines = [f"Source : {source_name}", f"Model  : {model_type.upper()}", sep]
    for rank, idx in enumerate(top_idx, 1):
        lbl = _get_label(idx2label, idx) if idx2label else f"[class {int(idx)}]"
        lines.append(f"  Top-{rank}: {lbl}  ({probs[int(idx)]*100:.2f}%)")
    lines.append(sep)
    return "\n".join(lines)


def _build_log_block(top_idx, probs, idx2label, source_name, top_k):
    """Log block with timestamp and top-k results."""
    ts    = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    sep   = "=" * 60
    lines = [sep, f"[{ts}]  {source_name}"]
    for rank, idx in enumerate(top_idx, 1):
        lbl = _get_label(idx2label, idx) if idx2label else "?"
        lbl = lbl.encode("utf-8", errors="replace").decode("utf-8")
        lines.append(f"  Top-{rank:>2}: Class {int(idx):>4d} | "
                     f"{lbl:<45s} | {probs[int(idx)]*100:6.2f}%")
    lines.append("")
    return "\n".join(lines) + "\n"


def _append_log(path, text):
    try:
        with open(path, "a", encoding="utf-8-sig") as f:
            f.write(text)
    except Exception as e:
        print(f"[WARNING] Could not write log: {e}")


def _make_video_writer(output_path: Path, fps: float, width: int, height: int):
    """Create a cv2.VideoWriter with MP4V codec."""
    fourcc = cv2.VideoWriter_fourcc(*"mp4v")
    return cv2.VideoWriter(str(output_path), fourcc, fps, (width, height))


def _save_feature_log(kp_list, inputs, probs, top_idx, idx2label,
                      source_name, output_dir, mirror):
    """
    Save full diagnostic information for one classification event.

    Creates two files in output_dir/features/:
      {source_name}_raw_kp.npy      — raw (un-normalized) stacked keypoints (T,75,3)
      {source_name}_model_input.npy — the actual array fed to the ONNX model
      {source_name}_features.txt    — human-readable statistics for debugging

    The .txt file contains:
      - Shoulder positions and detection status (are they non-zero?)
      - Shoulder width (normalization scale — should be ~0.3-0.7 if detected)
      - Per-axis statistics of normalized features
      - Fraction of frames with active hand movement
      - Mirror setting used
      - Top-k classification results
      - Comparison hint vs expected training distribution
    """
    import os
    feat_dir = output_dir / "features"
    feat_dir.mkdir(parents=True, exist_ok=True)

    # Raw keypoints before normalization
    raw_kp = np.stack(kp_list, axis=0).astype(np.float32)   # (T, 75, 3)
    T      = raw_kp.shape[0]

    # Save .npy files
    np.save(str(feat_dir / f"{source_name}_raw_kp.npy"), raw_kp)

    # Also save the model input (already normalized, cropped, padded)
    model_input_key = "keypoints"
    if model_input_key in inputs:
        np.save(str(feat_dir / f"{source_name}_model_input.npy"),
                inputs[model_input_key])

    # --- Build text report ---
    L_SH, R_SH = 11, 12
    l_sh = raw_kp[:, L_SH, :]   # (T, 3)
    r_sh = raw_kp[:, R_SH, :]   # (T, 3)

    # How many frames had shoulders detected (non-zero)?
    l_detected = (np.abs(l_sh).sum(axis=1) > 1e-6).sum()
    r_detected = (np.abs(r_sh).sum(axis=1) > 1e-6).sum()

    # Shoulder width distribution (normalization scale)
    dist = np.linalg.norm(r_sh - l_sh, axis=1)  # (T,)
    dist_nonzero = dist[dist > 1e-6]

    # Hand activity: fraction of frames where hands are non-zero
    lh_block = raw_kp[:, 33:54, :]
    rh_block = raw_kp[:, 54:75, :]
    lh_active = (np.abs(lh_block).sum(axis=(1,2)) > 1e-6).sum()
    rh_active = (np.abs(rh_block).sum(axis=(1,2)) > 1e-6).sum()

    # Normalized keypoints for distribution check
    norm_kp = normalize_sequence(raw_kp)
    # For a correctly framed frontal signer:
    #   Left  shoulder (idx 11) normalized x should be ~ +0.4 to +0.6
    #   Right shoulder (idx 12) normalized x should be ~ -0.6 to -0.4
    l_sh_norm = norm_kp[:, L_SH, :]
    r_sh_norm = norm_kp[:, R_SH, :]

    lines = [
        "=" * 65,
        f"Feature Diagnostic Log — {source_name}",
        f"Timestamp : {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}",
        f"Mirror    : {'YES — webcam flipped before MediaPipe' if mirror else 'NO — raw frame used'}",
        "=" * 65,
        "",
        "── Sequence info ───────────────────────────────────────────",
        f"  Total frames         : {T}",
        f"  Left  shoulder detected: {l_detected}/{T} frames ({l_detected/T*100:.0f}%)",
        f"  Right shoulder detected: {r_detected}/{T} frames ({r_detected/T*100:.0f}%)",
        f"  Left  hand active    : {lh_active}/{T} frames ({lh_active/T*100:.0f}%)",
        f"  Right hand active    : {rh_active}/{T} frames ({rh_active/T*100:.0f}%)",
        "",
        "── Shoulder positions (RAW, image space [0,1]) ─────────────",
        f"  Left  shoulder x mean : {l_sh[:,0].mean():.4f}  std={l_sh[:,0].std():.4f}",
        f"  Right shoulder x mean : {r_sh[:,0].mean():.4f}  std={r_sh[:,0].std():.4f}",
        f"  Left  shoulder y mean : {l_sh[:,1].mean():.4f}",
        f"  Right shoulder y mean : {r_sh[:,1].mean():.4f}",
        "",
        "  EXPECTED for frontal un-mirrored signer:",
        "    Left  shoulder x ~ 0.55–0.65 (camera right = signer left)",
        "    Right shoulder x ~ 0.35–0.45 (camera left  = signer right)",
        "  If values are REVERSED → webcam is mirrored → use --mirror",
        "",
        "── Shoulder width (normalization scale) ────────────────────",
    ]
    if len(dist_nonzero) > 0:
        lines += [
            f"  Shoulder width mean : {dist_nonzero.mean():.4f}",
            f"  Shoulder width std  : {dist_nonzero.std():.4f}",
            f"  Shoulder width range: {dist_nonzero.min():.4f} – {dist_nonzero.max():.4f}",
            "  EXPECTED: 0.15–0.50 (in normalized image coords)",
            "  If near 0 → shoulders not detected → normalization fails",
        ]
    else:
        lines.append("  !! Shoulder width is ZERO in ALL frames — shoulders NOT detected!")
        lines.append("     Normalization falls back to dist=1.0 → features NOT normalized correctly.")

    lines += [
        "",
        "── Normalized shoulder positions (after normalize_sequence) ─",
        f"  Left  shoulder norm x mean : {l_sh_norm[:,0].mean():.4f}",
        f"  Right shoulder norm x mean : {r_sh_norm[:,0].mean():.4f}",
        "  EXPECTED after correct normalization:",
        "    Left  shoulder norm x ~  +0.5  (signer's left = positive x)",
        "    Right shoulder norm x ~  -0.5  (signer's right = negative x)",
        "  If REVERSED (left~-0.5, right~+0.5) → mirror flip needed",
        "",
        "── Full keypoint distribution (normalized) ──────────────────",
        f"  All joints x: mean={norm_kp[:,:,0].mean():.4f}  std={norm_kp[:,:,0].std():.4f}"
        f"  min={norm_kp[:,:,0].min():.4f}  max={norm_kp[:,:,0].max():.4f}",
        f"  All joints y: mean={norm_kp[:,:,1].mean():.4f}  std={norm_kp[:,:,1].std():.4f}"
        f"  min={norm_kp[:,:,1].min():.4f}  max={norm_kp[:,:,1].max():.4f}",
        f"  All joints z: mean={norm_kp[:,:,2].mean():.4f}  std={norm_kp[:,:,2].std():.4f}"
        f"  min={norm_kp[:,:,2].min():.4f}  max={norm_kp[:,:,2].max():.4f}",
        "",
        "── Classification results ───────────────────────────────────",
    ]
    for rank, idx in enumerate(top_idx, 1):
        lbl = _get_label(idx2label, idx) if idx2label else "?"
        lbl_safe = lbl.encode("utf-8", errors="replace").decode("utf-8")
        lines.append(f"  Top-{rank}: Class {int(idx):>4d} | {lbl_safe:<50s} | {probs[int(idx)]*100:.2f}%")

    lines += [
        "",
        "── Files saved ──────────────────────────────────────────────",
        f"  Raw keypoints : features/{source_name}_raw_kp.npy      shape=({T},75,3)",
        f"  Model input   : features/{source_name}_model_input.npy",
        "=" * 65,
        "",
    ]

    txt_path = feat_dir / f"{source_name}_features.txt"
    with open(txt_path, "w", encoding="utf-8-sig") as f:
        f.write("\n".join(lines))
    print(f"[INFO] Feature log → {txt_path}")


# ─── Video processing ─────────────────────────────────────────────────────────

def run_video(model, prep, extractor, renderer, video_path,
              top_k=5, log_file=None, idx2label=None,
              output_dir=None, debug=False, mirror=False,
              save_features=False):
    """
    Process a single video file.  Saves two output videos:
      {stem}_clean.mp4  —  label overlay only
      {stem}_debug.mp4  —  label overlay + MediaPipe skeleton lines

    Matches infer.py / process_video() exactly:
      every frame included (zeros for undetected joints),
      normalize after stacking, centre-crop or zero-pad.
    """
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        print(f"[ERROR] Cannot open video: {video_path}")
        return

    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    fps_src      = cap.get(cv2.CAP_PROP_FPS) or 30.0
    w            = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    h            = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    stem         = Path(video_path).stem

    print(f"[INFO] {Path(video_path).name} — {total_frames} frames @ {fps_src:.1f} fps")
    print(f"[INFO] MediaPipe pose complexity: {extractor.model_complexity}"
          f"  (must match training; training default=2)")

    # --- Pass 1: extract all keypoints + collect frames for output video ---
    kp_list    = []
    raw_frames = []   # store for writing output videos after prediction
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        kp_list.append(extractor.process_frame(frame, mirror=mirror))
        raw_frames.append(frame.copy())
    cap.release()

    if not kp_list:
        print("[WARNING] No frames could be read from the video.")
        return

    T_raw = len(kp_list)
    T_use = min(T_raw, model.max_seq_len)
    print(f"[INFO] {T_raw} frames extracted → using {T_use} "
          f"({'centre-cropped' if T_raw > model.max_seq_len else 'zero-padded to'}"
          f" {model.max_seq_len})")

    # --- Inference ---
    inputs  = prep(kp_list, model.model_type)
    logits  = model.predict(inputs)
    probs   = ONNXModel.softmax(logits)
    top_idx = np.argsort(probs)[::-1][:top_k]

    # --- Console output ---
    print("\n" + _build_console_block(top_idx, probs, idx2label,
                                      model.model_type, Path(video_path).name, top_k))

    # --- Log file ---
    if log_file:
        _append_log(log_file,
                    _build_log_block(top_idx, probs, idx2label,
                                     Path(video_path).name, top_k))

    # --- Save keypoint feature log ---
    if save_features and output_dir:
        _save_feature_log(
            kp_list    = kp_list,
            inputs     = inputs,
            probs      = probs,
            top_idx    = top_idx,
            idx2label  = idx2label,
            source_name= Path(video_path).stem,
            output_dir = Path(output_dir),
            mirror     = mirror,
        )

    # --- Save output videos ---
    if output_dir:
        out_dir = Path(output_dir)
        out_dir.mkdir(parents=True, exist_ok=True)

        overlay_text, conf = _build_overlay_text(top_idx, probs, idx2label, renderer, top_k)
        color              = _pred_color(conf)

        clean_path = out_dir / f"{stem}_clean.mp4"
        debug_path = out_dir / f"{stem}_debug.mp4"
        vw_clean   = _make_video_writer(clean_path, fps_src, w, h)
        vw_debug   = _make_video_writer(debug_path, fps_src, w, h)

        for i, raw in enumerate(raw_frames):
            # Clean: label overlay only
            clean_frame = renderer.put_text(raw.copy(), overlay_text,
                                            position=(10, 10), color=color)
            vw_clean.write(clean_frame)

            # Debug: re-run MediaPipe on this frame to get landmarks,
            # then draw skeleton + label
            debug_frame = raw.copy()
            extractor.process_frame(debug_frame)       # updates _last_pose/_last_hands
            extractor.draw_landmarks(debug_frame)       # draws skeleton in-place
            debug_frame = renderer.put_text(debug_frame, overlay_text,
                                            position=(10, 10), color=color)
            vw_debug.write(debug_frame)

        vw_clean.release()
        vw_debug.release()
        print(f"[INFO] Saved  clean  → {clean_path}")
        print(f"[INFO] Saved  debug  → {debug_path}")


# ─── Webcam processing ────────────────────────────────────────────────────────

def run_webcam(model, prep, extractor, renderer, idx2label,
               cam_idx=0, top_k=5, log_file=None,
               buffer_sec=2.5, output_dir=None,
               mirror=True, save_features=False):
    """
    Live webcam sign recognition.

    Fix [INFER-C]: time-based buffer (buffer_sec seconds), matching infer.py.
    Fix [MIRROR]:  mirror=True (default) flips frames before MediaPipe so
                   coordinates match the training data orientation.

    Keys:  Q = quit  |  C = clear buffer manually
    """
    cap = cv2.VideoCapture(cam_idx)
    if not cap.isOpened():
        print(f"[ERROR] Cannot open webcam {cam_idx}")
        return

    fps_meta = cap.get(cv2.CAP_PROP_FPS) or 30.0
    w        = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    h        = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

    # Measure ACTUAL processing fps during a short warm-up so buf_frames
    # correctly represents buffer_sec real-world seconds (MediaPipe complexity=2
    # can be slow on CPU, making effective fps much lower than camera fps).
    print(f"[INFO] Measuring actual processing speed (warm-up)...")
    import time
    warmup_times = []
    for _ in range(10):
        ret, frame = cap.read()
        if not ret:
            break
        t0 = time.perf_counter()
        extractor.process_frame(frame, mirror=mirror)
        warmup_times.append(time.perf_counter() - t0)
    actual_fps = 1.0 / (sum(warmup_times) / len(warmup_times)) if warmup_times else fps_meta
    actual_fps = min(actual_fps, fps_meta)   # can't be faster than camera

    buf_frames = max(20, int(actual_fps * buffer_sec))
    print(f"[INFO] Camera fps: {fps_meta:.1f}   Actual processing fps: {actual_fps:.1f}")
    print(f"[INFO] Buffer: {buf_frames} frames = {buffer_sec:.1f}s of real signing")
    print(f"[INFO] Mirror: {'YES (webcam flip enabled — matches training)' if mirror else 'NO (raw webcam, disable if predictions wrong)'}")
    print(f"[INFO] MediaPipe pose complexity: {extractor.model_complexity}")
    print(f"[INFO] Q = quit | C = clear buffer manually")

    # ── Optional video writers ──
    ts_str     = datetime.now().strftime("%Y%m%d_%H%M%S")
    vw_clean   = vw_debug = None
    if output_dir:
        out_dir = Path(output_dir)
        out_dir.mkdir(parents=True, exist_ok=True)
        clean_path = out_dir / f"webcam_{ts_str}_clean.mp4"
        debug_path = out_dir / f"webcam_{ts_str}_debug.mp4"
        vw_clean   = _make_video_writer(clean_path, fps_meta, w, h)
        vw_debug   = _make_video_writer(debug_path, fps_meta, w, h)
        print(f"[INFO] Recording clean → {clean_path}")
        print(f"[INFO] Recording debug → {debug_path}")

    kp_buffer  = []
    last_text  = "Perform a sign — buffer filling..."
    last_color = (200, 200, 200)
    classify_count = 0
    total_frames_seen  = 0
    pose_detected_count = 0

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        # Extract keypoints — always append (Fix [INFER-A])
        kp = extractor.process_frame(frame, mirror=mirror)
        kp_buffer.append(kp)
        total_frames_seen += 1

        # Track whether pose was detected this frame (for status display)
        pose_this_frame = (
            extractor._last_pose is not None and
            extractor._last_pose.pose_landmarks is not None
        )
        if pose_this_frame:
            pose_detected_count += 1
        detect_rate = pose_detected_count / total_frames_seen * 100

        # Fix [INFER-C]: classify when buffer reaches buf_frames, then clear
        if len(kp_buffer) >= buf_frames:
            inputs  = prep(list(kp_buffer), model.model_type)
            logits  = model.predict(inputs)
            probs   = ONNXModel.softmax(logits)
            top_idx = np.argsort(probs)[::-1][:top_k]

            overlay_text, conf = _build_overlay_text(
                top_idx, probs, idx2label, renderer, top_k
            )
            last_text  = overlay_text
            last_color = _pred_color(conf)
            classify_count += 1

            # Console output
            print(f"\n[Prediction #{classify_count}]")
            print(_build_console_block(top_idx, probs, idx2label,
                                       model.model_type, f"webcam frame", top_k))

            # Log file — top_k results
            if log_file:
                _append_log(log_file,
                            _build_log_block(top_idx, probs, idx2label,
                                             f"webcam #{classify_count}", top_k))

            # Feature log
            if save_features and output_dir:
                _save_feature_log(
                    kp_list    = list(kp_buffer),   # already cleared below, snapshot here
                    inputs     = inputs,
                    probs      = probs,
                    top_idx    = top_idx,
                    idx2label  = idx2label,
                    source_name= f"webcam_{ts_str}_#{classify_count:03d}",
                    output_dir = Path(output_dir),
                    mirror     = mirror,
                )

            kp_buffer.clear()   # start fresh for next sign

        # ── Build display frames ──
        # Progress bar towards buf_frames (not max_seq_len)
        fill   = int(len(kp_buffer) / buf_frames * w)
        pct    = len(kp_buffer) / buf_frames * 100

        # Pose detection status: green=detected, red=not detected
        pose_status_text  = f"Pose: {'OK' if pose_this_frame else 'NOT DETECTED'}"
        pose_status_color = (0, 200, 0) if pose_this_frame else (0, 0, 255)
        detect_rate_text  = f"Detect rate: {detect_rate:.0f}%"

        # Clean frame: label + progress bar + pose status
        clean_frame = renderer.put_text(frame.copy(), last_text,
                                        position=(10, 10), color=last_color)
        cv2.putText(clean_frame, pose_status_text,
                    (10, 55), cv2.FONT_HERSHEY_SIMPLEX, 0.6, pose_status_color, 2)
        cv2.putText(clean_frame, detect_rate_text,
                    (10, 80), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (180, 180, 180), 1)
        cv2.rectangle(clean_frame, (0, h - 12), (fill, h), (0, 180, 255), -1)
        cv2.putText(clean_frame, f"buf {len(kp_buffer)}/{buf_frames}  ({pct:.0f}%)",
                    (10, h - 15), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (220, 220, 220), 1)

        # Debug frame: skeleton + label + pose status + progress bar
        debug_frame = frame.copy()
        extractor.draw_landmarks(debug_frame)
        debug_frame = renderer.put_text(debug_frame, last_text,
                                        position=(10, 10), color=last_color)
        cv2.putText(debug_frame, pose_status_text,
                    (10, 55), cv2.FONT_HERSHEY_SIMPLEX, 0.6, pose_status_color, 2)
        cv2.putText(debug_frame, detect_rate_text,
                    (10, 80), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (180, 180, 180), 1)
        cv2.rectangle(debug_frame, (0, h - 12), (fill, h), (0, 180, 255), -1)
        cv2.putText(debug_frame, f"buf {len(kp_buffer)}/{buf_frames}  ({pct:.0f}%)",
                    (10, h - 15), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (220, 220, 220), 1)

        # Write to video files
        if vw_clean: vw_clean.write(clean_frame)
        if vw_debug: vw_debug.write(debug_frame)

        # Show the debug view on screen (most useful for real-time feedback)
        cv2.imshow("MSL Recognition  [Q=quit  C=clear]", debug_frame)

        key = cv2.waitKey(1) & 0xFF
        if key == ord("q"):
            break
        elif key == ord("c"):
            kp_buffer.clear()
            last_text  = "Buffer cleared — perform a sign"
            last_color = (200, 200, 200)
            print("[INFO] Buffer manually cleared.")

    cap.release()
    if vw_clean: vw_clean.release()
    if vw_debug: vw_debug.release()
    cv2.destroyAllWindows()

    print(f"\n[INFO] Webcam session summary:")
    print(f"  Total frames     : {total_frames_seen}")
    print(f"  Pose detected    : {pose_detected_count}  ({pose_detected_count/max(total_frames_seen,1)*100:.0f}%)")
    print(f"  Classifications  : {classify_count}")
    if total_frames_seen > 0 and pose_detected_count / total_frames_seen < 0.3:
        print(f"  [WARNING] Low detection rate ({pose_detected_count/total_frames_seen*100:.0f}%).")
        print(f"            Tips: move closer, improve lighting, ensure upper body is visible.")
    if output_dir:
        print(f"  Saved  clean  → {clean_path}")
        print(f"  Saved  debug  → {debug_path}")


# ─── CLI ──────────────────────────────────────────────────────────────────────

def main():
    parser = argparse.ArgumentParser(
        description="MSL Sign Language Inference — ONNX Runtime",
        formatter_class=argparse.RawDescriptionHelpFormatter,
        epilog="""
Examples:
  # Single video, save both clean and debug outputs
  python infer_onnx.py --onnx_model onnx_models/bilstm_msl.onnx \\
      --video sample.mp4 --top_k 5 --output_dir predictions_output

  # Webcam, 3-second buffer, save recordings + log
  python infer_onnx.py --onnx_model onnx_models/transformer_msl.onnx \\
      --webcam 0 --buffer_sec 3.0 --top_k 5 \\
      --output_dir predictions_output --log_file webcam_log.txt
""")

    parser.add_argument("--onnx_model",   required=True,
                        help="Path to .onnx model file (NOT the .json metadata)")
    parser.add_argument("--metadata",     default=None,
                        help="Path to .json metadata (default: same stem as .onnx)")
    parser.add_argument("--label_map",    default=None,
                        help="Path to label_map.json (default: next to .onnx)")
    parser.add_argument("--output_dir",   default="predictions_output",
                        help="Directory for saved videos (default: predictions_output/)")
    parser.add_argument("--log_file",     default=None,
                        help="Append top-k results to this text file")
    parser.add_argument("--top_k",        type=int, default=5,
                        help="Number of top predictions to show/log (default: 5)")
    parser.add_argument("--font_size",    type=int, default=32,
                        help="Myanmar font size for overlay (default: 32)")
    parser.add_argument("--model_complexity", type=int, default=None,
                        help="MediaPipe pose complexity: 0/1/2. "
                             "Default: 2 for video (matches training), 1 for webcam.")

    grp = parser.add_mutually_exclusive_group(required=True)
    grp.add_argument("--video",  help="Path to a video file")
    grp.add_argument("--webcam", type=int, metavar="CAM_IDX",
                     help="Webcam device index (0 = default camera)")

    parser.add_argument("--buffer_sec", type=float, default=2.5,
                        help="[Webcam] Seconds of frames to accumulate before "
                             "each classification (default: 2.5, same as infer.py)")
    parser.add_argument("--debug",       action="store_true",
                        help="Print extra diagnostic information")
    args = parser.parse_args()

    # ── Validate ONNX path ────────────────────────────────────────────────────
    model_p = Path(args.onnx_model)
    if model_p.suffix.lower() == ".json":
        sys.exit(
            f"[ERROR] --onnx_model must be a .onnx file, not a .json file.\n"
            f"        You passed : {model_p}\n"
            f"        Did you mean: {model_p.with_suffix('.onnx')}"
        )
    if not model_p.exists():
        sys.exit(f"[ERROR] ONNX model not found: {model_p}")

    meta_p = args.metadata or str(model_p.with_suffix(".json"))
    lmap_p = args.label_map or str(model_p.parent / "label_map.json")

    if not Path(lmap_p).exists():
        sys.exit(f"[ERROR] label_map.json not found: {lmap_p}\n"
                 f"        Copy it from your Linux server alongside the .onnx file.")

    # ── Load resources ────────────────────────────────────────────────────────
    idx2label = load_idx2label(lmap_p)
    print(f"[INFO] Loaded {len(idx2label)} labels from label map.")

    model    = ONNXModel(str(model_p), meta_p)
    prep     = Preprocessor(max_seq_len=model.max_seq_len)
    renderer = TextRenderer(font_size=args.font_size)

    if args.log_file:
        print(f"[INFO] Logging top-{args.top_k} results to: {args.log_file}")
    if args.output_dir:
        print(f"[INFO] Saving output videos to: {args.output_dir}/")

    # Fix [INFER-F] + webcam detection fix:
    # complexity=2 (heavy model) for BOTH video and webcam.
    # Webcam needs the heavy model — the lite model (complexity=1) failed to detect
    # any pose landmarks in real webcam conditions (0% detection rate confirmed
    # by comparing debug videos: skeleton drawn 100% on video file, 0% on webcam).
    # The heavy model is more robust to webcam image quality, lighting variation,
    # and different distances. Slower, but actually detects the signer.
    #
    # min_detection_confidence lowered to 0.3 for webcam: live conditions are
    # harder than pre-recorded videos, so a more permissive threshold is needed.
    # Video keeps 0.5 to match training (extract_keypoints.py default).
    if args.model_complexity is not None:
        video_complexity  = args.model_complexity
        webcam_complexity = args.model_complexity
    else:
        video_complexity  = 2   # matches training (extract_keypoints.py --complexity 2)
        webcam_complexity = 2   # FIXED: was 1 (lite), now 2 (heavy) — see note above

    try:
        if args.video:
            extractor = KeypointExtractor(
                model_complexity=video_complexity,
                min_detection_confidence=0.5,   # matches training
                min_tracking_confidence=0.5,
            )
            try:
                run_video(model, prep, extractor, renderer, args.video,
                          top_k      = args.top_k,
                          log_file   = args.log_file,
                          idx2label  = idx2label,
                          output_dir = args.output_dir,
                          debug      = args.debug)
            finally:
                extractor.close()
        else:
            extractor = KeypointExtractor(
                model_complexity=webcam_complexity,
                min_detection_confidence=0.3,   # lower threshold for live webcam conditions
                min_tracking_confidence=0.3,
            )
            try:
                run_webcam(model, prep, extractor, renderer, idx2label,
                           cam_idx    = args.webcam,
                           top_k      = args.top_k,
                           log_file   = args.log_file,
                           buffer_sec = args.buffer_sec,
                           output_dir = args.output_dir)
            finally:
                extractor.close()
    finally:
        print("[INFO] Done.")


if __name__ == "__main__":
    main()

```

## Testing Failed Again!

## Debugging Log  

```
Let me check what's left to update in the file.Good. Now I have the full picture. I need to: (1) add `--mirror` / `--no_mirror` and `--save_features` CLI args, (2) wire them into both `run_video` and `run_webcam` calls, and (3) update the epilog with new examples.Now wire `mirror` and `save_features` into both `run_video` and `run_webcam` calls in `main()`:Now update the epilog examples to show the new flags:Now verify everything is correct:Here is a complete explanation of everything done and why.

---

## Root-cause analysis of the webcam failures

Comparing the two debug videos frame-by-frame with pixel-level analysis revealed the following:

**What the video analysis actually showed:**

| | `idx20-30_debug.mp4` | `webcam_..._debug.mp4` |
|---|---|---|
| MediaPipe skeleton drawn | 100% of frames | 0% of frames |
| Skeleton pixel color found | red landmarks + green connections | only text overlay pixels |
| Max confidence (BiLSTM) | **80.73%** | **4.58%** |

Zero skeleton in 575 webcam frames while a person fills 95% of the frame. Even after the `complexity=2` fix, the current webcam results still show 4–5% confidence on BiLSTM/Transformer — which is far above pure random (0.18% for 556 classes) but still wrong. This pattern — STGCN at 29% while BiLSTM/Transformer at 4% — is the definitive signature of features that exist but have the **wrong spatial orientation**. STGCN's BatchNorm partially compensates for distribution shifts; LayerNorm cannot.

**Three fixes applied:**

---

### Fix 1 — Mirror flip (`--mirror` / `--no_mirror`, default ON for webcam)

Most laptop webcams output a **mirrored (selfie-view)** image. Training videos were recorded with a normal camera, un-mirrored. This means in the webcam image, the signer's left shoulder appears on the right side of the frame and vice versa.

After `normalize_sequence()`, the left shoulder should always be at **x ≈ +0.5** (signer's left = positive x in training). With a mirrored webcam:
- Left shoulder appears at x ≈ 0.35–0.40 instead of 0.55–0.65
- After normalization: left shoulder x ≈ **−0.5** instead of **+0.5**
- Every x-coordinate is spatially flipped
- Hand slots (33 = left hand, 54 = right hand) are also assigned to wrong positions

The fix: `cv2.flip(frame, 1)` **before** passing to MediaPipe — only for the MediaPipe call, not the display frame. The displayed video is NOT flipped (so the user still sees a normal mirror view of themselves), but the keypoint coordinates are extracted from the un-mirrored orientation matching training.

---

### Fix 2 — Actual FPS measurement for buffer timing

MediaPipe complexity=2 on a Windows CPU takes ~0.1s per frame, giving an actual processing rate of ~10 fps. But `cap.get(CAP_PROP_FPS)` always returns 30. With the old code, `buf_frames = int(30 × 2.5) = 75` frames takes 7.5 real-world seconds to fill at 10fps. The user would sign for 2 seconds, then be idle for 5.5 seconds — the model receives mostly a resting pose, not the actual sign.

Fix: a 10-frame warm-up loop at startup measures actual per-frame processing time. `buf_frames` is then calculated from the real fps, so the buffer always represents exactly `buffer_sec` seconds of real signing time regardless of hardware speed.

---

### Fix 3 — Keypoint feature logging (`--save_features`)

Every classification now optionally saves to `output_dir/features/`:
- `*_raw_kp.npy` — raw (un-normalized) keypoints (T, 75, 3)
- `*_model_input.npy` — the exact array fed to the ONNX model
- `*_features.txt` — human-readable diagnostics including:
  - Left/right shoulder positions in image space with expected range
  - Shoulder width (normalization scale) — near-zero means detection failed
  - Normalized shoulder positions with "EXPECTED +0.5 / −0.5" hint
  - Whether values are **reversed** (→ mirror needed)
  - Per-axis statistics of all normalized joints
  - Hand detection rate per frame

**How to use it to diagnose the webcam:**

```bash
# Step 1: generate feature log from a known-good video
python infer_onnx.py --onnx_model onnx_models\bilstm_msl.onnx \
    --video sample4inference\idx20-30.mp4 --save_features \
    --output_dir predictions_output

# Step 2: generate feature log from webcam (default: --mirror ON)
python infer_onnx.py --onnx_model onnx_models\bilstm_msl.onnx \
    --webcam 0 --save_features \
    --output_dir predictions_output --log_file webcam_log.txt

# Step 3: open both _features.txt files and compare:
# - "Left shoulder norm x": video should be ~+0.5, webcam should also be ~+0.5
#   If webcam shows ~-0.5 → mirror is wrong direction → try --no_mirror
# - "Shoulder width": should be 0.15–0.50 on both. Near 0 → not detected.
# - Hand active frames: should be >50% during active signing.
```

In [103]:
from IPython.display import Markdown

file_path = './src/infer_onnx.py'

with open(file_path, 'r') as f:
    code = f.read()

# 'python' keyword triggers the syntax highlighter
display(Markdown(f"```python\n{code}\n```"))

```python
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
infer_onnx.py  –  Real-time MSL recognition on Windows using ONNX Runtime

Bugs fixed in this version
───────────────────────────
  [INFER-A] process_frame skipped frames without pose detection.
      Training always includes every frame (zeros for undetected joints).
      Fix: process_frame always returns the keypoint array, never None.

  [INFER-B] normalize_sequence used 2-D scale and left z unscaled.
      Fix: exact mirror of extract_keypoints.normalize_sequence() —
      full 3-D midpoint and 3-D Euclidean shoulder width, all axes.

  [INFER-C] *** ROOT CAUSE OF WEBCAM FAILURE ***
      Webcam classified every classify_every=15 frames = 0.5 seconds.
      Signs take 1–4 seconds; feeding 15 frames to a model trained on
      full-length signs produces garbage predictions.

      infer.py uses a TIME-BASED buffer: accumulate buffer_sec=2.5 sec
      of frames (~75 at 30 fps), then classify and clear.  This gives
      the model a realistic-duration sign sequence matching training.

      Fix: replaced the frame-count trigger with a time-based trigger
      using --buffer_sec (default 2.5), identical to infer.py.

  [INFER-D] np.int64 / int dict key mismatch.
      Fix: always cast to int() before idx2label lookup.

  [INFER-E] ONNXModel.predict returned a scalar instead of 1-D array.
      Fix: return output[0] — shape (num_classes,).

  [INFER-F] model_complexity=1 everywhere; training used complexity=2.
      Fix: complexity=2 for video (matching training), 1 for webcam
      (speed tradeoff, same as infer.py).

  [INFER-G] extractor.pose.model_complexity AttributeError on Windows.
      MediaPipe's Pose object does not expose model_complexity publicly.
      Fix: stored as self.model_complexity on KeypointExtractor.

New features
────────────
  • Video saving — two output files per inference:
      {name}_clean.mp4  —  label/confidence overlay only
      {name}_debug.mp4  —  label overlay + MediaPipe skeleton lines
    Works for both --video and --webcam modes.
    Saved to --output_dir (default: predictions_output/).

  • Webcam log — every classification event writes top-k results to the
    log file with timestamp and separator, matching the video log format.
"""

import argparse
import importlib
import json
import sys
from datetime import datetime
from pathlib import Path

import cv2
import numpy as np
import onnxruntime as ort

try:
    from PIL import Image, ImageDraw, ImageFont
    HAS_PIL = True
except ImportError:
    HAS_PIL = False

# ── Constants (must match extract_keypoints.py exactly) ───────────────────────
POSE_N          = 33
HAND_N          = 21
TOTAL_JOINTS    = POSE_N + HAND_N * 2   # 75
COORDS          = 3
FEAT_DIM        = TOTAL_JOINTS * COORDS  # 225

POSE_LEFT_SHOULDER  = 11
POSE_RIGHT_SHOULDER = 12
LEFT_HAND_START     = POSE_N            # 33
RIGHT_HAND_START    = POSE_N + HAND_N   # 54


# ─── MediaPipe import ─────────────────────────────────────────────────────────

def _import_mp():
    for prefix in ["mediapipe.solutions", "mediapipe.python.solutions"]:
        try:
            pose  = importlib.import_module(f"{prefix}.pose")
            hands = importlib.import_module(f"{prefix}.hands")
            draw  = importlib.import_module(f"{prefix}.drawing_utils")
            return pose, hands, draw
        except (ImportError, ModuleNotFoundError):
            continue
    try:
        import mediapipe as _mp
        return _mp.solutions.pose, _mp.solutions.hands, _mp.solutions.drawing_utils
    except AttributeError:
        pass
    raise ImportError(
        "Cannot import MediaPipe solutions.\n"
        "Try: pip uninstall mediapipe -y && pip install mediapipe==0.10.18"
    )

mp_pose, mp_hands, mp_drawing = _import_mp()


# ─── Myanmar text rendering ──────────────────────────────────────────────────

class TextRenderer:
    _FONT_CANDIDATES = [
        "C:\\Windows\\Fonts\\Padauk.ttf",
        "C:\\Windows\\Fonts\\NotoSansMyanmar-Regular.ttf",
        "C:\\Windows\\Fonts\\NotoSansMyanmar.ttf",
        "C:\\Windows\\Fonts\\myanmar3.ttf",
        "/usr/share/fonts/truetype/padauk/Padauk-Regular.ttf",
        "/usr/share/fonts/truetype/noto/NotoSansMyanmar-Regular.ttf",
        "fonts\\Padauk.ttf",
        "fonts/NotoSansMyanmar-Regular.ttf",
    ]

    def __init__(self, font_size: int = 32):
        self.font = self._load_font(font_size)
        self.can_render_myanmar = (self.font is not None)
        if not self.can_render_myanmar:
            print()
            print("  ╔═══════════════════════════════════════════════════════════╗")
            print("  ║  INFO: No Myanmar shaping font found.                    ║")
            print("  ║  Displaying readable Class IDs instead of Myanmar text.  ║")
            print("  ╚═══════════════════════════════════════════════════════════╝")
            print()

    def _load_font(self, size):
        if not HAS_PIL:
            return None
        for path in self._FONT_CANDIDATES:
            if Path(path).exists():
                try:
                    return ImageFont.truetype(path, size)
                except Exception:
                    continue
        return None

    def put_text(self, frame, text, position=(20, 20), color=(0, 255, 0)):
        if self.font and HAS_PIL:
            pil = Image.fromarray(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
            d   = ImageDraw.Draw(pil)
            r, g, b = color[2], color[1], color[0]
            for dx in [-1, 0, 1]:
                for dy in [-1, 0, 1]:
                    if dx or dy:
                        d.text((position[0]+dx, position[1]+dy), text,
                               font=self.font, fill=(0, 0, 0))
            d.text(position, text, font=self.font, fill=(r, g, b))
            return cv2.cvtColor(np.array(pil), cv2.COLOR_RGB2BGR)
        cv2.putText(frame, text, position, cv2.FONT_HERSHEY_SIMPLEX,
                    0.75, color, 2, cv2.LINE_AA)
        return frame


# ─── Keypoint extraction ──────────────────────────────────────────────────────

class KeypointExtractor:
    """
    Extracts (TOTAL_JOINTS, 3) keypoint arrays from BGR frames.

    Fix [INFER-G]: model_complexity is stored as self.model_complexity
    because mediapipe's Pose object does not expose it as a public attribute
    on all Windows builds (AttributeError otherwise).

    Fix [INFER-A]: process_frame always returns an array, never None.
    Matches extract_keypoints.py which includes every frame, using zeros
    for any joint that MediaPipe failed to detect.

    Hand convention (matches extract_keypoints.py):
      MediaPipe reports handedness from the camera perspective (mirrored).
      label='Right' → signer's LEFT  hand → slot LEFT_HAND_START  (33).
      label='Left'  → signer's RIGHT hand → slot RIGHT_HAND_START (54).
    """

    def __init__(self, model_complexity: int = 2,
                 min_detection_confidence: float = 0.5,
                 min_tracking_confidence:  float = 0.5):
        # Fix [INFER-G]: store complexity ourselves — mediapipe's Pose object
        # does NOT expose a public .model_complexity on all Windows builds.
        self.model_complexity = model_complexity

        self.pose  = mp_pose.Pose(
            static_image_mode=False,
            model_complexity=model_complexity,
            smooth_landmarks=True,
            enable_segmentation=False,
            min_detection_confidence=min_detection_confidence,
            min_tracking_confidence=min_tracking_confidence,
        )
        self.hands = mp_hands.Hands(
            static_image_mode=False,
            max_num_hands=2,
            model_complexity=min(model_complexity, 1),  # Hands caps at 1
            min_detection_confidence=min_detection_confidence,
            min_tracking_confidence=min_tracking_confidence,
        )
        self._last_pose  = None
        self._last_hands = None

    def process_frame(self, frame: np.ndarray,
                      mirror: bool = False) -> np.ndarray:
        """
        Returns (TOTAL_JOINTS, 3) float32 for every frame.
        Zeros for any undetected joint/hand — never returns None.

        Args:
            frame:  BGR frame from cv2.VideoCapture.
            mirror: If True, flip the frame horizontally before processing.
                    Many laptop/webcam cameras output a mirrored (selfie-view)
                    image.  Training videos were likely recorded un-mirrored
                    (normal camera view).  Setting mirror=True corrects this so
                    the normalised coordinates match the training distribution.
                    The DISPLAYED frame is NOT flipped — only the frame sent to
                    MediaPipe is flipped, so landmark positions are in the
                    coordinate space of the un-mirrored view.
        """
        if mirror:
            frame = cv2.flip(frame, 1)   # horizontal flip only for MediaPipe
        rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        rgb.flags.writeable = False
        p_res = self.pose.process(rgb)
        h_res = self.hands.process(rgb)
        self._last_pose  = p_res
        self._last_hands = h_res

        kp = np.zeros((TOTAL_JOINTS, COORDS), dtype=np.float32)

        if p_res.pose_landmarks:
            for i, lm in enumerate(p_res.pose_landmarks.landmark):
                kp[i] = [lm.x, lm.y, lm.z]

        if h_res.multi_hand_landmarks and h_res.multi_handedness:
            for lm_list, handedness in zip(
                h_res.multi_hand_landmarks, h_res.multi_handedness
            ):
                label  = handedness.classification[0].label
                offset = LEFT_HAND_START if label == 'Right' else RIGHT_HAND_START
                for i, lm in enumerate(lm_list.landmark):
                    kp[offset + i] = [lm.x, lm.y, lm.z]

        return kp  # always an array, never None

    def draw_landmarks(self, frame: np.ndarray) -> np.ndarray:
        """Draw MediaPipe skeleton onto frame IN-PLACE and return it."""
        if self._last_pose and self._last_pose.pose_landmarks:
            mp_drawing.draw_landmarks(
                frame, self._last_pose.pose_landmarks, mp_pose.POSE_CONNECTIONS
            )
        if self._last_hands and self._last_hands.multi_hand_landmarks:
            for lm in self._last_hands.multi_hand_landmarks:
                mp_drawing.draw_landmarks(frame, lm, mp_hands.HAND_CONNECTIONS)
        return frame

    def close(self):
        self.pose.close()
        self.hands.close()


# ─── Normalization ────────────────────────────────────────────────────────────

def normalize_sequence(seq: np.ndarray) -> np.ndarray:
    """
    Fix [INFER-B]: exact mirror of extract_keypoints.normalize_sequence().
    Full 3-D shoulder-midpoint subtraction and 3-D shoulder-width scaling,
    applied to all three coordinate axes.
    """
    l_sh = seq[:, POSE_LEFT_SHOULDER,  :]   # (T, 3)
    r_sh = seq[:, POSE_RIGHT_SHOULDER, :]   # (T, 3)
    mid  = (l_sh + r_sh) / 2.0
    dist = np.linalg.norm(r_sh - l_sh, axis=1, keepdims=True)
    dist = np.where(dist < 1e-6, 1.0, dist)
    return ((seq - mid[:, None, :]) / dist[:, None, :]).astype(np.float32)


# ─── Preprocessor ─────────────────────────────────────────────────────────────

class Preprocessor:
    """
    Stack raw (75, 3) keypoint frames → normalize → crop/pad → model input.

    Processing order matches infer.py exactly:
      1. Stack   → (T, 75, 3)
      2. Normalize (3-D shoulder-centred, full 3-D scale)
      3. Centre-crop  if T > max_seq_len
      4. Zero-pad     if T < max_seq_len
      5. Build mask   (True = padded position)
      6. Reshape for model type
    """

    def __init__(self, max_seq_len: int = 200):
        self.max_seq_len = max_seq_len

    def __call__(self, kp_list: list, model_type: str) -> dict:
        if not kp_list:
            return {}

        seq = np.stack(kp_list, axis=0).astype(np.float32)  # (T, 75, 3)
        seq = normalize_sequence(seq)
        T   = seq.shape[0]

        if T > self.max_seq_len:
            start = (T - self.max_seq_len) // 2
            seq   = seq[start: start + self.max_seq_len]
            T     = self.max_seq_len

        real_len = T

        if T < self.max_seq_len:
            pad = np.zeros((self.max_seq_len - T, TOTAL_JOINTS, COORDS), dtype=np.float32)
            seq = np.concatenate([seq, pad], axis=0)

        mask = np.zeros(self.max_seq_len, dtype=np.bool_)
        mask[real_len:] = True

        if model_type == "stgcn":
            kp_out = np.expand_dims(seq.transpose(2, 0, 1), axis=0).astype(np.float32)
            return {"keypoints": kp_out}
        else:
            kp_out = np.expand_dims(
                seq.reshape(self.max_seq_len, FEAT_DIM), axis=0
            ).astype(np.float32)
            return {"keypoints": kp_out, "mask": mask[np.newaxis]}


# ─── ONNX model wrapper ───────────────────────────────────────────────────────

class ONNXModel:
    def __init__(self, onnx_path: str, meta_path: str = None):
        available = ort.get_available_providers()
        preferred = ["CUDAExecutionProvider", "DmlExecutionProvider", "CPUExecutionProvider"]
        providers = [p for p in preferred if p in available]

        self.session = ort.InferenceSession(onnx_path, providers=providers)
        print(f"[INFO] ONNX loaded: {Path(onnx_path).name}"
              f"  (provider: {self.session.get_providers()[0]})")

        self.meta = {}
        if meta_path and Path(meta_path).exists():
            try:
                with open(meta_path, encoding="utf-8") as f:
                    self.meta = json.load(f)
            except Exception as e:
                print(f"[WARNING] Could not read metadata: {e}")
        else:
            print(f"[INFO] Metadata not found at '{meta_path}', using defaults.")

        self.model_type  = self.meta.get("model_type",  "transformer")
        self.max_seq_len = self.meta.get("max_seq_len", 200)
        self.num_classes = self.meta.get("num_classes", 558)

    def predict(self, inputs: dict) -> np.ndarray:
        """Fix [INFER-E]: return (num_classes,) not scalar."""
        return self.session.run(None, inputs)[0][0]

    @staticmethod
    def softmax(logits: np.ndarray) -> np.ndarray:
        e = np.exp(logits - logits.max())
        return e / e.sum()


# ─── Label map ────────────────────────────────────────────────────────────────

def load_idx2label(path: str) -> dict:
    with open(path, encoding="utf-8") as f:
        data = json.load(f)
    return {int(k): v for k, v in data.get("idx2label", {}).items()}


def _get_label(idx2label: dict, class_id) -> str:
    """Fix [INFER-D]: always cast to int() before dict lookup."""
    return idx2label.get(int(class_id), f"[class {int(class_id)}]")


# ─── Helpers ──────────────────────────────────────────────────────────────────

def _overlay_text(renderer, frame, text, color):
    """Put prediction text at top-left of frame."""
    return renderer.put_text(frame, text, position=(10, 10), color=color)


def _pred_color(confidence: float) -> tuple:
    """Green ≥80%, yellow ≥50%, red otherwise."""
    if confidence >= 0.80: return (0, 200,   0)
    if confidence >= 0.50: return (0, 200, 200)
    return (0, 0, 255)


def _build_overlay_text(top_idx, probs, idx2label, renderer, top_k):
    """Single-line text for the video overlay (top-1 label + confidence)."""
    best = top_idx[0]
    conf = probs[int(best)]
    lbl  = _get_label(idx2label, best) if idx2label else f"Class {int(best)}"
    if renderer.can_render_myanmar:
        return f"{lbl}  ({conf*100:.1f}%)", conf
    return f"Class {int(best)}  ({conf*100:.1f}%)", conf


def _build_console_block(top_idx, probs, idx2label, model_type, source_name, top_k):
    """Multi-line console output block, same format as run_video."""
    sep   = "─" * 60
    lines = [f"Source : {source_name}", f"Model  : {model_type.upper()}", sep]
    for rank, idx in enumerate(top_idx, 1):
        lbl = _get_label(idx2label, idx) if idx2label else f"[class {int(idx)}]"
        lines.append(f"  Top-{rank}: {lbl}  ({probs[int(idx)]*100:.2f}%)")
    lines.append(sep)
    return "\n".join(lines)


def _build_log_block(top_idx, probs, idx2label, source_name, top_k):
    """Log block with timestamp and top-k results."""
    ts    = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    sep   = "=" * 60
    lines = [sep, f"[{ts}]  {source_name}"]
    for rank, idx in enumerate(top_idx, 1):
        lbl = _get_label(idx2label, idx) if idx2label else "?"
        lbl = lbl.encode("utf-8", errors="replace").decode("utf-8")
        lines.append(f"  Top-{rank:>2}: Class {int(idx):>4d} | "
                     f"{lbl:<45s} | {probs[int(idx)]*100:6.2f}%")
    lines.append("")
    return "\n".join(lines) + "\n"


def _append_log(path, text):
    try:
        with open(path, "a", encoding="utf-8-sig") as f:
            f.write(text)
    except Exception as e:
        print(f"[WARNING] Could not write log: {e}")


def _make_video_writer(output_path: Path, fps: float, width: int, height: int):
    """Create a cv2.VideoWriter with MP4V codec."""
    fourcc = cv2.VideoWriter_fourcc(*"mp4v")
    return cv2.VideoWriter(str(output_path), fourcc, fps, (width, height))


def _save_feature_log(kp_list, inputs, probs, top_idx, idx2label,
                      source_name, output_dir, mirror):
    """
    Save full diagnostic information for one classification event.

    Creates two files in output_dir/features/:
      {source_name}_raw_kp.npy      — raw (un-normalized) stacked keypoints (T,75,3)
      {source_name}_model_input.npy — the actual array fed to the ONNX model
      {source_name}_features.txt    — human-readable statistics for debugging

    The .txt file contains:
      - Shoulder positions and detection status (are they non-zero?)
      - Shoulder width (normalization scale — should be ~0.3-0.7 if detected)
      - Per-axis statistics of normalized features
      - Fraction of frames with active hand movement
      - Mirror setting used
      - Top-k classification results
      - Comparison hint vs expected training distribution
    """
    import os
    feat_dir = output_dir / "features"
    feat_dir.mkdir(parents=True, exist_ok=True)

    # Raw keypoints before normalization
    raw_kp = np.stack(kp_list, axis=0).astype(np.float32)   # (T, 75, 3)
    T      = raw_kp.shape[0]

    # Save .npy files
    np.save(str(feat_dir / f"{source_name}_raw_kp.npy"), raw_kp)

    # Also save the model input (already normalized, cropped, padded)
    model_input_key = "keypoints"
    if model_input_key in inputs:
        np.save(str(feat_dir / f"{source_name}_model_input.npy"),
                inputs[model_input_key])

    # --- Build text report ---
    L_SH, R_SH = 11, 12
    l_sh = raw_kp[:, L_SH, :]   # (T, 3)
    r_sh = raw_kp[:, R_SH, :]   # (T, 3)

    # How many frames had shoulders detected (non-zero)?
    l_detected = (np.abs(l_sh).sum(axis=1) > 1e-6).sum()
    r_detected = (np.abs(r_sh).sum(axis=1) > 1e-6).sum()

    # Shoulder width distribution (normalization scale)
    dist = np.linalg.norm(r_sh - l_sh, axis=1)  # (T,)
    dist_nonzero = dist[dist > 1e-6]

    # Hand activity: fraction of frames where hands are non-zero
    lh_block = raw_kp[:, 33:54, :]
    rh_block = raw_kp[:, 54:75, :]
    lh_active = (np.abs(lh_block).sum(axis=(1,2)) > 1e-6).sum()
    rh_active = (np.abs(rh_block).sum(axis=(1,2)) > 1e-6).sum()

    # Normalized keypoints for distribution check
    norm_kp = normalize_sequence(raw_kp)
    # For a correctly framed frontal signer:
    #   Left  shoulder (idx 11) normalized x should be ~ +0.4 to +0.6
    #   Right shoulder (idx 12) normalized x should be ~ -0.6 to -0.4
    l_sh_norm = norm_kp[:, L_SH, :]
    r_sh_norm = norm_kp[:, R_SH, :]

    lines = [
        "=" * 65,
        f"Feature Diagnostic Log — {source_name}",
        f"Timestamp : {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}",
        f"Mirror    : {'YES — webcam flipped before MediaPipe' if mirror else 'NO — raw frame used'}",
        "=" * 65,
        "",
        "── Sequence info ───────────────────────────────────────────",
        f"  Total frames         : {T}",
        f"  Left  shoulder detected: {l_detected}/{T} frames ({l_detected/T*100:.0f}%)",
        f"  Right shoulder detected: {r_detected}/{T} frames ({r_detected/T*100:.0f}%)",
        f"  Left  hand active    : {lh_active}/{T} frames ({lh_active/T*100:.0f}%)",
        f"  Right hand active    : {rh_active}/{T} frames ({rh_active/T*100:.0f}%)",
        "",
        "── Shoulder positions (RAW, image space [0,1]) ─────────────",
        f"  Left  shoulder x mean : {l_sh[:,0].mean():.4f}  std={l_sh[:,0].std():.4f}",
        f"  Right shoulder x mean : {r_sh[:,0].mean():.4f}  std={r_sh[:,0].std():.4f}",
        f"  Left  shoulder y mean : {l_sh[:,1].mean():.4f}",
        f"  Right shoulder y mean : {r_sh[:,1].mean():.4f}",
        "",
        "  EXPECTED for frontal un-mirrored signer:",
        "    Left  shoulder x ~ 0.55–0.65 (camera right = signer left)",
        "    Right shoulder x ~ 0.35–0.45 (camera left  = signer right)",
        "  If values are REVERSED → webcam is mirrored → use --mirror",
        "",
        "── Shoulder width (normalization scale) ────────────────────",
    ]
    if len(dist_nonzero) > 0:
        lines += [
            f"  Shoulder width mean : {dist_nonzero.mean():.4f}",
            f"  Shoulder width std  : {dist_nonzero.std():.4f}",
            f"  Shoulder width range: {dist_nonzero.min():.4f} – {dist_nonzero.max():.4f}",
            "  EXPECTED: 0.15–0.50 (in normalized image coords)",
            "  If near 0 → shoulders not detected → normalization fails",
        ]
    else:
        lines.append("  !! Shoulder width is ZERO in ALL frames — shoulders NOT detected!")
        lines.append("     Normalization falls back to dist=1.0 → features NOT normalized correctly.")

    lines += [
        "",
        "── Normalized shoulder positions (after normalize_sequence) ─",
        f"  Left  shoulder norm x mean : {l_sh_norm[:,0].mean():.4f}",
        f"  Right shoulder norm x mean : {r_sh_norm[:,0].mean():.4f}",
        "  EXPECTED after correct normalization:",
        "    Left  shoulder norm x ~  +0.5  (signer's left = positive x)",
        "    Right shoulder norm x ~  -0.5  (signer's right = negative x)",
        "  If REVERSED (left~-0.5, right~+0.5) → mirror flip needed",
        "",
        "── Full keypoint distribution (normalized) ──────────────────",
        f"  All joints x: mean={norm_kp[:,:,0].mean():.4f}  std={norm_kp[:,:,0].std():.4f}"
        f"  min={norm_kp[:,:,0].min():.4f}  max={norm_kp[:,:,0].max():.4f}",
        f"  All joints y: mean={norm_kp[:,:,1].mean():.4f}  std={norm_kp[:,:,1].std():.4f}"
        f"  min={norm_kp[:,:,1].min():.4f}  max={norm_kp[:,:,1].max():.4f}",
        f"  All joints z: mean={norm_kp[:,:,2].mean():.4f}  std={norm_kp[:,:,2].std():.4f}"
        f"  min={norm_kp[:,:,2].min():.4f}  max={norm_kp[:,:,2].max():.4f}",
        "",
        "── Classification results ───────────────────────────────────",
    ]
    for rank, idx in enumerate(top_idx, 1):
        lbl = _get_label(idx2label, idx) if idx2label else "?"
        lbl_safe = lbl.encode("utf-8", errors="replace").decode("utf-8")
        lines.append(f"  Top-{rank}: Class {int(idx):>4d} | {lbl_safe:<50s} | {probs[int(idx)]*100:.2f}%")

    lines += [
        "",
        "── Files saved ──────────────────────────────────────────────",
        f"  Raw keypoints : features/{source_name}_raw_kp.npy      shape=({T},75,3)",
        f"  Model input   : features/{source_name}_model_input.npy",
        "=" * 65,
        "",
    ]

    txt_path = feat_dir / f"{source_name}_features.txt"
    with open(txt_path, "w", encoding="utf-8-sig") as f:
        f.write("\n".join(lines))
    print(f"[INFO] Feature log → {txt_path}")


# ─── Video processing ─────────────────────────────────────────────────────────

def run_video(model, prep, extractor, renderer, video_path,
              top_k=5, log_file=None, idx2label=None,
              output_dir=None, debug=False, mirror=False,
              save_features=False):
    """
    Process a single video file.  Saves two output videos:
      {stem}_clean.mp4  —  label overlay only
      {stem}_debug.mp4  —  label overlay + MediaPipe skeleton lines

    Matches infer.py / process_video() exactly:
      every frame included (zeros for undetected joints),
      normalize after stacking, centre-crop or zero-pad.
    """
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        print(f"[ERROR] Cannot open video: {video_path}")
        return

    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    fps_src      = cap.get(cv2.CAP_PROP_FPS) or 30.0
    w            = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    h            = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    stem         = Path(video_path).stem

    print(f"[INFO] {Path(video_path).name} — {total_frames} frames @ {fps_src:.1f} fps")
    print(f"[INFO] MediaPipe pose complexity: {extractor.model_complexity}"
          f"  (must match training; training default=2)")

    # --- Pass 1: extract all keypoints + collect frames for output video ---
    kp_list    = []
    raw_frames = []   # store for writing output videos after prediction
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        kp_list.append(extractor.process_frame(frame, mirror=mirror))
        raw_frames.append(frame.copy())
    cap.release()

    if not kp_list:
        print("[WARNING] No frames could be read from the video.")
        return

    T_raw = len(kp_list)
    T_use = min(T_raw, model.max_seq_len)
    print(f"[INFO] {T_raw} frames extracted → using {T_use} "
          f"({'centre-cropped' if T_raw > model.max_seq_len else 'zero-padded to'}"
          f" {model.max_seq_len})")

    # --- Inference ---
    inputs  = prep(kp_list, model.model_type)
    logits  = model.predict(inputs)
    probs   = ONNXModel.softmax(logits)
    top_idx = np.argsort(probs)[::-1][:top_k]

    # --- Console output ---
    print("\n" + _build_console_block(top_idx, probs, idx2label,
                                      model.model_type, Path(video_path).name, top_k))

    # --- Log file ---
    if log_file:
        _append_log(log_file,
                    _build_log_block(top_idx, probs, idx2label,
                                     Path(video_path).name, top_k))

    # --- Save keypoint feature log ---
    if save_features and output_dir:
        _save_feature_log(
            kp_list    = kp_list,
            inputs     = inputs,
            probs      = probs,
            top_idx    = top_idx,
            idx2label  = idx2label,
            source_name= Path(video_path).stem,
            output_dir = Path(output_dir),
            mirror     = mirror,
        )

    # --- Save output videos ---
    if output_dir:
        out_dir = Path(output_dir)
        out_dir.mkdir(parents=True, exist_ok=True)

        overlay_text, conf = _build_overlay_text(top_idx, probs, idx2label, renderer, top_k)
        color              = _pred_color(conf)

        clean_path = out_dir / f"{stem}_clean.mp4"
        debug_path = out_dir / f"{stem}_debug.mp4"
        vw_clean   = _make_video_writer(clean_path, fps_src, w, h)
        vw_debug   = _make_video_writer(debug_path, fps_src, w, h)

        for i, raw in enumerate(raw_frames):
            # Clean: label overlay only
            clean_frame = renderer.put_text(raw.copy(), overlay_text,
                                            position=(10, 10), color=color)
            vw_clean.write(clean_frame)

            # Debug: re-run MediaPipe on this frame to get landmarks,
            # then draw skeleton + label
            debug_frame = raw.copy()
            extractor.process_frame(debug_frame)       # updates _last_pose/_last_hands
            extractor.draw_landmarks(debug_frame)       # draws skeleton in-place
            debug_frame = renderer.put_text(debug_frame, overlay_text,
                                            position=(10, 10), color=color)
            vw_debug.write(debug_frame)

        vw_clean.release()
        vw_debug.release()
        print(f"[INFO] Saved  clean  → {clean_path}")
        print(f"[INFO] Saved  debug  → {debug_path}")


# ─── Webcam processing ────────────────────────────────────────────────────────

def run_webcam(model, prep, extractor, renderer, idx2label,
               cam_idx=0, top_k=5, log_file=None,
               buffer_sec=2.5, output_dir=None,
               mirror=True, save_features=False):
    """
    Live webcam sign recognition.

    Fix [INFER-C]: time-based buffer (buffer_sec seconds), matching infer.py.
    Fix [MIRROR]:  mirror=True (default) flips frames before MediaPipe so
                   coordinates match the training data orientation.

    Keys:  Q = quit  |  C = clear buffer manually
    """
    cap = cv2.VideoCapture(cam_idx)
    if not cap.isOpened():
        print(f"[ERROR] Cannot open webcam {cam_idx}")
        return

    fps_meta = cap.get(cv2.CAP_PROP_FPS) or 30.0
    w        = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    h        = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

    # Measure ACTUAL processing fps during a short warm-up so buf_frames
    # correctly represents buffer_sec real-world seconds (MediaPipe complexity=2
    # can be slow on CPU, making effective fps much lower than camera fps).
    print(f"[INFO] Measuring actual processing speed (warm-up)...")
    import time
    warmup_times = []
    for _ in range(10):
        ret, frame = cap.read()
        if not ret:
            break
        t0 = time.perf_counter()
        extractor.process_frame(frame, mirror=mirror)
        warmup_times.append(time.perf_counter() - t0)
    actual_fps = 1.0 / (sum(warmup_times) / len(warmup_times)) if warmup_times else fps_meta
    actual_fps = min(actual_fps, fps_meta)   # can't be faster than camera

    buf_frames = max(20, int(actual_fps * buffer_sec))
    print(f"[INFO] Camera fps: {fps_meta:.1f}   Actual processing fps: {actual_fps:.1f}")
    print(f"[INFO] Buffer: {buf_frames} frames = {buffer_sec:.1f}s of real signing")
    print(f"[INFO] Mirror: {'YES (webcam flip enabled — matches training)' if mirror else 'NO (raw webcam, disable if predictions wrong)'}")
    print(f"[INFO] MediaPipe pose complexity: {extractor.model_complexity}")
    print(f"[INFO] Q = quit | C = clear buffer manually")

    # ── Optional video writers ──
    ts_str     = datetime.now().strftime("%Y%m%d_%H%M%S")
    vw_clean   = vw_debug = None
    if output_dir:
        out_dir = Path(output_dir)
        out_dir.mkdir(parents=True, exist_ok=True)
        clean_path = out_dir / f"webcam_{ts_str}_clean.mp4"
        debug_path = out_dir / f"webcam_{ts_str}_debug.mp4"
        vw_clean   = _make_video_writer(clean_path, fps_meta, w, h)
        vw_debug   = _make_video_writer(debug_path, fps_meta, w, h)
        print(f"[INFO] Recording clean → {clean_path}")
        print(f"[INFO] Recording debug → {debug_path}")

    kp_buffer  = []
    last_text  = "Perform a sign — buffer filling..."
    last_color = (200, 200, 200)
    classify_count = 0
    total_frames_seen  = 0
    pose_detected_count = 0

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        # Extract keypoints — always append (Fix [INFER-A])
        kp = extractor.process_frame(frame, mirror=mirror)
        kp_buffer.append(kp)
        total_frames_seen += 1

        # Track whether pose was detected this frame (for status display)
        pose_this_frame = (
            extractor._last_pose is not None and
            extractor._last_pose.pose_landmarks is not None
        )
        if pose_this_frame:
            pose_detected_count += 1
        detect_rate = pose_detected_count / total_frames_seen * 100

        # Fix [INFER-C]: classify when buffer reaches buf_frames, then clear
        if len(kp_buffer) >= buf_frames:
            inputs  = prep(list(kp_buffer), model.model_type)
            logits  = model.predict(inputs)
            probs   = ONNXModel.softmax(logits)
            top_idx = np.argsort(probs)[::-1][:top_k]

            overlay_text, conf = _build_overlay_text(
                top_idx, probs, idx2label, renderer, top_k
            )
            last_text  = overlay_text
            last_color = _pred_color(conf)
            classify_count += 1

            # Console output
            print(f"\n[Prediction #{classify_count}]")
            print(_build_console_block(top_idx, probs, idx2label,
                                       model.model_type, f"webcam frame", top_k))

            # Log file — top_k results
            if log_file:
                _append_log(log_file,
                            _build_log_block(top_idx, probs, idx2label,
                                             f"webcam #{classify_count}", top_k))

            # Feature log
            if save_features and output_dir:
                _save_feature_log(
                    kp_list    = list(kp_buffer),   # already cleared below, snapshot here
                    inputs     = inputs,
                    probs      = probs,
                    top_idx    = top_idx,
                    idx2label  = idx2label,
                    source_name= f"webcam_{ts_str}_#{classify_count:03d}",
                    output_dir = Path(output_dir),
                    mirror     = mirror,
                )

            kp_buffer.clear()   # start fresh for next sign

        # ── Build display frames ──
        # Progress bar towards buf_frames (not max_seq_len)
        fill   = int(len(kp_buffer) / buf_frames * w)
        pct    = len(kp_buffer) / buf_frames * 100

        # Pose detection status: green=detected, red=not detected
        pose_status_text  = f"Pose: {'OK' if pose_this_frame else 'NOT DETECTED'}"
        pose_status_color = (0, 200, 0) if pose_this_frame else (0, 0, 255)
        detect_rate_text  = f"Detect rate: {detect_rate:.0f}%"

        # Clean frame: label + progress bar + pose status
        clean_frame = renderer.put_text(frame.copy(), last_text,
                                        position=(10, 10), color=last_color)
        cv2.putText(clean_frame, pose_status_text,
                    (10, 55), cv2.FONT_HERSHEY_SIMPLEX, 0.6, pose_status_color, 2)
        cv2.putText(clean_frame, detect_rate_text,
                    (10, 80), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (180, 180, 180), 1)
        cv2.rectangle(clean_frame, (0, h - 12), (fill, h), (0, 180, 255), -1)
        cv2.putText(clean_frame, f"buf {len(kp_buffer)}/{buf_frames}  ({pct:.0f}%)",
                    (10, h - 15), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (220, 220, 220), 1)

        # Debug frame: skeleton + label + pose status + progress bar
        debug_frame = frame.copy()
        extractor.draw_landmarks(debug_frame)
        debug_frame = renderer.put_text(debug_frame, last_text,
                                        position=(10, 10), color=last_color)
        cv2.putText(debug_frame, pose_status_text,
                    (10, 55), cv2.FONT_HERSHEY_SIMPLEX, 0.6, pose_status_color, 2)
        cv2.putText(debug_frame, detect_rate_text,
                    (10, 80), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (180, 180, 180), 1)
        cv2.rectangle(debug_frame, (0, h - 12), (fill, h), (0, 180, 255), -1)
        cv2.putText(debug_frame, f"buf {len(kp_buffer)}/{buf_frames}  ({pct:.0f}%)",
                    (10, h - 15), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (220, 220, 220), 1)

        # Write to video files
        if vw_clean: vw_clean.write(clean_frame)
        if vw_debug: vw_debug.write(debug_frame)

        # Show the debug view on screen (most useful for real-time feedback)
        cv2.imshow("MSL Recognition  [Q=quit  C=clear]", debug_frame)

        key = cv2.waitKey(1) & 0xFF
        if key == ord("q"):
            break
        elif key == ord("c"):
            kp_buffer.clear()
            last_text  = "Buffer cleared — perform a sign"
            last_color = (200, 200, 200)
            print("[INFO] Buffer manually cleared.")

    cap.release()
    if vw_clean: vw_clean.release()
    if vw_debug: vw_debug.release()
    cv2.destroyAllWindows()

    print(f"\n[INFO] Webcam session summary:")
    print(f"  Total frames     : {total_frames_seen}")
    print(f"  Pose detected    : {pose_detected_count}  ({pose_detected_count/max(total_frames_seen,1)*100:.0f}%)")
    print(f"  Classifications  : {classify_count}")
    if total_frames_seen > 0 and pose_detected_count / total_frames_seen < 0.3:
        print(f"  [WARNING] Low detection rate ({pose_detected_count/total_frames_seen*100:.0f}%).")
        print(f"            Tips: move closer, improve lighting, ensure upper body is visible.")
    if output_dir:
        print(f"  Saved  clean  → {clean_path}")
        print(f"  Saved  debug  → {debug_path}")


# ─── CLI ──────────────────────────────────────────────────────────────────────

def main():
    parser = argparse.ArgumentParser(
        description="MSL Sign Language Inference — ONNX Runtime",
        formatter_class=argparse.RawDescriptionHelpFormatter,
        epilog="""
Examples:
  # Single video with feature log for debugging
  python infer_onnx.py --onnx_model onnx_models\\bilstm_msl.onnx \\
      --video sample.mp4 --top_k 5 \\
      --output_dir predictions_output --log_file results.txt --save_features

  # Webcam with mirror (default) — try this first
  python infer_onnx.py --onnx_model onnx_models\\transformer_msl.onnx \\
      --webcam 0 --buffer_sec 2.5 --top_k 5 \\
      --output_dir predictions_output --log_file webcam_log.txt --save_features

  # Webcam without mirror — try if --mirror gives wrong results
  python infer_onnx.py --onnx_model onnx_models\\transformer_msl.onnx \\
      --webcam 0 --no_mirror --buffer_sec 2.5 --top_k 5 \\
      --output_dir predictions_output --log_file webcam_log.txt --save_features

  HOW TO DIAGNOSE WEBCAM FAILURES:
  1. Run with --save_features on BOTH video and webcam.
  2. Compare features/idx20-30_features.txt vs features/webcam_*_features.txt
  3. Check 'Left shoulder norm x': should be ~+0.5 for correct orientation.
     If it is ~-0.5 on webcam but +0.5 on video → use --mirror (already default).
     If it is ~-0.5 on BOTH → check camera framing and distance.
  4. Check 'Shoulder width': should be 0.15–0.50. Near 0 = not detected.
  5. Check detection rates: left/right hand active frames should be >50%.
""")

    parser.add_argument("--onnx_model",   required=True,
                        help="Path to .onnx model file (NOT the .json metadata)")
    parser.add_argument("--metadata",     default=None,
                        help="Path to .json metadata (default: same stem as .onnx)")
    parser.add_argument("--label_map",    default=None,
                        help="Path to label_map.json (default: next to .onnx)")
    parser.add_argument("--output_dir",   default="predictions_output",
                        help="Directory for saved videos (default: predictions_output/)")
    parser.add_argument("--log_file",     default=None,
                        help="Append top-k results to this text file")
    parser.add_argument("--top_k",        type=int, default=5,
                        help="Number of top predictions to show/log (default: 5)")
    parser.add_argument("--font_size",    type=int, default=32,
                        help="Myanmar font size for overlay (default: 32)")
    parser.add_argument("--model_complexity", type=int, default=None,
                        help="MediaPipe pose complexity: 0/1/2. "
                             "Default: 2 for video (matches training), 1 for webcam.")

    grp = parser.add_mutually_exclusive_group(required=True)
    grp.add_argument("--video",  help="Path to a video file")
    grp.add_argument("--webcam", type=int, metavar="CAM_IDX",
                     help="Webcam device index (0 = default camera)")

    parser.add_argument("--buffer_sec", type=float, default=2.5,
                        help="[Webcam] Seconds of frames to accumulate before "
                             "each classification (default: 2.5, same as infer.py)")
    parser.add_argument("--mirror", dest="mirror", action="store_true", default=True,
                        help="[Webcam] Flip frame horizontally before MediaPipe (default: ON). "
                             "Most webcams output a mirrored (selfie) image; flipping corrects "
                             "the x-coordinates to match the training data orientation.")
    parser.add_argument("--no_mirror", dest="mirror", action="store_false",
                        help="[Webcam] Disable the horizontal flip. Use if your webcam "
                             "already outputs an un-mirrored view, or if --mirror makes "
                             "predictions worse.")
    parser.add_argument("--save_features", action="store_true",
                        help="Save keypoint feature diagnostics to output_dir/features/. "
                             "Each classification event produces: "
                             "*_raw_kp.npy (raw keypoints), "
                             "*_model_input.npy (model input), "
                             "*_features.txt (human-readable stats). "
                             "Use to compare video vs webcam feature distributions.")
    parser.add_argument("--debug", action="store_true",
                        help="Print extra diagnostic information")
    args = parser.parse_args()
    # ── Validate ONNX path ────────────────────────────────────────────────────
    model_p = Path(args.onnx_model)
    if model_p.suffix.lower() == ".json":
        sys.exit(
            f"[ERROR] --onnx_model must be a .onnx file, not a .json file.\n"
            f"        You passed : {model_p}\n"
            f"        Did you mean: {model_p.with_suffix('.onnx')}"
        )
    if not model_p.exists():
        sys.exit(f"[ERROR] ONNX model not found: {model_p}")

    meta_p = args.metadata or str(model_p.with_suffix(".json"))
    lmap_p = args.label_map or str(model_p.parent / "label_map.json")

    if not Path(lmap_p).exists():
        sys.exit(f"[ERROR] label_map.json not found: {lmap_p}\n"
                 f"        Copy it from your Linux server alongside the .onnx file.")

    # ── Load resources ────────────────────────────────────────────────────────
    idx2label = load_idx2label(lmap_p)
    print(f"[INFO] Loaded {len(idx2label)} labels from label map.")

    model    = ONNXModel(str(model_p), meta_p)
    prep     = Preprocessor(max_seq_len=model.max_seq_len)
    renderer = TextRenderer(font_size=args.font_size)

    if args.log_file:
        print(f"[INFO] Logging top-{args.top_k} results to: {args.log_file}")
    if args.output_dir:
        print(f"[INFO] Saving output videos to: {args.output_dir}/")

    # Fix [INFER-F] + webcam detection fix:
    # complexity=2 (heavy model) for BOTH video and webcam.
    # Webcam needs the heavy model — the lite model (complexity=1) failed to detect
    # any pose landmarks in real webcam conditions (0% detection rate confirmed
    # by comparing debug videos: skeleton drawn 100% on video file, 0% on webcam).
    # The heavy model is more robust to webcam image quality, lighting variation,
    # and different distances. Slower, but actually detects the signer.
    #
    # min_detection_confidence lowered to 0.3 for webcam: live conditions are
    # harder than pre-recorded videos, so a more permissive threshold is needed.
    # Video keeps 0.5 to match training (extract_keypoints.py default).
    if args.model_complexity is not None:
        video_complexity  = args.model_complexity
        webcam_complexity = args.model_complexity
    else:
        video_complexity  = 2   # matches training (extract_keypoints.py --complexity 2)
        webcam_complexity = 2   # FIXED: was 1 (lite), now 2 (heavy) — see note above

    try:
        if args.video:
            extractor = KeypointExtractor(
                model_complexity=video_complexity,
                min_detection_confidence=0.5,   # matches training
                min_tracking_confidence=0.5,
            )
            try:
                run_video(model, prep, extractor, renderer, args.video,
                          top_k         = args.top_k,
                          log_file      = args.log_file,
                          idx2label     = idx2label,
                          output_dir    = args.output_dir,
                          debug         = args.debug,
                          mirror        = False,           # video files are not mirrored
                          save_features = args.save_features)
            finally:
                extractor.close()
        else:
            extractor = KeypointExtractor(
                model_complexity=webcam_complexity,
                min_detection_confidence=0.3,   # lower threshold for live webcam conditions
                min_tracking_confidence=0.3,
            )
            try:
                run_webcam(model, prep, extractor, renderer, idx2label,
                           cam_idx       = args.webcam,
                           top_k         = args.top_k,
                           log_file      = args.log_file,
                           buffer_sec    = args.buffer_sec,
                           output_dir    = args.output_dir,
                           mirror        = args.mirror,    # True by default
                           save_features = args.save_features)
            finally:
                extractor.close()
    finally:
        print("[INFO] Done.")


if __name__ == "__main__":
    main()


```

## Updated Commands

### Step 1: generate feature log from a known-good video
python infer_onnx.py --onnx_model onnx_models\bilstm_msl.onnx \
    --video sample4inference\idx20-30.mp4 --save_features \
    --output_dir predictions_output

### Step 2: generate feature log from webcam (default: --mirror ON)
python infer_onnx.py --onnx_model onnx_models\bilstm_msl.onnx \
    --webcam 0 --save_features \
    --output_dir predictions_output --log_file webcam_log.txt

### Step 3: open both _features.txt files and compare:
- "Left shoulder norm x": video should be ~+0.5, webcam should also be ~+0.5
- If webcam shows ~-0.5 → mirror is wrong direction → try --no_mirror
- "Shoulder width": should be 0.15–0.50 on both. Near 0 → not detected.
- Hand active frames: should be >50% during active signing.

## Debugging

In [104]:
from IPython.display import Markdown

file_path = './src/infer_onnx.py'

with open(file_path, 'r') as f:
    code = f.read()

# 'python' keyword triggers the syntax highlighter
display(Markdown(f"```python\n{code}\n```"))

```python
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
infer_onnx.py  –  Real-time MSL recognition on Windows using ONNX Runtime

Bugs fixed in this version
───────────────────────────
  [INFER-A] process_frame skipped frames without pose detection.
      Training always includes every frame (zeros for undetected joints).
      Fix: process_frame always returns the keypoint array, never None.

  [INFER-B] normalize_sequence used 2-D scale and left z unscaled.
      Fix: exact mirror of extract_keypoints.normalize_sequence() —
      full 3-D midpoint and 3-D Euclidean shoulder width, all axes.

  [INFER-C] *** ROOT CAUSE OF WEBCAM FAILURE ***
      Webcam classified every classify_every=15 frames = 0.5 seconds.
      Signs take 1–4 seconds; feeding 15 frames to a model trained on
      full-length signs produces garbage predictions.

      infer.py uses a TIME-BASED buffer: accumulate buffer_sec=2.5 sec
      of frames (~75 at 30 fps), then classify and clear.  This gives
      the model a realistic-duration sign sequence matching training.

      Fix: replaced the frame-count trigger with a time-based trigger
      using --buffer_sec (default 2.5), identical to infer.py.

  [INFER-D] np.int64 / int dict key mismatch.
      Fix: always cast to int() before idx2label lookup.

  [INFER-E] ONNXModel.predict returned a scalar instead of 1-D array.
      Fix: return output[0] — shape (num_classes,).

  [INFER-F] model_complexity=1 everywhere; training used complexity=2.
      Fix: complexity=2 for video (matching training), 1 for webcam
      (speed tradeoff, same as infer.py).

  [INFER-G] extractor.pose.model_complexity AttributeError on Windows.
      MediaPipe's Pose object does not expose model_complexity publicly.
      Fix: stored as self.model_complexity on KeypointExtractor.

New features
────────────
  • Video saving — two output files per inference:
      {name}_clean.mp4  —  label/confidence overlay only
      {name}_debug.mp4  —  label overlay + MediaPipe skeleton lines
    Works for both --video and --webcam modes.
    Saved to --output_dir (default: predictions_output/).

  • Webcam log — every classification event writes top-k results to the
    log file with timestamp and separator, matching the video log format.
"""

import argparse
import importlib
import json
import sys
from datetime import datetime
from pathlib import Path

import cv2
import numpy as np
import onnxruntime as ort

try:
    from PIL import Image, ImageDraw, ImageFont
    HAS_PIL = True
except ImportError:
    HAS_PIL = False

# ── Constants (must match extract_keypoints.py exactly) ───────────────────────
POSE_N          = 33
HAND_N          = 21
TOTAL_JOINTS    = POSE_N + HAND_N * 2   # 75
COORDS          = 3
FEAT_DIM        = TOTAL_JOINTS * COORDS  # 225

POSE_LEFT_SHOULDER  = 11
POSE_RIGHT_SHOULDER = 12
LEFT_HAND_START     = POSE_N            # 33
RIGHT_HAND_START    = POSE_N + HAND_N   # 54


# ─── MediaPipe import ─────────────────────────────────────────────────────────

def _import_mp():
    for prefix in ["mediapipe.solutions", "mediapipe.python.solutions"]:
        try:
            pose  = importlib.import_module(f"{prefix}.pose")
            hands = importlib.import_module(f"{prefix}.hands")
            draw  = importlib.import_module(f"{prefix}.drawing_utils")
            return pose, hands, draw
        except (ImportError, ModuleNotFoundError):
            continue
    try:
        import mediapipe as _mp
        return _mp.solutions.pose, _mp.solutions.hands, _mp.solutions.drawing_utils
    except AttributeError:
        pass
    raise ImportError(
        "Cannot import MediaPipe solutions.\n"
        "Try: pip uninstall mediapipe -y && pip install mediapipe==0.10.18"
    )

mp_pose, mp_hands, mp_drawing = _import_mp()


# ─── Myanmar text rendering ──────────────────────────────────────────────────

class TextRenderer:
    _FONT_CANDIDATES = [
        "C:\\Windows\\Fonts\\Padauk.ttf",
        "C:\\Windows\\Fonts\\NotoSansMyanmar-Regular.ttf",
        "C:\\Windows\\Fonts\\NotoSansMyanmar.ttf",
        "C:\\Windows\\Fonts\\myanmar3.ttf",
        "/usr/share/fonts/truetype/padauk/Padauk-Regular.ttf",
        "/usr/share/fonts/truetype/noto/NotoSansMyanmar-Regular.ttf",
        "fonts\\Padauk.ttf",
        "fonts/NotoSansMyanmar-Regular.ttf",
    ]

    def __init__(self, font_size: int = 32):
        self.font = self._load_font(font_size)
        self.can_render_myanmar = (self.font is not None)
        if not self.can_render_myanmar:
            print()
            print("  ╔═══════════════════════════════════════════════════════════╗")
            print("  ║  INFO: No Myanmar shaping font found.                    ║")
            print("  ║  Displaying readable Class IDs instead of Myanmar text.  ║")
            print("  ╚═══════════════════════════════════════════════════════════╝")
            print()

    def _load_font(self, size):
        if not HAS_PIL:
            return None
        for path in self._FONT_CANDIDATES:
            if Path(path).exists():
                try:
                    return ImageFont.truetype(path, size)
                except Exception:
                    continue
        return None

    def put_text(self, frame, text, position=(20, 20), color=(0, 255, 0)):
        if self.font and HAS_PIL:
            pil = Image.fromarray(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
            d   = ImageDraw.Draw(pil)
            r, g, b = color[2], color[1], color[0]
            for dx in [-1, 0, 1]:
                for dy in [-1, 0, 1]:
                    if dx or dy:
                        d.text((position[0]+dx, position[1]+dy), text,
                               font=self.font, fill=(0, 0, 0))
            d.text(position, text, font=self.font, fill=(r, g, b))
            return cv2.cvtColor(np.array(pil), cv2.COLOR_RGB2BGR)
        cv2.putText(frame, text, position, cv2.FONT_HERSHEY_SIMPLEX,
                    0.75, color, 2, cv2.LINE_AA)
        return frame


# ─── Keypoint extraction ──────────────────────────────────────────────────────

class KeypointExtractor:
    """
    Extracts (TOTAL_JOINTS, 3) keypoint arrays from BGR frames.

    Fix [INFER-G]: model_complexity is stored as self.model_complexity
    because mediapipe's Pose object does not expose it as a public attribute
    on all Windows builds (AttributeError otherwise).

    Fix [INFER-A]: process_frame always returns an array, never None.
    Matches extract_keypoints.py which includes every frame, using zeros
    for any joint that MediaPipe failed to detect.

    Hand convention (matches extract_keypoints.py):
      MediaPipe reports handedness from the camera perspective (mirrored).
      label='Right' → signer's LEFT  hand → slot LEFT_HAND_START  (33).
      label='Left'  → signer's RIGHT hand → slot RIGHT_HAND_START (54).
    """

    def __init__(self, model_complexity: int = 2,
                 min_detection_confidence: float = 0.5,
                 min_tracking_confidence:  float = 0.5):
        # Fix [INFER-G]: store complexity ourselves — mediapipe's Pose object
        # does NOT expose a public .model_complexity on all Windows builds.
        self.model_complexity = model_complexity

        self.pose  = mp_pose.Pose(
            static_image_mode=False,
            model_complexity=model_complexity,
            smooth_landmarks=True,
            enable_segmentation=False,
            min_detection_confidence=min_detection_confidence,
            min_tracking_confidence=min_tracking_confidence,
        )
        self.hands = mp_hands.Hands(
            static_image_mode=False,
            max_num_hands=2,
            model_complexity=min(model_complexity, 1),  # Hands caps at 1
            min_detection_confidence=min_detection_confidence,
            min_tracking_confidence=min_tracking_confidence,
        )
        self._last_pose  = None
        self._last_hands = None

    def process_frame(self, frame: np.ndarray,
                      mirror: bool = False) -> np.ndarray:
        """
        Returns (TOTAL_JOINTS, 3) float32 for every frame.
        Zeros for any undetected joint/hand — never returns None.

        Args:
            frame:  BGR frame from cv2.VideoCapture.
            mirror: If True, flip the frame horizontally before processing.
                    Many laptop/webcam cameras output a mirrored (selfie-view)
                    image.  Training videos were likely recorded un-mirrored
                    (normal camera view).  Setting mirror=True corrects this so
                    the normalised coordinates match the training distribution.
                    The DISPLAYED frame is NOT flipped — only the frame sent to
                    MediaPipe is flipped, so landmark positions are in the
                    coordinate space of the un-mirrored view.
        """
        if mirror:
            frame = cv2.flip(frame, 1)   # horizontal flip only for MediaPipe
        rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        rgb.flags.writeable = False
        p_res = self.pose.process(rgb)
        h_res = self.hands.process(rgb)
        self._last_pose  = p_res
        self._last_hands = h_res

        kp = np.zeros((TOTAL_JOINTS, COORDS), dtype=np.float32)

        if p_res.pose_landmarks:
            for i, lm in enumerate(p_res.pose_landmarks.landmark):
                kp[i] = [lm.x, lm.y, lm.z]

        if h_res.multi_hand_landmarks and h_res.multi_handedness:
            for lm_list, handedness in zip(
                h_res.multi_hand_landmarks, h_res.multi_handedness
            ):
                label  = handedness.classification[0].label
                offset = LEFT_HAND_START if label == 'Right' else RIGHT_HAND_START
                for i, lm in enumerate(lm_list.landmark):
                    kp[offset + i] = [lm.x, lm.y, lm.z]

        return kp  # always an array, never None

    def draw_landmarks(self, frame: np.ndarray) -> np.ndarray:
        """Draw MediaPipe skeleton onto frame IN-PLACE and return it."""
        if self._last_pose and self._last_pose.pose_landmarks:
            mp_drawing.draw_landmarks(
                frame, self._last_pose.pose_landmarks, mp_pose.POSE_CONNECTIONS
            )
        if self._last_hands and self._last_hands.multi_hand_landmarks:
            for lm in self._last_hands.multi_hand_landmarks:
                mp_drawing.draw_landmarks(frame, lm, mp_hands.HAND_CONNECTIONS)
        return frame

    def close(self):
        self.pose.close()
        self.hands.close()


# ─── Normalization ────────────────────────────────────────────────────────────

def normalize_sequence(seq: np.ndarray) -> np.ndarray:
    """
    Fix [INFER-B]: exact mirror of extract_keypoints.normalize_sequence().
    Full 3-D shoulder-midpoint subtraction and 3-D shoulder-width scaling,
    applied to all three coordinate axes.
    """
    l_sh = seq[:, POSE_LEFT_SHOULDER,  :]   # (T, 3)
    r_sh = seq[:, POSE_RIGHT_SHOULDER, :]   # (T, 3)
    mid  = (l_sh + r_sh) / 2.0
    dist = np.linalg.norm(r_sh - l_sh, axis=1, keepdims=True)
    dist = np.where(dist < 1e-6, 1.0, dist)
    return ((seq - mid[:, None, :]) / dist[:, None, :]).astype(np.float32)


# ─── Preprocessor ─────────────────────────────────────────────────────────────

class Preprocessor:
    """
    Stack raw (75, 3) keypoint frames → normalize → crop/pad → model input.

    Processing order matches infer.py exactly:
      1. Stack   → (T, 75, 3)
      2. Normalize (3-D shoulder-centred, full 3-D scale)
      3. Centre-crop  if T > max_seq_len
      4. Zero-pad     if T < max_seq_len
      5. Build mask   (True = padded position)
      6. Reshape for model type
    """

    def __init__(self, max_seq_len: int = 200):
        self.max_seq_len = max_seq_len

    def __call__(self, kp_list: list, model_type: str,
                 zero_z: bool = False) -> dict:
        if not kp_list:
            return {}

        seq = np.stack(kp_list, axis=0).astype(np.float32)  # (T, 75, 3)
        seq = normalize_sequence(seq)

        # Zero out z-coordinates for webcam use.
        # Rationale from feature analysis:
        #   Training data: z_mean ≈ +0.03 (nearly zero, low variance)
        #   Webcam data:   z_mean ≈ -0.5 to -0.8 (large systematic offset)
        # The z offset comes from the signer being closer to the webcam and
        # different posture/depth than training studio recordings.
        # Since z contributes little discriminative information (std≈0.74 vs
        # x_std≈0.89, y_std≈1.94) and is systematically wrong on webcam,
        # zeroing it removes harmful noise without losing sign-relevant features.
        if zero_z:
            seq[:, :, 2] = 0.0
        T   = seq.shape[0]

        if T > self.max_seq_len:
            start = (T - self.max_seq_len) // 2
            seq   = seq[start: start + self.max_seq_len]
            T     = self.max_seq_len

        real_len = T

        if T < self.max_seq_len:
            pad = np.zeros((self.max_seq_len - T, TOTAL_JOINTS, COORDS), dtype=np.float32)
            seq = np.concatenate([seq, pad], axis=0)

        mask = np.zeros(self.max_seq_len, dtype=np.bool_)
        mask[real_len:] = True

        if model_type == "stgcn":
            kp_out = np.expand_dims(seq.transpose(2, 0, 1), axis=0).astype(np.float32)
            return {"keypoints": kp_out}
        else:
            kp_out = np.expand_dims(
                seq.reshape(self.max_seq_len, FEAT_DIM), axis=0
            ).astype(np.float32)
            return {"keypoints": kp_out, "mask": mask[np.newaxis]}


# ─── ONNX model wrapper ───────────────────────────────────────────────────────

class ONNXModel:
    def __init__(self, onnx_path: str, meta_path: str = None):
        available = ort.get_available_providers()
        preferred = ["CUDAExecutionProvider", "DmlExecutionProvider", "CPUExecutionProvider"]
        providers = [p for p in preferred if p in available]

        self.session = ort.InferenceSession(onnx_path, providers=providers)
        print(f"[INFO] ONNX loaded: {Path(onnx_path).name}"
              f"  (provider: {self.session.get_providers()[0]})")

        self.meta = {}
        if meta_path and Path(meta_path).exists():
            try:
                with open(meta_path, encoding="utf-8") as f:
                    self.meta = json.load(f)
            except Exception as e:
                print(f"[WARNING] Could not read metadata: {e}")
        else:
            print(f"[INFO] Metadata not found at '{meta_path}', using defaults.")

        self.model_type  = self.meta.get("model_type",  "transformer")
        self.max_seq_len = self.meta.get("max_seq_len", 200)
        self.num_classes = self.meta.get("num_classes", 558)

    def predict(self, inputs: dict) -> np.ndarray:
        """Fix [INFER-E]: return (num_classes,) not scalar."""
        return self.session.run(None, inputs)[0][0]

    @staticmethod
    def softmax(logits: np.ndarray) -> np.ndarray:
        e = np.exp(logits - logits.max())
        return e / e.sum()


# ─── Label map ────────────────────────────────────────────────────────────────

def load_idx2label(path: str) -> dict:
    with open(path, encoding="utf-8") as f:
        data = json.load(f)
    return {int(k): v for k, v in data.get("idx2label", {}).items()}


def _get_label(idx2label: dict, class_id) -> str:
    """Fix [INFER-D]: always cast to int() before dict lookup."""
    return idx2label.get(int(class_id), f"[class {int(class_id)}]")


# ─── Helpers ──────────────────────────────────────────────────────────────────

def _overlay_text(renderer, frame, text, color):
    """Put prediction text at top-left of frame."""
    return renderer.put_text(frame, text, position=(10, 10), color=color)


def _pred_color(confidence: float) -> tuple:
    """Green ≥80%, yellow ≥50%, red otherwise."""
    if confidence >= 0.80: return (0, 200,   0)
    if confidence >= 0.50: return (0, 200, 200)
    return (0, 0, 255)


def _build_overlay_text(top_idx, probs, idx2label, renderer, top_k):
    """Single-line text for the video overlay (top-1 label + confidence)."""
    best = top_idx[0]
    conf = probs[int(best)]
    lbl  = _get_label(idx2label, best) if idx2label else f"Class {int(best)}"
    if renderer.can_render_myanmar:
        return f"{lbl}  ({conf*100:.1f}%)", conf
    return f"Class {int(best)}  ({conf*100:.1f}%)", conf


def _build_console_block(top_idx, probs, idx2label, model_type, source_name, top_k):
    """Multi-line console output block, same format as run_video."""
    sep   = "─" * 60
    lines = [f"Source : {source_name}", f"Model  : {model_type.upper()}", sep]
    for rank, idx in enumerate(top_idx, 1):
        lbl = _get_label(idx2label, idx) if idx2label else f"[class {int(idx)}]"
        lines.append(f"  Top-{rank}: {lbl}  ({probs[int(idx)]*100:.2f}%)")
    lines.append(sep)
    return "\n".join(lines)


def _build_log_block(top_idx, probs, idx2label, source_name, top_k):
    """Log block with timestamp and top-k results."""
    ts    = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    sep   = "=" * 60
    lines = [sep, f"[{ts}]  {source_name}"]
    for rank, idx in enumerate(top_idx, 1):
        lbl = _get_label(idx2label, idx) if idx2label else "?"
        lbl = lbl.encode("utf-8", errors="replace").decode("utf-8")
        lines.append(f"  Top-{rank:>2}: Class {int(idx):>4d} | "
                     f"{lbl:<45s} | {probs[int(idx)]*100:6.2f}%")
    lines.append("")
    return "\n".join(lines) + "\n"


def _append_log(path, text):
    try:
        with open(path, "a", encoding="utf-8-sig") as f:
            f.write(text)
    except Exception as e:
        print(f"[WARNING] Could not write log: {e}")


def _make_video_writer(output_path: Path, fps: float, width: int, height: int):
    """Create a cv2.VideoWriter with MP4V codec."""
    fourcc = cv2.VideoWriter_fourcc(*"mp4v")
    return cv2.VideoWriter(str(output_path), fourcc, fps, (width, height))


def _save_feature_log(kp_list, inputs, probs, top_idx, idx2label,
                      source_name, output_dir, mirror):
    """
    Save full diagnostic information for one classification event.

    Creates two files in output_dir/features/:
      {source_name}_raw_kp.npy      — raw (un-normalized) stacked keypoints (T,75,3)
      {source_name}_model_input.npy — the actual array fed to the ONNX model
      {source_name}_features.txt    — human-readable statistics for debugging

    The .txt file contains:
      - Shoulder positions and detection status (are they non-zero?)
      - Shoulder width (normalization scale — should be ~0.3-0.7 if detected)
      - Per-axis statistics of normalized features
      - Fraction of frames with active hand movement
      - Mirror setting used
      - Top-k classification results
      - Comparison hint vs expected training distribution
    """
    import os
    feat_dir = output_dir / "features"
    feat_dir.mkdir(parents=True, exist_ok=True)

    # Raw keypoints before normalization
    raw_kp = np.stack(kp_list, axis=0).astype(np.float32)   # (T, 75, 3)
    T      = raw_kp.shape[0]

    # Save .npy files
    np.save(str(feat_dir / f"{source_name}_raw_kp.npy"), raw_kp)

    # Also save the model input (already normalized, cropped, padded)
    model_input_key = "keypoints"
    if model_input_key in inputs:
        np.save(str(feat_dir / f"{source_name}_model_input.npy"),
                inputs[model_input_key])

    # --- Build text report ---
    L_SH, R_SH = 11, 12
    l_sh = raw_kp[:, L_SH, :]   # (T, 3)
    r_sh = raw_kp[:, R_SH, :]   # (T, 3)

    # How many frames had shoulders detected (non-zero)?
    l_detected = (np.abs(l_sh).sum(axis=1) > 1e-6).sum()
    r_detected = (np.abs(r_sh).sum(axis=1) > 1e-6).sum()

    # Shoulder width distribution (normalization scale)
    dist = np.linalg.norm(r_sh - l_sh, axis=1)  # (T,)
    dist_nonzero = dist[dist > 1e-6]

    # Hand activity: fraction of frames where hands are non-zero
    lh_block = raw_kp[:, 33:54, :]
    rh_block = raw_kp[:, 54:75, :]
    lh_active = (np.abs(lh_block).sum(axis=(1,2)) > 1e-6).sum()
    rh_active = (np.abs(rh_block).sum(axis=(1,2)) > 1e-6).sum()

    # Normalized keypoints for distribution check
    norm_kp = normalize_sequence(raw_kp)
    # For a correctly framed frontal signer:
    #   Left  shoulder (idx 11) normalized x should be ~ +0.4 to +0.6
    #   Right shoulder (idx 12) normalized x should be ~ -0.6 to -0.4
    l_sh_norm = norm_kp[:, L_SH, :]
    r_sh_norm = norm_kp[:, R_SH, :]

    lines = [
        "=" * 65,
        f"Feature Diagnostic Log — {source_name}",
        f"Timestamp : {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}",
        f"Mirror    : {'YES — webcam flipped before MediaPipe' if mirror else 'NO — raw frame used'}",
        "=" * 65,
        "",
        "── Sequence info ───────────────────────────────────────────",
        f"  Total frames         : {T}",
        f"  Left  shoulder detected: {l_detected}/{T} frames ({l_detected/T*100:.0f}%)",
        f"  Right shoulder detected: {r_detected}/{T} frames ({r_detected/T*100:.0f}%)",
        f"  Left  hand active    : {lh_active}/{T} frames ({lh_active/T*100:.0f}%)",
        f"  Right hand active    : {rh_active}/{T} frames ({rh_active/T*100:.0f}%)",
        "",
        "── Shoulder positions (RAW, image space [0,1]) ─────────────",
        f"  Left  shoulder x mean : {l_sh[:,0].mean():.4f}  std={l_sh[:,0].std():.4f}",
        f"  Right shoulder x mean : {r_sh[:,0].mean():.4f}  std={r_sh[:,0].std():.4f}",
        f"  Left  shoulder y mean : {l_sh[:,1].mean():.4f}",
        f"  Right shoulder y mean : {r_sh[:,1].mean():.4f}",
        "",
        "  EXPECTED for frontal un-mirrored signer:",
        "    Left  shoulder x ~ 0.55–0.65 (camera right = signer left)",
        "    Right shoulder x ~ 0.35–0.45 (camera left  = signer right)",
        "    Shoulder    y   ~ 0.45–0.55  (vertical CENTER of frame)",
        "  If x values are REVERSED → webcam is mirrored → add --mirror / remove --no_mirror",
        f"  {'!! WARNING: Shoulders too LOW in frame (y=' + f'{l_sh[:,1].mean():.2f}' + '). Move camera UP or sit further back.' if l_sh[:,1].mean() > 0.58 else ''}",
        f"  {'!! WARNING: Shoulders too HIGH in frame (y=' + f'{l_sh[:,1].mean():.2f}' + '). Move camera DOWN or closer.' if l_sh[:,1].mean() < 0.38 else ''}",
        "",
        "── Shoulder width (normalization scale) ────────────────────",
    ]
    if len(dist_nonzero) > 0:
        sw_warn = ""
        if dist_nonzero.mean() > 0.27:
            sw_warn = f"  !! WARNING: Too close to camera (width={dist_nonzero.mean():.3f} > 0.27). Move FURTHER BACK."
        elif dist_nonzero.mean() < 0.12:
            sw_warn = f"  !! WARNING: Too far from camera (width={dist_nonzero.mean():.3f} < 0.12). Move CLOSER."
        lines += [
            f"  Shoulder width mean : {dist_nonzero.mean():.4f}",
            f"  Shoulder width std  : {dist_nonzero.std():.4f}",
            f"  Shoulder width range: {dist_nonzero.min():.4f} – {dist_nonzero.max():.4f}",
            "  TARGET: 0.15–0.27 (training avg was 0.205; webcam was 0.313 = too close)",
            sw_warn if sw_warn else "  ✓ Shoulder width is in good range",
        ]
    else:
        lines.append("  !! Shoulder width is ZERO in ALL frames — shoulders NOT detected!")
        lines.append("     Normalization falls back to dist=1.0 → features NOT normalized correctly.")

    lines += [
        "",
        "── Normalized shoulder positions (after normalize_sequence) ─",
        f"  Left  shoulder norm x mean : {l_sh_norm[:,0].mean():.4f}",
        f"  Right shoulder norm x mean : {r_sh_norm[:,0].mean():.4f}",
        "  EXPECTED after correct normalization:",
        "    Left  shoulder norm x ~  +0.5  (signer's left = positive x)",
        "    Right shoulder norm x ~  -0.5  (signer's right = negative x)",
        "  If REVERSED (left~-0.5, right~+0.5) → mirror flip needed",
        "",
        "── Full keypoint distribution (normalized) ──────────────────",
        f"  All joints x: mean={norm_kp[:,:,0].mean():.4f}  std={norm_kp[:,:,0].std():.4f}"
        f"  min={norm_kp[:,:,0].min():.4f}  max={norm_kp[:,:,0].max():.4f}",
        f"  All joints y: mean={norm_kp[:,:,1].mean():.4f}  std={norm_kp[:,:,1].std():.4f}"
        f"  min={norm_kp[:,:,1].min():.4f}  max={norm_kp[:,:,1].max():.4f}",
        f"  All joints z: mean={norm_kp[:,:,2].mean():.4f}  std={norm_kp[:,:,2].std():.4f}"
        f"  min={norm_kp[:,:,2].min():.4f}  max={norm_kp[:,:,2].max():.4f}",
        "",
        "  Training reference (video idx20-30):",
        "    x mean=-0.44  y mean=+0.05  z mean=+0.03",
        f"  {'!! Z-AXIS OFFSET: z mean=' + f'{norm_kp[:,:,2].mean():.2f}' + ' (expected ~0). Use --zero_z to suppress depth noise.' if abs(norm_kp[:,:,2].mean()) > 0.2 else '  ✓ Z-axis looks reasonable'}",
        f"  {'!! Y-AXIS OFFSET: y mean=' + f'{norm_kp[:,:,1].mean():.2f}' + ' (expected ~0). Adjust vertical framing (see shoulder y tip above).' if abs(norm_kp[:,:,1].mean()) > 0.2 else '  ✓ Y-axis looks reasonable'}",
        f"  {'!! SEQUENCE LENGTH: only ' + str(T) + ' frames fed to model (training avg ~127). Increase --buf_frames or use complexity=1 for higher fps.' if T < 80 else '  ✓ Sequence length looks reasonable (' + str(T) + ' frames)'}",
        "",
        "── Classification results ───────────────────────────────────",
    ]
    for rank, idx in enumerate(top_idx, 1):
        lbl = _get_label(idx2label, idx) if idx2label else "?"
        lbl_safe = lbl.encode("utf-8", errors="replace").decode("utf-8")
        lines.append(f"  Top-{rank}: Class {int(idx):>4d} | {lbl_safe:<50s} | {probs[int(idx)]*100:.2f}%")

    lines += [
        "",
        "── Files saved ──────────────────────────────────────────────",
        f"  Raw keypoints : features/{source_name}_raw_kp.npy      shape=({T},75,3)",
        f"  Model input   : features/{source_name}_model_input.npy",
        "=" * 65,
        "",
    ]

    txt_path = feat_dir / f"{source_name}_features.txt"
    with open(txt_path, "w", encoding="utf-8-sig") as f:
        f.write("\n".join(lines))
    print(f"[INFO] Feature log → {txt_path}")


# ─── Video processing ─────────────────────────────────────────────────────────

def run_video(model, prep, extractor, renderer, video_path,
              top_k=5, log_file=None, idx2label=None,
              output_dir=None, debug=False, mirror=False,
              save_features=False):
    """
    Process a single video file.  Saves two output videos:
      {stem}_clean.mp4  —  label overlay only
      {stem}_debug.mp4  —  label overlay + MediaPipe skeleton lines

    Matches infer.py / process_video() exactly:
      every frame included (zeros for undetected joints),
      normalize after stacking, centre-crop or zero-pad.
    """
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        print(f"[ERROR] Cannot open video: {video_path}")
        return

    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    fps_src      = cap.get(cv2.CAP_PROP_FPS) or 30.0
    w            = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    h            = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    stem         = Path(video_path).stem

    print(f"[INFO] {Path(video_path).name} — {total_frames} frames @ {fps_src:.1f} fps")
    print(f"[INFO] MediaPipe pose complexity: {extractor.model_complexity}"
          f"  (must match training; training default=2)")

    # --- Pass 1: extract all keypoints + collect frames for output video ---
    kp_list    = []
    raw_frames = []   # store for writing output videos after prediction
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        kp_list.append(extractor.process_frame(frame, mirror=mirror))
        raw_frames.append(frame.copy())
    cap.release()

    if not kp_list:
        print("[WARNING] No frames could be read from the video.")
        return

    T_raw = len(kp_list)
    T_use = min(T_raw, model.max_seq_len)
    print(f"[INFO] {T_raw} frames extracted → using {T_use} "
          f"({'centre-cropped' if T_raw > model.max_seq_len else 'zero-padded to'}"
          f" {model.max_seq_len})")

    # --- Inference ---
    inputs  = prep(kp_list, model.model_type)
    logits  = model.predict(inputs)
    probs   = ONNXModel.softmax(logits)
    top_idx = np.argsort(probs)[::-1][:top_k]

    # --- Console output ---
    print("\n" + _build_console_block(top_idx, probs, idx2label,
                                      model.model_type, Path(video_path).name, top_k))

    # --- Log file ---
    if log_file:
        _append_log(log_file,
                    _build_log_block(top_idx, probs, idx2label,
                                     Path(video_path).name, top_k))

    # --- Save keypoint feature log ---
    if save_features and output_dir:
        _save_feature_log(
            kp_list    = kp_list,
            inputs     = inputs,
            probs      = probs,
            top_idx    = top_idx,
            idx2label  = idx2label,
            source_name= Path(video_path).stem,
            output_dir = Path(output_dir),
            mirror     = mirror,
        )

    # --- Save output videos ---
    if output_dir:
        out_dir = Path(output_dir)
        out_dir.mkdir(parents=True, exist_ok=True)

        overlay_text, conf = _build_overlay_text(top_idx, probs, idx2label, renderer, top_k)
        color              = _pred_color(conf)

        clean_path = out_dir / f"{stem}_clean.mp4"
        debug_path = out_dir / f"{stem}_debug.mp4"
        vw_clean   = _make_video_writer(clean_path, fps_src, w, h)
        vw_debug   = _make_video_writer(debug_path, fps_src, w, h)

        for i, raw in enumerate(raw_frames):
            # Clean: label overlay only
            clean_frame = renderer.put_text(raw.copy(), overlay_text,
                                            position=(10, 10), color=color)
            vw_clean.write(clean_frame)

            # Debug: re-run MediaPipe on this frame to get landmarks,
            # then draw skeleton + label
            debug_frame = raw.copy()
            extractor.process_frame(debug_frame)       # updates _last_pose/_last_hands
            extractor.draw_landmarks(debug_frame)       # draws skeleton in-place
            debug_frame = renderer.put_text(debug_frame, overlay_text,
                                            position=(10, 10), color=color)
            vw_debug.write(debug_frame)

        vw_clean.release()
        vw_debug.release()
        print(f"[INFO] Saved  clean  → {clean_path}")
        print(f"[INFO] Saved  debug  → {debug_path}")


# ─── Webcam processing ────────────────────────────────────────────────────────

def run_webcam(model, prep, extractor, renderer, idx2label,
               cam_idx=0, top_k=5, log_file=None,
               buffer_sec=2.5, buffer_frames=100, output_dir=None,
               mirror=True, save_features=False, zero_z=False):
    """
    Live webcam sign recognition.

    Fix [INFER-C]: time-based buffer (buffer_sec seconds), matching infer.py.
    Fix [MIRROR]:  mirror=True (default) flips frames before MediaPipe so
                   coordinates match the training data orientation.

    Keys:  Q = quit  |  C = clear buffer manually
    """
    cap = cv2.VideoCapture(cam_idx)
    if not cap.isOpened():
        print(f"[ERROR] Cannot open webcam {cam_idx}")
        return

    fps_meta = cap.get(cv2.CAP_PROP_FPS) or 30.0
    w        = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    h        = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

    # Use a FIXED frame count for the buffer instead of time-based fps measurement.
    #
    # Why: When complexity=2 was used, actual processing fps dropped to ~9fps.
    # At 9fps, buffer_sec=2.5 gives only 22-36 frames, but the model was trained
    # on sequences averaging ~127 frames. Feeding 36 frames (18% of max_seq_len)
    # means 82% of the model input is zero-padding — the model has never seen
    # sequences this short at test time, so predictions are garbage.
    #
    # Fix: complexity=1 (~25fps) + fixed buf_frames=100 (from --buf_frames arg).
    # At 25fps: 100 frames = 4 seconds per sign = realistic signing duration.
    # This matches the training distribution (avg ~127 frames) much better.
    buf_frames = max(20, min(buffer_frames, model.max_seq_len))

    print(f"[INFO] Webcam  {w}x{h} @ {fps_meta:.1f} fps  (complexity={extractor.model_complexity})")
    print(f"[INFO] Buffer: {buf_frames} frames fixed  "
          f"(~{buf_frames/fps_meta:.1f}s at camera fps — fill buffer THEN sign)")
    print(f"[INFO] Mirror: {'YES' if mirror else 'NO'}")
    print(f"[INFO] Zero-Z: {'YES — z coordinates zeroed (reduces depth noise)' if zero_z else 'NO'}")
    print(f"[INFO] Positioning tip: stand so your SHOULDERS are at the VERTICAL CENTER")
    print(f"                        of the frame (not too high/low). Fill 40-60% of width.")
    print(f"[INFO] Q = quit | C = clear buffer manually")

    # ── Optional video writers ──
    ts_str     = datetime.now().strftime("%Y%m%d_%H%M%S")
    vw_clean   = vw_debug = None
    if output_dir:
        out_dir = Path(output_dir)
        out_dir.mkdir(parents=True, exist_ok=True)
        clean_path = out_dir / f"webcam_{ts_str}_clean.mp4"
        debug_path = out_dir / f"webcam_{ts_str}_debug.mp4"
        vw_clean   = _make_video_writer(clean_path, fps_meta, w, h)
        vw_debug   = _make_video_writer(debug_path, fps_meta, w, h)
        print(f"[INFO] Recording clean → {clean_path}")
        print(f"[INFO] Recording debug → {debug_path}")

    kp_buffer  = []
    last_text  = "Perform a sign — buffer filling..."
    last_color = (200, 200, 200)
    classify_count = 0
    total_frames_seen  = 0
    pose_detected_count = 0

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        # Extract keypoints — always append (Fix [INFER-A])
        kp = extractor.process_frame(frame, mirror=mirror)
        kp_buffer.append(kp)
        total_frames_seen += 1

        # Track whether pose was detected this frame (for status display)
        pose_this_frame = (
            extractor._last_pose is not None and
            extractor._last_pose.pose_landmarks is not None
        )
        if pose_this_frame:
            pose_detected_count += 1
        detect_rate = pose_detected_count / total_frames_seen * 100

        # Fix: classify when buffer reaches buf_frames, then clear
        if len(kp_buffer) >= buf_frames:
            inputs  = prep(list(kp_buffer), model.model_type, zero_z=zero_z)
            logits  = model.predict(inputs)
            probs   = ONNXModel.softmax(logits)
            top_idx = np.argsort(probs)[::-1][:top_k]

            overlay_text, conf = _build_overlay_text(
                top_idx, probs, idx2label, renderer, top_k
            )
            last_text  = overlay_text
            last_color = _pred_color(conf)
            classify_count += 1

            # Console output
            print(f"\n[Prediction #{classify_count}]")
            print(_build_console_block(top_idx, probs, idx2label,
                                       model.model_type, f"webcam frame", top_k))

            # Log file — top_k results
            if log_file:
                _append_log(log_file,
                            _build_log_block(top_idx, probs, idx2label,
                                             f"webcam #{classify_count}", top_k))

            # Feature log
            if save_features and output_dir:
                _save_feature_log(
                    kp_list    = list(kp_buffer),   # already cleared below, snapshot here
                    inputs     = inputs,
                    probs      = probs,
                    top_idx    = top_idx,
                    idx2label  = idx2label,
                    source_name= f"webcam_{ts_str}_#{classify_count:03d}",
                    output_dir = Path(output_dir),
                    mirror     = mirror,
                )

            kp_buffer.clear()   # start fresh for next sign

        # ── Build display frames ──
        # Progress bar towards buf_frames (not max_seq_len)
        fill   = int(len(kp_buffer) / buf_frames * w)
        pct    = len(kp_buffer) / buf_frames * 100

        # Pose detection status: green=detected, red=not detected
        pose_status_text  = f"Pose: {'OK' if pose_this_frame else 'NOT DETECTED'}"
        pose_status_color = (0, 200, 0) if pose_this_frame else (0, 0, 255)
        detect_rate_text  = f"Detect rate: {detect_rate:.0f}%"

        # Clean frame: label + progress bar + pose status
        clean_frame = renderer.put_text(frame.copy(), last_text,
                                        position=(10, 10), color=last_color)
        cv2.putText(clean_frame, pose_status_text,
                    (10, 55), cv2.FONT_HERSHEY_SIMPLEX, 0.6, pose_status_color, 2)
        cv2.putText(clean_frame, detect_rate_text,
                    (10, 80), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (180, 180, 180), 1)
        cv2.rectangle(clean_frame, (0, h - 12), (fill, h), (0, 180, 255), -1)
        cv2.putText(clean_frame, f"buf {len(kp_buffer)}/{buf_frames}  ({pct:.0f}%)",
                    (10, h - 15), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (220, 220, 220), 1)

        # Debug frame: skeleton + label + pose status + progress bar
        debug_frame = frame.copy()
        extractor.draw_landmarks(debug_frame)
        debug_frame = renderer.put_text(debug_frame, last_text,
                                        position=(10, 10), color=last_color)
        cv2.putText(debug_frame, pose_status_text,
                    (10, 55), cv2.FONT_HERSHEY_SIMPLEX, 0.6, pose_status_color, 2)
        cv2.putText(debug_frame, detect_rate_text,
                    (10, 80), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (180, 180, 180), 1)
        cv2.rectangle(debug_frame, (0, h - 12), (fill, h), (0, 180, 255), -1)
        cv2.putText(debug_frame, f"buf {len(kp_buffer)}/{buf_frames}  ({pct:.0f}%)",
                    (10, h - 15), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (220, 220, 220), 1)

        # Write to video files
        if vw_clean: vw_clean.write(clean_frame)
        if vw_debug: vw_debug.write(debug_frame)

        # Show the debug view on screen (most useful for real-time feedback)
        cv2.imshow("MSL Recognition  [Q=quit  C=clear]", debug_frame)

        key = cv2.waitKey(1) & 0xFF
        if key == ord("q"):
            break
        elif key == ord("c"):
            kp_buffer.clear()
            last_text  = "Buffer cleared — perform a sign"
            last_color = (200, 200, 200)
            print("[INFO] Buffer manually cleared.")

    cap.release()
    if vw_clean: vw_clean.release()
    if vw_debug: vw_debug.release()
    cv2.destroyAllWindows()

    print(f"\n[INFO] Webcam session summary:")
    print(f"  Total frames     : {total_frames_seen}")
    print(f"  Pose detected    : {pose_detected_count}  ({pose_detected_count/max(total_frames_seen,1)*100:.0f}%)")
    print(f"  Classifications  : {classify_count}")
    if total_frames_seen > 0 and pose_detected_count / total_frames_seen < 0.3:
        print(f"  [WARNING] Low detection rate ({pose_detected_count/total_frames_seen*100:.0f}%).")
        print(f"            Tips: move closer, improve lighting, ensure upper body is visible.")
    if output_dir:
        print(f"  Saved  clean  → {clean_path}")
        print(f"  Saved  debug  → {debug_path}")


# ─── CLI ──────────────────────────────────────────────────────────────────────

def main():
    parser = argparse.ArgumentParser(
        description="MSL Sign Language Inference — ONNX Runtime",
        formatter_class=argparse.RawDescriptionHelpFormatter,
        epilog="""
Examples:
  # Single video with feature log for debugging
  python infer_onnx.py --onnx_model onnx_models\\bilstm_msl.onnx \\
      --video sample.mp4 --top_k 5 \\
      --output_dir predictions_output --log_file results.txt --save_features

  # Webcam — recommended starting command
  python infer_onnx.py --onnx_model onnx_models\\transformer_msl.onnx \\
      --webcam 0 --no_mirror --buf_frames 100 --zero_z --top_k 5 \\
      --output_dir predictions_output --log_file webcam_log.txt --save_features

  # Webcam — if no_mirror gives wrong results try with mirror
  python infer_onnx.py --onnx_model onnx_models\\transformer_msl.onnx \\
      --webcam 0 --buf_frames 100 --zero_z --top_k 5 \\
      --output_dir predictions_output --log_file webcam_log.txt --save_features

  POSITIONING TIPS (critical for correct webcam predictions):
  Feature analysis found these differences vs training data:
    Training shoulder y ~ 0.50 (frame center), width ~ 0.205
    Webcam   shoulder y ~ 0.63 (too low),      width ~ 0.313 (too close)
  1. Move camera UP or sit FURTHER BACK until shoulders are near frame center.
  2. Your upper body should fill about 40-60% of the frame width (move back).
  3. Perform the FULL sign within the buffer window (watch progress bar).
  4. Use --save_features and check *_features.txt to verify positioning.
     Good: shoulder y raw ~ 0.45-0.55, shoulder width ~ 0.15-0.25.
""")

    parser.add_argument("--onnx_model",   required=True,
                        help="Path to .onnx model file (NOT the .json metadata)")
    parser.add_argument("--metadata",     default=None,
                        help="Path to .json metadata (default: same stem as .onnx)")
    parser.add_argument("--label_map",    default=None,
                        help="Path to label_map.json (default: next to .onnx)")
    parser.add_argument("--output_dir",   default="predictions_output",
                        help="Directory for saved videos (default: predictions_output/)")
    parser.add_argument("--log_file",     default=None,
                        help="Append top-k results to this text file")
    parser.add_argument("--top_k",        type=int, default=5,
                        help="Number of top predictions to show/log (default: 5)")
    parser.add_argument("--font_size",    type=int, default=32,
                        help="Myanmar font size for overlay (default: 32)")
    parser.add_argument("--model_complexity", type=int, default=None,
                        help="MediaPipe pose complexity: 0/1/2. "
                             "Default: 2 for video (matches training), 1 for webcam.")

    grp = parser.add_mutually_exclusive_group(required=True)
    grp.add_argument("--video",  help="Path to a video file")
    grp.add_argument("--webcam", type=int, metavar="CAM_IDX",
                     help="Webcam device index (0 = default camera)")

    parser.add_argument("--buffer_sec", type=float, default=2.5,
                        help="[Webcam] Legacy: seconds-based buffer (overridden by --buf_frames)")
    parser.add_argument("--buf_frames", type=int, default=100,
                        help="[Webcam] Fixed number of frames to accumulate before classifying. "
                             "Default: 100 (matches training avg sequence length ~127 frames). "
                             "At complexity=1 (~25fps): 100 frames ≈ 4 seconds per sign. "
                             "Increase if signs are slow; decrease for faster response.")
    parser.add_argument("--zero_z", action="store_true",
                        help="[Webcam] Zero out the z (depth) coordinate before classification. "
                             "Feature analysis shows webcam z_mean ≈ -0.5 to -0.8 vs "
                             "training z_mean ≈ 0.03 — a large systematic offset caused by "
                             "the signer being closer to the webcam than in training recordings. "
                             "Zeroing z removes this noise. Recommended when webcam predictions "
                             "are wrong but x/y skeleton looks correct in debug video.")
    parser.add_argument("--mirror", dest="mirror", action="store_true", default=True,
                        help="[Webcam] Flip frame horizontally before MediaPipe (default: ON). "
                             "Most webcams output a mirrored (selfie) image; flipping corrects "
                             "the x-coordinates to match the training data orientation.")
    parser.add_argument("--no_mirror", dest="mirror", action="store_false",
                        help="[Webcam] Disable the horizontal flip. Use if your webcam "
                             "already outputs an un-mirrored view, or if --mirror makes "
                             "predictions worse.")
    parser.add_argument("--save_features", action="store_true",
                        help="Save keypoint feature diagnostics to output_dir/features/. "
                             "Each classification event produces: "
                             "*_raw_kp.npy (raw keypoints), "
                             "*_model_input.npy (model input), "
                             "*_features.txt (human-readable stats). "
                             "Use to compare video vs webcam feature distributions.")
    parser.add_argument("--debug", action="store_true",
                        help="Print extra diagnostic information")
    args = parser.parse_args()
    # ── Validate ONNX path ────────────────────────────────────────────────────
    model_p = Path(args.onnx_model)
    if model_p.suffix.lower() == ".json":
        sys.exit(
            f"[ERROR] --onnx_model must be a .onnx file, not a .json file.\n"
            f"        You passed : {model_p}\n"
            f"        Did you mean: {model_p.with_suffix('.onnx')}"
        )
    if not model_p.exists():
        sys.exit(f"[ERROR] ONNX model not found: {model_p}")

    meta_p = args.metadata or str(model_p.with_suffix(".json"))
    lmap_p = args.label_map or str(model_p.parent / "label_map.json")

    if not Path(lmap_p).exists():
        sys.exit(f"[ERROR] label_map.json not found: {lmap_p}\n"
                 f"        Copy it from your Linux server alongside the .onnx file.")

    # ── Load resources ────────────────────────────────────────────────────────
    idx2label = load_idx2label(lmap_p)
    print(f"[INFO] Loaded {len(idx2label)} labels from label map.")

    model    = ONNXModel(str(model_p), meta_p)
    prep     = Preprocessor(max_seq_len=model.max_seq_len)
    renderer = TextRenderer(font_size=args.font_size)

    if args.log_file:
        print(f"[INFO] Logging top-{args.top_k} results to: {args.log_file}")
    if args.output_dir:
        print(f"[INFO] Saving output videos to: {args.output_dir}/")

    # Complexity settings based on feature analysis:
    # - complexity=2 for video: matches training (extract_keypoints.py default=2)
    # - complexity=1 for webcam: much faster (~25fps vs ~9fps for complexity=2)
    #   Feature analysis showed the main webcam problems are sequence length
    #   and z-axis offset — NOT detection quality. Complexity=1 detects fine
    #   for a well-lit, close-up signer and gives 25fps → 100 frames in 4s.
    if args.model_complexity is not None:
        video_complexity  = args.model_complexity
        webcam_complexity = args.model_complexity
    else:
        video_complexity  = 2   # matches training
        webcam_complexity = 1   # faster fps -> more frames in buffer

    try:
        if args.video:
            extractor = KeypointExtractor(
                model_complexity=video_complexity,
                min_detection_confidence=0.5,   # matches training
                min_tracking_confidence=0.5,
            )
            try:
                run_video(model, prep, extractor, renderer, args.video,
                          top_k         = args.top_k,
                          log_file      = args.log_file,
                          idx2label     = idx2label,
                          output_dir    = args.output_dir,
                          debug         = args.debug,
                          mirror        = False,           # video files are not mirrored
                          save_features = args.save_features)
            finally:
                extractor.close()
        else:
            extractor = KeypointExtractor(
                model_complexity=webcam_complexity,
                min_detection_confidence=0.5,
                min_tracking_confidence=0.5,
            )
            try:
                run_webcam(model, prep, extractor, renderer, idx2label,
                           cam_idx        = args.webcam,
                           top_k          = args.top_k,
                           log_file       = args.log_file,
                           buffer_sec     = args.buffer_sec,
                           buffer_frames  = args.buf_frames,
                           output_dir     = args.output_dir,
                           mirror         = args.mirror,
                           save_features  = args.save_features,
                           zero_z         = args.zero_z)
            finally:
                extractor.close()
    finally:
        print("[INFO] Done.")


if __name__ == "__main__":
    main()

```

## Online Testing